# 🧬 **pdb2reaction**: End-to-End Reaction-Path Elucidation from PDB Structures Using Machine-Learning Interatomic Potentials

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/t-0hmura/pdb2reaction/blob/main/examples/pdb2reaction_colab.ipynb)

Build enzymatic reaction paths from PDB/mmCIF or small-molecule XYZ/GJF structures with automatic active-site model setup, minimum-energy-path search, transition-state optimization, IRC validation, thermochemistry, and optional DFT single-point calculations in four guided tabs:

**① Input → ② Setup → ③ Options → (Run) → ④ Results**

> Choose a **GPU** runtime, run **Installation** once, then run **Launch GUI**.

**Repository:** [t-0hmura/pdb2reaction](https://github.com/t-0hmura/pdb2reaction)

**Citation:** Ohmura, T.; Sato, H.; Terada, T. *pdb2reaction: End-to-End Reaction-Path Elucidation from PDB Structures Using Machine-Learning Interatomic Potentials*. ChemRxiv **2026**. [DOI: 10.26434/chemrxiv.15003538/v1](https://doi.org/10.26434/chemrxiv.15003538/v1)


In [ ]:
#@title  ⚙️ Installation — pdb2reaction + dependencies { display-mode: "form" }
#@markdown Select the backend and version, then run Installation (first run takes several minutes).
#@markdown Installs pdb2reaction, the selected MLIP backend, notebook UI dependencies, and optional DFT support.
backend = "uma"  #@param ["mace", "uma", "orb"]
pdb2reaction_version = "v0.4.12"  #@param {type:"string"}
#@markdown ><small style="display:block;color:#52647b;line-height:1.45;margin:4px 0 8px">Recommended backend: UMA (Hugging Face licence acceptance + read token required). Without a token, use ORB or MACE. Restart the runtime before switching backends. UMA uses fp32; ORB/MACE use fp64 by default.</small>
install_dft = True  #@param {type:"boolean"}
#@markdown ><small style="display:block;color:#52647b;line-height:1.45;margin:4px 0 8px">Keep `install_dft` checked to install PySCF + GPU4PySCF for the standalone `dft` workflow and the optional DFT stage in `all`. Clear it only when you intentionally want a lighter non-DFT runtime.</small><!-- QA: set the version field to exactly `debug` to use the matching adjacent source ZIP. No file picker is opened. -->
import os, sys, subprocess, time, importlib, importlib.util, warnings, re, zipfile, shutil
from pathlib import Path
# GPU4PySCF 1.8 reports its normal CuPy contraction-engine fallback as a
# UserWarning. Suppress only that known informational message; all other
# installation and runtime warnings remain visible.
warnings.filterwarnings(
    'ignore', message=r'using .* as the tensor contraction engine\.',
    category=UserWarning, module=r'gpu4pyscf\.lib\.cutensor')
_setup_started = time.monotonic()
def _phase(number, label):
    print('[%d/5] %s …' % (number, label), flush=True)
def _phase_done(label):
    print('      ✓ %s' % label, flush=True)
def _run_installer(command, label, check=True):
    """Run one quiet installer with a single concise status line."""
    print('      … %s' % label, flush=True)
    completed = subprocess.run(command)
    if check and completed.returncode:
        raise subprocess.CalledProcessError(completed.returncode, command)
    return completed.returncode
_installed_backend = globals().get('_INSTALL_BACKEND') or globals().get('BACKEND')
if _installed_backend not in (None, backend):
    raise RuntimeError('Backend switch requested. Restart the Colab runtime first, then rerun Installation.')
_INSTALL_BACKEND = backend
try: _gpu = subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True).stdout.strip()
except FileNotFoundError: _gpu = ''   # nvidia-smi absent on CPU runtimes -> don't crash Installation
if _gpu: print(_gpu)
elif install_dft:
    print('⚠️ No GPU detected. A GPU runtime is required while install_dft is checked (Runtime ▸ Change runtime type ▸ GPU).')
else:
    print('⚠️ No GPU detected. The GUI can open, but compute workflows require a GPU runtime.')
def pip(*a):
    packages = [str(value) for value in a if not str(value).startswith('-')]
    label = 'Installing ' + (', '.join(packages)[:92] or 'Python packages')
    _run_installer([sys.executable, '-m', 'pip', 'install', '-q', *a], label)
_DEFAULT_RELEASE_VERSION = 'v0.4.12'
_raw_version = str(pdb2reaction_version)
_debug_install = _raw_version == 'debug'
_release_version = _DEFAULT_RELEASE_VERSION if _debug_install else _raw_version.strip(' ._-')
if not _release_version or _release_version.lower() == 'v':
    _release_version = _DEFAULT_RELEASE_VERSION
_release_tag = _release_version if _release_version.lower().startswith('v') else 'v' + _release_version
_requested_version = _release_version.lstrip('v')
REPO_DIR = 'pdb2reaction-src'
SOURCE_BUNDLE_ID = '2ed463f68bd7160326e389748b231918c6f4431b83ad83320fd685417c89518c'
_source_zip = REPO_DIR + '.zip'
if _debug_install:
    # Exact `debug` input selects the matching local source snapshot. The
    # marker prevents a stale or wrong ZIP from entering the runtime.
    _phase(1, 'pdb2reaction %s from local debug source ZIP' % _release_version)
    _marker = Path(REPO_DIR, '.colab-debug-source')
    if not os.path.isfile(_source_zip):
        raise FileNotFoundError(
            'Debug mode needs the adjacent source ZIP at %s, but it was not found. '
            'Upload the matching pdb2reaction-src.zip in Colab Files and rerun Installation, '
            'or enter %s for a published release install.' %
            (Path(_source_zip).resolve(), _release_tag))
    with zipfile.ZipFile(_source_zip) as _archive:
        _marker_name = REPO_DIR + '/.colab-debug-source'
        try:
            _archive_id = _archive.read(_marker_name).decode('utf-8').strip()
        except KeyError as _error:
            raise RuntimeError('%s has no source-snapshot marker.' % _source_zip) from _error
        if _archive_id != SOURCE_BUNDLE_ID:
            raise RuntimeError('Notebook and ZIP are from different debug builds; replace both with the matching pair.')
        _root = Path('.').resolve()
        _repo_root = (_root / REPO_DIR).resolve()
        for _member in _archive.infolist():
            if not _member.filename.startswith(REPO_DIR + '/'):
                raise RuntimeError('%s contains a file outside its source directory.' % _source_zip)
            _target = (_root / _member.filename).resolve()
            if _target != _repo_root and _repo_root not in _target.parents:
                raise RuntimeError('%s contains an unsafe path.' % _source_zip)
            if (_member.external_attr >> 16) & 0o170000 == 0o120000:
                raise RuntimeError('%s contains an unsupported symbolic link.' % _source_zip)
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        _archive.extractall('.')
    _found_id = _marker.read_text(encoding='utf-8').strip() if _marker.is_file() else ''
    if _found_id != SOURCE_BUNDLE_ID:
        raise RuntimeError('%s did not extract the expected source snapshot.' % _source_zip)
    os.environ['SETUPTOOLS_SCM_PRETEND_VERSION'] = _requested_version
    try:
        if install_dft:
            print('      install_dft is ticked — adding the [dft] extra (PySCF + GPU4PySCF):', flush=True)
        pip('-e', './' + REPO_DIR + ('[dft]' if install_dft else ''))
    finally:
        os.environ.pop('SETUPTOOLS_SCM_PRETEND_VERSION', None)
    sys.path.insert(0, os.path.abspath(REPO_DIR))
    importlib.invalidate_caches()
    _install_origin = 'local debug source ZIP'
else:
    # Normal releases install the pinned wheel from PyPI. The wheel carries the
    # package only; GUI examples are fetched from the matching git tag on demand.
    _phase(1, 'pdb2reaction %s from PyPI' % _release_version)
    if install_dft:
        print('      install_dft is ticked — adding the [dft] extra (PySCF + GPU4PySCF):', flush=True)
    _release_spec = 'pdb2reaction%s==%s' % ('[dft]' if install_dft else '', _requested_version)
    try:
        pip(_release_spec)
    except subprocess.CalledProcessError as _pip_error:
        raise RuntimeError(
            'Could not install %s from PyPI. The requested pdb2reaction version %s may not '
            'be published, or pip reported another error above. Check the version field. '
            'For an unpublished debug build, upload the matching pdb2reaction_colab.ipynb '
            'and pdb2reaction-src.zip pair, then enter debug.' %
            (_release_spec, _release_tag)) from _pip_error
    _install_origin = 'PyPI'
# DMF is part of the default notebook installation. Colab does not guarantee
# cyipopt, so install its system IPOPT dependency before the Python module.
if importlib.util.find_spec('cyipopt') is None:
    if not os.path.isdir('/content'):
        raise RuntimeError('cyipopt is missing; install it before running this Colab notebook.')
    _ipopt_env = dict(os.environ, DEBIAN_FRONTEND='noninteractive')
    subprocess.run(['apt-get', 'update', '-qq', '-o', 'Acquire::Retries=3'],
                   check=True, env=_ipopt_env)
    subprocess.run(['apt-get', 'install', '-y', '-qq', '-o', 'Acquire::Retries=3',
                    'coinor-libipopt-dev', 'pkg-config'],
                   check=True, env=_ipopt_env)
    pip('cyipopt')
    importlib.invalidate_caches()
_missing_dmf = [name for name in ('dmf', 'cyipopt')
                if importlib.util.find_spec(name) is None]
if _missing_dmf:
    raise RuntimeError('DMF dependency installation failed: %s' % ', '.join(_missing_dmf))
if install_dft:
    from importlib.metadata import PackageNotFoundError, version as _dist_version
    _dft_packages = {'pyscf': 'pyscf', 'gpu4pyscf': 'gpu4pyscf-cuda12x'}
    _dft_missing = [module for module in _dft_packages
                    if importlib.util.find_spec(module) is None]
    _dft_versions = {}
    for _module, _distribution in _dft_packages.items():
        try: _dft_versions[_module] = _dist_version(_distribution)
        except PackageNotFoundError: _dft_missing.append(_module)
    if _dft_missing:
        raise RuntimeError('install_dft was selected, but these modules are missing: %s' %
                           ', '.join(sorted(set(_dft_missing))))
    try:
        _dft_imports = ('pyscf', 'basis_set_exchange', 'gpu4pyscf.dft')
        for _module in _dft_imports: importlib.import_module(_module)
        _cupy = importlib.import_module('cupy')
        _dft_gpu_count = int(_cupy.cuda.runtime.getDeviceCount())
    except Exception as _dft_exc:
        raise RuntimeError('DFT packages installed but failed their import/GPU check: %s' %
                           _dft_exc) from _dft_exc
    if _dft_gpu_count < 1:
        raise RuntimeError('DFT support needs a GPU runtime; no CUDA device is visible.')
    DFT_SETUP_READY = True
    try:
        _dft_device_name = _cupy.cuda.runtime.getDeviceProperties(0)['name']
        if isinstance(_dft_device_name, bytes):
            _dft_device_name = _dft_device_name.decode('utf-8', 'replace')
    except Exception:
        _dft_device_name = 'CUDA device 0'
    DFT_CUDA_DEVICE = str(_dft_device_name)
    print('DFT support installed: PySCF %s · GPU4PySCF %s · CUDA devices %d · device 0 %s' %
          (_dft_versions['pyscf'], _dft_versions['gpu4pyscf'],
           _dft_gpu_count, DFT_CUDA_DEVICE))
_phase_done('tool package installed')
_phase(2, '%s MLIP backend' % backend.upper())
if backend == 'mace':
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','-q','fairchem-core'], check=False)
    # Pin the Colab-tested MACE stack. Unbounded upgrades have changed model
    # download behaviour in the past and can break an otherwise unchanged notebook.
    pip('mace-torch==0.3.16', 'torchvision==0.23.0')
    print('Downloading MACE-OMOL-0 model weights (fp64); the ASL text below is an informational licence notice, not an input prompt...')
    subprocess.run([
        sys.executable, '-c',
        "from mace.calculators import mace_omol; "
        "mace_omol(model='extra_large', device='cpu', default_dtype='float64'); "
        "print('MACE-OMOL-0 model cache ready.')"
    ], check=True)
    print('MACE installed and its default model is ready (no login needed).')
elif backend == 'orb':
    # ORB coexists with the base install (the e3nn clash is MACE vs fairchem-core only).
    pip('orb-models'); print('ORB installed (no login needed; runs fp64 by default).')
else:
    # UMA is gated, but the token is only needed once a run downloads the
    # weights. Sign-in is therefore deferred to the end of Installation so no install
    # phase waits on a prompt.
    print('UMA installed (uses fairchem-core; Hugging Face sign-in runs at the end of Installation).')
_phase_done('backend installed')
_phase(3, 'notebook widgets')
pip('ipywidgets==7.7.2','anywidget==0.11.0','matplotlib')
_phase_done('GUI dependencies installed; Mol* loads in the browser')
# Plotly static-image export needs both a Chrome binary and its Linux shared
# libraries. Keep Installation deterministic: install the export stack and
# browser, but do not launch a second Chromium render process here.
_phase(4, 'plot export and Chromium bridge')
subprocess.run(['apt-get','update','-qq'], check=True)
_audio_pkg = ('libasound2t64' if subprocess.run(
    ['apt-cache','show','libasound2t64'], stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL).returncode == 0 else 'libasound2')
subprocess.run([
    'apt-get','install','-y','-qq',
    'libnss3','libatk-bridge2.0-0','libcups2','libxcomposite1',
    'libxdamage1','libxfixes3','libxrandr2','libgbm1','libxkbcommon0',
    'libpango-1.0-0','libcairo2',_audio_pkg,
], check=True)
pip('plotly==6.6.0','kaleido==1.2.0','choreographer==1.2.1')
_chrome = subprocess.run(['plotly_get_chrome','-y'], capture_output=True, text=True)
if _chrome.returncode != 0:
    _chrome = subprocess.run(['choreo_get_chrome'], capture_output=True, text=True)
if _chrome.returncode != 0:
    raise RuntimeError('Chrome installation for Plotly export failed:\n' +
                       (_chrome.stderr or _chrome.stdout or 'unknown error'))
_plot_probe_code = (
    "import os,tempfile; import plotly.graph_objects as go; "
    "p=tempfile.mktemp(suffix='.png'); "
    "go.Figure(go.Scatter(x=[0,1,2],y=[0,1,0])).write_image(p,width=320,height=240); "
    "assert os.path.getsize(p)>1000; print(os.path.getsize(p)); os.remove(p)"
)
_plot_probe = subprocess.run([sys.executable, '-c', _plot_probe_code],
                             capture_output=True, text=True, timeout=90)
PLOT_EXPORT_READY = _plot_probe.returncode == 0
if not PLOT_EXPORT_READY:
    _plot_error = (_plot_probe.stderr or _plot_probe.stdout or 'unknown error').strip()
    raise RuntimeError('Plotly PNG export failed its real render check: %s' % _plot_error[-800:])
print('PNG export verified (%s bytes).' % ((_plot_probe.stdout or '').strip() or 'non-empty'))
_phase_done('plot export verified')
_phase(5, 'installed-version verification')
INSTALL_DFT = install_dft
BACKEND = backend; TOOL = 'pdb2reaction'
from importlib.metadata import version
installed_version = version('pdb2reaction')
if installed_version != _requested_version:
    raise RuntimeError('Installed version %s does not match requested package version %s.' % (installed_version, _requested_version))
print('pdb2reaction', installed_version, '| version input', pdb2reaction_version, '| from', _install_origin)
_phase_done('version verified')
print('\n✅ Install done in %.1f min. backend = %s.' %
      ((time.monotonic() - _setup_started) / 60, BACKEND))
_hf_auth_ready = True
if backend == 'uma':
    # Deliberately the last step: accept the FAIR Chemistry License at
    # https://huggingface.co/facebook/UMA, then sign in here.
    from huggingface_hub import login, notebook_login
    try: from huggingface_hub import get_token as _hf_get_token
    except ImportError: _hf_get_token = lambda: None
    _hf_token = os.environ.get('HF_TOKEN') or ''
    if not _hf_token:
        try:
            from google.colab import userdata as _hf_userdata
            _hf_token = _hf_userdata.get('HF_TOKEN') or ''
        except Exception: _hf_token = ''   # no Colab secret, or access not granted
    if _hf_token:
        login(token=_hf_token, add_to_git_credential=False)
        print('🔑 Hugging Face: signed in with a stored token.')
    elif _hf_get_token():
        print('🔑 Hugging Face: reusing the token already cached in this runtime.')
    else:
        print('🔑 UMA is gated. Accept the licence at https://huggingface.co/facebook/UMA,\n   then paste a read token below — the install above is already complete.')
        print('   If login is unavailable, restart the runtime and rerun Installation with backend=orb or backend=mace.')
        try:
            notebook_login()
        except KeyboardInterrupt:
            _hf_auth_ready = False
            print('\n⚠️ Hugging Face sign-in was cancelled. Installation is complete, but UMA authentication is still pending.')
            print('   Rerun Installation to sign in again, set HF_TOKEN, or use backend=orb/backend=mace.')
        except Exception as _hf_login_exc:
            if type(_hf_login_exc).__name__ != 'DeviceCodeError':
                raise
            _hf_auth_ready = False
            print('\n⚠️ Hugging Face device code expired. Installation is complete, but UMA authentication is still pending.')
            print('   Rerun Installation to sign in again, set HF_TOKEN, or use backend=orb/backend=mace.')
if _hf_auth_ready:
    print('\nNow run "Launch GUI".')
else:
    print('\nYou may launch the GUI, but authenticate UMA before running a calculation.')


In [ ]:
#@title 🖥️ Launch GUI  (run once, after Installation) { display-mode: "form" }
import os, glob, json, shlex, math, shutil, subprocess, time, zipfile, hashlib, importlib.util, signal, html, tempfile, base64, gzip, csv, io, weakref, urllib.request, ast, threading, asyncio, warnings, queue, re
# Reduce CUDA allocator fragmentation in long Colab sessions. This is inherited by CLI subprocesses.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
# GPU4PySCF 1.8 emits this informational fallback while the CLI is imported.
# Filter only that exact warning family; do not silence unrelated warnings.
warnings.filterwarnings(
    'ignore', message=r'using .* as the tensor contraction engine\.',
    category=UserWarning, module=r'gpu4pyscf\.lib\.cutensor')
from pathlib import Path
import ipywidgets as W
from traitlets import validate as _trait_validate
from IPython.display import display, clear_output, Image, HTML
import click

# A notebook cell shares one Python global namespace across executions.  Do not
# let construction-time lookups call hooks belonging to the previous GUI before
# this execution has defined their replacements.
for _stale_builder_hook in ('_sync_capability_controls', '_render_scan_panel'):
    globals().pop(_stale_builder_hook, None)
try:
    import anywidget, traitlets
    _HAS_DROP_WIDGET = True
except ImportError:
    anywidget = traitlets = None
    _HAS_DROP_WIDGET = False
try: TOOL
except NameError: TOOL = 'pdb2reaction'
try: BACKEND
except NameError: BACKEND = 'uma'
try: REPO_DIR
except NameError: REPO_DIR = 'pdb2reaction-src'
IS_CLUSTER = True
_RUNTIME_DIR = Path(tempfile.mkdtemp(prefix='%s-colab-' % TOOL.replace('_', '-')))

def _runtime_path(*parts):
    """Allocate notebook-owned files away from uploads and run outputs."""
    path = _RUNTIME_DIR.joinpath(*parts)
    path.parent.mkdir(parents=True, exist_ok=True)
    return str(path)

def _unique_path(path):
    """Return path, or a numbered sibling, without replacing an existing file."""
    candidate = Path(path)
    if not candidate.exists(): return str(candidate)
    for number in range(2, 10000):
        numbered = candidate.with_name('%s_%d%s' % (candidate.stem, number, candidate.suffix))
        if not numbered.exists(): return str(numbered)
    raise RuntimeError('Could not allocate a unique path for %s.' % candidate.name)
# Detect Colab from the import alone.  Enabling the third-party widget
# manager is best-effort: core ipywidgets and the native upload bridge still work
# when a newly installed widget stack makes this optional call fail.
try:
    from google.colab import output as _colab_output
except ImportError:
    _colab_output = None
IN_COLAB = _colab_output is not None
# A Colab browser can use a local Jupyter kernel without google.colab.
# Keep this separate from IN_COLAB, which means the native backend API is available.
try:
    _colab_frontend = bool(get_ipython().parent_header.get('metadata', {}).get('colab'))
except Exception:
    _colab_frontend = False
IS_COLAB_FRONTEND = IN_COLAB or _colab_frontend
_cwm = _colab_output
# Capture the kernel loop on the cell thread. get_ipython() is not reliable
# inside Colab worker threads, so completion callbacks reuse this handle.
try:
    _UI_IOLOOP = getattr(getattr(get_ipython(), 'kernel', None), 'io_loop', None)
except Exception:
    _UI_IOLOOP = None
try:
    _UI_ASYNC_LOOP = asyncio.get_running_loop()
except Exception:
    _UI_ASYNC_LOOP = None
if IN_COLAB:
    try:
        _cwm.enable_custom_widget_manager()
    except Exception as _cwm_error:
        print('note: enable_custom_widget_manager failed (%s); using native Colab uploads.' %
              _cwm_error)

ACCENT = '#114b8a'
display(HTML("""<style>
.rxapp {
  --rx-ink:#172033; --rx-muted:#607087; --rx-line:#dfe6ef; --rx-soft:#f5f8fc;
  --rx-blue:#1268b3; --rx-blue-soft:#eaf4ff; --rx-green:#157347;
  --rx-amber:#b85d20; --rx-navy:#101827;
  width:100%; max-width:100%; box-sizing:border-box; padding:0 8px 8px;
  height:auto; min-height:0; overflow-x:clip; overflow-y:visible;
  display:flex !important; flex-flow:column nowrap !important;
}
.rxheader-host { flex:0 0 auto; min-height:0; padding-top:1px; }
.rxapp-main { flex:0 0 auto !important; min-height:0; overflow:visible; }
.rxpages { flex:0 0 auto !important; min-height:0; overflow:visible; position:relative; }
.rxpage {
  height:auto; max-height:none; min-height:0; overflow:visible;
  padding:0 0 2px; box-sizing:border-box;
}
/* Keep Setup mounted at its real responsive width while another step is
   active. Moving it off-screen prevents the display:none -> visible WebGL
   relayout that made the first and subsequent Setup switches stall. */
.rxpage-prewarm {
  display:flex !important; position:absolute !important; left:-200vw !important; top:0 !important;
  width:100% !important; visibility:hidden !important; opacity:0 !important;
  pointer-events:none !important; z-index:-1 !important;
}
/* The fixed-height viewer/inspector workspace below already prevents Mol*
   replacement from resizing the page; avoid reserving blank space beneath it. */
.rxviewer-page { min-height:0 !important; row-gap:6px !important; align-items:stretch !important; }
.rxviewer-page > * { width:100% !important; max-width:100% !important; min-width:0 !important;
  margin-left:0 !important; margin-right:0 !important; box-sizing:border-box !important; }
.rxheader {
  min-height:58px; display:flex; align-items:center; justify-content:space-between;
  gap:12px; flex-wrap:wrap; box-sizing:border-box;
  background:linear-gradient(118deg,#0b1220 0%,#111c30 58%,#102a43 100%);
  border:1px solid rgba(148,163,184,.18); border-radius:15px; padding:10px 14px 10px 16px;
  margin-bottom:6px; box-shadow:0 0 6px rgba(15,23,42,.10);
}
.rxheader-brand { display:flex; align-items:center; gap:9px; min-width:0; }
.rxheader-product {
  color:#f8fafc; font-size:22px; line-height:1.15; font-weight:760;
  letter-spacing:.1px; white-space:nowrap;
}
.rxheader-version {
  display:inline-flex; align-items:center; justify-content:center; min-height:23px;
  padding:2px 8px; box-sizing:border-box; border:1px solid rgba(147,197,253,.28);
  border-radius:999px; background:rgba(30,64,110,.58); color:#bfdbfe;
  font-size:11px; line-height:1; font-weight:720; letter-spacing:.35px; white-space:nowrap;
}
.rxheader-meta {
  display:flex; align-items:center; gap:6px; flex-wrap:nowrap; padding:4px 5px;
  border:1px solid rgba(148,163,184,.16); border-radius:999px;
  background:rgba(15,23,42,.42); color:#cbd5e1; font-size:12px; white-space:nowrap;
}
.rxheader-chip {
  display:inline-flex; align-items:center; min-height:28px; padding:3px 10px;
  border-radius:999px; background:rgba(37,99,235,.17); color:#bfdbfe;
  font-weight:720; letter-spacing:.3px;
}
.rxheader-backend {
  display:inline-flex; align-items:baseline; gap:5px; padding:3px 8px 3px 5px;
}
.rxheader-key { color:#94a3b8; }
.rxheader-backend b { color:#f8fafc; font-weight:740; }
@media (max-width:640px) {
  .rxheader { align-items:flex-start; padding:11px 12px; }
  .rxheader-brand, .rxheader-meta { width:100%; }
  .rxheader-meta { justify-content:space-between; }
  .rxheader-product { font-size:20px; }
}
.rxtabs { display:flex !important; flex-flow:row nowrap !important; width:100% !important; max-width:none !important; align-self:stretch !important; margin:5px 0 8px !important; gap:6px !important; }
.rxtabs .rxtab-button { flex:1 1 0 !important; width:auto !important; max-width:none !important; min-width:0 !important; min-height:42px !important; }
.rxtabs .rxtab-button.mod-active { background:var(--rx-blue) !important; color:#fff !important;
  border-color:var(--rx-blue) !important; box-shadow:0 0 0 2px rgba(18,104,179,.13) !important; }
.rxtab-loading { margin:0 0 7px !important; }
.rxtab-loading-banner { display:flex; align-items:center; gap:8px; min-height:38px; padding:7px 10px;
  border:1px solid #bfdbfe; border-radius:9px; background:#eff6ff; color:#1e3a5f; box-sizing:border-box; }
.rxtab-spinner { width:14px; height:14px; flex:0 0 14px; border:2px solid #bfdbfe; border-top-color:var(--rx-blue);
  border-radius:50%; animation:rxtab-spin .75s linear infinite; }
@keyframes rxtab-spin { to { transform:rotate(360deg); } }
.rxpages-loading { opacity:.18 !important; pointer-events:none !important; }
.rxfold { margin:8px 0 6px; }
.rxapp .rxcard { padding:10px 12px !important; margin:0 !important; }
.rxinspector > .rxactive-card {
  border-color:#7cb7f2 !important;
  background:#fff !important;
  box-shadow:0 0 0 1px rgba(37,99,235,.12),0 0 16px rgba(30,64,175,.09) !important;
}
.rxtab-notice { margin:0 0 7px !important; }
.rxtab-notice .widget-html-content > div {
  padding:7px 10px; border:1px solid #fde68a; border-radius:9px; background:#fffbeb; color:#92400e;
}
.rxmanual-notice { position:sticky; top:0; z-index:35; }
.rxmanual-notice .widget-html-content > div { padding:9px 12px; border:1px solid #f59e0b;
  border-radius:10px; background:#fffbeb; color:#78350f; box-shadow:0 0 7px rgba(120,53,15,.12); }
.rxapp .rxoptional-card { margin-top:10px !important; }
.rxapp .rxoptional-card > .widget-hbox { flex-flow:row wrap !important; align-items:flex-start !important; gap:7px !important; overflow:visible !important; }
.rxapp .rxoptional-card .rxflagrow { flex:1 1 270px !important; min-width:0 !important; align-items:flex-start !important; }
.rxapp .rxoptional-card .rxflagrow:not(:has(.rxinfo-details[open])) { flex-wrap:nowrap !important; }
.rxapp .rxoptional-card .rxflagrow:has(.rxinfo-details[open]) { flex-wrap:wrap !important; }
.rxapp .rxoptional-card .rxflagrow > .widget-checkbox { flex:1 1 auto !important; width:auto !important; min-width:0 !important; height:auto !important; align-items:flex-start !important; }
.rxapp .rxoptional-card .widget-checkbox .widget-label,
.rxapp .rxoptional-card .widget-checkbox .widget-label-basic { white-space:normal !important; overflow:visible !important; text-overflow:clip !important; line-height:1.35 !important; height:auto !important; }
.rxapp .rxoptional-card .rxflagrow:not(:has(.rxinfo-details[open])) > .rxinfo { flex:0 0 26px !important; }
.rxworkspace { row-gap:10px !important; }
.rxapp .widget-vbox > .widget-html:empty { display:none; }
.rxapp .widget-html-content > div { margin:0; }
.rxinspector, .rxviewer { min-width:0; }
.rxfold > button { min-height:38px !important; padding:7px 12px !important;
  display:flex !important; align-items:center !important; background:#f8fafc !important;
  border:1px solid #e2e8f0 !important; color:#334155 !important;
  font-weight:600 !important; text-align:left !important; }
.rxfold-toggle::before { content:'▶'; display:inline-block; flex:0 0 auto; margin-right:9px;
  font-size:1.08em; line-height:1; transform:translateY(-.02em); color:#475569; }
.rxfold-toggle.rxfold-open::before { content:'▼'; font-size:1.04em; transform:translateY(-.01em); }
.rxapp, .rxapp .widget-label, .rxapp .widget-html-content, .rxapp .widget-readout {
  font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Helvetica,Arial,sans-serif !important;
  color:#1f2937; }
.rxapp .widget-html-content code { background:#f1f5f9; padding:1px 5px; border-radius:5px;
  font-family:"DejaVu Sans Mono","Liberation Mono",Consolas,"Courier New",monospace; font-size:12px; color:#334155;
  overflow-wrap:anywhere; word-break:break-word; white-space:normal; }
.rxapp .widget-button { border-radius:9px !important; font-weight:500 !important;
  box-shadow:none !important; min-height:38px; }
/* Button labels are complete; hide Font Awesome glyphs that become tofu boxes
   when a Colab/static frontend does not provide the icon font. */
.rxapp .widget-button .fa, .rxapp .widget-upload .fa { display:none !important; }
.rxapp button.jupyter-button, .rxapp .widget-upload > button { min-height:38px !important; }
.rxapp .widget-dropdown select, .rxapp .widget-text input, .rxapp .widget-textarea textarea,
.rxapp input[type="number"] { border-radius:9px !important; border:1px solid #e2e8f0 !important; }
.rxapp .widget-button:focus-visible, .rxapp select:focus-visible,
.rxapp input:focus-visible, .rxapp textarea:focus-visible {
  outline:3px solid rgba(37,99,235,.28) !important; outline-offset:2px !important; }
.rxapp .widget-tab > .rxapp-tabs, .rxapp .p-TabBar-tab { font-weight:500; }
.rxcard { border:1px solid var(--rx-line); border-radius:12px; padding:10px 12px; background:#ffffff;
  box-shadow:0 0 5px rgba(16,24,40,0.065); box-sizing:border-box; flex:0 0 auto; min-width:0; }
.rxapp .rxworkspace {
  display:grid !important;
  grid-template-columns:minmax(0,2fr) minmax(260px,1fr);
  align-items:start !important; justify-content:stretch !important;
  column-gap:10px; row-gap:10px !important;
  width:100%; height:auto !important; min-height:0;
  flex:0 0 auto !important; overflow-x:clip !important; overflow-y:visible !important;
  margin:0 0 10px !important;
}
/* Give molecular inspection priority: a large square canvas plus a compact editor. */
.rxworkspace > .rxviewer > .rxcard { margin:0; }
.rxworkspace > * { width:100% !important; max-width:100% !important; min-width:0 !important;
  margin-left:0 !important; margin-right:0 !important; box-sizing:border-box !important; }
.rxviewer {
  width:100% !important; min-width:0; max-width:none;
  position:static; align-self:start;
}
.rxinspector {
  width:100% !important; min-width:0;
  container-type:inline-size;
  align-items:stretch !important;
  height:calc(clamp(460px,50vw,660px) + 52px); max-height:712px;
  overflow-y:auto !important; overflow-x:hidden !important;
  scrollbar-gutter:auto; scrollbar-width:auto; scrollbar-color:#2563eb #dbeafe;
  overscroll-behavior:contain;
  padding-right:7px; box-sizing:border-box;
  padding-left:0;
  border:0; border-radius:0; background:transparent; outline:none; box-shadow:none;
}
.rxinspector::-webkit-scrollbar { width:14px; }
.rxinspector::-webkit-scrollbar-track { background:#dbeafe; border:1px solid #bfdbfe; border-radius:999px; box-shadow:inset 0 0 0 1px rgba(255,255,255,.55); }
.rxinspector::-webkit-scrollbar-thumb { min-height:52px; background:#2563eb; border:3px solid #dbeafe; border-radius:999px; }
.rxinspector::-webkit-scrollbar-thumb:hover { background:#1d4ed8; }
.rxinspector > * {
  width:100% !important; min-width:0 !important; max-width:100% !important;
  margin-left:0 !important; margin-right:0 !important; box-sizing:border-box !important;
}
.rxinspector > .widget-html > .widget-html-content,
.rxinspector > .widget-html > .widget-html-content > div {
  width:100% !important; max-width:100% !important; margin:0 !important; box-sizing:border-box !important;
}
.rxinspector > .rxcard { width:100%; min-width:0; max-width:100%; padding:10px 12px !important;
  box-sizing:border-box !important; overflow-x:hidden !important; overflow-y:visible !important; }
.rxinspector { row-gap:10px !important; }
.rxinspector > .rxcenter-panel { row-gap:9px !important; }
.rxcenter-panel { overflow-x:clip !important; overflow-y:visible !important; }
.rxcenter-panel > *, .rxcenter-panel .widget-hbox, .rxcenter-panel .widget-vbox {
  min-width:0 !important; max-width:100% !important; box-sizing:border-box !important; }
.rxcenter-panel .widget-select-multiple { width:100% !important; min-width:0 !important; max-width:100% !important; box-sizing:border-box !important; }
.rxcenter-panel, .rxcenter-panel > * { scrollbar-width:none !important; overscroll-behavior-x:none; }
.rxcenter-panel::-webkit-scrollbar, .rxcenter-panel > *::-webkit-scrollbar { width:0 !important; height:0 !important; display:none !important; }
.rxselected-resn-row:not(:has(.rxinfo-details[open])) {
  flex-wrap:nowrap !important; align-items:center !important; column-gap:6px !important;
}
.rxselected-resn-row:not(:has(.rxinfo-details[open])) > .widget-text {
  flex:1 1 auto !important; width:auto !important; min-width:0 !important;
}
.rxselected-resn-row:not(:has(.rxinfo-details[open])) > .rxinfo { flex:0 0 26px !important; }
.rxselected-resn-row:has(.rxinfo-details[open]) { flex-wrap:wrap !important; }
.rxcenter-actions { flex-flow:row nowrap !important; column-gap:6px !important; row-gap:6px !important; }
.rxcenter-actions > .widget-button { min-width:0 !important; margin:0 !important; }
.rxcenter-actions > .rxpickresn-button { flex:1 1 174px !important; width:auto !important; padding-inline:6px !important; }
.rxcenter-actions > .rxclear-center { flex:0 0 68px !important; width:68px !important; padding-inline:6px !important; }
@container (max-width:250px) {
  .rxcenter-actions { flex-wrap:wrap !important; }
  .rxcenter-actions > .rxpickresn-button { flex-basis:100% !important; }
}
.rxapp .rxcenter-guide {
  background:transparent !important; border:0 !important;
  padding:2px 0 !important; margin:0 !important; column-gap:4px !important;
  color:#52647b !important; font-size:12px !important; line-height:1.4 !important;
}
/* Keep the closed info icon immediately after the instruction. When opened,
   the detail panel may take a full row below without squeezing the sentence. */
.rxcenter-guide:not(:has(.rxinfo-details[open])) { flex-wrap:nowrap !important; }
.rxcenter-guide:not(:has(.rxinfo-details[open])) > .widget-html {
  flex:0 1 auto !important; min-width:0 !important;
}
.rxcenter-guide:not(:has(.rxinfo-details[open])) > .rxinfo { flex:0 0 26px !important; }
.rxcenter-guide:has(.rxinfo-details[open]) { flex-wrap:wrap !important; }
.rxworkflow-card {
  padding:8px 10px !important;
  overflow-x:hidden !important;
}
.rxworkflow-card > .rxhelp-row { min-height:28px; }
.rxworkflow-card .widget-label { margin-right:6px !important; }
.rxworkflow-card .widget-inline-hbox,
.rxworkflow-card .widget-hbox { column-gap:8px !important; }
.rxworkflow-card .widget-button,
.rxworkflow-card button.jupyter-button { min-height:36px !important; }
.rxworkflow-card small { line-height:1.4; }
.rxviewer-toolbar {
  width:100%; flex:0 0 auto !important; flex-wrap:wrap !important;
  align-items:flex-start !important; column-gap:6px !important; row-gap:4px !important;
  margin:4px 0 6px !important;
}
.rxviewer-toolbar > .widget-dropdown { flex:1 1 15rem !important; width:auto !important; min-width:12rem !important; max-width:100% !important; }
.rxviewer-toolbar > .widget-hbox { flex:1 1 19rem !important; width:auto !important; min-width:0 !important; max-width:100% !important; }
.rxviewer-toolbar > .rxinfo { flex:0 0 auto !important; }
.rxview-controls { flex-flow:row nowrap !important; align-items:center !important; column-gap:6px !important; }
.rxview-controls > .widget-dropdown { flex:1 1 12rem !important; width:auto !important; min-width:9rem !important; max-width:100% !important; }
.rxview-controls > .widget-checkbox { flex:0 0 82px !important; min-width:82px !important; }
.rxview-controls > .rxinfo { flex:0 0 26px !important; }
.rxviewer-toolbar .rxfold { margin:0; }
.rxviewer { overflow-x:hidden !important; }
.rxviewer .jupyter-widgets-output-area, .rxviewer iframe { width:100% !important; max-width:100% !important; box-sizing:border-box !important; }
.rxviewer .rxmolstar-embed, .rxresults .rxmolstar-embed { width:100%; max-width:100%; min-width:0; overflow:hidden; box-sizing:border-box; }
.rxviewer .rxmolstar-embed { width:100%; margin-inline:0; border:0;
  border-radius:10px; background:#fff; outline:1px solid #E0DDD4; outline-offset:-1px;
  box-shadow:inset 0 0 8px rgba(51,43,31,.055); }
.rxviewer > .widget-html, .rxviewer > .rxpick-footer {
  width:100% !important; max-width:100% !important; margin:0 !important; padding:0 !important;
  box-sizing:border-box !important; min-width:0 !important; }
.rxviewer > .widget-html > .widget-html-content,
.rxpick-footer .widget-html-content, .rxpick-footer [role="status"] {
  width:100% !important; max-width:100% !important; box-sizing:border-box !important; }
.rxviewer > .rxpick-footer { margin-top:clamp(8px,.7vw,12px) !important; }
.rxviewer .rxmolstar-frame, .rxresults .rxmolstar-frame,
.rxresults .rxenergy-frame {
  display:block; width:100% !important; max-width:100% !important; height:auto !important;
  box-sizing:border-box !important; overflow:hidden !important;
}
.rxviewer .rxmolstar-frame {
  height:clamp(460px,50vw,660px) !important; max-height:660px !important;
  aspect-ratio:auto !important; border:0 !important;
}
.rxresults .rxmolstar-frame, .rxresults .rxenergy-frame {
  height:clamp(300px,31vw,420px) !important; max-height:420px !important;
  aspect-ratio:auto !important;
}
.rxresult-primary {
  display:flex;align-items:center;gap:8px;flex-wrap:wrap;min-height:32px;
  margin:2px 0 6px;padding:5px 8px;border:1px solid #e2e8f0;border-radius:9px;
  background:#fff;color:#52647b;font-size:12px;box-sizing:border-box;
}
.rxresult-primary b { color:#172033; }
.rxlevel-diagram {
  height:clamp(300px,31vw,420px);min-height:300px;width:100%;
  display:flex;align-items:center;justify-content:center;padding:8px;box-sizing:border-box;
}
.rxlevel-diagram svg { width:100% !important;height:100% !important;display:block; }
.rxresults { width:100% !important; max-width:100% !important; min-width:0 !important; }
.rxresults .jupyter-widgets-output-area, .rxresults iframe,
.rxresults canvas { width:100% !important; max-width:100% !important; box-sizing:border-box !important; }
.rxapp img, .rxapp svg, .rxapp iframe { max-width:100%; }
.rxcmd {
  height:fit-content !important; min-height:48px !important; overflow:visible !important;
}
.rxcmd textarea {
  field-sizing:content; width:100% !important; height:auto !important; min-height:36px !important;
  overflow-x:hidden !important; overflow-y:hidden !important; resize:none !important;
  white-space:pre-wrap !important; overflow-wrap:anywhere !important;
  font-family:"DejaVu Sans Mono","Liberation Mono",Consolas,"Courier New",monospace !important;
  font-size:13px !important; background:#0f172a !important; color:#7ee787 !important;
  border-radius:11px !important; line-height:1.5 !important; border:1px solid #1e293b !important;
  padding:9px 11px !important; }
.rxlog { width:100% !important; min-width:0 !important; }
.rxlog pre, .rxlog code, .rxlog .output_stream, .rxlog .output_text {
  font-family:"DejaVu Sans Mono","Liberation Mono",Consolas,"Courier New",monospace !important;
}
.rxlog .rxlog-console, .rxlog .widget-html-content {
  box-sizing:border-box; width:100%; height:clamp(280px,52vh,420px); min-height:180px;
  overflow-y:scroll !important; overflow-x:hidden !important;
  scrollbar-gutter:stable; overscroll-behavior:contain; overflow-anchor:auto;
  border:1px solid #263449; border-radius:10px; background:#0f172a;
}
.rxlog pre {
  box-sizing:border-box; width:100%; min-height:100%; margin:0; padding:10px 12px;
  white-space:pre-wrap; overflow-wrap:anywhere; border:0; background:transparent;
  color:#d1fae5; font-size:12px; line-height:1.5;
}
.rxlog .rxlog-console::-webkit-scrollbar,
.rxlog .widget-html-content::-webkit-scrollbar { width:12px; }
.rxlog .rxlog-console::-webkit-scrollbar-track,
.rxlog .widget-html-content::-webkit-scrollbar-track { background:#111827; border-radius:999px; }
.rxlog .rxlog-console::-webkit-scrollbar-thumb,
.rxlog .widget-html-content::-webkit-scrollbar-thumb {
  background:#64748b; border:3px solid #111827; border-radius:999px;
}
.rxscan-panel { row-gap:10px !important; }
.rxscan-pair-controls, .rxscan-add-actions,
.rxscan-stage-head, .rxscan-stage-actions,
.rxscan-coordinate-row {
  width:100%; flex-wrap:nowrap !important; align-items:center !important;
  column-gap:8px !important; row-gap:0 !important;
}
.rxscan-pair-controls {
  display:grid !important;
  grid-template-columns:minmax(0,1fr) minmax(0,1fr) 54px;
  gap:6px !important;
}
.rxscan-add-actions {
  display:grid !important; grid-template-columns:minmax(0,.85fr) minmax(0,1.15fr); gap:6px !important;
}
.rxscan-pair-controls, .rxscan-add-actions {
  min-width:0 !important; max-width:100% !important; box-sizing:border-box !important;
  overflow-x:hidden !important; overflow-y:visible !important;
}
.rxscan-pair-controls .widget-button,
.rxscan-add-actions .widget-button { min-width:0 !important; width:100% !important; }
.rxscan-pair-controls button, .rxscan-add-actions button { padding-inline:4px !important; }
.rxscan-pair-readout {
  display:flex; align-items:center; gap:14px; flex-wrap:wrap;
  min-height:34px; padding:5px 0; font-size:14px;
}
.rxscan-pair-readout > span { min-width:0; }
.rxscan-pair-readout b { font-size:15px; }
.rxscan-pair-readout code { font-size:13px !important; }
.rxscan-pair-state { margin-top:2px; line-height:1.4; }
.rxscan-target-row { width:100%; align-items:center !important; }
.rxscan-target-row > .widget-inline-hbox { flex:1 1 100% !important; width:100% !important; }
.rxscan-stages { width:100%; row-gap:10px !important; }
.rxscan-stage {
  border:2px solid #93c5fd; border-radius:12px; padding:12px; margin:8px 0;
  background:#eff6ff; box-shadow:0 0 6px rgba(37,99,235,.10);
  row-gap:10px !important; flex:0 0 auto !important; height:auto !important;
  min-height:0 !important; overflow:visible !important;
}
.rxscan-stage-title { min-width:0; }
.rxscan-stage-actions { width:auto; flex:0 0 auto; }
.rxscan-stage-actions .widget-button { min-width:40px !important; }
.rxscan-bond {
  border-top:1px solid #bfdbfe; padding:8px 0 0;
  row-gap:6px !important; flex:0 0 auto !important; height:auto !important;
  min-height:0 !important; overflow:visible !important;
}
.rxscan-bond > * { flex:0 1 auto !important; min-height:0 !important; }
.rxscan-coordinate-row { justify-content:flex-start !important; }
.rxscan-coordinate-remove { margin-left:auto !important; min-width:36px !important; }
.rxscan-clear-all { align-self:flex-start; margin-top:2px; }
.rxhelp-table-scroll { max-width:100%; overflow-x:auto; }
.rxhelp-panel table { border-collapse:collapse; margin-top:6px; width:100%; }
.rxhelp-panel th, .rxhelp-panel td {
  border:1px solid #cbd5e1; padding:3px 5px; text-align:left; white-space:nowrap;
}
.rxcommand-dock {
  width:100% !important; max-width:100% !important; min-width:0;
  margin-left:0 !important; margin-right:0 !important; box-sizing:border-box;
  flex:0 0 auto !important; min-height:0; max-height:none;
  overflow:visible;
  border-top:1px solid var(--rx-line); padding-top:13px; row-gap:4px !important;
}
.rxcommand-head {
  width:100%; min-height:24px; align-items:center !important;
  column-gap:8px !important; row-gap:3px !important;
}
.rxapp .rxcommand-editor {
  height:fit-content !important; overflow:visible !important;
  padding:8px 10px !important; row-gap:8px !important; align-items:stretch !important;
}
.rxcommand-editor > * { width:100% !important; max-width:100% !important; min-width:0 !important;
  margin-left:0 !important; margin-right:0 !important; box-sizing:border-box !important; }
.rxcommand-footer {
  width:100%; align-items:stretch !important; row-gap:4px !important;
}
.rxcommand-actions {
  width:100%; flex-wrap:nowrap !important; align-items:center !important;
  column-gap:6px !important; overflow-x:hidden !important;
}
.rxcommand-tools, .rxsession-upload, .rxrun, .rxcommand-more {
  flex-wrap:nowrap !important; align-items:center !important; column-gap:6px !important;
}
.rxrun { margin-left:auto !important; }
.rxcommand-footer .rxfold { margin:0; }
.rxdrop { position:relative; min-height:112px; overflow:hidden;
  border:2px dashed #cbd5e1; border-radius:14px; padding:0; background:#f8fafc;
  transition:border-color .15s ease,background .15s ease;
  align-items:center !important; justify-content:center !important; text-align:center; box-sizing:border-box; }
.rxdrop:hover, .rxdrop:focus-within, .rxdrop:has(.rxnative-drop.rxdrag) {
  border-color:#2563eb; background:#eff6ff; }
.rxdrop.rxdrag { border-color:#2563eb; background:#eff6ff; }
.rxdrop .rxnative-input { display:none !important; }
.rxdrop .rxnative-drop { position:relative; min-height:108px; width:100%; display:flex;
  flex-direction:column; align-items:center; justify-content:center; text-align:center;
  box-sizing:border-box; padding:13px; color:#52647b; }
.rxdrop .rxnative-drop.rxbusy { opacity:.72; }
.rxdrop .rxnative-drop input[type="file"] { display:none !important; }
.rxdrop .rxnative-prompt, .rxdrop .rxnative-formats { pointer-events:none; }
.rxapp .rxnative-button {
  align-self:center; min-width:180px; min-height:36px; margin:8px auto 2px;
  padding:7px 18px; display:inline-flex; align-items:center; justify-content:center;
  border:1px solid #2563eb; border-radius:9px; box-sizing:border-box;
  background:linear-gradient(180deg,#ffffff 0%,#f8fbff 100%); color:#1d4ed8;
  font:650 12px/1.2 system-ui,sans-serif; text-align:center; cursor:pointer;
  box-shadow:0 0 5px rgba(37,99,235,.12); transition:background .15s ease,box-shadow .15s ease;
}
.rxapp .rxnative-button:hover { background:#eff6ff; box-shadow:0 0 8px rgba(37,99,235,.16); }
.rxapp .rxnative-button:focus-visible { outline:3px solid rgba(37,99,235,.25); outline-offset:2px; }
.rxapp .rxnative-status { margin-top:4px; font-size:12px; color:#475569; min-height:16px; }
.rxapp .rxmodel-upload .rxnative-button, .rxapp .rxsession-upload .rxnative-button {
  margin-top:0; }
.rxapp .rxsession-upload .rxnative-button {
  width:130px; min-width:130px; max-width:130px;
}
.rxapp .rxsession-upload .rxnative-status:empty { display:none; }
.rxdrop .rxnative-prompt { font-weight:650; color:#334155; }
.rxdrop .rxnative-formats { margin-top:3px; font-size:12px; color:#64748b; }
.rxdrop .rxdrop-prompt { width:100%; padding-top:12px; color:#52647b;
  line-height:1.35; pointer-events:none; }
.rxdrop .rxdrop-prompt b { color:#334155; }
.rxdrop .widget-upload { width:220px !important; max-width:100% !important;
  align-self:center !important; margin:4px auto 12px !important; }
.rxdrop .widget-upload > button { width:220px !important; max-width:100% !important; }
.rxapp .rxfile-list {
  width:100%; row-gap:7px !important; padding:2px 0 5px; box-sizing:border-box;
}
.rxapp .rxfile {
  width:100%; min-height:50px; flex-wrap:nowrap !important; align-items:center !important;
  border:1px solid #dbe5f0; border-radius:11px; padding:7px 8px 7px 12px;
  background:linear-gradient(180deg,#ffffff 0%,#fbfdff 100%); box-sizing:border-box;
  box-shadow:0 0 4px rgba(15,23,42,.045); transition:border-color .15s ease,box-shadow .15s ease;
}
.rxapp .rxfile:hover { border-color:#bfd3e8; box-shadow:0 0 7px rgba(30,64,110,.07); }
.rxapp .rxfile > .widget-html {
  flex:1 1 auto !important; min-width:0 !important; min-height:34px;
  display:flex !important; align-items:center !important; padding-right:8px;
}
.rxapp .rxfile .rxfilename .widget-html-content {
  width:100%; min-height:34px; margin:0 !important;
  display:flex !important; align-items:center !important; line-height:1.35;
}
.rxapp .rxfile .rxfilename .widget-html-content > span {
  display:block !important; width:100%; line-height:1.35;
}
.rxapp .rxfile > .widget-hbox {
  flex:0 0 auto !important; flex-wrap:nowrap !important; align-items:center !important;
  column-gap:6px !important;
}
.rxapp .rxfile button {
  position:relative !important; flex:0 0 34px !important;
  width:34px !important; min-width:34px !important; max-width:34px !important;
  height:34px !important; min-height:34px !important; max-height:34px !important;
  margin:0 !important; padding:0 !important; box-sizing:border-box !important;
  display:inline-flex !important; align-items:center !important; justify-content:center !important;
  border:1px solid #d9e3ee !important; border-radius:9px !important;
  background:#f8fafc !important; color:#334155 !important;
  font-size:0 !important; line-height:0 !important; box-shadow:none !important;
}
.rxapp .rxfile button::after { content:none !important; }
.rxapp .rxfile button .fa {
  display:inline-flex !important; align-items:center !important; justify-content:center !important;
  width:18px !important; height:18px !important; margin:0 !important; padding:0 !important;
  font-size:18px !important; line-height:18px !important; color:#334155;
}
.rxapp .rxfile .rxmove-earlier:hover:not(:disabled),
.rxapp .rxfile .rxmove-later:hover:not(:disabled) {
  border-color:#93c5fd !important; background:#eff6ff !important;
}
.rxapp .rxfile .rxmove-earlier:hover:not(:disabled) .fa,
.rxapp .rxfile .rxmove-later:hover:not(:disabled) .fa { color:#1d4ed8; }
.rxapp .rxfile .rxremove-file .fa { color:#b42318; }
.rxapp .rxfile .rxremove-file:hover:not(:disabled) {
  border-color:#fda4af !important; background:#fff1f2 !important;
}
.rxapp .rxfile button:disabled {
  opacity:.36 !important; border-color:#e2e8f0 !important; background:#f8fafc !important;
}
.rxhelp-row, .rxflagrow, .rxinfo, .rxinfo .widget-html-content {
  position:relative; overflow:visible !important;
}
.rxflag-tier { display:flex; align-items:baseline; gap:9px; padding:12px 10px 7px; margin-top:4px; border-bottom:1px solid var(--rx-line); }
.rxflag-tier:first-child { margin-top:0; }
.rxflag-tier small { color:var(--rx-muted); }
.rxapp .rxhelp-row, .rxapp .rxflagrow {
  flex-wrap:wrap !important; align-items:center !important; width:100% !important;
}
.rxapp .rxhelp-row:not(:has(.rxinfo-details[open])) {
  flex-wrap:nowrap !important; align-items:flex-start !important; column-gap:4px !important;
}
.rxapp .rxhelp-row:not(:has(.rxinfo-details[open])) > .widget-html:first-child {
  flex:1 1 auto !important; min-width:0 !important;
}
.rxapp .rxhelp-row:not(:has(.rxinfo-details[open])) > .rxinfo { flex:0 0 26px !important; }
.rxapp .rxhelp-row:has(.rxinfo-details[open]) { flex-wrap:wrap !important; }
.rxapp .rxinfo { flex:0 0 26px; width:26px; min-width:26px; }
.rxapp .rxinfo:has(.rxinfo-details[open]) {
  flex:1 1 100%; width:100%; min-width:100%;
}
.rxinfo-details > summary {
  width:24px; min-width:24px; height:24px; box-sizing:border-box;
  display:flex; align-items:center; justify-content:center;
  cursor:pointer; list-style:none; color:#52647b; background:transparent;
  border:0; border-radius:4px; font-size:17px; font-weight:600; line-height:1;
}
.rxinfo-details > summary::-webkit-details-marker { display:none; }
.rxinfo-details > summary:hover, .rxinfo-details > summary:focus-visible,
.rxinfo-details[open] > summary {
  color:#1d4ed8; background:#eff6ff; outline:none;
}
.rxinfo-details > summary:focus-visible {
  box-shadow:0 0 0 3px rgba(37,99,235,.24);
}
.rxinfo-details .rxhelp-panel {
  position:relative; border:1px solid #bfdbfe; border-radius:9px;
  background:#eff6ff; color:#1e3a5f; padding:8px 10px;
  margin:5px 0 5px; line-height:1.4;
}
.rxinfo-details .rxhelp-panel::before {
  content:''; position:absolute; top:-6px; left:7px; width:10px; height:10px;
  background:#eff6ff; border-left:1px solid #bfdbfe;
  border-top:1px solid #bfdbfe; transform:rotate(45deg);
}
.rxsr-only { position:absolute !important; width:1px !important; height:1px !important;
  padding:0 !important; margin:-1px !important; overflow:hidden !important;
  clip:rect(0,0,0,0) !important; white-space:nowrap !important; border:0 !important; }
.rxchip button { border-radius:999px !important; font-size:12px !important; padding:1px 10px !important; }
.rxrun button { font-weight:700 !important; }
.rxrun > .widget-button:not(.rxcancel-run):disabled {
  background:#f8fafc !important; border-color:#e2e8f0 !important;
  color:#a3afc2 !important; box-shadow:none !important; opacity:.72 !important;
}
.rxreadiness-error { display:inline-flex; align-items:center; gap:5px;
  background:#fff7ed; border:1px solid #fb923c; color:#9a3412;
  padding:4px 10px; border-radius:999px; font-size:12px; font-weight:750; line-height:1.25;
}
.rxrun > .widget-button:last-child {
  background:#b42318 !important;
  border-color:#8f1d14 !important;
  color:#fff !important;
  box-shadow:0 0 7px rgba(180,35,24,.28) !important;
}
.rxrun > .widget-button:last-child:hover:not(:disabled) {
  background:#8f1d14 !important;
}
.rxrun > .widget-button:last-child:disabled {
  background:#dc5b52 !important;
  border-color:#b42318 !important;
  color:#fff !important;
  box-shadow:0 0 5px rgba(180,35,24,.20) !important;
  opacity:1 !important;
}
.rxapp .widget-hbox {
  flex-wrap:wrap !important; column-gap:8px !important;
  row-gap:8px; align-items:center;
}
.rxapp .widget-hbox.rxfile, .rxapp .rxfile > .widget-hbox {
  flex-wrap:nowrap !important; }
.rxapp .rxworkflow-controls { display:grid !important;
  grid-template-columns:minmax(230px,270px) minmax(140px,1fr) 246px;
  gap:2px 4px; align-items:center; min-width:0; }
.rxapp .rxworkflow-controls > * { min-width:0 !important; }
.rxapp .rxworkflow-controls .widget-toggle-buttons > div {
  flex-wrap:nowrap !important;
}
.rxapp .rxworkflow-contract { display:grid !important;
  grid-template-columns:minmax(220px,.78fr) minmax(0,2.22fr); gap:2px 4px;
  min-width:0; }
.rxapp .rxworkflow-contract > * { min-width:0 !important; }
.rxpath {
  border:1px solid #cfe0f2; border-radius:14px; background:#f8fbff;
  padding:12px !important; margin-top:8px;
  box-shadow:0 0 12px rgba(30,64,110,.065); box-sizing:border-box;
}
.rxpath-head {
  width:100%; min-height:0; align-items:stretch !important;
  justify-content:flex-start !important; row-gap:8px !important; min-width:0;
  margin:0 0 4px !important; overflow:hidden; box-sizing:border-box;
}
.rxpath-head > * { width:100% !important; min-width:0 !important; max-width:100% !important; }
.rxpath-head > .widget-hbox:first-child { justify-content:flex-start !important; align-items:center !important; flex-wrap:wrap !important; gap:6px !important; }
.rxpath-title { font-size:16px; line-height:1.35; font-weight:750; color:var(--rx-ink); }
.rxpath-subtitle { color:var(--rx-muted); font-size:13px; line-height:1.4; margin-left:8px; }
.rxresult-selector {
  width:100% !important; display:grid;
  grid-template-columns:minmax(260px,720px) minmax(150px,max-content);
  gap:10px; align-items:center !important; padding:7px 9px !important;
  border:1px solid #dbe7f3; border-radius:11px; background:#fff; box-sizing:border-box;
}
.rxresult-selector > * { min-width:0 !important; max-width:100% !important; margin:0 !important; }
.rxresult-loading { width:100% !important; margin:0 !important; }
.rxresult-loading .widget-html-content { margin:0 !important; }
.rxresult-loading-banner { display:flex; align-items:center; gap:9px; min-height:34px; padding:7px 10px;
  border:1px solid #bfdbfe; border-radius:10px; background:#eff6ff; color:#334155; box-sizing:border-box; }
.rxresult-choice { width:100% !important; align-items:center !important; }
.rxresult-choice > .widget-label { min-width:44px; margin:0 7px 0 0 !important; font-weight:700; color:#334155; }
.rxresult-choice select { min-height:36px; font-weight:650; color:#172033; border-radius:8px; }
.rxresult-count .widget-html-content { display:flex; align-items:center; gap:8px; color:#64748b; white-space:nowrap; }
.rxresult-count-pill { display:inline-flex; align-items:center; min-height:26px; padding:2px 9px;
  border-radius:999px; background:#e6f5f1; color:#0f766e; font-size:12px; font-weight:750; }
.rxpath-state { width:100% !important; display:grid !important; grid-template-columns:minmax(0,1fr) 36px; gap:8px; align-items:center !important; max-width:100%; min-width:0; overflow:hidden !important; box-sizing:border-box; }
.rxpath-state > * { min-width:0 !important; max-width:100% !important; margin:0 !important; }
.rxpath-state > .widget-html { width:auto !important; overflow:hidden !important; }
.rxpath-state > :last-child { width:36px !important; min-width:36px !important; height:100% !important; justify-self:end; align-self:stretch; display:flex !important; align-items:center !important; justify-content:center !important; }
.rxpath-state > :last-child .widget-html-content { display:flex !important; align-items:center !important; justify-content:center !important; min-height:100% !important; padding:0 !important; overflow:visible !important; background:transparent !important; border:0 !important; }
.rxpath-state > :last-child .rxinfo-details { display:flex; align-items:center; justify-content:center; min-height:100%; }
.rxpath-state .widget-html-content {
  display:block; width:100%; min-height:30px; box-sizing:border-box;
  background:#fff; color:#334155; border:1px solid #cad8e8;
  border-radius:10px; padding:5px 10px; white-space:normal; line-height:1.45;
  overflow-wrap:anywhere; word-break:normal; max-width:100%; overflow:hidden;
}
.rxpath-controls {
  display:grid; grid-template-columns:40px 40px 116px minmax(0,1fr);
  width:100%; min-height:52px; align-items:center !important; gap:10px !important;
  padding:6px 8px !important; box-sizing:border-box; overflow:hidden;
  background:#f4f8fc; border:1px solid #e1e9f2; border-radius:11px;
}
.rxpath-controls > * { min-width:0 !important; max-width:100% !important; margin:0 !important; }
.rxpath-controls > .widget-button {
  width:40px !important; min-width:40px !important; height:38px !important; min-height:38px !important;
  padding:0 !important; font-size:22px !important; color:#334155 !important;
  background:#fff !important; border:1px solid #d7e2ee !important; border-radius:9px !important;
  box-shadow:0 0 4px rgba(15,23,42,.06) !important;
}
.rxpath-controls > .widget-button:disabled { opacity:.42 !important; box-shadow:none !important; }
.rxpath-controls > :nth-child(3) { display:flex !important; width:116px !important; min-width:116px !important; max-width:116px !important; }
.rxpath-controls > :nth-child(4) { display:flex !important; width:100% !important; min-width:0 !important; max-width:none !important; box-sizing:border-box; padding:0 10px !important; overflow:visible !important; }
.rxpath-controls > :nth-child(4) .slider-container { min-width:0 !important; width:100% !important; margin:0 !important; overflow:visible !important; }
.rxpath-controls .widget-play { height:38px !important; min-height:38px !important; }
.rxpath-controls .noUi-target { height:6px !important; border:0 !important; border-radius:999px !important; background:#cbd8e6 !important; box-shadow:none !important; }
.rxpath-controls .noUi-connect { background:#0f766e !important; }
.rxpath-controls .noUi-horizontal .noUi-handle { width:18px !important; height:18px !important; right:-9px !important; top:-6px !important; border:2px solid #0f766e !important; border-radius:50% !important; background:#fff !important; box-shadow:0 0 5px rgba(15,118,110,.22) !important; }
.rxpath-controls .noUi-handle::before, .rxpath-controls .noUi-handle::after { display:none !important; }
.rxpath-controls .widget-label { margin-top:0 !important; margin-bottom:0 !important; }
.rxpath-grid {
  width:100%; max-width:100%; min-width:0; overflow:hidden; box-sizing:border-box;
  display:grid !important;
  grid-template-columns:minmax(0,1.05fr) minmax(0,.95fr);
  gap:12px; align-items:stretch !important; margin:0;
}
.rxpath-grid.rxplot-only, .rxpath-grid.rxstructure-only { grid-template-columns:minmax(0,1fr); }
.rxpath-grid.rxplot-only .rxenergy-frame {
  height:auto!important; min-height:320px!important; max-height:560px!important; aspect-ratio:16/10;
}
.rxscan-pes-frame { display:block!important; width:100%!important; max-width:100%!important; min-width:0!important; height:auto!important; min-height:320px!important; max-height:520px!important; aspect-ratio:16/10; box-sizing:border-box!important; overflow:hidden!important; background:#fff; }
.rxscan-axis-controls { display:flex; flex-direction:column!important; gap:6px!important; width:100%; padding:8px 10px!important; box-sizing:border-box; background:#f4f8fc; border:1px solid #e1e9f2; border-radius:11px; }
.rxscan-axis-row { display:grid; grid-template-columns:122px minmax(0,1fr); align-items:center!important; gap:10px!important; width:100%; min-width:0; }
.rxscan-axis-row > :first-child { width:122px!important; min-width:122px!important; color:#334155; font-weight:650; }
.rxscan-axis-row > :last-child { width:100%!important; min-width:0!important; max-width:none!important; }
.rxpath-panel {
  min-width:0; height:100%; overflow:hidden;
  border:1px solid var(--rx-line); border-radius:12px; background:#fff;
  display:flex; flex-direction:column !important; align-self:stretch !important;
  box-shadow:0 0 4px rgba(15,23,42,.045);
}
.rxpath-panel-title {
  min-height:44px; padding:8px 12px; border-bottom:1px solid #edf1f6;
  color:#44546a; font-size:14px; line-height:1.35; font-weight:700;
  letter-spacing:.15px; display:flex; align-items:center; box-sizing:border-box;
}
.rxpath-panel-head {
  min-height:44px; width:100%; display:grid !important;
  grid-template-columns:minmax(116px,auto) minmax(170px,1fr);
  align-items:center !important; gap:8px; padding:4px 10px;
  border-bottom:1px solid #edf1f6; box-sizing:border-box;
}
.rxpath-panel-head .rxpath-panel-title { min-height:34px; padding:0; border:0; }
.rxstructure-panel-head {
  min-height:44px; width:100%; padding:0 10px 0 0; border-bottom:1px solid #edf1f6;
  box-sizing:border-box; flex-wrap:wrap !important; align-items:center !important;
}
.rxstructure-panel-head > .widget-html:first-child { flex:1 1 auto; min-width:0; }
.rxstructure-panel-head .rxpath-panel-title { min-height:43px; border:0; }
.rxstructure-panel-head > .rxinfo { flex:0 0 26px; }
.rxstructure-panel-head:has(.rxinfo-details[open]) > .rxinfo { flex:1 1 100%; }
.rxpath-panel > .widget-output {
  flex:1 1 auto !important; min-height:0; padding:0 6px 6px;
  box-sizing:border-box;
}
.rxpath-panel svg, .rxpath-panel img, .rxpath-panel canvas {
  width:100% !important; height:auto !important;
}
.rxenergy-choice {
  width:100% !important; min-height:34px;
  margin:0 !important; align-items:center !important;
  box-sizing:border-box;
}
.rxenergy-choice > .widget-label {
  min-width:38px; margin:0 6px 0 0 !important; align-self:center !important;
}
.rxenergy-choice select { font-weight:650; color:#172033; }
.rxartifact { margin-top:6px; }
.rxartifact > button { background:#fff !important; }
.rxartifact-image-wrap { width:100%; overflow:hidden; text-align:center; }
.rxartifact-image, .rxpath-panel img.rxartifact-image { display:block; width:50% !important; max-width:50% !important; height:auto !important; margin:0 auto; object-fit:contain; }
@media (max-width:720px) { .rxartifact-image, .rxpath-panel img.rxartifact-image { width:100% !important; max-width:100% !important; } }
.rxapp hr { border:none; border-top:1px solid #eef0f4; }
@media (min-width: 821px) and (max-height: 900px) {
  .rxheader { padding:5px 12px; margin-bottom:3px; }
  .rxtabs { margin:3px 0 5px !important; }
  .rxtabs .rxtab-button { min-height:34px !important; }
  .rxcommand-dock { padding-top:13px; row-gap:2px !important; }
  .rxcommand-head { min-height:20px; row-gap:2px !important; }
  .rxapp .rxcommand-editor { padding:6px 8px !important; row-gap:6px !important; }
  .rxcmd { height:fit-content !important; min-height:36px !important; }
  .rxcmd textarea { padding:5px 8px !important; }
  .rxcommand-dock .widget-button,
  .rxcommand-dock button.jupyter-button { min-height:32px !important; }
}
@container (max-width: 400px) {
  .rxscan-pair-controls { grid-template-columns:minmax(0,1fr) minmax(0,1fr) 46px !important; gap:4px !important; }
  .rxscan-add-actions { grid-template-columns:minmax(0,.82fr) minmax(0,1.18fr) !important; gap:4px !important; }
  .rxscan-stage-head { flex-wrap:wrap !important; row-gap:8px !important; }
  .rxscan-stage-actions { width:100% !important; }
  .rxscan-coordinate-row { flex-wrap:wrap !important; row-gap:6px !important; }
}
@container (max-width: 280px) {
  .rxscan-pair-controls { grid-template-columns:repeat(2,minmax(0,1fr)) !important; }
  .rxscan-pair-controls > :last-child { grid-column:1 / -1; }
  .rxscan-add-actions { grid-template-columns:1fr !important; }
}
@media (max-width: 940px) {
  .rxapp .rxworkspace { display:flex !important; flex-flow:row wrap !important; }
  .rxviewer, .rxinspector { flex:1 1 100% !important; min-width:0 !important; }
  .rxviewer { max-width:none; }
  .rxinspector { height:auto; max-height:none; overflow-y:visible !important; padding:0; border:0; border-radius:0; background:transparent; box-shadow:none; }
  .rxpath-grid { grid-template-columns:1fr; }
  .rxpath-controls { grid-template-columns:40px 40px 108px minmax(0,1fr) !important; gap:7px !important; }
  .rxpath-controls > :nth-child(3) { width:108px !important; min-width:108px !important; max-width:108px !important; }
}
@media (max-width: 720px) {
  .rxpath-head { row-gap:6px !important; }
  .rxpath-subtitle { display:block; margin:2px 0 0 !important; }
  .rxresult-selector { grid-template-columns:1fr !important; gap:5px; }
  .rxresult-count .widget-html-content { white-space:normal; }
  .rxpath-panel-head { grid-template-columns:1fr !important; gap:2px; padding:6px 10px 8px; }
  .rxviewer-toolbar { flex-wrap:wrap !important; }
  .rxviewer-toolbar > * { flex:1 1 190px !important; min-width:0 !important; max-width:100% !important; }
  .rxcommand-actions { display:grid !important; grid-template-columns:auto minmax(260px,1fr); gap:6px !important; overflow:visible !important; }
  .rxcommand-actions > * { min-width:0 !important; max-width:100% !important; margin-left:0 !important; }
  .rxcommand-actions .rxrun { grid-column:1 / -1; justify-content:flex-end !important; }
  .rxscan-pair-controls { display:grid !important; grid-template-columns:minmax(0,1fr) minmax(0,1fr) 54px; gap:6px !important; }
  .rxscan-add-actions { display:grid !important; grid-template-columns:minmax(0,.85fr) minmax(0,1.15fr); gap:6px !important; }
  .rxviewer .rxmolstar-frame { height:clamp(340px,72vw,460px) !important; }
  .rxresults .rxmolstar-frame, .rxresults .rxenergy-frame, .rxlevel-diagram { height:clamp(300px,64vw,380px) !important; }
  .rxapp .rxworkflow-controls, .rxapp .rxworkflow-contract {
    display:flex !important; flex-flow:row wrap !important;
  }
  .rxapp .rxworkflow-controls > * { flex:1 1 220px !important; width:auto !important; }
  .rxapp .rxworkflow-contract > * { flex:1 1 100% !important; width:100% !important; }
}
@media (max-width: 600px) {
  .rxapp .rxfilename .widget-html-content > span { white-space:normal !important; overflow-wrap:anywhere; }
  .rxapp .widget-dropdown, .rxapp .widget-text,
  .rxapp .widget-select-multiple { max-width:100% !important; }
  .rxapp .lm-TabBar-tab, .rxapp .p-TabBar-tab { min-width:0 !important; }
  .rxapp .lm-TabBar-tabLabel, .rxapp .p-TabBar-tabLabel {
    overflow:hidden; text-overflow:ellipsis; }
}
@media (max-width: 480px) {
  .rxapp { padding-left:4px; padding-right:4px; }
  .rxapp .rxfile { flex-wrap:wrap !important; row-gap:5px !important; padding-left:8px; }
  .rxapp .rxfile > .widget-html { flex:1 1 100% !important; width:100% !important; padding-right:0; }
  .rxapp .rxfile > .widget-hbox { margin-left:auto !important; }
  .rxcommand-actions { grid-template-columns:1fr !important; }
  .rxcommand-actions > *, .rxcommand-tools, .rxsession-upload, .rxrun { width:100% !important; justify-content:stretch !important; }
  .rxcommand-actions .widget-button, .rxsession-upload > * { flex:1 1 0 !important; min-width:0 !important; max-width:none !important; }
  .rxpath-controls { grid-template-columns:38px 38px 96px minmax(0,1fr) !important; gap:6px !important; padding-left:5px !important; padding-right:5px !important; }
  .rxpath-controls > :nth-child(3) { width:96px !important; min-width:96px !important; max-width:96px !important; }
}
@media (prefers-reduced-motion: reduce) {
  .rxapp * { transition:none !important; animation:none !important; }
}
</style>"""))

# Give immediate feedback while the Viewer, Results router, and browser bridges are built.
_gui_launch_status = W.HTML(
    '<div role="status" aria-live="polite" style="margin:6px 8px;padding:9px 12px;'
    'border:1px solid #dbe7f3;border-radius:10px;background:#f8fbff;color:#475569;'
    'font:13px -apple-system,BlinkMacSystemFont,Segoe UI,sans-serif">Building GUI…</div>')
display(_gui_launch_status)

CLI = 'pdb2reaction'
from pdb2reaction.cli import cli as PRODUCT_CLI
from pdb2reaction.domain.residue_data import AMINO_ACIDS as _AA_DATA, ION as _ION_CHARGES, WATER_RES as _WATER_DATA
_INITIAL_MODEL = {'mace': 'MACE-OMOL-0', 'uma': 'uma-s-1p2',
                  'orb': 'orb_v3_conservative_omol'}.get(BACKEND, 'MACE-OMOL-0')
def _dft_runtime_ready():
    if globals().get('DFT_SETUP_READY') is True: return True
    required = ('pyscf', 'basis_set_exchange', 'gpu4pyscf.dft', 'cupy')
    if (importlib.util.find_spec('pyscf') is None or
            importlib.util.find_spec('gpu4pyscf') is None): return False
    try:
        for module in required[:-1]: importlib.import_module(module)
        cupy = importlib.import_module(required[-1])
        return int(cupy.cuda.runtime.getDeviceCount()) > 0
    except Exception:
        return False
DFT_READY = _dft_runtime_ready()
DMF_READY = (importlib.util.find_spec('dmf') is not None and
             importlib.util.find_spec('cyipopt') is not None)
S = {'tool': TOOL, 'backend': BACKEND, 'model': _INITIAL_MODEL,
     'mode': None, 'inputs': [], 'subcmd': 'all',
     'advanced_overrides': {},
     'parm': None, 'model_pdb': None, 'ref_pdbs': [], 'center': [], 'center_ids': [], 'selected_resn': '', 'lcharge': {},
     '_pre_extract': None,
     'scan_atoms': [None, None], 'scan_target': 1.6, 'scan_preset': '', 'scan_stages': [], 'scan_axes': [],
     'freeze_buf': [None, None], 'freeze_pairs': [], 'freeze_atoms': [], 'charge': 0,
     'charge_explicit': False, '_charge_scope': None,
     'tsopt': True, 'thermo': True, 'out_dir': './result_all/',
     '_last_out_dir': None, '_last_subcmd': None, '_last_argv': [], '_last_files': [], '_last_manifest': {}, '_last_log': '',
     '_pdb_path': None, '_pdb_text': '', '_view_format': 'pdb', '_view_source_format': 'pdb',
     '_hetero': [], '_atoms': {}, '_atom_meta': [],
     '_last_pick': None, '_pick_history': [],
     '_last_pick_message': '', '_last_pick_tone': 'ok', '_view_input_index': 0, '_view_mapping_ok': True,
     '_primary_atom_signatures': [], '_primary_atom_meta': [],
     '_uploaded_paths': set(), '_installed_backend': BACKEND,
     '_session_file_identities': {}, '_session_primary_atom_signatures': [],
     'show_water': True, 'surface': False, 'spin': False,
     'measure_atoms': [], 'rep': 'cartoon', 'color': 'element',
     'viewer_width': 720, 'viewer_height': 540}

_AA = set(_AA_DATA)
_WATER = set(_WATER_DATA) | {'T3P','TIP3P','TIP4P','TIP5P','SPC','SPCE','OPC','OPC3'}

def _pdb_coordinate_rows(text):
    rows = []
    for line in text.splitlines():
        if line[0:6].strip() not in ('ATOM', 'HETATM'): continue
        try:
            rows.append((float(line[30:38]), float(line[38:46]), float(line[46:54])))
        except ValueError:
            raise ValueError('Viewer bridge contains an invalid PDB coordinate row.')
    return rows

# Link/cap hydrogens are appended by `extract` as HL atoms in residue LKH.
# They are neither a ligand (no -l charge) nor a sensible extraction center.
_CAP = {'LKH'}


_EXAMPLE_URL = 'https://raw.githubusercontent.com/t-0hmura/pdb2reaction/%s/examples/%s'


def _example_file(relpath):
    """Return a local path to a bundled example structure.

    A source checkout (debug builds) has them on disk; a PyPI install does not,
    because the wheel ships the package only. In that case fetch the file from
    the git tag matching the installed release, so example and code agree.
    """
    local = os.path.join(REPO_DIR, 'examples', relpath)
    if os.path.exists(local):
        return local
    dest = _runtime_path('examples', relpath)
    if not os.path.exists(dest):
        import urllib.request
        try:
            urllib.request.urlretrieve(_EXAMPLE_URL % (_release_tag, relpath), dest)
        except Exception as exc:
            raise RuntimeError('Could not download the example for release %s; retry or use your own files.' % _release_tag) from exc
    return dest

def parse_residues(text, metadata=None):
    allr, het = set(), set()
    if metadata:
        for atom in metadata:
            r = str(atom.get('resname') or '').strip().upper()
            if not r or r in _CAP: continue
            allr.add(r)
            if r not in _WATER and r not in _AA: het.add(r)
        return sorted(allr), sorted(het)
    for ln in text.splitlines():
        if ln[0:6].strip() not in ('ATOM', 'HETATM'): continue
        r = ln[17:20].strip()
        if r in _CAP: continue
        allr.add(r)
        if r not in _WATER and r not in _AA: het.add(r)
    return sorted(allr), sorted(het)

def parse_atoms(text, metadata=None):
    if metadata:
        coords = _pdb_coordinate_rows(text)
        if len(coords) != len(metadata):
            raise ValueError('Viewer PDB atom count does not match retained structure metadata.')
        out = {}
        for index, (meta, xyz) in enumerate(zip(metadata, coords)):
            meta['xyz'] = xyz
            meta['index'] = index
            key = (str(meta.get('chain') or ''), str(meta.get('resname') or ''),
                   str(meta.get('resseq')), str(meta.get('name') or ''))
            out[key] = xyz
        return out
    d = {}
    for ln in text.splitlines():
        if ln[0:6].strip() in ('ATOM', 'HETATM'):
            try:
                d[(ln[21:22].strip(), ln[17:20].strip(), ln[22:26].strip(), ln[12:16].strip())] = \
                    (float(ln[30:38]), float(ln[38:46]), float(ln[46:54]))
            except ValueError:
                continue
    return d

def _load_view_structure(path):
    """Return a safe viewer PDB plus original/auth atom metadata."""
    from pdb2reaction.core.utils import prepare_input_structure, load_pdb_atom_metadata
    with prepare_input_structure(Path(path)) as prepared:
        source = Path(prepared.geom_path)
        text = source.read_text(encoding='utf-8', errors='replace')
        metadata = load_pdb_atom_metadata(source)
        if not metadata:
            raise ValueError('Structure contains no coordinate atoms.')
        viewer_path = Path(_runtime_path('viewer_input.pdb'))
        viewer_path.write_text(text, encoding='utf-8')
    parse_atoms(text, metadata)       # attach stable 0-based index + coordinates
    return text, metadata, str(viewer_path)

def _load_small_view_structure(path):
    """Return the first XYZ geometry plus stable metadata for small molecules."""
    from pdb2reaction.core.utils import prepare_input_structure
    with prepare_input_structure(Path(path)) as prepared:
        source = Path(prepared.geom_path)
        raw = source.read_text(encoding='utf-8', errors='replace')
    lines = raw.splitlines()
    try: atom_count = int(lines[0].strip())
    except (IndexError, ValueError):
        raise ValueError('Small-molecule viewer needs XYZ/GJF coordinates.')
    if atom_count < 1 or len(lines) < atom_count + 2:
        raise ValueError('XYZ input is incomplete.')
    frame = lines[:atom_count + 2]
    metadata = []
    for index, line in enumerate(frame[2:]):
        fields = line.split()
        if len(fields) < 4:
            raise ValueError('XYZ atom row %d is incomplete.' % (index + 1))
        element = fields[0].strip()
        try: xyz = tuple(float(value) for value in fields[1:4])
        except ValueError:
            raise ValueError('XYZ atom row %d has invalid coordinates.' % (index + 1))
        metadata.append({'serial': index + 1, 'chain': '', 'resname': 'MOL',
                         'resseq': 1, 'icode': '', 'name': '%s%d' % (element.upper(), index + 1),
                         'element': element, 'index': index, 'xyz': xyz})
    text = '\n'.join(frame) + '\n'
    viewer_path = Path(_runtime_path('viewer_input.xyz'))
    viewer_path.write_text(text, encoding='utf-8')
    return text, metadata, str(viewer_path)

def _aspec(x):
    chain = str(x.get('chain') or '').strip()
    icode = str(x.get('icode') or '').strip()
    # Exact viewer picks already carry a stable 0-based atom index. Small
    # molecules have no PDB metadata, and a blank-chain insertion code is not
    # representable by the CLI's three-field residue selector, so serialize
    # those two cases as the CLI's default 1-based integer index.
    if S.get('mode') in ('small', 'xyz') or (not chain and icode):
        return int(x['index']) + 1
    if chain:
        return '%s:%s:%s:%s' % (chain, x['resn'], x['resi'], x['atom'])
    return '%s %s %s' % (x['resn'], x['resi'], x['atom'])

def _atom_identity(atom):
    if not atom: return None
    try:
        if atom.get('index') is not None: return ('index', int(atom['index']))
    except (TypeError, ValueError):
        pass
    return ('name', str(atom.get('chain') or ''), str(atom.get('resn') or '').upper(),
            str(atom.get('resi') or ''), str(atom.get('atom') or '').upper())

def _same_atom(first, second):
    return bool(first and second and _atom_identity(first) == _atom_identity(second))

def _stored_atom_records():
    seen = set()
    def emit(value):
        if isinstance(value, dict) and id(value) not in seen:
            seen.add(id(value)); return value
        return None
    for atom in S.get('scan_atoms', []):
        record = emit(atom)
        if record is not None: yield record
    for stage in S.get('scan_stages', []):
        for bond in stage:
            for atom in (bond.get('a'), bond.get('b')):
                record = emit(atom)
                if record is not None: yield record
    for axis in S.get('scan_axes', []):
        for atom in (axis.get('a'), axis.get('b')):
            record = emit(atom)
            if record is not None: yield record
    for atom in S.get('freeze_buf', []):
        record = emit(atom)
        if record is not None: yield record
    for pair in S.get('freeze_pairs', []):
        for atom in (pair.get('a'), pair.get('b')):
            record = emit(atom)
            if record is not None: yield record
    for atom in S.get('measure_atoms', []):
        record = emit(atom)
        if record is not None: yield record
    for atom in S.get('_pick_history', []):
        record = emit(atom)
        if record is not None: yield record
    record = emit(S.get('_last_pick'))
    if record is not None: yield record

def _remap_stored_atom_coordinates(metadata):
    for atom in _stored_atom_records():
        try: index = int(atom.get('index'))
        except (TypeError, ValueError): continue
        if not (0 <= index < len(metadata)): continue
        current = metadata[index]
        atom.update(chain=str(current.get('chain') or ''),
                    resn=str(current.get('resname') or ''),
                    resi=str(current.get('resseq')) + str(current.get('icode') or ''),
                    atom=str(current.get('name') or ''), xyz=current.get('xyz'))

def _assert_distinct_pairs(sub=None):
    """Validate only selectors emitted by the currently selected workflow."""
    sub = sub or _wv('dd_subcmd', S.get('subcmd', 'all'))
    panels = set(SPEC.get(sub, {}).get('panels', ()))
    if sub == 'all' and _wv('all_mode', 'mep') != 'scan':
        panels.discard('scan')
    pairs = []
    if 'scan' in panels and sub in ('all', 'scan'):
        a, b = S.get('scan_atoms', [None, None])
        if a and b: pairs.append(('scan bond', a, b))
        for stage in S.get('scan_stages', []):
            pairs.extend(('staged scan bond', bond.get('a'), bond.get('b')) for bond in stage)
    if 'scan' in panels and sub in ('scan2d', 'scan3d'):
        pairs.extend(('scan axis', axis.get('a'), axis.get('b'))
                     for axis in S.get('scan_axes', []))
    if sub == 'opt':
        pairs.extend(('distance restraint', pair.get('a'), pair.get('b'))
                     for pair in S.get('freeze_pairs', []))
    for label, first, second in pairs:
        if _same_atom(first, second):
            raise ValueError('%s needs two different atoms.' % label.capitalize())

def _quoted_selector(atom):
    """Use JSON double quotes inside a valid Python literal (clean shell display)."""
    return json.dumps(_aspec(atom), ensure_ascii=False)

def _scan_literal_bonds(value):
    try: parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError): return []
    if isinstance(parsed, tuple) and len(parsed) in (3, 4):
        parsed = [parsed]
    if not isinstance(parsed, (list, tuple)): return []
    bonds = []
    for bond in parsed:
        if (not isinstance(bond, (list, tuple)) or len(bond) not in (3, 4) or
                isinstance(bond[2], bool) or not isinstance(bond[2], (int, float))):
            return []
        bonds.append(tuple(bond))
    return bonds

def scan_literals():
    """One literal per stage (multiple = staged); tuples within a stage = concerted."""
    def bond_literal(bond):
        return '(%s,%s,%g)' % (
            _quoted_selector(bond['a']), _quoted_selector(bond['b']), bond['t'])
    if S['scan_stages']:
        return ['[' + ','.join(bond_literal(bond) for bond in stage) + ']'
                for stage in S['scan_stages']]
    if S['scan_preset'] and not all(S['scan_atoms']): return [S['scan_preset']]
    a, b = S['scan_atoms']
    if a and b:
        return ['[(%s,%s,%g)]' % (
            _quoted_selector(a), _quoted_selector(b), S['scan_target'])]
    return []

def scan2d_literal():
    """scan2d/scan3d need ONE -s with per-axis quadruples: [(a,b,low,high),...]."""
    ax = S.get('scan_axes', [])
    if not ax: return ''
    return '[' + ','.join('(%s,%s,%g,%g)' % (
        _quoted_selector(a['a']), _quoted_selector(a['b']), a['lo'], a['hi'])
        for a in ax) + ']'

def freeze_pair_lit():
    if not S['freeze_pairs']: return ''
    parts = []
    for p in S['freeze_pairs']:
        if p.get('t') is not None:
            parts.append('(%r,%r,%g)' % (_aspec(p['a']), _aspec(p['b']), p['t']))
        else:
            parts.append('(%r,%r)' % (_aspec(p['a']), _aspec(p['b'])))
    return '[' + ','.join(parts) + ']'

def scan_distance():
    a, b = S['scan_atoms']
    if a and b and a.get('xyz') and b.get('xyz'):
        return math.dist(a['xyz'], b['xyz'])
    return None

def _xyz(d):
    if not d: return None
    if d.get('xyz') is not None: return d['xyz']
    return S['_atoms'].get((str(d.get('chain') or ''), d['resn'], str(d['resi']), d['atom']))

CLUSTER_SUBS = ['all', 'opt', 'sp', 'tsopt', 'freq', 'irc', 'dft', 'scan', 'scan2d', 'scan3d',
            'path-opt', 'path-search', 'extract', 'fix-altloc', 'add-elem-info',
            'energy-diagram', 'bond-summary', 'trj2fig']
MLMM_SUBS = ['all', 'opt', 'sp', 'tsopt', 'freq', 'irc', 'dft', 'scan', 'scan2d', 'scan3d',
             'path-opt', 'path-search', 'extract', 'define-layer', 'mm-parm', 'oniom-export',
             'oniom-import', 'fix-altloc', 'add-elem-info', 'energy-diagram', 'bond-summary', 'trj2fig']
SUBS = CLUSTER_SUBS if IS_CLUSTER else MLMM_SUBS
COMPUTE = {'all', 'opt', 'sp', 'tsopt', 'freq', 'irc', 'dft', 'scan', 'scan2d', 'scan3d',
           'path-opt', 'path-search'}
MLIP_COMPUTE = COMPUTE - {'dft'}
# Subcommands a non-advanced user can run entirely from the GUI (no CLI typing):
BASIC_SUBS = ['all', 'opt', 'sp', 'tsopt', 'freq', 'irc', 'dft', 'scan', 'scan2d', 'scan3d',
              'path-opt', 'path-search']
SUB_LABELS = {
    'all': 'Full mechanism · all', 'opt': 'Geometry optimization · opt',
    'sp': 'Single-point energy · sp', 'tsopt': 'TS optimization · tsopt',
    'freq': 'Frequencies / thermochemistry · freq', 'irc': 'IRC endpoints · irc',
    'dft': 'DFT single point · dft', 'scan': '1D bond scan · scan',
    'scan2d': '2D scan grid · scan2d', 'scan3d': '3D scan grid · scan3d',
    'path-opt': 'Optimize one path · path-opt', 'path-search': 'Search a multi-step path · path-search',
    'extract': 'Extract a model · extract', 'define-layer': 'Define ONIOM layers · define-layer',
    'mm-parm': 'Build MM parameters · mm-parm', 'oniom-export': 'Export ONIOM input · oniom-export',
    'oniom-import': 'Import ONIOM result · oniom-import',
    'fix-altloc': 'Resolve alternate locations · fix-altloc',
    'add-elem-info': 'Repair element columns · add-elem-info',
    'energy-diagram': 'Draw an energy diagram · energy-diagram',
    'bond-summary': 'Compare bonds · bond-summary', 'trj2fig': 'Plot a trajectory · trj2fig',
}
def _sub_options(names):
    return [(SUB_LABELS.get(name, name), name) for name in names]
if not DFT_READY:
    SUBS = [name for name in SUBS if name != 'dft']
    BASIC_SUBS = [name for name in BASIC_SUBS if name != 'dft']
# ── SPEC: what every subcommand needs, produces and accepts ────────────────
# One declarative table drives every branch: the input requirement hint, which
# Viewer panels apply, which Options are shown, and what the Results page
# looks for. Verified against the CLI itself (click introspection) and docs/.
#   n_in   : (min, max) input files; max None = unbounded
#   panels : Viewer-page panels that apply ('center' | 'scan' | 'freeze')
#   out    : the deliverables worth pointing the user at
SPEC = {
    'all':      dict(n_in=(1, None), panels=('center', 'scan', 'freeze'),
                     req='R + P structures (MEP) · or 1 file + scan-lists · or 1 TS + TS-only mode',
                     out=('summary.log', 'mep.pdb', 'energy_diagram_MEP.png', 'segments/seg_NN/')),
    'extract':  dict(n_in=(1, None), panels=('center',),
                     req='a complex PDB/mmCIF + center residues (-c, required)',
                     out=('the extracted cluster model (-o)',)),
    'opt':      dict(n_in=(1, 1), panels=('freeze',), req='one structure to optimize',
                     out=('final_geometry.{xyz,pdb,cif}', 'optimization_trj.xyz')),
    'sp':       dict(n_in=(1, 1), panels=(), req='one structure (single-point E/F)',
                     out=('stdout energy', 'forces.npy', 'hessian.npy (--hess)')),
    'tsopt':    dict(n_in=(1, 1), panels=('freeze',), req='one TS-candidate structure',
                     out=('final_geometry.{xyz,pdb,cif}', 'vib/imag_*cm-1 (expect exactly one)')),
    'freq':     dict(n_in=(1, 1), panels=('freeze',), req='one optimized structure',
                     out=('frequencies_cm-1.txt', 'mode_*', 'thermoanalysis.yaml (--dump/--thermo)')),
    'irc':      dict(n_in=(1, 1), panels=('freeze',), req='one TS structure',
                     out=('*finished_irc_trj.xyz', 'forward / backward branches')),
    'dft':      dict(n_in=(1, 1), panels=(), req='one structure (DFT single-point)',
                     out=('result.yaml', 'result.json (--out-json)')),
    'scan':     dict(n_in=(1, 1), panels=('scan', 'freeze'),
                     req='1 file + scan-lists (pick atoms A & B in Setup)',
                     out=('stage_XX/result.*', 'scan_trj.xyz')),
    'scan2d':   dict(n_in=(1, 1), panels=('scan', 'freeze'),
                     req='one structure + 2 scan axes (pick bonds + low/high in Setup)',
                     out=('scan grid outputs',)),
    'scan3d':   dict(n_in=(1, 1), panels=('scan', 'freeze'),
                     req='one structure + 3 scan axes, or one precomputed CSV for plotting',
                     out=('scan grid outputs',)),
    'path-opt': dict(n_in=(2, 2), panels=('freeze',), req='exactly two structures (segment endpoints; -i takes both)',
                     out=('final_geometries_trj.xyz', 'hei.xyz (highest-energy image)')),
    'path-search': dict(n_in=(2, None), panels=('freeze',), req='two or more structures (reactant … product)',
                        out=('per-segment path outputs',)),
}
SPEC.update({
    'fix-altloc': dict(n_in=(1, 1), panels=(), req='one PDB (or directory of PDB files) with alternate locations',
                       out=('cleaned structure (-o)',)),
    'add-elem-info': dict(n_in=(1, 1), panels=(), req='one PDB with missing/incorrect element columns',
                          out=('element-repaired PDB (-o)',)),
    'energy-diagram': dict(n_in=(0, None), panels=(), req='label/energy values supplied with repeated -i',
                           out=('energy diagram image (-o)',)),
    'bond-summary': dict(n_in=(2, None), panels=(), req='two or more structures to compare',
                         out=('bond table or JSON on stdout (--json)',)),
    'trj2fig': dict(n_in=(1, 1), panels=(), req='one XYZ trajectory with per-frame energies',
                    out=('energy profile image / HTML / CSV',)),
})
SUBREQ = {k: v['req'] for k, v in SPEC.items()}
# Keep the basic GUI chemically scoped: bare MACE "small/medium/large" aliases are
# materials checkpoints, not MACE-OMOL-0 variants. Advanced users can still type a
# different --backend-model in the editable command line.
MODELS = {'mace': ['MACE-OMOL-0'],
          'uma': ['uma-s-1p2', 'uma-s-1p1', 'uma-m-1p1'],
          'orb': ['orb_v3_conservative_omol']}
DEFAULT_MODEL = {'mace': 'MACE-OMOL-0', 'uma': 'uma-s-1p2', 'orb': 'orb_v3_conservative_omol'}
# Which subcommand accepts which surfaced flag (verified against the CLI, not
# guessed). Widget name -> accepting subcommands; a flag is hidden elsewhere.
FLAG_SUBS = {
    'adv_mep':     {'all', 'path-opt', 'path-search'},
    'adv_dmf':     {'all', 'path-opt', 'path-search'},
    'adv_refine':  {'all'},                                   # --refine-path: path-opt -> path-search
    'adv_flatten': {'all', 'opt', 'tsopt'},
    'adv_thresh':  {'all', 'opt', 'tsopt', 'scan', 'scan2d', 'scan3d', 'path-opt', 'path-search'},
    'adv_radius':  {'all', 'extract'},
    'adv_dft':     {'all'},
    'adv_prec':    MLIP_COMPUTE,
    'adv_det':     MLIP_COMPUTE,
    'adv_mult':    COMPUTE,
}
TOOL_CAPABILITIES = {                    # derived view kept for existing consumers
    'mep_mode': FLAG_SUBS['adv_mep'],
    'threshold': FLAG_SUBS['adv_thresh'],
}
OUT_JSON_SUBS = {'opt', 'sp', 'tsopt', 'freq', 'irc', 'dft', 'scan',
                 'scan2d', 'scan3d', 'path-opt', 'extract'}
_SUBCOMMAND_OUT_DEFAULTS = {
    'all': './result_all/', 'opt': './result_opt/', 'tsopt': './result_tsopt/',
    'freq': './result_freq/', 'irc': './result_irc/', 'scan': './result_scan/',
    'scan2d': './result_scan2d/', 'scan3d': './result_scan3d/',
    'path-opt': './result_path_opt/', 'path-search': './result_path_search/',
    'sp': './result_sp/', 'dft': './result_dft/',
}
AUTOFILL_UTILS = {'fix-altloc', 'add-elem-info', 'bond-summary', 'trj2fig'}
_PREP_SUBS = set(COMPUTE)

def _wv(name, default=None):
    """Safely read a widget's .value by global name (widget may not exist yet)."""
    w = globals().get(name)
    return getattr(w, 'value', default) if w is not None else default

def _apply_subcommand_output_default(v):
    target = _SUBCOMMAND_OUT_DEFAULTS.get(v)
    if not target: return
    S['out_dir'] = target
    widget = globals().get('w_out')
    if widget is not None and widget.value != target: widget.value = target

def set_subcmd(v):
    # Once the dropdown exists, let its observer own the whole transition.
    # Pre-setting S here made the observer miss the change and left out dir
    # one workflow behind.
    dd = globals().get('dd_subcmd')
    if dd is not None and dd.value != v:
        dd.value = v
        return
    previous = S.get('subcmd')
    S['subcmd'] = v
    if previous != v: _apply_subcommand_output_default(v)

def _residue_id_selector(meta):
    """Return one selector in the CLI's chain/sequence-number dialect."""
    resi = str(meta.get('resseq')) + str(meta.get('icode') or '')
    chain = str(meta.get('chain') or '').strip()
    return ('%s:%s' % (chain, resi)) if chain else resi

def _rich_residue_selector(meta):
    chain = str(meta.get('chain') or '').strip()
    resname = str(meta.get('resname') or '').strip().upper()
    resi = str(meta.get('resseq')) + str(meta.get('icode') or '')
    return '%s:%s:%s' % (chain, resname, resi) if chain else resi

def _center_cli_selectors(center_names=None, exact_ids=None, metadata=None):
    """Render -c without mixing the CLI's name and residue-ID grammars.

    Exact picks are kept as rich CHAIN:RESNAME:RESSEQ labels in the GUI.  Once
    one is present, broad residue-name selections are expanded through the
    primary input metadata so the emitted value uses one coherent CLI grammar.
    """
    names = [str(value).strip().upper() for value in
             (S.get('center', []) if center_names is None else center_names) if str(value).strip()]
    exact = [str(value).strip() for value in
             (S.get('center_ids', []) if exact_ids is None else exact_ids) if str(value).strip()]
    if not exact:
        return list(dict.fromkeys(names))
    metadata = S.get('_primary_atom_meta', []) if metadata is None else metadata
    residues = []; seen_residues = set()
    for meta in metadata:
        key = (str(meta.get('chain') or ''), str(meta.get('resname') or '').strip().upper(),
               str(meta.get('resseq')), str(meta.get('icode') or ''))
        if key not in seen_residues:
            seen_residues.add(key); residues.append((key, meta))
    selected = set(); matched_names = set()
    for key, meta in residues:
        if key[1] in names: selected.add(key); matched_names.add(key[1])
    missing = [name for name in names if name not in matched_names]
    if missing:
        raise ValueError('Return to the primary input to resolve center name(s): %s.' % ', '.join(missing))
    for value in exact:
        parts = value.split(':'); matches = []
        for key, meta in residues:
            resi = key[2] + key[3]
            if ((len(parts) == 3 and key[0] == parts[0] and key[1] == parts[1].upper() and resi == parts[2]) or
                (len(parts) == 2 and key[0] == parts[0] and resi == parts[1]) or
                (len(parts) == 1 and resi == parts[0])):
                matches.append(key)
        if len(parts) == 1 and len(matches) > 1:
            raise ValueError('Exact blank-chain center %s is ambiguous across residues/chains; '
                             'use a chain-labelled structure or a residue-name selection.' % value)
        if not matches:
            raise ValueError('Exact center %s is not present in the primary input.' % value)
        selected.update(matches)
    chosen = [(key, meta) for key, meta in residues if key in selected]
    use_rich = bool(chosen) and all(key[0] for key, _ in chosen)
    # Both CLI selector grammars treat an omitted insertion code as “any code”.
    # Reject a token if that would silently include a residue outside the GUI set.
    for key, _ in chosen:
        if key[3]: continue
        broadened = {other for other, _meta in residues
                     if other[2] == key[2] and
                     ((other[0] == key[0] and other[1] == key[1]) if use_rich else
                      (other[0] == key[0] if key[0] else True))}
        if not broadened.issubset(selected):
            raise ValueError('The CLI cannot express exact center %s without also selecting an insertion-code sibling.' %
                             _rich_residue_selector(dict(chain=key[0], resname=key[1], resseq=key[2], icode=key[3])))
    selectors = [(_rich_residue_selector(meta) if use_rich else _residue_id_selector(meta))
                 for key, meta in chosen]
    return list(dict.fromkeys(selectors))

def _input_count_error(sub, count):
    if sub == 'all':
        mode = _wv('all_mode', 'mep')
        required = 2 if mode == 'mep' else 1
        if count != required and not (mode == 'mep' and count > required):
            return ('all MEP needs 2 or more structures.' if mode == 'mep' else
                    'all %s needs exactly 1 input structure.' % ('Scan' if mode == 'scan' else 'TS-only'))
        return ''
    bounds = SPEC.get(sub, {}).get('n_in')
    if bounds is None: return ''
    lo, hi = bounds
    if count >= lo and (hi is None or count <= hi): return ''
    limit = str(lo) if lo == hi else ('%d or more' % lo if hi is None else '%d-%d' % (lo, hi))
    return '%s needs %s input file(s).' % (sub, limit)

_ADV_OWNED_SUBS = {
    'backend_model': set(MLIP_COMPUTE), 'deterministic': set(MLIP_COMPUTE),
    'backend': set(MLIP_COMPUTE), 'charge': set(COMPUTE), 'charge_override': {'all'},
    'center_spec': {'all'}, 'substrate_pdb': {'extract'},
    'dist_freeze_raw': {'opt'}, 'do_dft': {'all'}, 'do_thermo': {'all'}, 'do_tsopt': {'all'},
    'dft_func_basis': {'all'}, 'func_basis': {'dft'},
    'dmf_backend': set(FLAG_SUBS['adv_dmf']), 'flatten': set(FLAG_SUBS['adv_flatten']),
    'freeze_atoms_text': {s for s, spec in SPEC.items() if 'freeze' in spec.get('panels', ())},
    'ligand_charge': set(COMPUTE) | {'extract'},
    'mep_mode': set(FLAG_SUBS['adv_mep']),
    'precision': set(MLIP_COMPUTE), 'radius': {'all', 'extract'},
    'selected_resn': {'all', 'extract'},
    'refine_path': set(FLAG_SUBS['adv_refine']), 'spin': set(COMPUTE),
    'scan_lists_raw': {'all', 'scan'}, 'scan_list_raw': {'scan2d', 'scan3d'},
    'thresh': set(FLAG_SUBS['adv_thresh']),
}
_ADV_GENERATED = {'dry_run'}
_ADV_BLOCKED = {'help_advanced'}
_ADV_AUTO_IO_SUBS = set(COMPUTE) | {'extract'} | set(AUTOFILL_UTILS)
_ADV_IO_FLAGS = {'-i', '--input', '-o', '--out', '--output', '--out-dir', '--out-prefix'}
_ADV_COMMAND_CACHE = {}
_ADV_OPTIONS_CACHE = {}
_ADV_BOOL_CACHE = {}
def _advanced_command(sub):
    if sub not in _ADV_COMMAND_CACHE:
        try:
            import contextlib, io
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                _ADV_COMMAND_CACHE[sub] = PRODUCT_CLI.get_command(click.Context(PRODUCT_CLI), sub)
        except Exception: _ADV_COMMAND_CACHE[sub] = None
    return _ADV_COMMAND_CACHE[sub]

def _advanced_options(sub):
    """Every Click option, including the normally visible and hidden pages."""
    if sub not in _ADV_OPTIONS_CACHE:
        command = _advanced_command(sub)
        _ADV_OPTIONS_CACHE[sub] = tuple(
            param for param in (getattr(command, 'params', ()) or ())
            if isinstance(param, click.Option)) if command else ()
    return _ADV_OPTIONS_CACHE[sub]

def _advanced_bool_metadata(sub):
    if sub not in _ADV_BOOL_CACHE:
        try:
            import contextlib, io
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                _ADV_BOOL_CACHE[sub] = PRODUCT_CLI._resolve_bool_options(
                    click.Context(PRODUCT_CLI), sub)
        except Exception:
            _ADV_BOOL_CACHE[sub] = (set(), set(), {}, set())
    return _ADV_BOOL_CACHE[sub]

def _advanced_status(sub, param):
    name = param.name
    if name in _ADV_BLOCKED: return 'blocked'
    if name == 'out_json' and sub in OUT_JSON_SUBS: return 'generated'
    if name in _ADV_GENERATED: return 'generated'
    if sub in _ADV_AUTO_IO_SUBS and set(param.opts + param.secondary_opts) & _ADV_IO_FLAGS: return 'owned'
    if sub in _ADV_OWNED_SUBS.get(name, set()): return 'owned'
    return 'rendered'

def _named_flag_applies(widget_name, sub):
    if sub not in FLAG_SUBS.get(widget_name, set()): return False
    if sub != 'all': return True
    mode = _wv('all_mode', 'mep')
    if widget_name in {'adv_mep','adv_dmf','adv_refine','adv_thresh'}:
        return mode != 'tsonly'
    if widget_name == 'adv_flatten':
        return mode == 'tsonly' or bool(_wv('w_ts', S.get('tsopt')))
    return True

def _advanced_flag(param):
    return next((opt for opt in param.opts if opt.startswith('--')), param.opts[0])

def _advanced_argv(sub):
    saved = S.get('advanced_overrides', {}).get(sub, {})
    if not saved: return []
    command = _advanced_command(sub)
    bool_values, bool_toggles, negative_aliases, bool_single = _advanced_bool_metadata(sub)
    argv = []
    for param in _advanced_options(sub):
        if _advanced_status(sub, param) != 'rendered' or param.name not in saved:
            continue
        value = saved[param.name]; flag = _advanced_flag(param)
        is_bool = param.is_bool_flag or isinstance(param.type, click.types.BoolParamType)
        if is_bool:
            if value is True:
                argv += [flag] if flag in (set(bool_toggles) | set(bool_single)) or param.is_bool_flag else [flag, 'true']
            elif value is False:
                negative = (next((opt for opt in param.secondary_opts if opt.startswith('--')), None)
                            or negative_aliases.get(flag))
                if negative:
                    argv += [negative]
                elif not param.is_bool_flag:
                    argv += [flag, 'false']
                elif param.default not in (False, None):
                    raise ValueError('%s has no CLI form for disabling its true default.' % flag)
        elif param.multiple:
            values = shlex.split(value) if isinstance(value, str) else list(value)
            nargs = max(1, int(getattr(param, 'nargs', 1)))
            if nargs > 1 and len(values) % nargs:
                raise ValueError('%s expects repeated groups of %d values.' % (flag, nargs))
            if nargs > 1:
                for start in range(0, len(values), nargs):
                    argv += [flag] + [str(item) for item in values[start:start + nargs]]
            else:
                for item in values: argv += [flag, str(item)]
        elif int(getattr(param, 'nargs', 1)) > 1 and value not in (None, ''):
            values = shlex.split(value) if isinstance(value, str) else list(value)
            if len(values) != int(param.nargs):
                raise ValueError('%s expects exactly %d values.' % (flag, int(param.nargs)))
            argv += [flag] + [str(item) for item in values]
        elif value not in (None, ''):
            argv += [flag, str(value)]
    return argv

def _utility_autofill_complete(argv):
    """Return whether the GUI supplied a semantically valid utility command."""
    if len(argv) < 2: return False
    sub = argv[1]
    if sub in AUTOFILL_UTILS: return True
    if sub != 'energy-diagram': return False
    command = _advanced_command(sub)
    if command is None: return False
    try:
        import contextlib, io
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            with command.make_context(
                    sub, list(argv[2:]), resilient_parsing=False) as context:
                if list(getattr(context, 'args', ()) or ()): return False
                callback = command.callback
                while hasattr(callback, '__wrapped__'):
                    callback = callback.__wrapped__
                callback_globals = getattr(callback, '__globals__', {})
                parse_energies = callback_globals.get('_parse_numeric_inputs')
                parse_labels = callback_globals.get('_parse_label_x')
                if not callable(parse_energies) or not callable(parse_labels):
                    return False
                energies = parse_energies(context.params.get('input_values') or ())
                labels = parse_labels(context.params.get('label_x') or ())
                return not labels or len(labels) == len(energies)
    except (click.ClickException, TypeError, ValueError):
        return False

def _advanced_coverage(sub):
    return {param.name: _advanced_status(sub, param) for param in _advanced_options(sub)}


_CHARGE_VERIFY_GUARD = {'active': False}

def _current_ligand_charge_state():
    values = dict(S.get('lcharge') or {})
    if charge_rows is not None:
        values.update({
            name: (_ION_CHARGES[name] if row.get('auto') else row['val'].value)
            for name, row in charge_rows.items() if row['use'].value})
    unconfirmed = sorted(
        name for name, row in (charge_rows or {}).items()
        if not row.get('auto') and not row['use'].value)
    return values, unconfirmed

def _ligand_charge_cli(require_confirmation=False):
    values, unconfirmed = _current_ligand_charge_state()
    if require_confirmation and unconfirmed:
        raise ValueError(
            'Confirm each ligand charge in Setup, including zero: %s.' %
            ', '.join(unconfirmed))
    return ','.join(
        '%s:%g' % (name, values[name]) for name in sorted(values))

_CHARGE_EXTRACT_OVERRIDE_KEYS = {
    'radius_het2het', 'include_h2o', 'exclude_backbone', 'add_linkh',
    'modified_residue',
}
_CHARGE_LAYER_OVERRIDE_KEYS = {
    'model_indices', 'model_indices_str', 'model_indices_one_based',
    'detect_layer', 'config', 'config_yaml',
}

def _charge_region_overrides(sub):
    saved = S.get('advanced_overrides', {}).get(sub, {})
    keys = set()
    if sub == 'all': keys.update(_CHARGE_EXTRACT_OVERRIDE_KEYS)
    if not IS_CLUSTER and sub in COMPUTE:
        keys.update(_CHARGE_LAYER_OVERRIDE_KEYS)
    return {name: saved[name] for name in sorted(keys) if name in saved}

def _charge_scope_fingerprint(sub=None):
    sub = sub or _wv('dd_subcmd', S.get('subcmd', 'all')) or 'all'
    identity = globals().get('_file_identity')
    def file_token(path):
        if not path: return None
        current = identity(path) if identity is not None else None
        return current or {'path': os.path.abspath(str(path)), 'missing': True}
    values, _unconfirmed = _current_ligand_charge_state()
    centers = sorted(_center_cli_selectors())
    region_overrides = _charge_region_overrides(sub)
    if IS_CLUSTER:
        region_kind = ('extracted-system'
                       if sub == 'all' and centers else 'input-system')
    elif sub == 'oniom-export':
        region_kind = ('qm-model-pdb' if S.get('model_pdb')
                       else 'qm-layered-input')
    elif sub == 'all':
        region_kind = ('ml-model-pdb' if S.get('model_pdb') else
                       'ml-extracted' if centers else
                       'ml-layered-input' if region_overrides.get('detect_layer')
                       else 'ml-full-input')
    else:
        region_kind = ('ml-model-pdb' if S.get('model_pdb') else
                       'ml-indices' if (region_overrides.get('model_indices') or
                                        region_overrides.get('model_indices_str'))
                       else 'ml-layered-input')
    payload = {
        'role': ('system' if IS_CLUSTER else
                 'qm' if sub == 'oniom-export' else 'ml-region'),
        'region_kind': region_kind,
        'charge': int(S.get('charge', 0)),
        'inputs': [file_token(path) for path in S.get('inputs', [])],
        'ref_pdbs': [file_token(path) for path in S.get('ref_pdbs', [])],
        'parm': file_token(S.get('parm')),
        'model_pdb': file_token(S.get('model_pdb')),
        'centers': centers,
        'radius': float(_wv('adv_radius', 0.0) or 0.0),
        'selected_resn': str(_wv('selected_resn', S.get('selected_resn', '')) or '').strip(),
        'ligand_charges': sorted((str(name), float(value))
                                 for name, value in values.items()),
        'region_overrides': region_overrides,
    }
    encoded = json.dumps(
        payload, sort_keys=True, separators=(',', ':'), default=str).encode()
    return hashlib.sha256(encoded).hexdigest()

def _invalidate_charge_confirmation():
    S['charge_explicit'] = False
    S['_charge_scope'] = None
    widget = globals().get('w_charge_ok')
    if widget is not None and widget.value and not _CHARGE_VERIFY_GUARD['active']:
        _CHARGE_VERIFY_GUARD['active'] = True
        try: widget.value = False
        finally: _CHARGE_VERIFY_GUARD['active'] = False
    sync = globals().get('_sync_charge_controls')
    if sync is not None: sync()

def _charge_is_current(sub=None):
    return (bool(S.get('charge_explicit')) and
            S.get('_charge_scope') == _charge_scope_fingerprint(sub))

def _effective_run_inputs(sub=None):
    """Resolve the files the selected workflow will actually receive."""
    sub = sub or (_wv('dd_subcmd', S.get('subcmd', 'all')) or 'all')
    inputs = list(S.get('inputs', []))
    if sub == 'all' and _wv('all_mode', 'mep') in ('scan', 'tsonly') and len(inputs) > 1:
        index = max(0, min(int(S.get('_view_input_index', 0)), len(inputs) - 1))
        return [inputs[index]]
    bounds = SPEC.get(sub, {}).get('n_in')
    if bounds and bounds[1] == 1 and len(inputs) > 1:
        return [inputs[0]]
    return inputs

def build_cmd(validate=True):
    if center_widget is not None: S['center'] = list(center_widget.value)
    if charge_rows is not None:
        S['lcharge'] = {r: x['val'].value for r, x in charge_rows.items()
                        if x['use'].value and not x.get('auto')}
    sub = _wv('dd_subcmd', S['subcmd']) or 'all'
    previous_sub = S.get('subcmd')
    S['subcmd'] = sub
    if previous_sub != sub:
        _apply_subcommand_output_default(sub)
        # Click feedback belongs to the workflow that produced it.  Keep
        # persistent selections, but never carry a stale action message
        # (for example, a freeze-atom click) into a scan builder.
        S.update(_last_pick=None, _pick_history=[], _last_pick_message='',
                 _last_pick_tone='ok')
        _rlp = globals().get('_render_last_pick_status')
        if _rlp is not None: _rlp()
    run_inputs = _effective_run_inputs(sub)
    _assert_distinct_pairs(sub)
    count_error = _input_count_error(sub, len(run_inputs))
    if validate and count_error:
        raise ValueError(count_error + (' Put structures in reaction order.' if sub in COMPUTE else ''))
    if validate and (sub in COMPUTE or sub == 'extract' or sub in AUTOFILL_UTILS):
        missing = [path for path in run_inputs if not os.path.isfile(path)]
        if missing:
            raise ValueError('Re-upload missing input file(s): %s.' % ', '.join(os.path.basename(p) for p in missing))
        if S.get('model_pdb') and not os.path.isfile(S['model_pdb']):
            raise ValueError('Re-upload the missing prepared model: %s.' % os.path.basename(S['model_pdb']))
    csv_plot = sub == 'scan3d' and bool(S.get('advanced_overrides', {}).get('scan3d', {}).get('csv_path'))
    if csv_plot:
        return [CLI, 'scan3d', '-o', S['out_dir'], '--out-json'] + _advanced_argv('scan3d')
    bk = S.get('backend', BACKEND)
    if validate and sub in MLIP_COMPUTE and bk != S.get('_installed_backend'):
        raise ValueError('Backend %s is not installed in this runtime; rerun Installation with backend=%s.' % (bk, bk))
    if validate and sub == 'path-search' and len(run_inputs) < 2:
        raise ValueError('path-search needs two or more structures in reaction order.')
    if validate and sub == 'path-opt' and len(run_inputs) != 2:
        raise ValueError('path-opt needs exactly two endpoint structures.')
    if sub == 'all':
        all_kind = _wv('all_mode', 'mep')
        if validate and all_kind == 'mep' and len(run_inputs) < 2:
            raise ValueError('all MEP mode needs two or more structures.')
        if validate and all_kind == 'scan' and (len(run_inputs) != 1 or not scan_literals()):
            raise ValueError('all scan mode needs one structure and a picked scan bond.')
        if validate and all_kind == 'tsonly' and len(run_inputs) != 1:
            raise ValueError('all TS-only mode needs exactly one TS candidate.')
    if validate and sub == 'scan' and not scan_literals():
        raise ValueError('scan needs a picked atom pair and target distance.')
    if sub in ('scan2d', 'scan3d'):
        expected = 2 if sub == 'scan2d' else 3
        if validate and len(S.get('scan_axes', [])) != expected:
            raise ValueError('%s needs exactly %d scan axes.' % (sub, expected))
    if validate and (sub == 'dft' or (sub == 'all' and _wv('adv_dft', False))) and not DFT_READY:
        raise ValueError('DFT support is not installed; rerun Installation with install_dft enabled.')
    center_all = _center_cli_selectors()
    if validate and sub == 'extract' and not center_all:
        raise ValueError('extract needs a center residue (-c); choose one in Setup.')
    charge_values, unconfirmed = _current_ligand_charge_state()
    requires_ligand_confirmation = (sub == 'extract' or
                                   (sub in COMPUTE and bool(center_all)))
    if (validate and requires_ligand_confirmation and S.get('mode') in ('pdb', 'mmcif')
            and unconfirmed):
        raise ValueError('Confirm each ligand charge in Setup, including zero: %s.' %
                         ', '.join(unconfirmed))
    lc = ','.join('%s:%g' % (key, charge_values[key]) for key in sorted(charge_values))
    gjf_supplies_charge = bool(run_inputs) and all(Path(path).suffix.lower() == '.gjf' for path in run_inputs)
    ligand_supplies_charge = bool(lc)
    charge_override_current = _charge_is_current(sub)
    # An override is scoped to the current structure/extraction setup.
    if S.get('charge_explicit') and not charge_override_current:
        _invalidate_charge_confirmation()
        charge_override_current = False
    emit_system_charge = (sub in COMPUTE and not gjf_supplies_charge and
                          (not ligand_supplies_charge or charge_override_current))
    extra = _advanced_argv(sub)
    if sub not in COMPUTE and sub != 'extract':
        cmd = [CLI, sub]
        inputs = run_inputs
        if sub == 'fix-altloc':
            src = Path(inputs[0]); cmd += ['-i', str(src), '-o', str(src.with_name(src.stem + '_altloc_fixed.pdb'))]
        elif sub == 'add-elem-info':
            src = Path(inputs[0]); cmd += ['-i', str(src)]
            # p2r's --overwrite is an explicit in-place request. Keep the safe
            # generated output unless the user selected that advanced flag.
            if not bool(S.get('advanced_overrides', {}).get(sub, {}).get('overwrite')):
                cmd += ['-o', str(src.with_name(src.stem + '_elements.pdb'))]
        elif sub == 'bond-summary':
            for path in inputs: cmd += ['-i', path]
        elif sub == 'trj2fig':
            src = Path(inputs[0])
            if validate and src.suffix.lower() != '.xyz': raise ValueError('trj2fig needs an uploaded XYZ trajectory.')
            cmd += ['-i', str(src), '-o', str(src.with_name(src.stem + '_energy.png'))]
        return cmd + extra
    cmd = [CLI, sub, '-i', *run_inputs]
    if sub in MLIP_COMPUTE: cmd += ['-b', bk]
    mdl = S.get('model')
    if sub in MLIP_COMPUTE and mdl and mdl != DEFAULT_MODEL.get(bk):
        cmd += ['--backend-model', mdl]
    output_arg = S['out_dir']
    if sub == 'extract' and Path(output_arg).suffix.lower() not in ('.pdb', '.ent', '.cif', '.mmcif'):
        output_arg = os.path.join(output_arg, 'cluster.pdb')
    cmd += ['-o', output_arg]
    if sub in ('all', 'extract') and center_all: cmd += ['-c', ','.join(center_all)]
    if lc: cmd += ['-l', lc]
    if sub in OUT_JSON_SUBS: cmd += ['--out-json']
    if sub in ('scan2d', 'scan3d'):
        if scan2d_literal(): cmd += ['-s', scan2d_literal()]  # one -s, per-axis (i,j,low,high) quadruples
    elif (sub == 'scan') or (sub == 'all' and all_kind == 'scan'):
        lits = scan_literals(); cmd += (['-s'] + lits) if lits else []        # single -s followed by staged literals
    # Without -l, -q is the direct system charge. With -l it is emitted only
    # when the Viewer override checkbox is selected.
    if emit_system_charge:
        cmd += ['-q', str(S['charge'])]
    m = _wv('adv_mult', 1)
    if sub in COMPUTE and m not in (None, 1): cmd += ['-m', str(int(m))]
    precision = _wv('adv_prec', 'auto')
    if sub in MLIP_COMPUTE and precision in ('fp32', 'fp64'): cmd += ['--precision', precision]
    if sub in MLIP_COMPUTE and _wv('adv_det', False): cmd += ['--deterministic']
    r = float(_wv('adv_radius', 0.0))
    # Radius controls cluster extraction; zero is an explicit valid cutoff.
    radius_applies = (sub == 'extract' or
                      (sub == 'all' and S.get('mode') in ('pdb', 'mmcif')))
    if radius_applies: cmd += ['-r', str(r)]
    selected = str(_wv('selected_resn', S.get('selected_resn', '')) or '').strip()
    if sub in ('all', 'extract') and selected:
        cmd += ['--selected-resn', selected]
    th = _wv('adv_thresh', '(default)')
    if _named_flag_applies('adv_thresh', sub) and th and th != '(default)': cmd += ['--thresh', th]
    th_post = _wv('adv_thresh_post', '(default)')
    if sub == 'all' and th != '(default)' and th_post and th_post != '(default)':
        cmd += ['--thresh-post', th_post]
    mep = _wv('adv_mep', '(default)')
    if _named_flag_applies('adv_mep', sub) and mep and mep != '(default)':
        if validate and mep == 'dmf' and not DMF_READY:
            raise ValueError('DMF needs pydmf + cyipopt; rerun Installation.')
        cmd += ['--mep-mode', mep]
        dmf_backend = _wv('adv_dmf', '(default)')
        if mep == 'dmf' and dmf_backend != '(default)':
            cmd += ['--dmf-backend', dmf_backend]
    if _named_flag_applies('adv_flatten', sub) and _wv('adv_flatten', False): cmd += ['--flatten']
    if _named_flag_applies('adv_refine', sub) and _wv('adv_refine', False): cmd += ['--refine-path']
    if sub == 'opt' and freeze_pair_lit(): cmd += ['--dist-freeze', freeze_pair_lit()]
    if 'freeze' in SPEC.get(sub, {}).get('panels', ()) and S['freeze_atoms']:
        cmd += ['--freeze-atoms', ','.join(str(i) for i in S['freeze_atoms'])]
    if sub == 'all':
        do_tsopt = bool(S['tsopt']); do_thermo = bool(S['thermo'])
        do_dft = bool(_wv('adv_dft', False))
        if validate and (do_thermo or do_dft) and not do_tsopt:
            raise ValueError('Thermochemistry and DFT require TS optimization.')
        if do_tsopt: cmd.append('--tsopt')
        if do_thermo: cmd.append('--thermo')
        if do_dft:
            cmd.append('--dft')
            fb = _wv('adv_dftfb', '')
            if fb: cmd += ['--dft-func-basis', fb]
    elif sub == 'dft':
        fb = _wv('adv_dftfb', '')
        if fb: cmd += ['--func-basis', fb]
    cmd += extra
    return cmd

# -------- PyMOL-style editable command line + readiness chip ------------------
ready_chip = W.HTML()
manual_mode_notice = W.HTML(
    '<div role="alert"><b>Manual command mode.</b> Changes in Setup and Options are not applied. '
    'Use <b>Rebuild</b> in the command bar to return to GUI control.</div>',
    layout=W.Layout(display='none'))
manual_mode_notice.add_class('rxmanual-notice')
toast = W.HTML()
_ACTION_STATE = {'running': False, 'auto_ready': False, 'ready_before_run': ''}
def _manual_command_ready():
    line = cmd_box.value.strip() if 'cmd_box' in globals() else ''
    if not line or line.startswith('#'): return False
    try: argv = shlex.split(line)
    except ValueError: return False
    return bool(argv and os.path.basename(argv[0]) == CLI)

def _current_command_subcommand():
    line = cmd_box.value.strip() if 'cmd_box' in globals() else ''
    try: argv = shlex.split(line)
    except ValueError: return ''
    if len(argv) < 2 or os.path.basename(argv[0]) != CLI: return ''
    normalizer = globals().get('_normalized_scope_argv')
    normalized = normalizer(argv) if normalizer is not None else argv
    return normalized[1] if len(normalized) > 1 else ''

def _sync_action_enabled():
    run = globals().get('b_run'); validate = globals().get('b_validate')
    if run is None or validate is None: return
    usable = (_ACTION_STATE['auto_ready'] if _auto.get('on', True) else _manual_command_ready())
    blocked = _ACTION_STATE['running'] or not usable
    run.disabled = blocked
    validate.disabled = blocked or _current_command_subcommand() not in COMPUTE

run_status = W.HTML(value='<div role="status" aria-live="polite" aria-atomic="true" style="min-height:24px"></div>')
_RUN_STATE = {'validated_fingerprint': None, 'validation_log': '', 'kind': ''}
# A rerun of Launch GUI replaces the previous widget tree. Stop any task or
# child process owned by that tree so stale callbacks cannot blank or lock it.
_previous_execution = globals().get('_RUN_EXECUTION')
if isinstance(_previous_execution, dict):
    _previous_task = _previous_execution.get('task')
    if _previous_task is not None and not _previous_task.done():
        _previous_task.cancel()
    _previous_process = _previous_execution.get('process')
    if _previous_process is not None:
        try:
            _previous_active = (_previous_process.poll() is None if hasattr(_previous_process, 'poll')
                                else _previous_process.returncode is None)
            if _previous_active: _previous_process.terminate()
        except Exception:
            pass
_RUN_EXECUTION = {'thread': None, 'task': None, 'process': None,
                  'cancel': threading.Event(), 'argv': None,
                  'deferred_deletes': set()}
_RUN_TONES = {'info': '#2563eb', 'ok': '#1f7a3d', 'warn': '#b45309', 'error': '#a00', 'muted': '#64748b'}

def _set_run_status(text='', tone='muted', kind='', _on_ioloop=False):
    """Update one consistently announced status region without blocking a worker."""
    _RUN_STATE.update(kind=kind, text=str(text), tone=tone)
    badge = (('<span style="background:%s;color:#fff;padding:2px 9px;border-radius:11px;">%s</span>' %
              (_RUN_TONES.get(tone, _RUN_TONES['muted']), html.escape(str(text)))) if text else '')
    markup = ('<div role="status" aria-live="polite" aria-atomic="true" '
              'style="min-height:24px">%s</div>' % badge)
    def emit():
        run_status.value = markup
        chip = globals().get('ready_chip')
        if chip is not None: chip.value = badge
        if chip is not None:
            try: chip.send_state()
            except Exception: pass
    if _on_ioloop:
        return emit()
    dispatcher = globals().get('_dispatch_ui')
    return dispatcher(emit) if dispatcher is not None else emit()

def _command_fingerprint(text):
    return hashlib.sha256(str(text or '').encode('utf-8')).hexdigest()

def _invalidate_last_run(reason='Inputs changed; run again to populate Results.'):
    """Detach Results once; avoid repainting hidden panes for a pristine GUI."""
    had_run = bool(S.get('_last_manifest') or S.get('_last_files') or S.get('_last_log'))
    had_validation = bool(_RUN_STATE.get('validated_fingerprint') or
                          _RUN_STATE.get('validation_log'))
    S.update(_last_out_dir=None, _last_subcmd=None, _last_argv=[], _last_files=[],
             _last_manifest={}, _last_log='', _results_presented_dir=None,
             _results_notice=(reason if had_run else ''))
    _RUN_STATE['validated_fingerprint'] = None
    _RUN_STATE['validation_log'] = ''
    # Setup text fields can emit one trait change per keystroke.  Before a
    # run or validation exists there is no Results DOM to invalidate; touching
    # every hidden Results widget here makes Colab repaint the whole app and
    # can recreate the live Mol* iframe.
    if not had_run and not had_validation:
        return
    if had_run:
        status = ('command changed · validate and run again'
                  if str(reason).lower().startswith('command') else
                  'inputs changed · validate and run again')
        _set_run_status(status, 'warn', 'changed')
        log_emit = globals().get('_run_log_emit')
        if callable(log_emit):
            log_emit('', clear=True, flush=True)
    guard = globals().get('_result_pick_guard')
    if guard is not None: guard['active'] = True
    try:
        for name in ('artifact_choice', 'traj_choice'):
            widget = globals().get(name)
            if widget is not None:
                widget.options = []; widget.disabled = True
        slider = globals().get('frame_slider')
        if slider is not None:
            slider.max = 1; slider.value = 0; slider.disabled = True
        for name in ('frame_prev', 'frame_next'):
            button = globals().get(name)
            if button is not None: button.disabled = True
    finally:
        if guard is not None: guard['active'] = False
    traj = globals().get('_TRAJ')
    if traj is not None:
        traj.update(frames=[], energies=[], energy_provenance=[],
                    energy_unit='hartree', path=None, semantics={})
    for name in ('res_out', 'artifact_out', 'traj_out', 'plot_out'):
        output = globals().get(name)
        if output is None:
            continue
        # Results panes can be either an Output widget or an HTML widget.
        # Do not use ``with output`` here: W.HTML is not a context manager.
        if hasattr(output, 'outputs'):
            # Colab does not reliably capture Output.clear_output() from a
            # widget callback; it can clear the entire Launch GUI cell.
            output.outputs = ()
        elif hasattr(output, 'value'):
            output.value = ''
    for name in ('artifact_fold', 'trajectory_box'):
        box = globals().get(name)
        if box is not None: box.layout.display = 'none'
    empty = globals().get('results_empty')
    if empty is not None:
        empty.layout.display = ''
        empty.value = ('<div role="status" style="padding:10px;border:1px dashed #64748b;'
                       'border-radius:10px;">%s</div>' % html.escape(reason if had_run else 'No results yet.'))
    context = globals().get('result_context')
    if context is not None: context.value = ''
    for name in ('traj_label', 'frame_state', 'trajectory_intro'):
        widget = globals().get(name)
        if widget is not None: widget.value = ''
    button = globals().get('dl_btn')
    if button is not None: button.disabled = True
    button = globals().get('res_btn')
    if button is not None:
        button.description = 'Load results'; button.disabled = False
cmd_box = W.Textarea(value='# Preparing the %s command…' % CLI,
                     placeholder='the generated command appears here — edit only when needed',
                     layout=W.Layout(width='100%', height='auto', min_height='48px',
                                     overflow='visible'))
cmd_box.add_class('rxcmd')
_RUNNING = '<span role="status" style="background:#b45309;color:#fff;padding:2px 9px;border-radius:11px;font-size:12px;">⏳ running…</span>'
_OK = '<span style="background:#1f7a3d;color:#fff;padding:2px 9px;border-radius:11px;font-size:12px;">● ready to run</span>'
_NO = '<span role="alert" class="rxreadiness-error"><b aria-hidden="true">!</b> %s</span>'
_MANUAL = '<span role="status" style="background:#7c5c00;color:#fff;padding:2px 9px;border-radius:11px;font-size:12px;">◆ add utility arguments below</span>'
_MANUAL_EDIT = '<span role="status" style="background:#7c5c00;color:#fff;padding:2px 9px;border-radius:11px;font-size:12px;">◆ manual command · GUI changes are detached</span>'
_auto = {'on': True, 'guard': False}
def _has_current_run():
    return bool(S.get('_last_manifest') or S.get('_last_files') or S.get('_last_log'))
def _matches_last_run(text):
    if not S.get('_last_argv'): return not _has_current_run()
    try: return shlex.split(str(text or '')) == list(S['_last_argv'])
    except ValueError: return False
def _set_cmd(txt):
    changed = cmd_box.value != txt
    _auto['guard'] = True; cmd_box.value = txt; _auto['guard'] = False
    if changed: toast.value = ''
    if _has_current_run() and not _matches_last_run(txt):
        _invalidate_last_run('Command changed; run again to populate Results.')
def refresh(_=None, *, surface_only=False):
    manual_mode_notice.layout.display = 'none' if _auto['on'] else ''
    command_ready = False
    try:
        built = build_cmd()
        if _auto['on']: _set_cmd(' '.join(shlex.quote(c) for c in built))
        is_utility = len(built) > 1 and built[1] not in COMPUTE and built[1] != 'extract'
        manual_utility = is_utility and not _utility_autofill_complete(built)
        ready_chip.value = (_RUNNING if _ACTION_STATE['running'] else
                            (_MANUAL if manual_utility and _auto['on'] else
                             (_OK if _auto['on'] else _MANUAL_EDIT)))
        command_ready = not manual_utility
    except Exception as e:
        if _auto['on']:
            # Keep the editable command synchronized even when the current
            # combination is not runnable (for example Scan with 2 inputs).
            # Readiness stays blocked and reports the exact validation error.
            try:
                preview = build_cmd(validate=False)
                _set_cmd(' '.join(shlex.quote(c) for c in preview))
            except Exception:
                _sub = _wv('dd_subcmd', S.get('subcmd', 'all')) or 'all'
                _set_cmd('# %s %s — %s' % (CLI, _sub, str(e)))
        ready_chip.value = (_RUNNING if _ACTION_STATE['running'] else
                            (_NO % html.escape(str(e)) if _auto['on'] else _MANUAL_EDIT))
        if not _auto['on']: command_ready = _manual_command_ready()
    _ACTION_STATE['auto_ready'] = command_ready
    _sync_action_enabled()
    if not surface_only:
        _rs = globals().get('_render_summary')
        if _rs is not None: _rs()
        _rc = globals().get('_render_chips')
        if _rc is not None: _rc()
        _ro = globals().get('_render_output_note')
        if _ro is not None: _ro()
    valid = _RUN_STATE.get('validated_fingerprint')
    valid_command = valid[0] if isinstance(valid, tuple) else valid
    if valid and valid_command != _command_fingerprint(cmd_box.value):
        _RUN_STATE['validated_fingerprint'] = None
        _RUN_STATE['validation_log'] = ''
        log_emit = globals().get('_run_log_emit')
        if callable(log_emit): log_emit('', clear=True, flush=True)
        _set_run_status('command changed · validate again', 'warn', 'changed')
def _on_cmd_edit(_):
    if not _auto['guard']:
        _auto['on'] = False   # user took manual control of the line
        ready_chip.value = _MANUAL_EDIT
        manual_mode_notice.layout.display = ''
        _sync_action_enabled()
        if _has_current_run():
            _invalidate_last_run('Command changed; run again to populate Results.')
        elif _RUN_STATE.get('validated_fingerprint'):
            _RUN_STATE['validated_fingerprint'] = None
            _RUN_STATE['validation_log'] = ''
            log_emit = globals().get('_run_log_emit')
            if callable(log_emit): log_emit('', clear=True, flush=True)
            _set_run_status('command changed · validate again', 'warn', 'changed')
cmd_box.observe(_on_cmd_edit, names='value')

# ============================================================== INPUT tab
input_msg = W.HTML()
_OPERATION_LOADING = {'depth': 0, 'label': ''}
def _set_operation_loading(label, active):
    """Show one shared, tab-style busy banner while input identity is changing."""
    loading = globals().get('_tab_loading'); body = globals().get('_tab_body')
    if active:
        _OPERATION_LOADING['depth'] += 1; _OPERATION_LOADING['label'] = str(label)
        if loading is not None:
            loading.value = (
                '<div class="rxtab-loading-banner" role="status" aria-live="polite" aria-atomic="true">'
                '<span class="rxtab-spinner" aria-hidden="true"></span>'
                'Loading <b>%s</b>…</div>' % html.escape(str(label)))
        if body is not None: body.add_class('rxpages-loading')
    else:
        _OPERATION_LOADING['depth'] = max(0, _OPERATION_LOADING['depth'] - 1)
        if _OPERATION_LOADING['depth'] == 0:
            if loading is not None: loading.value = ''
            if body is not None: body.remove_class('rxpages-loading')
    for widget in (loading, body):
        try:
            if widget is not None: widget.send_state()
        except Exception: pass
def _clear_structure_bound_state(preserve_model=None):
    """Clear selections before input identity changes; preserve a reused model upload."""
    global center_widget, charge_rows
    previous_model = S.get('model_pdb')
    _invalidate_last_run('Input identity changed; validate and run the new system.')
    center_widget = None; charge_rows = None
    S.update(center=[], center_ids=[], selected_resn='', lcharge={}, model_pdb=None, _pre_extract=None,
             scan_atoms=[None, None], scan_preset='', scan_stages=[], scan_axes=[],
             freeze_buf=[None, None], freeze_pairs=[], freeze_atoms=[], measure_atoms=[],
             charge=0, charge_explicit=False, _charge_scope=None,
             _pdb_path=None, _pdb_text='', _view_format='pdb', _view_source_format='pdb',
             _hetero=[], _atoms={}, _atom_meta=[], _last_out_dir=None,
             _last_subcmd=None, _last_argv=[], _last_files=[], _last_manifest={}, _last_log='',
             _last_pick=None, _pick_history=[],
             _last_pick_message='', _last_pick_tone='ok',
             _view_input_index=0, _view_mapping_ok=True,
             _primary_atom_signatures=[], _primary_atom_meta=[],
             _session_file_identities={}, _session_primary_atom_signatures=[])
    charge_widget = globals().get('w_q')
    if charge_widget is not None: charge_widget.value = 0
    charge_ok = globals().get('w_charge_ok')
    if charge_ok is not None: charge_ok.value = False
    S['charge_explicit'] = False
    selected_resn_widget = globals().get('selected_resn')
    if selected_resn_widget is not None:
        S['_resetting_structure_state'] = True
        try:
            selected_resn_widget.value = ''
        finally:
            S.pop('_resetting_structure_state', None)
    delete_owned = globals().get('_delete_owned_uploads')
    if (delete_owned is not None and
            _path_identity(previous_model) != _path_identity(preserve_model)):
        delete_owned([previous_model])
    revert = globals().get('b_revert')
    if revert is not None: revert.layout.display = 'none'
    for name, message in (('center_panel', 'Load a primary PDB/mmCIF to choose center residues.'),
                          ('charge_panel', 'Ligand charges appear after the primary structure loads.')):
        panel = globals().get(name)
        if panel is not None: panel.children = [W.HTML('<small>%s</small>' % message)]

def load_pdb(paths, parm=None, center=None, lcharge=None, scan_preset='', mode='pdb',
             append=False, keep_subcmd=False):
    previous_subcmd = S.get('subcmd', 'all')
    previous_parm = S.get('parm')
    target_subcmd = previous_subcmd if (append or keep_subcmd) else 'all'
    paths = [p for p in paths if p]
    if append:
        _invalidate_last_run('Input files changed; validate and run the updated system.')
        _retire_session_binding()
        paths = list(S.get('inputs', [])) + [p for p in paths if p not in S.get('inputs', [])]
        S.update(inputs=paths, parm=(parm or S.get('parm')), mode=mode, subcmd=target_subcmd)
        if center is not None: S['center'] = list(center)
        if lcharge is not None: S['lcharge'] = dict(lcharge)
        if scan_preset: S['scan_preset'] = scan_preset
    else:
        _clear_structure_bound_state()
        if not keep_subcmd:
            globals().get('_ALL_MODE_STATE', {})['user'] = False
        S.update(inputs=paths, parm=parm, mode=mode, subcmd=target_subcmd,
                 center=list(center or []), center_ids=[], lcharge=dict(lcharge or {}),
                 scan_atoms=[None, None], scan_preset=scan_preset)
    if _path_identity(previous_parm) != _path_identity(S.get('parm')):
        _delete_owned_uploads([previous_parm])
    set_subcmd(target_subcmd)
    am = globals().get('all_mode')
    mode_state = globals().get('_ALL_MODE_STATE', {})
    if (am is not None and not keep_subcmd and target_subcmd == 'all' and
            (not append or not mode_state.get('user'))):
        mode_state['sync'] = True
        try: am.value = 'mep' if len(paths) >= 2 else 'scan'
        finally: mode_state['sync'] = False
    _riq = globals().get('_render_input_queue')
    if _riq is not None: _riq()
    build_selection()
    _scc = globals().get('_sync_capability_controls')
    if _scc is not None: _scc()
    refresh()   # stay on Input: more files may still be added

_acc = '.pdb,.ent,.cif,.mmcif,.xyz,.gjf,.csv'
_drop_formats_detail = '.pdb / .cif / .xyz / .gjf'
_drop_formats = 'Structures (.pdb/.cif/.xyz/.gjf)'
# Hosted Colab needs its browser-native upload bridge: it does not render
# AnyWidget and drops a FileUpload's binary buffers. A Colab local runtime
# reliably transfers FileUpload bytes, so use that standard widget there.
_UPLOAD_MODE = ('colab' if IN_COLAB else
                ('basic' if IS_COLAB_FRONTEND else
                 ('anywidget' if _HAS_DROP_WIDGET else 'basic')))
if _UPLOAD_MODE == 'anywidget':
    class _DropUpload(anywidget.AnyWidget):
        """A native file target whose picker and drop events share one FIFO path."""
        _esm = r"""
        function render({model, el, signal}) {
          el.classList.add('rxnative-drop');
          let prompt = document.createElement('div');
          prompt.className = 'rxnative-prompt';
          prompt.setAttribute('role', 'status');
          prompt.setAttribute('aria-live', 'polite');
          prompt.setAttribute('aria-atomic', 'true');
          prompt.textContent = 'Drop files here';
          let formats = document.createElement('div');
          formats.className = 'rxnative-formats';
          formats.textContent = model.get('formats');
          formats.title = model.get('formats_detail');
          let browse = document.createElement('button');
          browse.type = 'button';
          browse.className = 'rxnative-button';
          browse.textContent = 'Upload files';
          browse.setAttribute('aria-label', 'Upload input files');
          let input = document.createElement('input');
          input.type = 'file';
          input.multiple = true;
          input.accept = model.get('accept');
          input.setAttribute('aria-label', 'Choose input files');
          function openPicker(event) {
            event.preventDefault();
            event.stopPropagation();
            input.click();
          }
          browse.addEventListener('click', openPicker);
          let busy = false, active = null, dragDepth = 0, queue = [];
          let generation = model.get('generation');
          const opaqueId = () => (globalThis.crypto && globalThis.crypto.randomUUID) ?
            globalThis.crypto.randomUUID() :
            (Date.now().toString(36) + '-' + Math.random().toString(36).slice(2));
          function submit(files) {
            if (model.get('disabled')) return;
            const selected = Array.from(files || []);
            input.value = '';
            if (!selected.length || signal.aborted) return;
            const total = selected.reduce((sum, file) => sum + file.size, 0);
            if (selected.length > model.get('max_files') ||
                selected.some(file => file.size > model.get('max_file_bytes')) ||
                total > model.get('max_batch_bytes')) {
              prompt.textContent = 'Not added — upload size or file-count limit exceeded';
              return;
            }
            queue.push({id: opaqueId(), files: selected, generation});
            if (busy) {
              prompt.textContent = 'Queued ' + queue.length + ' upload batch' +
                                   (queue.length === 1 ? '' : 'es');
            }
            pump();
          }
          async function pump() {
            if (busy || !queue.length || signal.aborted) return;
            const item = queue.shift(), selected = item.files;
            if (item.generation !== generation) { pump(); return; }
            active = item.id;
            busy = true;
            el.classList.add('rxbusy');
            prompt.textContent = 'Adding ' + selected.length +
                                 (selected.length === 1 ? ' file…' : ' files…');
            try {
              const buffers = await Promise.all(selected.map(file => file.arrayBuffer()));
              if (signal.aborted || active !== item.id) return;
              model.send({
                event: 'upload',
                batch: item.id,
                generation: item.generation,
                files: selected.map(file => ({
                  name: file.name, size: file.size, type: file.type || ''
                }))
              }, undefined, buffers);
            } catch (_error) {
              busy = false;
              active = null;
              el.classList.remove('rxbusy');
              prompt.textContent = 'Upload failed — click or drop to retry';
              pump();
            }
          }
          function complete(message) {
            if (!message || message.event !== 'upload-complete' ||
                message.generation !== generation || message.batch !== active) return;
            busy = false;
            active = null;
            el.classList.remove('rxbusy');
            prompt.textContent = message.ok ? 'Add more files — drop or click' :
                                              'Not added — drop or click to retry';
            pump();
          }
          function onInputChange(event) { submit(event.target.files); }
          input.addEventListener('change', onInputChange);
          function onDragEnter(event) {
            event.preventDefault();
            event.stopPropagation();
            dragDepth += 1;
            el.classList.add('rxdrag');
            if (event.dataTransfer) event.dataTransfer.dropEffect = 'copy';
          }
          function onDragOver(event) {
            event.preventDefault();
            event.stopPropagation();
            el.classList.add('rxdrag');
            if (event.dataTransfer) event.dataTransfer.dropEffect = 'copy';
          }
          function onDragLeave(event) {
            event.preventDefault();
            event.stopPropagation();
            dragDepth = Math.max(0, dragDepth - 1);
            if (!dragDepth || !el.contains(event.relatedTarget)) {
              dragDepth = 0;
              el.classList.remove('rxdrag');
            }
          }
          function onDrop(event) {
            event.preventDefault();
            event.stopPropagation();
            dragDepth = 0;
            el.classList.remove('rxdrag');
            if (model.get('disabled')) return;
            submit(event.dataTransfer ? event.dataTransfer.files : []);
          }
          el.addEventListener('dragenter', onDragEnter);
          el.addEventListener('dragover', onDragOver);
          el.addEventListener('dragleave', onDragLeave);
          el.addEventListener('drop', onDrop);
          function cancelPending() {
            generation = model.get('generation');
            queue = [];
            busy = false;
            active = null;
            el.classList.remove('rxbusy', 'rxdrag');
            prompt.textContent = 'Drop files here';
          }
          function syncDisabled() {
            const disabled = !!model.get('disabled');
            browse.disabled = disabled;
            input.disabled = disabled;
            el.setAttribute('aria-disabled', disabled ? 'true' : 'false');
          }
          model.on('msg:custom', complete);
          model.on('change:generation', cancelPending);
          model.on('change:disabled', syncDisabled);
          syncDisabled();
          signal.addEventListener('abort', () => {
            queue = [];
            busy = false;
            active = null;
            model.off('msg:custom', complete);
            model.off('change:generation', cancelPending);
            model.off('change:disabled', syncDisabled);
            el.removeEventListener('dragenter', onDragEnter);
            el.removeEventListener('dragover', onDragOver);
            el.removeEventListener('dragleave', onDragLeave);
            el.removeEventListener('drop', onDrop);
            browse.removeEventListener('click', openPicker);
            input.removeEventListener('change', onInputChange);
          }, {once: true});
          el.append(prompt, formats, browse, input);
          return () => model.off('msg:custom', complete);
        }
        export default {render};
        """
        accept = traitlets.Unicode().tag(sync=True)
        formats = traitlets.Unicode().tag(sync=True)
        formats_detail = traitlets.Unicode().tag(sync=True)
        max_files = traitlets.Int(64).tag(sync=True)
        max_file_bytes = traitlets.Int(256 * 1024 * 1024).tag(sync=True)
        max_batch_bytes = traitlets.Int(512 * 1024 * 1024).tag(sync=True)
        generation = traitlets.Int(0).tag(sync=True)
        disabled = traitlets.Bool(False).tag(sync=True)
    upl = _DropUpload(accept=_acc, formats=_drop_formats, formats_detail=_drop_formats_detail)
else:
    # Colab, and any frontend without anywidget, use the core upload widget: its
    # button stays visible in the middle of the dashed zone and its own input
    # receives both picked and dropped files.
    upl = W.FileUpload(accept=_acc, multiple=True, description='Upload files', icon='upload',
                       layout=W.Layout(width='220px'))
def _widget_upload_pairs(raw):
    """Unpack an ipywidgets upload value (7.x dict or 8.x tuple) into pairs."""
    return [(name, meta['content']) for name, meta in raw]

def _save_upload(name, content):
    """Write one browser file to a collision-free path without an overwrite race."""
    clean = os.path.basename(str(name).replace('\\', '/')) or 'upload.dat'
    if '\x00' in clean: raise ValueError('Invalid upload filename.')
    path = Path(clean)
    for number in range(1, 10000):
        target = path if number == 1 else path.with_name('%s_%d%s' % (path.stem, number, path.suffix))
        try:
            with open(target, 'xb') as fh: fh.write(memoryview(content))
            return str(target)
        except FileExistsError:
            continue
        except Exception:
            try: target.unlink()
            except OSError: pass
            raise
    raise RuntimeError('Could not allocate a unique name for %s.' % clean)

def _discard_rejected_uploads(paths):
    """Remove only newly saved browser files that no UI role accepted."""
    attached = {os.path.abspath(path) for path in list(S.get('inputs', [])) +
                [p for p in (S.get('parm'), S.get('model_pdb')) if p]}
    for path in paths:
        if os.path.abspath(path) not in attached and os.path.isfile(path): os.remove(path)

def _remember_uploaded_paths(paths):
    """Track only browser-created runtime copies; source/example paths stay external."""
    S.setdefault('_uploaded_paths', set()).update(
        os.path.abspath(str(path)) for path in paths if path)

def _path_identity(path):
    """Canonical identity for relative/absolute/symlink aliases of one file."""
    if not path: return None
    return os.path.realpath(os.path.abspath(str(path)))

def _active_input_identities():
    """Return browser paths still owned by the command currently executing."""
    argv = _RUN_EXECUTION.get('argv')
    collector = globals().get('_command_input_files')
    if not argv or collector is None: return set()
    return {_path_identity(path) for path in collector(argv) if path}

def _referenced_upload_identities():
    paths = list(S.get('inputs', [])) + list(S.get('ref_pdbs', []))
    paths += [S.get('parm'), S.get('model_pdb')]
    return {_path_identity(path) for path in paths if path}

def _delete_owned_uploads(paths):
    """Delete detached browser copies after any active command releases them."""
    owned = S.setdefault('_uploaded_paths', set())
    deferred = _RUN_EXECUTION.setdefault('deferred_deletes', set())
    targets = {_path_identity(path) for path in paths if path}
    protected = _active_input_identities()
    for absolute in list(owned):
        identity = _path_identity(absolute)
        if identity not in targets: continue
        if identity in protected:
            deferred.add(absolute)
            continue
        try:
            if os.path.isfile(absolute): os.remove(absolute)
        finally:
            owned.discard(absolute)
            deferred.discard(absolute)

def _flush_deferred_upload_deletes():
    """Release detached browser copies when execution ends, unless reattached."""
    owned = S.setdefault('_uploaded_paths', set())
    deferred = _RUN_EXECUTION.setdefault('deferred_deletes', set())
    referenced = _referenced_upload_identities()
    for absolute in list(deferred):
        if _path_identity(absolute) in referenced:
            deferred.discard(absolute)
            continue
        try:
            if os.path.isfile(absolute): os.remove(absolute)
        finally:
            owned.discard(absolute)
            deferred.discard(absolute)

def _reset_file_upload(widget):
    value = widget.value
    if isinstance(value, dict):
        value.clear()
        if hasattr(widget, '_counter'): widget._counter = 0
    else:
        widget.value = ()

def _preflight_structures(paths):
    for path in paths:
        _load_view_structure(path)

def _rebind_missing_uploads(loaded):
    """Reattach missing saved roles by content identity, committing one batch atomically."""
    remaining = list(loaded)
    references = [('input', i, path) for i, path in enumerate(S.get('inputs', []))
                  if not os.path.isfile(path)]
    references += [('ref', i, path) for i, path in enumerate(S.get('ref_pdbs', []))
                   if not os.path.isfile(path)]
    if S.get('parm') and not os.path.isfile(S['parm']):
        references.append(('parm', None, S['parm']))
    if S.get('model_pdb') and not os.path.isfile(S['model_pdb']):
        references.append(('model', None, S['model_pdb']))
    if not references:
        return remaining, []

    plan = {}
    for candidate in remaining:
        matches = [
            ref for ref in references
            if _identity_matches(candidate, _expected_session_identity(ref[0], ref[1]))
        ]
        if not matches:
            raise ValueError(
                'This session is waiting for saved files; uploaded content does not match '
                'any pending SHA-256 identity: %s.' % os.path.basename(candidate))
        for ref in matches:
            key = (ref[0], ref[1])
            previous = plan.get(key)
            if previous is not None and previous != candidate:
                raise ValueError(
                    'More than one uploaded file matches the same pending session role: %s.' %
                    os.path.basename(ref[2]))
            plan[key] = candidate

    candidate_state = dict(S)
    candidate_state['inputs'] = list(S.get('inputs', []))
    candidate_state['ref_pdbs'] = list(S.get('ref_pdbs', []))
    for (kind, index), candidate in plan.items():
        if kind == 'input':
            candidate_state['inputs'][index] = candidate
        elif kind == 'ref':
            candidate_state['ref_pdbs'][index] = candidate
        elif kind == 'parm':
            candidate_state['parm'] = candidate
        else:
            candidate_state['model_pdb'] = candidate

    expected_primary = S.get('_session_primary_atom_signatures') or []
    actual_primary = _session_primary_signature_for(candidate_state)
    if expected_primary and actual_primary is not None and actual_primary != expected_primary:
        raise ValueError('Re-attached primary atom identity/order does not match the saved session.')

    S['inputs'] = candidate_state['inputs']
    S['ref_pdbs'] = candidate_state['ref_pdbs']
    S['parm'] = candidate_state.get('parm')
    S['model_pdb'] = candidate_state.get('model_pdb')
    rebound = [
        '%s → %s' % (
            os.path.basename(next(ref[2] for ref in references
                                  if ref[0] == kind and ref[1] == index)),
            os.path.basename(candidate))
        for (kind, index), candidate in plan.items()
    ]
    _invalidate_last_run('Missing session files were re-attached; validate before running.')
    return [], rebound

def _ingest_saved_files(loaded, source='upload'):
    """Attach one saved browser/drop batch without replacing earlier batches."""
    previous_parm = S.get('parm')
    loaded = [str(path) for path in loaded if path]
    structures = [p for p in loaded if p.lower().endswith(('.pdb', '.ent', '.cif', '.mmcif'))]
    parm_files = [p for p in loaded if p.lower().endswith('.parm7')]
    smalls = [p for p in loaded if p.lower().endswith(('.xyz', '.gjf'))]
    csv_files = [p for p in loaded if p.lower().endswith('.csv')]
    known = set(structures + parm_files + smalls + csv_files)
    unsupported = [p for p in loaded if p not in known]
    mixed_utility = bool(csv_files) and len(loaded) != 1
    if unsupported or len(parm_files) > 1 or (structures and smalls) or (parm_files and IS_CLUSTER) or mixed_utility:
        reason = ('parm7 belongs in the mlmm notebook, not pdb2reaction' if parm_files and IS_CLUSTER else
                  'unsupported file: %s' % ', '.join(os.path.basename(p) for p in unsupported)
                  if unsupported else 'attach one file type at a time')
        input_msg.value = '<div role="alert" style="color:#991b1b">%s; existing files were kept.</div>' % html.escape(reason)
        return False
    try:
        _preflight_structures(structures)
        for small_path in smalls: _load_small_view_structure(small_path)
    except Exception as exc:
        input_msg.value = ('<div role="alert" style="color:#991b1b">Structure was not attached: %s</div>' %
                           html.escape(str(exc)))
        return False
    loaded, rebound = _rebind_missing_uploads(loaded)
    if rebound and not loaded:
        _render_input_queue()
        referenced = (list(S.get('inputs', [])) + list(S.get('ref_pdbs', [])) +
                      [path for path in (S.get('parm'), S.get('model_pdb')) if path])
        pending = [path for path in referenced if not os.path.isfile(path)]
        if not pending and S.get('inputs') and os.path.isfile(S['inputs'][0]):
            if not build_selection(): raise ValueError('Re-attached structure could not be loaded.')
            _verify_session_primary_signature()
        refresh()
        waiting = ('<br><small>Still needed: %s</small>' %
                   html.escape(', '.join(os.path.basename(path) for path in pending))
                   if pending else '')
        input_msg.value = ('✅ re-attached <b>%s</b>%s' %
                           (html.escape(', '.join(rebound)), waiting))
        if _path_identity(previous_parm) != _path_identity(S.get('parm')):
            _delete_owned_uploads([previous_parm])
        return True
    structures = [p for p in loaded if p.lower().endswith(('.pdb', '.ent', '.cif', '.mmcif'))]
    parm_files = [p for p in loaded if p.lower().endswith('.parm7')]
    smalls = [p for p in loaded if p.lower().endswith(('.xyz', '.gjf'))]
    csv_files = [p for p in loaded if p.lower().endswith('.csv')]
    parm = parm_files[0] if parm_files else None
    if csv_files:
        displaced = (list(S.get('inputs', [])) + list(S.get('ref_pdbs', [])) +
                     [p for p in (S.get('parm'), S.get('model_pdb')) if p])
        _clear_structure_bound_state()
        csv_path = csv_files[0]
        S.update(mode='utility', inputs=[csv_path], subcmd='scan3d', parm=None)
        S.setdefault('advanced_overrides', {}).setdefault('scan3d', {})['csv_path'] = csv_path
        _delete_owned_uploads(displaced)
        set_subcmd('scan3d')
        input_msg.value = '✅ 3D scan surface <b>%s</b> attached for plot-only mode.' % html.escape(csv_path)
        _render_input_queue(); _sync_capability_controls(); render_viewer(); refresh()
    elif structures:
        if S.get('inputs') and S.get('mode') not in (None, 'pdb', 'mmcif'):
            input_msg.value = '<div role="alert" style="color:#991b1b">Remove the current small-molecule files before adding structures.</div>'
            return False
        adding = bool(S.get('inputs'))
        load_pdb(structures, parm=(parm or S.get('parm')), append=adding)
        input_msg.value = '✅ %s <b>%s</b> (%s)' % (
            'appended' if adding else 'loaded', html.escape(', '.join(structures)), html.escape(source))
    elif smalls:
        if S.get('inputs') and S.get('mode') not in (None, 'small'):
            input_msg.value = '<div role="alert" style="color:#991b1b">Remove the current structures before adding small molecules.</div>'
            return False
        adding = bool(S.get('inputs'))
        previous_sub = S.get('subcmd', 'opt')
        if adding:
            merged = list(S.get('inputs', [])) + [p for p in smalls if p not in S.get('inputs', [])]
            _invalidate_last_run('Input files changed; validate and run the updated system.')
            _retire_session_binding()
        else:
            merged = smalls; _clear_structure_bound_state()
        S.update(mode='small', inputs=merged, charge_explicit=False)
        if not adding and not globals().get('_rep_user_set', {}).get('value', False):
            S['rep'] = 'stick'
            _rep_widget = globals().get('dd_rep')
            if _rep_widget is not None and _rep_widget.value != 'stick':
                globals().get('_REP_SYNC', {})['active'] = True
                try: _rep_widget.value = 'stick'
                finally: globals().get('_REP_SYNC', {})['active'] = False
        compatible = adding and not _input_count_error(previous_sub, len(merged))
        chosen_sub = previous_sub if compatible else ('opt' if len(merged) == 1 else 'all')
        if chosen_sub == 'all' and len(merged) >= 2:
            state = globals().get('_ALL_MODE_STATE', {})
            state['sync'] = True
            try:
                if globals().get('all_mode') is not None: all_mode.value = 'mep'
            finally:
                state['sync'] = False
        set_subcmd(chosen_sub)
        _scc = globals().get('_sync_capability_controls')
        if _scc is not None: _scc()
        input_msg.value = '✅ %s <b>%s</b> (%s) — set total charge in ② Setup → System charge & multiplicity.' % (
            'appended' if adding else 'loaded', html.escape(', '.join(smalls)), html.escape(source))
        _render_input_queue()
        if not build_selection(): return False
        refresh()
    elif parm:
        _invalidate_last_run('Amber topology changed; validate and run again.')
        _retire_session_binding()
        S['parm'] = parm; _render_input_queue(); refresh()
        input_msg.value = '✅ topology <b>%s</b> attached (%s)' % (html.escape(parm), html.escape(source))
    if _path_identity(previous_parm) != _path_identity(S.get('parm')):
        _delete_owned_uploads([previous_parm])
    return bool(structures or smalls or csv_files or parm)

def _accept_upload_pairs(pairs, source):
    renamed, loaded = [], []
    try:
        for name, content in pairs:
            target = _save_upload(name, content); loaded.append(target)
            if target != os.path.basename(name): renamed.append('%s → %s' % (name, target))
        accepted = _ingest_saved_files(loaded, source)
    except Exception as exc:
        accepted = False
        input_msg.value = ('<div role="alert" style="color:#991b1b">Upload failed: %s; '
                           'existing files were kept.</div>' % html.escape(str(exc)))
    if not accepted: _discard_rejected_uploads(loaded)
    else: _remember_uploaded_paths(loaded)
    if accepted and renamed:
        input_msg.value += ('<br><small>Existing files kept; uploaded as %s</small>' %
                            html.escape(', '.join(renamed)))
    return bool(accepted)

_upload_guard = {'active': False}
def _on_upload(change):
    if _upload_guard['active']: return
    items = change.get('new', upl.value)
    if not items: return
    raw = list(items.items()) if isinstance(items, dict) else [(f['name'], f) for f in items]
    pairs = _widget_upload_pairs(raw)
    _set_operation_loading('files', True)
    try:
        _accept_upload_pairs(pairs, 'file picker')
    finally:
        _upload_guard['active'] = True
        try: _reset_file_upload(upl)
        finally: _upload_guard['active'] = False
        _set_operation_loading('files', False)

def _on_drop_upload(widget, content, buffers):
    if content.get('event') != 'upload': return
    batch = content.get('batch')
    generation = content.get('generation')
    claimed, _reason = _claim_drop_batch(batch, generation)
    if not claimed or generation != widget.generation:
        widget.send({'event': 'upload-complete', 'batch': batch,
                     'generation': generation, 'ok': False})
        return
    ok = False
    _set_operation_loading('files', True)
    try:
        files = content.get('files') or []
        if not files or len(files) > 64:
            raise ValueError('Upload batch file-count limit exceeded.')
        if len(files) != len(buffers):
            raise ValueError('Incomplete browser upload batch.')
        pairs, total = [], 0
        for meta, buffer in zip(files, buffers):
            name = meta.get('name') if isinstance(meta, dict) else None
            if not name: raise ValueError('A browser file has no name.')
            view = memoryview(buffer)
            size = view.nbytes; total += size
            if size > 256 * 1024 * 1024 or total > 512 * 1024 * 1024:
                raise ValueError('Upload size limit exceeded.')
            if int(meta.get('size', size)) != size:
                raise ValueError('Browser file size changed during transfer: %s' % name)
            pairs.append((name, view))
        ok = _accept_upload_pairs(pairs, 'browser upload')
    except Exception as exc:
        input_msg.value = ('<div role="alert" style="color:#991b1b">Upload failed: %s; '
                           'existing files were kept.</div>' % html.escape(str(exc)))
    finally:
        widget.send({'event': 'upload-complete', 'batch': batch,
                     'generation': generation, 'ok': bool(ok)})
        _set_operation_loading('files', False)

if _UPLOAD_MODE == 'anywidget': upl.on_msg(_on_drop_upload)
else: upl.observe(_on_upload, names='value')

_EX = (['BezA methyltransferase - cluster MEP (R->P)',
        'Aromatic Claisen rearrangement - small molecule']
       if IS_CLUSTER else ['methyltransferase complex - scan mode'])
ex_choice = W.Dropdown(options=_EX, value=_EX[0], description='example',
                       style={'description_width': 'initial'},
                       layout=W.Layout(width='400px', max_width='100%'))
ex_btn = W.Button(description='Load example', icon='flask', layout=W.Layout(width='150px'))
ex_btn.add_class('rxoperation-trigger')
example_msg = W.HTML()
def _load_example_impl(_):
    sel = ex_choice.value
    if IS_CLUSTER and sel.startswith('Aromatic Claisen'):
        try:
            reactant = _example_file('aromatic_claisen/reactant.xyz')
            product = _example_file('aromatic_claisen/product.xyz')
        except Exception as exc:
            example_msg.value = '⚠️ could not fetch the example: %s' % exc
            return
        _queue_change(clear=True)
        _ingest_saved_files([reactant, product], 'example')
        S.update(charge=0, charge_explicit=False, _charge_scope=None)
        set_subcmd('all'); all_mode.value = 'mep'
        _scc = globals().get('_sync_capability_controls')
        if _scc is not None: _scc()
        wq = globals().get('w_q')
        if wq is not None: wq.value = 0
        adv_refine.value = False
        example_msg.value = '✅ Aromatic Claisen rearrangement loaded. Confirm charge 0 in ② Setup → System charge & multiplicity.'
        input_msg.value = '<small>Loaded example: <b>Aromatic Claisen rearrangement</b> (2 structures, R → P).</small>'
        _render_input_queue()
        def _sync_example_command():
            _auto['on'] = True
            refresh()
            publisher = globals().get('_publish_run_widget_state')
            if publisher is not None: publisher()
        try:
            loop = getattr(getattr(get_ipython(), 'kernel', None), 'io_loop', None)
            if loop is None: _sync_example_command()
            else: loop.add_callback(_sync_example_command)
        except Exception:
            _sync_example_command()
        return
    if IS_CLUSTER:
        try:
            _r, _p = _example_file('1.R.pdb'), _example_file('3.P.pdb')
        except Exception as exc:
            example_msg.value = '⚠️ could not fetch the example: %s' % exc; return
        _queue_change(clear=True)
        # BezA proceeds through an intermediate, so the example ships with
        # recursive path-search on: each extra MEP segment is one step.
        adv_refine.value = True
        example_msg.value = ('⭐ BezA methyltransferase (R→P MEP). --refine-path is on: '
                             'recursive path-search reports every elementary step it finds.')
        load_pdb([_r, _p], center=['SAM', 'GPP', 'MG'], lcharge={'SAM': 1, 'GPP': -3})
        input_msg.value = '<small>Loaded example: <b>BezA methyltransferase</b> (2 structures, R → P).</small>'
    else:
        try:
            _c = _example_file('methyltransferase/complex.pdb')
            _pm = _example_file('methyltransferase/complex.parm7')
        except Exception as exc:
            example_msg.value = '⚠️ could not fetch the example: %s' % exc; return
        S['scan_target'] = 1.3
        adv_refine.value = False
        example_msg.value = '⭐ Methyltransferase (scan mode).'
        load_pdb([_c], parm=_pm,
                 center=['SAM', 'PHN'], lcharge={'SAM': 1, 'PHN': -1},
                 scan_preset="[('SAM 359 CS1','PHN 360 C8',1.3)]")
        input_msg.value = '<small>Loaded example: <b>Methyltransferase scan</b>.</small>'
def _load_example(_):
    _set_operation_loading('example', True); ex_btn.disabled = True
    try: return _load_example_impl(_)
    finally:
        ex_btn.disabled = False; _set_operation_loading('example', False)
def _on_colab_example():
    """Run one example load to completion for the browser-native Colab bridge."""
    _load_example(None)
    return {'ok': True}
ex_btn.on_click(_load_example)
_input_formats = _drop_formats_detail
_drop_prompt = W.HTML(
    '<div class="rxdrop-prompt"><b>Drop files here</b> or use the button below'
    '<br><small title="%s">Structures (.pdb/.cif/.xyz/.gjf)</small></div>' %
    html.escape(_input_formats, quote=True))
# The Colab zone reports transfer state here; the browser script fills it in.
_drop_status = W.HTML('<div class="rxnative-status" role="status" aria-live="polite"></div>')
_DROP_STATE = {'generation': 0, 'seen': set()}

def _claim_drop_batch(batch, generation):
    batch = str(batch or '')
    try: generation = int(generation)
    except (TypeError, ValueError): return False, 'invalid generation'
    if generation != _DROP_STATE['generation']: return False, 'stale generation'
    if not batch: return False, 'missing batch id'
    if batch in _DROP_STATE['seen']: return False, 'duplicate batch'
    _DROP_STATE['seen'].add(batch)
    return True, ''

def _bump_drop_generation():
    _DROP_STATE['generation'] += 1
    _DROP_STATE['seen'].clear()
    if _UPLOAD_MODE == 'anywidget': upl.generation = _DROP_STATE['generation']

_drop_children = ([upl] if _UPLOAD_MODE == 'anywidget'
                  else [_drop_prompt, _drop_status] if _UPLOAD_MODE == 'colab'
                  else [_drop_prompt, upl])
_drop = W.VBox(_drop_children,
               layout=W.Layout(width='100%', align_items='center', justify_content='center'))
_drop.add_class('rxdrop')
input_file_rows = W.VBox(layout=W.Layout(width='100%'))
input_file_rows.add_class('rxfile-list')
input_order_note = W.HTML()

def _queue_change(path=None, delta=0, remove=False, clear=False, kind='input'):
    if kind == 'parm':
        _invalidate_last_run('Amber topology removed; validate and run again.')
        _retire_session_binding()
        previous_parm = S.get('parm'); S['parm'] = None
        _delete_owned_uploads([previous_parm])
        input_msg.value = '<small>Topology removed.</small>'
        _render_input_queue(); refresh(); return
    if kind == 'model':
        clear_model = globals().get('_clear_uploaded_model')
        if clear_model is not None: clear_model()
        return
    old_paths = list(S.get('inputs', [])); paths = list(old_paths)
    old_aux_paths = [S.get('parm'), S.get('model_pdb')]
    old_view_index = max(0, min(int(S.get('_view_input_index', 0)),
                                max(0, len(old_paths) - 1)))
    old_view_path = old_paths[old_view_index] if old_paths else None
    if clear:
        _bump_drop_generation()
        paths = []; S['parm'] = None; S['model_pdb'] = None
    elif path in paths:
        i = paths.index(path)
        if remove: paths.pop(i)
        else:
            j = max(0, min(len(paths) - 1, i + delta))
            paths[i], paths[j] = paths[j], paths[i]
    if old_paths != paths:
        _retire_session_binding()
    primary_changed = old_paths[:1] != paths[:1]
    if old_view_path in paths:
        S['_view_input_index'] = paths.index(old_view_path)
        displayed_changed = False
    else:
        S['_view_input_index'] = min(old_view_index, max(0, len(paths) - 1))
        new_view_path = paths[S['_view_input_index']] if paths else None
        displayed_changed = old_view_path != new_view_path
    if primary_changed:
        globals().get('_ALL_MODE_STATE', {})['user'] = False
        _clear_structure_bound_state()
    else:
        _invalidate_last_run('Input order changed; validate and run again.')
        if displayed_changed:
            S.update(_last_pick=None, _pick_history=[], _last_pick_message='',
                     _last_pick_tone='ok')
    detached = [item for item in old_paths if item not in paths]
    if clear: detached.extend(old_aux_paths)
    _delete_owned_uploads(detached)
    S['inputs'] = paths
    input_msg.value = ('<small>Primary input changed — residue/atom selections, charge, and prepared model were cleared.</small>'
                       if primary_changed else
                       '<small>Input order updated; selections tied to input 1 were kept.</small>')
    if not paths:
        rep_state = globals().get('_rep_user_set')
        if rep_state is not None: rep_state['value'] = False
        S['rep'] = 'cartoon'
        rep_widget = globals().get('dd_rep')
        if rep_widget is not None and rep_widget.value != 'cartoon':
            globals().get('_REP_SYNC', {})['active'] = True
            try: rep_widget.value = 'cartoon'
            finally: globals().get('_REP_SYNC', {})['active'] = False
        S.update(mode=None, _pdb_text='', _pdb_path=None, _view_format='pdb',
                 _view_source_format='pdb', _atom_meta=[], _atoms={})
    _render_input_queue()
    if paths and S.get('mode') in ('pdb', 'mmcif', 'small'): build_selection()
    else: render_viewer()
    _scc = globals().get('_sync_capability_controls')
    if _scc is not None: _scc()
    refresh()

def _file_row(path, role, index=None, total=None, kind='input'):
    label = W.HTML('<span style="display:inline-block;overflow:hidden;text-overflow:ellipsis;'
                   'white-space:nowrap;max-width:100%%" title="%s"><b>%s</b> · %s</span>' %
                   (html.escape(path), html.escape(role), html.escape(os.path.basename(path))))
    label.add_class('rxfilename')
    controls = []
    if kind == 'input':
        up = W.Button(description='Move earlier', icon='arrow-up', tooltip='Move earlier', disabled=(index == 0),
                      layout=W.Layout(width='38px'))
        down = W.Button(description='Move later', icon='arrow-down', tooltip='Move later', disabled=(index == total - 1),
                        layout=W.Layout(width='38px'))
        up.add_class('rxmove-earlier'); down.add_class('rxmove-later')
        up.on_click(lambda _, p=path: _queue_change(path=p, delta=-1))
        down.on_click(lambda _, p=path: _queue_change(path=p, delta=1))
        controls.extend([up, down])
    close = W.Button(description='Remove file', icon='times', tooltip='Remove %s' % os.path.basename(path),
                     layout=W.Layout(width='38px'))
    close.add_class('rxremove-file')
    close.on_click(lambda _, p=path, k=kind: _queue_change(path=p, remove=True, kind=k))
    controls.append(close)
    row = W.HBox([label, W.HBox(controls, layout=W.Layout(flex_flow='row nowrap'))],
                 layout=W.Layout(width='100%', flex_flow='row nowrap', align_items='center'))
    row.add_class('rxfile')
    return row

def _render_input_queue():
    paths = list(S.get('inputs', [])); rows = []
    reaction_order = (_wv('dd_subcmd', S.get('subcmd', 'all')) in ('path-opt', 'path-search') or
                      (_wv('dd_subcmd', S.get('subcmd', 'all')) == 'all' and
                       _wv('all_mode', 'mep') == 'mep'))
    for i, path in enumerate(paths):
        role = ('Input' if len(paths) == 1 or not reaction_order else
                ('Reactant' if i == 0 else ('Product' if i == len(paths) - 1 else 'Intermediate %d' % i)))
        if len(paths) > 1 and not reaction_order: role = 'Input %d' % (i + 1)
        rows.append(_file_row(path, '%d · %s' % (i + 1, role), i, len(paths)))
    if S.get('parm'): rows.append(_file_row(S['parm'], 'topology', kind='parm'))
    if S.get('model_pdb'): rows.append(_file_row(S['model_pdb'], 'prepared model', kind='model'))
    input_file_rows.children = rows or [W.HTML('<small style="color:#64748b">No files attached yet.</small>')]
    if not paths:
        order = '<b>No structures loaded</b>'
    elif len(paths) == 1:
        order = '<b>1 structure</b> · single-structure input'
    else:
        order = ('<b>%d structures</b> · reaction order shown above' % len(paths) if reaction_order else
                 '<b>%d input files</b>' % len(paths))
    if S.get('parm'):
        order += ' · topology: <code>%s</code>' % html.escape(os.path.basename(S['parm']))
    input_order_note.value = '<small>%s</small>' % order
    clear_button = globals().get('b_clear_inputs')
    if clear_button is not None:
        has_attached = bool(paths or S.get('parm') or S.get('model_pdb'))
        clear_button.disabled = not has_attached
        clear_button.layout.display = '' if has_attached else 'none'
    tab_notice = globals().get('_tab_notice')
    if tab_notice is not None and paths:
        tab_notice.value = ''
    sync_view = globals().get('_sync_view_input_widget')
    if sync_view is not None: sync_view()
b_clear_inputs = W.Button(description='Clear files', icon='eraser', layout=W.Layout(width='120px'))
b_clear_inputs.on_click(lambda _: _queue_change(clear=True))
_render_input_queue()
_input_box_children = [
    W.HTML('<b>Upload files</b> <small>· drag and drop, or choose files below</small>'),
    _drop, input_msg, input_file_rows, W.HBox([b_clear_inputs]), input_order_note]

# Colab renders ipywidgets' selection containers (Tab, Accordion) as an empty
# block, so every collapsible below is a Button + VBox that works everywhere.
def _info_markup(tip, revision=0, rich=False):
    """Browser-native disclosure: click opens the panel; hover shows the same help."""
    raw = str(tip)
    plain = ' '.join(__import__('re').sub(r'<[^>]+>', ' ', raw).split())
    safe = html.escape(plain, quote=True)
    body = raw if rich else html.escape(' '.join(raw.split()))
    return ('<details class="rxinfo-details" data-revision="%d">'
            '<summary aria-label="More information: %s" title="%s">&#9432;</summary>'
            '<div class="rxhelp-panel" role="note"><small>%s</small></div></details>' %
            (int(revision), safe, safe, body))

_INFO_CONTROLS = weakref.WeakSet()
def _close_info(control=None):
    """Rebuild one or all disclosures in their closed state."""
    controls = [control] if control is not None else list(_INFO_CONTROLS)
    for item in controls:
        if item is not None and hasattr(item, '_rx_tip'):
            item._rx_info_revision = getattr(item, '_rx_info_revision', 0) + 1
            item.value = _info_markup(
                item._rx_tip, item._rx_info_revision, getattr(item, '_rx_rich', False))

def _info_control(tip, target=None, rich=False):
    """A native details/summary info disclosure that works without Python clicks."""
    control = target if target is not None else W.HTML()
    control._rx_tip = str(tip)
    control._rx_rich = bool(rich)
    control._rx_info_revision = getattr(control, '_rx_info_revision', 0) + 1
    control.value = _info_markup(control._rx_tip, control._rx_info_revision, control._rx_rich)
    control.add_class('rxinfo')
    _INFO_CONTROLS.add(control)
    return control

def _set_info_text(control, tip):
    control._rx_tip = str(tip)
    control._rx_rich = False
    control._rx_info_revision = getattr(control, '_rx_info_revision', 0) + 1
    control.value = _info_markup(control._rx_tip, control._rx_info_revision, False)

def _close_info_target(target):
    _close_info(target)

def _hdr(content, tip):
    row = W.HBox([W.HTML(content), _info_control(tip, rich=True)],
                 layout=W.Layout(width='100%', flex_flow='row wrap', align_items='center'))
    row.add_class('rxhelp-row'); return row

def _flag_row(widget, tip, info_target=None, rich=False):
    row = W.HBox([widget, _info_control(tip, info_target, rich=rich)],
                 layout=W.Layout(width='100%', flex_flow='row wrap', align_items='center'))
    row.layout.display = widget.layout.display or ''
    widget._rx_flag_row = row
    row.add_class('rxflagrow'); return row

def _set_flag_visible(widget, visible):
    display_value = '' if visible else 'none'
    widget.layout.display = display_value
    row = getattr(widget, '_rx_flag_row', None)
    if row is not None: row.layout.display = display_value


def _collapsible(title, child, on_open=None):
    _btn = W.Button(layout=W.Layout(width='auto'))
    _btn.add_class('rxfold-toggle')
    _body = W.VBox([child])
    _st = {'open': False}
    def _sync():
        _body.layout.display = '' if _st['open'] else 'none'
        _btn.description = ('Hide ' if _st['open'] else 'Show ') + title
        _btn.remove_class('rxfold-open')
        if _st['open']: _btn.add_class('rxfold-open')
    def _click(_):
        _st['open'] = not _st['open']; _sync()
        if _st['open'] and on_open is not None: on_open()
    _btn.on_click(_click); _sync()
    _box = W.VBox([_btn, _body]); _box.add_class('rxfold')
    def _set_open(opened=True):
        changed = _st['open'] != bool(opened)
        _st['open'] = bool(opened); _sync()
        if changed and _st['open'] and on_open is not None: on_open()
    _box._rx_set_open = _set_open; _box._rx_button = _btn; _box._rx_body = _body
    return _box

example_fold = _collapsible('Examples', W.VBox([
    W.HTML('<small>Start with prepared inputs.</small>'),
    W.HBox([ex_choice, ex_btn]), example_msg]))
example_fold.add_class('rxexample')
_input_box_children.append(example_fold)

if not IS_CLUSTER:
    _input_box_children.insert(5, _collapsible('Manual ML-region PDB settings', model_upload_box))
input_box = W.VBox(_input_box_children)

# ============================================================== VIEWER (3D pick)
viewer_out = W.HTML(layout=W.Layout(width='100%', min_width='0'))
viewer_signal_out = W.HTML(layout=W.Layout(
    height='1px', min_height='1px', overflow='hidden'))
viewer_status = W.HTML()
view_input = W.Dropdown(options=[], description='viewing', style={'description_width': 'initial'},
                        layout=W.Layout(width='260px', max_width='100%'))
view_input_note = W.HTML()
_view_input_guard = {'active': False}
center_panel, charge_panel, scan_panel, freeze_panel, system_charge_panel = (W.VBox() for _ in range(5))
for _p in (center_panel, charge_panel, scan_panel, freeze_panel, system_charge_panel): _p.add_class('rxcard')
center_panel.add_class('rxcenter-panel')
scan_panel.add_class('rxscan-panel')
b_clear_center = W.Button(
    description='Clear',
    tooltip='Clear the center, ligand charges, and force-included residues.',
    layout=W.Layout(width='68px', flex='0 0 68px'))
b_clear_center.add_class('rxclear-center')
b_clear_scan = W.Button(description='Clear scan selections', layout=W.Layout(width='175px'))
center_ids_html = W.HTML()
center_widget = None
prep_radius = W.BoundedFloatText(
    value=2.6, min=0.0, max=1000000.0, step=0.5,
    description='-r radius Å', style={'description_width': 'initial'},
    layout=W.Layout(width='180px', max_width='100%'))
def _make_radius_stepper(widget):
    # Keep the native number-input spinner.  The Colab bridge primes its
    # step base from the current value so 2.6 -> 3.1 instead of 3.0.
    widget.add_class('rxhalf-step')
    return widget
prep_radius_control = _make_radius_stepper(prep_radius)
try:
    prep_radius.description_tooltip = ('Extraction radius in Å. Zero is valid and remains explicit '
                                       'in the command line.')
except Exception:
    pass
selected_resn = W.Text(
    value=str(S.get('selected_resn', '') or ''), description='--selected-resn',
    placeholder='123, A:456', style={'description_width': 'initial'},
    layout=W.Layout(width='100%', max_width='100%', min_width='0'))
def _sync_selected_resn(change):
    S['selected_resn'] = str(change.get('new') or '').strip()
    if S.get('_resetting_structure_state'):
        return
    if not globals().get('_SESSION_APPLY', {}).get('active', False):
        _invalidate_charge_confirmation()
        refresh(surface_only=True)
        render_summary = globals().get('_render_summary')
        if render_summary is not None: render_summary()
        render_chips = globals().get('_render_chips')
        if render_chips is not None: render_chips()
selected_resn.observe(_sync_selected_resn, names='value')
b_pick_selected_resn = W.Button(
    description='Pick force-included residues', icon='mouse-pointer',
    tooltip='Add or remove residue IDs in --selected-resn by clicking the sequence or 3D viewer.',
    layout=W.Layout(width='auto', max_width='100%', flex='1 1 174px', min_width='0'))
b_pick_selected_resn.add_class('rxpickresn-button')
# Compatibility alias: selected-resn picking now uses one transforming button.
b_done_selected_resn = b_pick_selected_resn
def _selected_resn_tokens(value=None):
    return [token for token in re.split(r'[,\s;]+',
                                         str(selected_resn.value if value is None else value).strip())
            if token]


def _center_values():
    """The center picker's options are (label, value) pairs so ligands can be
    flagged in the label; callers that match on residue names need the values."""
    if center_widget is None: return []
    return [o[1] if isinstance(o, tuple) else o for o in center_widget.options]
charge_rows = None
_PICK_ACTIONS = (
    ('Set extraction center (-c)', 'center', 'center'), ('Ligand charge (-l)', 'ligand', 'center'),
    ('Force-include residue (--selected-resn)', 'selectedresn', 'center'),
    ('Scan bond · atom A', 'scanA', 'scan'), ('Scan bond · atom B', 'scanB', 'scan'),
    ('Freeze pair · atom A', 'freezeA', 'freeze'), ('Freeze pair · atom B', 'freezeB', 'freeze'),
    ('Freeze/unfreeze atom (Cartesian)', 'freezeatom', 'freeze'),
)
pick_action = W.Dropdown(
    options=[(label, value) for label, value, _panel in _PICK_ACTIONS],
    value='center', description='Click action', style={'description_width': 'initial'},
    layout=W.Layout(width='320px', max_width='100%'))
_PICK_ACTION_STATE = {'user': False, 'sync': False, 'freeze_active': False}
def _sync_selected_resn_picker_button():
    active = pick_action.value == 'selectedresn'
    b_pick_selected_resn.description = ('Done picking' if active
                                        else 'Pick force-included residues')
    b_pick_selected_resn.icon = 'check' if active else 'mouse-pointer'
    b_pick_selected_resn.button_style = 'primary' if active else ''
    b_pick_selected_resn.tooltip = (
        'Finish --selected-resn picking and return clicks to extraction-center selection.' if active
        else 'Add or remove residue IDs in --selected-resn by clicking the sequence or 3D viewer.')
def _pick_selected_resn(_=None):
    if not _view_is_editable(): return
    pick_action.value = 'selectedresn'
    _render_pick_hint()
def _done_selected_resn(_=None):
    pick_action.value = 'center'
    _render_pick_hint()
def _toggle_selected_resn_picker(_=None):
    if pick_action.value == 'selectedresn':
        _done_selected_resn()
    else:
        _pick_selected_resn()
b_pick_selected_resn.on_click(_toggle_selected_resn_picker)
def _remember_pick_action(change):
    if not _PICK_ACTION_STATE['sync'] and change.get('old') != change.get('new'):
        _PICK_ACTION_STATE['user'] = True
        _PICK_ACTION_STATE['freeze_active'] = change.get('new') == 'freezeatom'
    elif change.get('new') != 'freezeatom':
        _PICK_ACTION_STATE['freeze_active'] = False
    _sync_selected_resn_picker_button()
    renderer = globals().get('_render_freeze_panel')
    if renderer is not None: renderer()
    highlighter = globals().get('_sync_active_selection_card')
    if highlighter is not None: highlighter()
pick_action.observe(_remember_pick_action, names='value')
_sync_selected_resn_picker_button()
exact_atom = W.Text(value='', description='exact atom',
                    placeholder='1-based index or A:SAM:359:C1',
                    style={'description_width': 'initial'},
                    layout=W.Layout(width='360px', max_width='100%'))
exact_atom_btn = W.Button(description='set current pick', icon='crosshairs',
                          layout=W.Layout(width='165px'),
                          tooltip='Use this atom for the current selection.')
exact_atom_msg = W.HTML()

_MOLSTAR_VERSION = '5.11.0'
# Colab occasionally leaves a CDN request pending forever inside about:srcdoc.
# Load asynchronously, retry on the alternate pinned CDN, and give every GUI
# execution a fresh cache key.  The persistent iframe pays this cost only once.
_MOLSTAR_JS = ('https://unpkg.com/molstar@%s/build/viewer/molstar.js'
               % _MOLSTAR_VERSION)
_MOLSTAR_CSS = ('https://unpkg.com/molstar@%s/build/viewer/molstar.css'
                % _MOLSTAR_VERSION)
_MOLSTAR_JS_FALLBACK = ('https://cdn.jsdelivr.net/npm/molstar@%s/build/viewer/molstar.js'
                        % _MOLSTAR_VERSION)
_MOLSTAR_CSS_FALLBACK = ('https://cdn.jsdelivr.net/npm/molstar@%s/build/viewer/molstar.css'
                         % _MOLSTAR_VERSION)
_MOLSTAR_ASSET_TOKEN = format(time.time_ns(), 'x')

_MOLSTAR_ASSET_CACHE = {'js': None, 'css': None, 'source': '', 'error': ''}

def _download_molstar_asset(url, label):
    request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(request, timeout=45) as response:
        payload = response.read(8 * 1024 * 1024 + 1)
    if len(payload) > 8 * 1024 * 1024:
        raise RuntimeError('%s asset is unexpectedly large' % label)
    text = payload.decode('utf-8')
    minimum = 500000 if label == 'JavaScript' else 5000
    if len(text) < minimum:
        raise RuntimeError('%s asset is unexpectedly short' % label)
    return text

def _inline_molstar_assets():
    """Fetch pinned browser assets in Python once, avoiding srcdoc CDN hangs."""
    if _MOLSTAR_ASSET_CACHE['js'] and _MOLSTAR_ASSET_CACHE['css']:
        return _MOLSTAR_ASSET_CACHE['js'], _MOLSTAR_ASSET_CACHE['css']
    errors = []
    # jsDelivr is attempted first from Python; unpkg remains the pinned backup.
    pairs = [
        (_MOLSTAR_JS_FALLBACK, _MOLSTAR_CSS_FALLBACK, 'jsDelivr'),
        (_MOLSTAR_JS, _MOLSTAR_CSS, 'unpkg'),
    ]
    for js_url, css_url, source_name in pairs:
        try:
            js_text = _download_molstar_asset(js_url, 'JavaScript')
            css_text = _download_molstar_asset(css_url, 'CSS')
            _MOLSTAR_ASSET_CACHE.update(
                js=js_text, css=css_text, source=source_name, error='')
            return js_text, css_text
        except Exception as exc:
            errors.append('%s: %s' % (source_name, exc))
    _MOLSTAR_ASSET_CACHE['error'] = '; '.join(errors)
    return None, None
_VIEWER_GENERATION = {'value': 0}
_VIEWER_MOUNTED = {'value': False}
_VIEWER_CONTENT_KEY = {'value': None}
_REP_SYNC = {'active': False}
_rep_user_set = {'value': False}
_INCREMENTAL_PICK = {'active': False}

def _pick_text(pick=None):
    pick = pick or S.get('_last_pick') or {}
    residue = '%s:%s:%s' % (pick.get('chain'), pick.get('resn'), pick.get('resi')) \
        if pick.get('chain') else '%s:%s' % (pick.get('resn'), pick.get('resi'))
    suffix = ' · atom #%d' % (int(pick['index']) + 1) \
        if pick.get('index') is not None and int(pick['index']) >= 0 else ''
    return '%s · %s%s' % (residue, pick.get('atom') or '?', suffix)

def _pick_key(pick):
    if not pick: return None
    try: return ('index', int(pick.get('index')))
    except (TypeError, ValueError):
        return ('atom', str(pick.get('chain') or ''), str(pick.get('resn') or '').upper(),
                str(pick.get('resi') or ''), str(pick.get('atom') or '').upper())

def _remember_pick(pick):
    """Keep each transient Mol* pick once, in click order."""
    history = S.setdefault('_pick_history', [])
    key = _pick_key(pick)
    for position, previous in enumerate(history):
        if _pick_key(previous) == key:
            history[position] = pick
            return False
    history.append(pick)
    return True

# Keep each document self-contained. Widget HTML updates do not reliably execute
# adjacent scripts in Colab, so the iframe itself owns its bootstrap. Large raw
# srcdoc values can remain permanently blank in Chromium; store a deterministic
# gzip payload on the iframe and keep srcdoc below that startup boundary.
_DOC_FRAME_SEQUENCE = {'value': 0}
_DOC_FRAME_INLINE_LIMIT = 1000000
_DOC_FRAME_BOOTSTRAP = r'''<!doctype html><html><head><meta charset="utf-8">
<style>html,body{margin:0;width:100%;height:100%;background:#fff}body{display:flex;align-items:center;justify-content:center;font:13px -apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;color:#475569}</style>
</head><body>Loading molecular viewer…<script>(async()=>{try{const frame=window.frameElement;const packed=frame&&frame.getAttribute('data-rx-document');if(!packed)throw new Error('Missing embedded viewer payload.');if(typeof DecompressionStream!=='function')throw new Error('This browser cannot unpack the embedded viewer.');const raw=atob(packed);const bytes=Uint8Array.from(raw,c=>c.charCodeAt(0));const stream=new Blob([bytes]).stream().pipeThrough(new DecompressionStream('gzip'));const text=await new Response(stream).text();frame.removeAttribute('data-rx-document');document.open();document.write(text);document.close();}catch(error){document.body.textContent='Viewer bootstrap failed: '+String(error&&error.message||error);}})();</script></body></html>'''
def _document_iframe(document, attributes, style):
    """One self-contained iframe with a small, reliably bootable srcdoc."""
    _DOC_FRAME_SEQUENCE['value'] += 1
    frame_id = 'rxdocframe-%d' % _DOC_FRAME_SEQUENCE['value']
    payload_attr = ''
    embedded = document
    if len(document) > _DOC_FRAME_INLINE_LIMIT:
        packed = base64.b64encode(gzip.compress(
            document.encode('utf-8'), compresslevel=9, mtime=0)).decode('ascii')
        payload_attr = ' data-rx-document="%s"' % packed
        embedded = _DOC_FRAME_BOOTSTRAP
    return '<iframe id="%s" %s%s srcdoc="%s" style="%s"></iframe>' % (
        frame_id, attributes, payload_attr, html.escape(embedded, quote=True), style)

def _plotly_document_iframe(document, attributes, style):
    """Mount a large Plotly document once in a script-independent iframe."""
    _DOC_FRAME_SEQUENCE['value'] += 1
    frame_id = 'rxplotframe-%d' % _DOC_FRAME_SEQUENCE['value']
    return '<iframe id="%s" %s srcdoc="%s" style="%s"></iframe>' % (
        frame_id, attributes, html.escape(document, quote=True), style)

def _binary_iframe(mime, encoded, attributes, style):
    """Same idea for a base64 payload: Colab also strips `src="data:..."`."""
    _DOC_FRAME_SEQUENCE['value'] += 1
    frame_id = 'rxdocframe-%d' % _DOC_FRAME_SEQUENCE['value']
    return ('<iframe id="%s" %s src="data:%s;base64,%s" style="%s"></iframe>'
            '<script>(function(){'
            'var self=document.currentScript;'
            'var frame=document.getElementById(%s)||(self&&self.previousElementSibling);'
            'if(!frame||frame.tagName!=="IFRAME"||frame.getAttribute("src"))return;'
            'var raw=atob(%s), bytes=new Uint8Array(raw.length);'
            'for(var i=0;i<raw.length;i++)bytes[i]=raw.charCodeAt(i);'
            'var url=URL.createObjectURL(new Blob([bytes],{type:%s}));'
            'if(frame._rxObjectUrl)try{URL.revokeObjectURL(frame._rxObjectUrl);}catch(_error){}'
            'frame._rxObjectUrl=url;frame.src=url;'
            'var root=document.documentElement||document.body;'
            'if(root&&typeof MutationObserver!=="undefined"){'
            'var observer=new MutationObserver(function(){if(!frame.isConnected){'
            'try{URL.revokeObjectURL(url);}catch(_error){}observer.disconnect();}});'
            'observer.observe(root,{childList:true,subtree:true});}'
            '})();</script>' %
            (frame_id, attributes, mime, encoded, style,
             json.dumps(frame_id), json.dumps(encoded), json.dumps(mime)))

def _molstar_default_preset(source, fmt='pdb'):
    """Choose cartoon only when complete backbone residues form a real run."""
    normalized = str(fmt or 'pdb').lower()
    if normalized in ('cif', 'mmcif'):
        return 'polymer-and-ligand'
    if normalized != 'pdb':
        return 'auto'
    backbone = {}
    for line in str(source or '').splitlines():
        if not line.startswith('ATOM  ') or len(line) < 27:
            continue
        altloc = line[16:17].strip()
        if altloc not in ('', 'A'):
            continue
        atom = line[12:16].strip().upper()
        if atom not in ('N', 'CA', 'C'):
            continue
        try:
            resseq = int(line[22:26])
        except ValueError:
            continue
        key = (line[21:22], resseq, line[26:27])
        backbone.setdefault(key, set()).add(atom)
    complete = sorted((chain, resseq, icode) for (chain, resseq, icode), atoms in backbone.items()
                      if {'N', 'CA', 'C'} <= atoms)
    total = len(complete)
    longest = run = 0
    previous = None
    for chain, resseq, icode in complete:
        contiguous = (previous is not None and chain == previous[0] and
                      resseq == previous[1] + 1 and not icode.strip() and
                      not previous[2].strip())
        run = run + 1 if contiguous else 1
        longest = max(longest, run)
        previous = (chain, resseq, icode)
    return ('polymer-and-ligand' if longest >= 3 and longest * 2 >= total
            else 'ball-and-stick')

def _normalize_representation(value):
    normalized = str(value or '').strip().lower()
    if normalized == 'cartoon':
        return 'cartoon'
    if normalized in ('stick', 'sticks', 'ball+stick', 'ball-and-stick',
                      'spheres', 'sphere', 'line', 'lines'):
        return 'stick'
    return 'stick'

def _auto_representation(source, fmt='pdb'):
    return ('cartoon' if _molstar_default_preset(source, fmt) == 'polymer-and-ligand'
            else 'stick')

def _representation_preset(representation):
    return ('polymer-and-ligand' if _normalize_representation(representation) == 'cartoon'
            else 'ball-and-stick')

def _viewer_pdb_with_conect(source, fmt='pdb', source_format=None):
    """Add distance-inferred CONECT to a viewer-only atom-cluster PDB copy."""
    text = str(source or '')
    lines = text.splitlines()
    origin = str(source_format or fmt or '').lower()
    if (str(fmt or '').lower() != 'pdb' or origin in ('cif', 'mmcif') or
            _molstar_default_preset(text, fmt) != 'ball-and-stick' or
            any(line.startswith('CONECT') for line in lines) or
            sum(line.startswith('MODEL ') for line in lines) > 1):
        return text
    atom_lines = [line for line in lines if line.startswith(('ATOM  ', 'HETATM'))]
    if not atom_lines or len(atom_lines) > 2000:
        return text
    try:
        from ase.data import atomic_numbers, covalent_radii
    except Exception:
        return text
    atoms = []; serials = set()
    try:
        for line in atom_lines:
            serial = int(line[6:11])
            xyz = tuple(float(line[start:end]) for start, end in
                        ((30, 38), (38, 46), (46, 54)))
            element = line[76:78].strip().title() if len(line) >= 78 else ''
            if not element:
                field = line[12:16] if len(line) >= 16 else ''
                letters = ''.join(ch for ch in field if ch.isalpha())
                one_letter = field[:1].isspace() or field[:1].isdigit()
                element = (letters[:1] if one_letter else letters[:2]).title()
            if element not in atomic_numbers and len(element) > 1:
                element = element[:1]
            if (serial <= 0 or serial in serials or element not in atomic_numbers or
                    not all(math.isfinite(value) for value in xyz)):
                return text
            serials.add(serial); atoms.append((serial, element, xyz))
    except (IndexError, TypeError, ValueError):
        return text
    neighbours = {serial: [] for serial, _element, _xyz in atoms}
    for index, (serial_a, element_a, xyz_a) in enumerate(atoms):
        for serial_b, element_b, xyz_b in atoms[index + 1:]:
            if element_a == element_b == 'H':
                continue
            distance2 = sum((a - b) ** 2 for a, b in zip(xyz_a, xyz_b))
            cutoff = (float(covalent_radii[atomic_numbers[element_a]]) +
                      float(covalent_radii[atomic_numbers[element_b]]) + 0.45)
            if 0.01 < distance2 <= cutoff * cutoff:
                neighbours[serial_a].append(serial_b)
                neighbours[serial_b].append(serial_a)
    if not any(neighbours.values()):
        return text
    records = []
    for serial in sorted(neighbours):
        bonded = sorted(set(neighbours[serial]))
        for offset in range(0, len(bonded), 4):
            records.append('CONECT%5d%s' %
                           (serial, ''.join('%5d' % item for item in bonded[offset:offset + 4])))
    body = [line for line in lines if line.strip() != 'END']
    return '\n'.join(body + records + ['END']) + '\n'

def _molstar_document(source, fmt='pdb', *, interactive=False, generation=0,
                      show_water=False, show_sequence=None, frame_count=None,
                      representation=None, source_format=None):
    """Build an isolated, pinned Mol* viewer using its standard interface."""
    callback_ns = 'pdb2reaction_gui'
    rep = _normalize_representation(
        representation if representation is not None else _auto_representation(source, fmt))
    source = _viewer_pdb_with_conect(source, fmt, source_format)
    config = {
        'format': ('xyz' if fmt == 'xyz' else
                   ('mmcif' if fmt in ('cif', 'mmcif') else 'pdb')),
        'interactive': bool(interactive),
        'generation': int(generation),
        'callback': callback_ns if (interactive or int(frame_count or 1) > 1) else '',
        'showWater': bool(show_water),
        'showSequence': bool(fmt != 'xyz' if show_sequence is None else show_sequence),
        'frameCount': max(1, int(frame_count or 1)),
        'representationPreset': _representation_preset(rep),
    }
    template = r"""<!doctype html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
__MOLSTAR_STYLE__
<style>
html,body,#molstar-host{width:100%;height:100%;margin:0;overflow:hidden;background:#fff}
body{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Helvetica,Arial,sans-serif}
#molstar-host{position:absolute;inset:0}
#molstar-state{position:absolute;z-index:20;inset:0;display:flex;align-items:center;
 justify-content:center;background:#f8fafc;color:#475569;font-size:13px}
#molstar-state.error{color:#991b1b;padding:18px;text-align:center;box-sizing:border-box}
.msp-plugin{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Helvetica,Arial,sans-serif}
/* A single unnamed chain has no alternative to select. Keep the control
   for real chain names and multi-chain structures. */
.msp-sequence-select select.rx-empty-chain-select{display:none!important}
</style>
</head>
<body>
<div id="molstar-host"></div>
<div id="molstar-state" role="status">Loading structure…</div>
__MOLSTAR_LOADER__
<script>
(async function(){
  'use strict';
  const cfg=__CONFIG__;
  let source=__SOURCE__;
  const stateNode=document.getElementById('molstar-state');
  let ready=false, viewer=null, ignoreStartupEmpty=true, clickQueue=Promise.resolve();
  let loadQueue=Promise.resolve(), pendingFrame=0, applyingFrame=false;
  let structureRequestKey=JSON.stringify([Number(cfg.generation)||0,String(source||''),
    String(cfg.format||'pdb'),String(cfg.representationPreset||'auto'),
    !!cfg.showWater,!!cfg.showSequence,
    Math.max(1,Number(cfg.frameCount)||1)]);
  let lastReportedFrame=-1, modelUiTimer=0;
  let userModelActionUntil=0, programmaticUntil=0;

  function kernel(){
    let found=null, scope=window;
    for(let depth=0;depth<8&&scope;depth++){
      try{
        if(scope.google&&scope.google.colab&&scope.google.colab.kernel)
          found={api:scope.google.colab.kernel,scope:scope};
        scope=(scope.parent&&scope.parent!==scope)?scope.parent:null;
      }catch(_error){scope=null;}
    }
    return found;
  }
  function invoke(suffix,args){
    if(!cfg.callback)return Promise.resolve(null);
    const bridge=kernel();
    if(bridge){
      try{
        const safeArgs=bridge.scope.JSON.parse(JSON.stringify(args));
        const kwargs=bridge.scope.JSON.parse('{}');
        return bridge.api.invokeFunction(cfg.callback+'.'+suffix,safeArgs,kwargs);
      }catch(_error){}
    }
    // Retain the trusted parent relay as a fallback when the kernel is hidden.
    if(window.parent&&window.parent!==window){
      try{
        window.parent.postMessage({type:'rx-kernel-invoke',suffix:suffix,args:args},'*');
      }catch(_error){}
    }
    return Promise.resolve(null);
  }
  function broadcastFrame(index){
    try{
      const host=window.parent&&window.parent.document;if(!host)return;
      const selector='iframe[data-rx-channel="trajectory"]'+
        '[data-rx-generation="'+String(cfg.generation)+'"]';
      const message={type:'rx-set-frame',generation:Number(cfg.generation),index:Number(index)};
      for(const target of host.querySelectorAll(selector)){
        if(target&&target.contentWindow&&target.contentWindow!==window)
          target.contentWindow.postMessage(message,'*');
      }
    }catch(_error){}
  }
  function readModelUiFrame(){
    if(!ready||!viewer||applyingFrame||Number(cfg.frameCount||1)<2)return;
    // Reverse synchronization belongs only to a direct user action on
    // Mol*'s model controls. Programmatic DOM updates are never allowed
    // to write a stale model number back into the playback slider.
    if(Date.now()>userModelActionUntil)return;
    const text=String(document.getElementById('molstar-host').textContent||'');
    const match=text.match(/Model\s+(\d+)\s*\/\s*(\d+)/i);
    if(!match)return;
    const index=Math.max(0,Math.min(Number(cfg.frameCount||1)-1,Number(match[1])-1));
    if(index===lastReportedFrame)return;
    if(index!==pendingFrame&&Date.now()<programmaticUntil)return;
    lastReportedFrame=index;
    pendingFrame=index;
    broadcastFrame(index);
    invoke('set_frame',[cfg.generation,index]).catch(error=>console.error(error));
  }
  function queueModelUiFrame(delay=80){
    clearTimeout(modelUiTimer);
    modelUiTimer=setTimeout(readModelUiFrame,delay);
  }
  async function applyRequestedFrame(target){
    if(!ready||!viewer)return;
    const state=viewer.plugin.state.data;
    const transform=molstar.lib.plugin.StateTransforms.Model.ModelFromTrajectory;
    const models=state.selectQ(q=>q.ofTransformer(transform));
    if(!models.length)return;
    const maximum=Math.max(0,Number(cfg.frameCount||1)-1);
    const modelIndex=Math.max(0,Math.min(maximum,Number(target)||0));
    const update=state.build();
    let changed=false;
    for(const model of models){
      const params=model.transform&&model.transform.params;
      if(params&&Number(params.modelIndex)===modelIndex)continue;
      update.to(model).update(old=>({...old,modelIndex}));
      changed=true;
    }
    if(changed)await update.commit({doNotLogTiming:true});
    // Mol* updates the visible `Model N / M` label after the state commit.
    // Keep a short echo-suppression window while that DOM catches up.
    lastReportedFrame=modelIndex;
    programmaticUntil=Date.now()+800;
  }
  async function drainRequestedFrames(){
    if(!ready||applyingFrame)return;
    applyingFrame=true;
    try{
      while(ready){
        const target=pendingFrame;
        await applyRequestedFrame(target);
        if(target===pendingFrame)break;
      }
    }finally{
      applyingFrame=false;
    }
  }
  window.addEventListener('message',event=>{
    const message=event&&event.data;
    if(!message)return;
    // Linked trajectory controls live in a sibling iframe under Colab's widget host.
    if(message.type==='rx-load-structure'){
      const request={
        generation:Number(message.generation)||0,
        source:String(message.source||''),
        format:String(message.format||'pdb'),
        showWater:!!message.showWater,
        showSequence:!!message.showSequence,
        frameCount:Math.max(1,Number(message.frameCount)||1),
        representationPreset:String(message.representationPreset||'auto')
      };
      const requestKey=JSON.stringify([request.generation,request.source,request.format,
        request.representationPreset,request.showWater,request.showSequence,request.frameCount]);
      // A one-shot sibling bridge can be reattached when another widget is
      // updated. Ignore that duplicate instead of reparsing the same structure.
      if(requestKey===structureRequestKey)return;
      structureRequestKey=requestKey;
      // Update routing immediately so a following rx-set-frame message can be
      // queued while the replacement trajectory is still being parsed.
      cfg.generation=request.generation;
      cfg.frameCount=request.frameCount;
      loadQueue=loadQueue.then(async()=>{
        if(request.generation<Number(cfg.generation))return;
        cfg.format=request.format;
        cfg.showWater=request.showWater;
        cfg.representationPreset=request.representationPreset;
        cfg.showSequence=request.showSequence;
        source=request.source;
        await loadCurrentStructure();
      }).catch(error=>{
        if(structureRequestKey===requestKey)structureRequestKey='';
        console.error(error);
        stateNode.className='error';
        stateNode.textContent=String(error&&error.message||error);
      });
      return;
    }
    if(message.type!=='rx-set-frame')return;
    if(Number(message.generation)!==Number(cfg.generation))return;
    pendingFrame=Math.max(0,Math.min(Number(cfg.frameCount||1)-1,
                                   Number(message.index)||0));
    void drainRequestedFrames();
  });
  function properties(loc){
    const SP=molstar.lib.structure.StructureProperties;
    return {
      sourceIndex:Number(SP.atom.sourceIndex(loc)),
      serial:Number(SP.atom.id(loc)),
      chain:String(SP.chain.auth_asym_id(loc)||'').trim(),
      seq:Number(SP.residue.auth_seq_id(loc)),
      ins:String(SP.residue.pdbx_PDB_ins_code(loc)||'').trim(),
      resn:String(SP.residue.auth_comp_id(loc)||SP.residue.label_comp_id(loc)||'').trim(),
      atom:String(SP.atom.auth_atom_id(loc)||SP.atom.label_atom_id(loc)||'').trim(),
      element:String(SP.atom.type_symbol(loc)||'').trim()
    };
  }
  async function setCartoonAlpha(){
    // Keep Mol*'s native colour themes while making every cartoon
    // representation clearly translucent. Component keys differ between
    // structures, so match the representation type rather than one key.
    const structures=viewer.plugin.managers.structure.hierarchy.current.structures;
    const state=viewer.plugin.state.data, update=state.build();
    let changed=false;
    for(const structure of structures){
      for(const component of structure.components){
        for(const representation of component.representations||[]){
          const params=representation.cell&&representation.cell.transform.params;
          if(!params||!params.values||!params.values.type||
             params.values.type.name!=='cartoon')continue;
          update.to(representation.cell).update(old=>({
            ...old,type:{...old.type,params:{...old.type.params,alpha:0.4}}
          }));
          changed=true;
        }
      }
    }
    if(changed)await update.commit();
  }
  function syncEmptyChainSelector(){
    const selectors=document.querySelectorAll('.msp-sequence-select select');
    for(const select of selectors){
      const title=String(select.getAttribute('title')||'');
      const only=select.options&&select.options.length===1?select.options[0]:null;
      const useless=title.startsWith('[Chain]')&&only&&
                    !String(only.textContent||'').trim();
      select.classList.toggle('rx-empty-chain-select',!!useless);
      if(useless)select.setAttribute('aria-hidden','true');
      else select.removeAttribute('aria-hidden');
    }
  }
  async function configureWater(){
    const hierarchy=viewer.plugin.managers.structure.hierarchy;
    for(const structure of hierarchy.current.structures){
      let component=structure.components.find(
        item=>item.key==='structure-component-static-water'
      );
      if(!component&&cfg.showWater){
        const water=await viewer.plugin.builders.structure.tryCreateComponentStatic(
          structure.cell,'water',{label:'Water'}
        );
        if(water)await viewer.plugin.builders.structure.representation.addRepresentation(
          water,{type:'ball-and-stick'}
        );
        continue;
      }
      if(!component)continue;
      const hidden=!!(component.cell&&component.cell.state&&component.cell.state.isHidden);
      if(hidden===cfg.showWater)
        viewer.plugin.managers.structure.component.toggleVisibility([component]);
      if(cfg.showWater&&!component.representations.length)
        await viewer.plugin.managers.structure.component.addRepresentation(
          [component],'ball-and-stick'
        );
    }
  }

  async function loadStructureWithPreset(){
    const data=await viewer.plugin.builders.data.rawData(
      {data:source,label:'input structure'}
    );
    const trajectory=await viewer.plugin.builders.structure.parseTrajectory(data,cfg.format);
    if(cfg.representationPreset!=='ball-and-stick'){
      await viewer.plugin.builders.structure.hierarchy.applyPreset(trajectory,'default',{
        representationPreset:cfg.representationPreset||'auto'
      });
      return;
    }
    await viewer.plugin.builders.structure.hierarchy.applyPreset(trajectory,'default',{
      representationPreset:'empty'
    });
    const componentTypes=['polymer','ligand','non-standard','branched','ion','lipid','coarse'];
    for(const structure of viewer.plugin.managers.structure.hierarchy.current.structures){
      for(const type of componentTypes){
        const component=await viewer.plugin.builders.structure.tryCreateComponentStatic(
          structure.cell,type,{label:type}
        );
        if(component)await viewer.plugin.builders.structure.representation.addRepresentation(
          component,{type:'ball-and-stick'}
        );
      }
    }
  }

  async function loadCurrentStructure(){
    // Reactant/Product share coordinates. Preserve the complete Mol* camera
    // snapshot so switching structures keeps the user's angle, center and zoom.
    const previousCamera=(ready&&viewer&&viewer.plugin&&viewer.plugin.canvas3d)
      ? viewer.plugin.canvas3d.camera.getSnapshot() : null;
    ready=false;
    ignoreStartupEmpty=true;
    stateNode.className='';
    stateNode.style.display='flex';
    stateNode.textContent='Loading structure…';
    if(viewer.plugin&&typeof viewer.plugin.clear==='function')
      await viewer.plugin.clear();
    await loadStructureWithPreset();
    await setCartoonAlpha();
    syncEmptyChainSelector();
    await configureWater();
    if(previousCamera)
      viewer.plugin.managers.camera.setSnapshot(previousCamera,0);
    else
      viewer.plugin.managers.camera.reset({},0);
    if(!document.querySelector('#molstar-host canvas'))
      throw new Error('WebGL is unavailable in this browser session.');
    ready=true;
    stateNode.style.display='none';
    await drainRequestedFrames();
  }

  try{
    await window.rxMolstarReady;
    if(!window.molstar||!molstar.Viewer)
      throw new Error('Mol* did not load. Check the network connection and rerun Launch GUI.');
    viewer=window.rxMolstarViewer=await molstar.Viewer.create('molstar-host',{
      layoutIsExpanded:false,
      layoutShowControls:true,
      layoutShowRemoteState:false,
      layoutShowSequence:cfg.showSequence,
      layoutShowLog:false,
      layoutShowLeftPanel:false,
      collapseLeftPanel:true,
      collapseRightPanel:true,
      viewportShowControls:true,
      viewportShowSelectionMode:true,
      viewportBackgroundColor:'#ffffff'
    });
    const molstarHost=document.getElementById('molstar-host');
    const sequenceObserver=new MutationObserver(()=>{
      syncEmptyChainSelector();
      queueModelUiFrame();
    });
    sequenceObserver.observe(molstarHost,
                             {childList:true,subtree:true,characterData:true});
    // Observe Mol*'s built-in model arrows without polling programmatic
    // updates. A trusted button action opens a bounded reporting window;
    // the MutationObserver then reports the resulting model exactly once.
    molstarHost.addEventListener('click',event=>{
      if(!event.isTrusted||!event.target.closest('button'))return;
      userModelActionUntil=Date.now()+1200;
      programmaticUntil=0;
      queueModelUiFrame(100);
      setTimeout(readModelUiFrame,300);
    },true);
    syncEmptyChainSelector();
    await loadCurrentStructure();

    if(cfg.interactive){
      async function handleClick(event){
        if(!ready)return;
        const loci=event&&event.current&&event.current.loci;
        if(!loci||loci.kind==='empty-loci'){
          if(ignoreStartupEmpty){ignoreStartupEmpty=false;return;}
          await invoke('clear_highlights',[cfg.generation]);
          return;
        }
        ignoreStartupEmpty=false;
        const SE=molstar.lib.structure.StructureElement;
        let loc=null, exact=false;
        if(loci.kind==='element-loci'){
          loc=SE.Loci.getFirstLocation(loci);
          exact=SE.Loci.size(loci)===1;
        }else if(loci.kind==='bond-loci'){
          const bond=loci.bonds&&loci.bonds[0];
          if(!bond)return;
          loc=SE.Location.create(bond.aStructure,bond.aUnit,bond.aUnit.elements[bond.aIndex]);
          exact=true;
        }else return;
        if(!loc)return;
        const item=properties(loc);
        await invoke('on_click',[
          String(item.sourceIndex),item.resn,String(item.seq),item.chain,item.atom,
          String(item.serial),item.ins,true,cfg.generation,exact
        ]);
      }
      // Sequence-panel clicks are emitted through the interaction behavior. Canvas3D
      // forwards the same event object there, so retain the direct embedded-Colab
      // fallback while deduplicating that exact object before queueing the callback.
      const seenClickEvents=new WeakSet();
      function queueInteractiveClick(event){
        if(event&&(typeof event==='object'||typeof event==='function')){
          if(seenClickEvents.has(event))return;
          seenClickEvents.add(event);
        }
        clickQueue=clickQueue.then(()=>handleClick(event)).catch(error=>console.error(error));
      }
      viewer.plugin.behaviors.interaction.click.subscribe(queueInteractiveClick);
      viewer.plugin.canvas3d.interaction.click.subscribe(queueInteractiveClick);
    }
    window.addEventListener('resize',()=>viewer&&viewer.handleResize());
  }catch(error){
    console.error(error);
    stateNode.className='error';
    stateNode.style.display='flex';
    stateNode.textContent=String(error&&error.message||error);
  }
})();
</script>
</body>
</html>"""
    cache_query = '?rx=' + _MOLSTAR_ASSET_TOKEN
    css_primary = _MOLSTAR_CSS + cache_query
    css_fallback = _MOLSTAR_CSS_FALLBACK + cache_query
    js_urls = [_MOLSTAR_JS + cache_query, _MOLSTAR_JS_FALLBACK + cache_query]
    inline_js, inline_css = _inline_molstar_assets()
    if inline_js and inline_css:
        style_tag = '<style id="molstar-css">%s</style>' % (
            inline_css.replace('</style', '<\\/style'))
        loader = (
            '<script>%s</script>'
            '<script>window.rxMolstarReady=Promise.resolve();</script>' %
            inline_js.replace('</script', '<\\/script'))
    else:
        style_tag = '<link id="molstar-css" rel="stylesheet" href="%s">' % (
            html.escape(css_primary, quote=True))
        loader = r"""<script>
(function(){
  const css=document.getElementById('molstar-css');
  if(css)css.addEventListener('error',()=>{css.href=%s;},{once:true});
  const urls=%s;
  window.rxMolstarReady=new Promise((resolve,reject)=>{
    let index=0;
    function attempt(){
      if(index>=urls.length){
        reject(new Error('Mol* could not be loaded from either pinned CDN.'));
        return;
      }
      const script=document.createElement('script');
      const url=urls[index++];
      let settled=false;
      const timer=setTimeout(()=>{
        if(settled)return;
        settled=true;
        script.remove();
        attempt();
      },12000);
      script.async=true;
      script.src=url;
      script.onload=()=>{
        if(settled)return;
        settled=true;
        clearTimeout(timer);
        resolve();
      };
      script.onerror=()=>{
        if(settled)return;
        settled=true;
        clearTimeout(timer);
        script.remove();
        attempt();
      };
      document.head.appendChild(script);
    }
    attempt();
  });
})();
</script>""" % (
            json.dumps(css_fallback).replace('<', '\\u003c'),
            json.dumps(js_urls, separators=(',', ':')).replace('<', '\\u003c'))
    return (template
            .replace('__MOLSTAR_STYLE__', style_tag)
            .replace('__MOLSTAR_LOADER__', loader)
            .replace('__CONFIG__', json.dumps(config, separators=(',', ':')).replace('<', '\\u003c'))
            .replace('__SOURCE__', json.dumps(str(source)).replace('<', '\\u003c')))

def _molstar_iframe(source, fmt='pdb', **kwargs):
    channel = str(kwargs.pop('channel', 'viewer'))
    generation = int(kwargs.get('generation', 0))
    document = _molstar_document(source, fmt, **kwargs)
    # The wrapper gives IPython a normal HTML fragment instead of a bare iframe,
    # avoiding its visible "use IFrame" warning.
    return ('<div class="rxmolstar-embed">%s</div>' % _document_iframe(
        document,
        'class="rxmolstar-frame" title="Mol* molecular viewer" '
        'data-rx-channel="%s" data-rx-generation="%d" allow="fullscreen"' %
        (html.escape(channel, quote=True), generation),
        'width:100%;aspect-ratio:1/1;border:1px solid #dfe6ef;'
        'border-radius:10px;background:#fff;'))

def _molstar_update_script(source, fmt, generation, show_water=False, representation=None,
                           source_format=None):
    rep = _normalize_representation(
        representation if representation is not None else _auto_representation(source, fmt))
    source = _viewer_pdb_with_conect(source, fmt, source_format)
    message = {
        'type': 'rx-load-structure',
        'source': str(source),
        'format': ('xyz' if fmt == 'xyz' else
                   ('mmcif' if fmt in ('cif', 'mmcif') else 'pdb')),
        'generation': int(generation),
        'showWater': bool(show_water),
        'showSequence': bool(fmt != 'xyz'),
        'representationPreset': _representation_preset(rep),
    }
    payload = json.dumps(message, separators=(',', ':')).replace('<', '\\u003c')
    return r"""<script>(function(){
      let root=document;
      try{if(window.parent&&window.parent.document)root=window.parent.document;}catch(_){}
      const frames=root.querySelectorAll(
        'iframe.rxmolstar-frame[data-rx-channel="viewer"]');
      const frame=frames.length?frames[frames.length-1]:null;
      if(!frame)return;
      frame.dataset.rxGeneration=String(%d);
      frame.contentWindow.postMessage(%s,'*');
    })();</script>""" % (int(generation), payload)

def render_viewer():
    source = S.get('_pdb_text') or ''
    if not source:
        if _VIEWER_MOUNTED['value']:
            _VIEWER_GENERATION['value'] += 1
        viewer_status.layout.display = ''
        viewer_status.value = (
            '<div role="status" style="min-height:180px;display:flex;align-items:center;'
            'justify-content:center;text-align:center;border:1px dashed #cbd5e1;'
            'border-radius:12px;background:#f8fafc;color:#64748b;padding:16px;">'
            '<div><b>No structure loaded</b><br><small>Load a structure in Input, '
            'then use Mol* here.</small></div></div>')
        viewer_out.value = ''
        viewer_signal_out.value = ''
        _VIEWER_MOUNTED['value'] = False
        _VIEWER_CONTENT_KEY['value'] = None
        return
    fmt = S.get('_view_format', 'pdb')
    source_format = S.get('_view_source_format', fmt)
    show_water = bool(S.get('show_water', True))
    if not _rep_user_set['value']:
        automatic = _auto_representation(source, fmt)
        S['rep'] = automatic
        rep_widget = globals().get('dd_rep')
        if rep_widget is not None and rep_widget.value != automatic:
            _REP_SYNC['active'] = True
            try: rep_widget.value = automatic
            finally: _REP_SYNC['active'] = False
    representation = _normalize_representation(S.get('rep'))
    S['rep'] = representation
    content_key = (len(source), hash(source), fmt, source_format, show_water, representation)
    # Selection and option widgets call render_viewer as part of their shared
    # refresh path.  Do not reload Mol* unless the displayed structure itself
    # changed; reloading here discards the camera and flashes Loading structure.
    if _VIEWER_MOUNTED['value'] and content_key == _VIEWER_CONTENT_KEY['value']:
        return
    _VIEWER_GENERATION['value'] += 1
    generation = _VIEWER_GENERATION['value']
    viewer_status.value = ''
    viewer_status.layout.display = 'none'
    if not _VIEWER_MOUNTED['value']:
        viewer_out.value = _molstar_iframe(
            source, fmt,
            interactive=True, generation=generation,
            show_water=show_water, representation=representation,
            source_format=source_format)
        try: viewer_out.send_state()
        except Exception: pass
        _VIEWER_MOUNTED['value'] = True
        _VIEWER_CONTENT_KEY['value'] = content_key
        return
    update_document = ('<!doctype html><html><body>' +
                       _molstar_update_script(
                           source, fmt, generation, show_water=show_water,
                           representation=representation, source_format=source_format) +
                       '</body></html>')
    viewer_signal_out.value = _document_iframe(
        update_document, 'title="Mol* update bridge" aria-hidden="true"',
        'position:absolute;width:1px;height:1px;opacity:0;pointer-events:none;border:0;')
    try: viewer_signal_out.send_state()
    except Exception: pass
    _VIEWER_CONTENT_KEY['value'] = content_key

def _render_center_ids():
    ids = S.get('center_ids', [])
    center_ids_html.value = ('<small>exact residues: <code>%s</code></small>' % ','.join(ids)) if ids else ''
    center_ids_html.layout.display = '' if ids else 'none'

def _resolve_click_meta(atom_index, serial='', resn='', resi='', chain='', atom='', icode=''):
    """Map a browser atom to retained input metadata without trusting source-index order."""
    metadata = S.get('_atom_meta', [])
    if S.get('_view_format') == 'xyz':
        try: return metadata[int(atom_index)]
        except (IndexError, TypeError, ValueError): return None
    norm = lambda value: str(value or '').strip()
    try: serial_int = int(serial)
    except (TypeError, ValueError): serial_int = None
    if serial_int is not None:
        found = [meta for meta in metadata if meta.get('serial') == serial_int]
        if len(found) == 1: return found[0]
    expected = (norm(chain), norm(resn).upper(), norm(resi), norm(icode), norm(atom).upper())
    if any(expected):
        found = [meta for meta in metadata
                 if (norm(meta.get('chain')), norm(meta.get('resname')).upper(),
                     norm(meta.get('resseq')), norm(meta.get('icode')),
                     norm(meta.get('name')).upper()) == expected]
        if len(found) == 1: return found[0]
        return None
    try: return metadata[int(atom_index)]
    except (IndexError, TypeError, ValueError): return None

def on_click(atom_index, resn='', resi='', chain='', atom='', serial='', icode='',
             live_marked=False, viewer_generation=None, exact=True):
    if viewer_generation is not None:
        try: current_generation = int(viewer_generation)
        except (TypeError, ValueError): return
        if current_generation != _VIEWER_GENERATION['value']: return
    meta = _resolve_click_meta(atom_index, serial, resn, resi, chain, atom, icode)
    if meta is None:
        exact_atom_msg.value = ('<small role="alert" style="color:#991b1b">Could not map this viewer atom '
                                'to the input; workflow selections were not changed.</small>')
        return
    try: viewer_index = int(atom_index)
    except (TypeError, ValueError): viewer_index = -1
    index = int(meta['index'])
    resn = str(meta.get('resname') or resn).strip()
    resi = str(meta.get('resseq') if meta.get('resseq') is not None else resi)
    icode = str(meta.get('icode') or icode).strip(); resi += icode
    chain = str(meta.get('chain') or chain).strip()
    atom = str(meta.get('name') or atom).strip()
    serial = meta.get('serial', serial); xyz = meta.get('xyz')
    exact_atom_msg.value = ''
    act = pick_action.value
    if act == 'freezeatom' and not _PICK_ACTION_STATE['freeze_active']:
        act = None
    if exact in (False, 0, 'false', 'False') and act not in ('center', 'ligand', 'selectedresn'):
        S['_last_pick'] = {'chain': chain, 'resn': resn, 'resi': resi, 'atom': 'residue',
                           'serial': serial, 'icode': icode, 'xyz': xyz, 'index': index,
                           'viewer_index': viewer_index, 'action': act}
        S['_last_pick_message'] = ('Mol* focused this residue. Click one atom in its '
                                   'ball-and-stick view to assign the active role.')
        S['_last_pick_tone'] = 'warn'
        renderer = globals().get('_render_last_pick_status')
        if renderer is not None: renderer()
        refresh()
        return
    pick = {'chain': chain, 'resn': resn, 'resi': resi, 'atom': atom,
            'serial': serial, 'icode': icode, 'xyz': xyz, 'index': index,
            'viewer_index': viewer_index, 'action': act}
    message, tone = '', 'ok'
    if S.get('_view_input_index', 0) and not S.get('_view_mapping_ok', False):
        message, tone = 'view-only: atom identifiers differ from the first input; return to R/input 1 to edit selections', 'warn'
    elif act == 'center':
        rid = ('%s:%s:%s' % (chain, resn, resi)) if chain else str(resi)
        narrowed = False
        if center_widget is not None and resn in set(center_widget.value):
            _INCREMENTAL_PICK['active'] = True
            try:
                center_widget.value = tuple(value for value in center_widget.value if value != resn)
            finally:
                _INCREMENTAL_PICK['active'] = False
            narrowed = True
        elif resn in S.get('center', []):
            S['center'] = [value for value in S['center'] if value != resn]
            narrowed = True
        if rid not in S['center_ids']:
            S['center_ids'].append(rid); _invalidate_charge_confirmation(); _render_center_ids()
            message = ('replaced the %s name selection with this exact center (-c)' % resn
                       if narrowed else 'added as an exact center (-c)')
        else:
            message = 'already selected as an exact center (-c)'
    elif act == 'selectedresn':
        # The CLI selector for a viewer-picked residue is its residue ID.
        # Prefixing the residue name (for example GLU:186) is interpreted as
        # a chain-qualified selector and therefore fails to match.
        selector = str(resi)
        tokens = _selected_resn_tokens()
        if selector in tokens:
            tokens = [token for token in tokens if token != selector]
            message = 'removed from --selected-resn'
        else:
            tokens.append(selector)
            message = 'force-included with --selected-resn'
        selected_resn.value = ', '.join(tokens)
    elif act == 'ligand' and charge_rows is not None and resn in charge_rows:
        row = charge_rows[resn]
        if row.get('auto'):
            message = 'uses the built-in %s charge (%+g)' % (resn, row['val'].value)
        else:
            row['use'].value = True; message = 'enabled its ligand-charge (-l) row'
    elif act == 'ligand':
        message, tone = 'not a detected ligand/cofactor; choose a hetero residue', 'warn'
    elif act in ('scanA', 'scanB'):
        slot = 0 if act == 'scanA' else 1
        picked = {'chain': chain, 'resn': resn, 'resi': resi, 'icode': icode,
                  'atom': atom, 'xyz': xyz, 'index': index}
        if slot == 1 and _same_atom(S['scan_atoms'][0], picked):
            message, tone = 'scan atom B must differ from atom A', 'warn'
        else:
            # The first manual replacement owns the scan definition. Keeping a
            # complete preset here would leave a stale runnable command while
            # the new pair is visibly incomplete.
            S['scan_preset'] = ''
            S['scan_atoms'][slot] = picked
            message = 'set scan atom %s' % ('A' if slot == 0 else 'B')
            if slot == 0:
                pick_action.value = 'scanB'; message += '; next click sets atom B'
        _render_scan_panel()
    elif act in ('freezeA', 'freezeB'):
        slot = 0 if act == 'freezeA' else 1
        picked = {'chain': chain, 'resn': resn, 'resi': resi, 'icode': icode,
                  'atom': atom, 'xyz': xyz, 'index': index}
        if slot == 1 and _same_atom(S['freeze_buf'][0], picked):
            message, tone = 'restraint endpoint B must differ from endpoint A', 'warn'
        else:
            S['freeze_buf'][slot] = picked
            message = 'set restraint endpoint %s' % ('A' if slot == 0 else 'B')
            if slot == 0:
                pick_action.value = 'freezeB'; message += '; next click sets endpoint B'
        _render_freeze_panel()
    elif act == 'freezeatom':
        idx = index + 1 if index >= 0 else None
        if idx is None:
            message, tone = 'could not resolve the atom index', 'warn'
        elif idx in S['freeze_atoms']:
            S['freeze_atoms'] = [value for value in S['freeze_atoms'] if value != idx]
            _render_freeze_panel(); message = 'removed frozen atom #%d' % idx
        else:
            S['freeze_atoms'] = sorted(set(S['freeze_atoms']) | {idx})
            _render_freeze_panel(); message = 'added frozen atom #%d' % idx
    S['_last_pick'] = pick
    _remember_pick(pick)
    S['_last_pick_message'] = message or 'selected'
    S['_last_pick_tone'] = tone
    _rlp = globals().get('_render_last_pick_status')
    if _rlp is not None: _rlp()
    # Mol* owns live click rendering in the browser. Only the manual exact-atom
    # fallback rebuilds the iframe from Python.
    if not live_marked:
        render_viewer()
    refresh()

def _clear_highlights_from_browser(viewer_generation=None):
    if viewer_generation is not None:
        try: current_generation = int(viewer_generation)
        except (TypeError, ValueError): return
        if current_generation != _VIEWER_GENERATION['value']: return
    S.update(_last_pick=None, _pick_history=[], _last_pick_message='', _last_pick_tone='ok')
    renderer = globals().get('_render_last_pick_status')
    if renderer is not None: renderer()
    refresh()

try:
    from google.colab import output as _co
    _co.register_callback('pdb2reaction_gui.on_click', on_click)
    _co.register_callback('pdb2reaction_gui.clear_highlights', _clear_highlights_from_browser)
except Exception:
    _co = None              # click->Python only registers inside Google Colab

def _render_scan_panel():
    a, b = S['scan_atoms']
    preset_bonds = (_scan_literal_bonds(S.get('scan_preset', ''))
                    if not a and not b else [])
    preset_pair = preset_bonds[0] if len(preset_bonds) == 1 else None
    def fmt(atom): return str(_aspec(atom)) if atom else 'not set'
    display_a = fmt(a) if a else (str(preset_pair[0]) if preset_pair else 'not set')
    display_b = fmt(b) if b else (str(preset_pair[1]) if preset_pair else 'not set')
    distance = scan_distance()
    pair_ready = bool(a and b and not _same_atom(a, b))
    pair_state = (
        '<span style="color:#166534"><b>Ready</b> · current distance %.2f Å</span>' % distance
        if pair_ready and distance is not None else
        '<span style="color:#475569">Loaded example · Clear to redefine.</span>'
        if preset_pair else
        '<span style="color:#92400e">Pick atom A, then atom B.</span>')
    def _choose(slot):
        def activate(_):
            if not _view_is_editable(): return
            _PICK_ACTION_STATE['sync'] = True
            try: pick_action.value = 'scanA' if slot == 0 else 'scanB'
            finally: _PICK_ACTION_STATE['sync'] = False
            _render_scan_panel()
        return activate
    pick_a = W.Button(description='① Pick atom A',
                      tooltip='Choose the first atom in Mol*.',
                      layout=W.Layout(width='auto', flex='1 1 0', min_width='0'))
    pick_b = W.Button(description='② Pick atom B',
                      tooltip='Choose the second atom in Mol*.',
                      layout=W.Layout(width='auto', flex='1 1 0', min_width='0'))
    pick_a.button_style = 'primary' if pick_action.value == 'scanA' else ''
    pick_b.button_style = 'primary' if pick_action.value == 'scanB' else ''
    pick_a.on_click(_choose(0)); pick_b.on_click(_choose(1))
    def _reset_pair(render=True):
        S['scan_atoms'] = [None, None]; S['scan_preset'] = ''
        _PICK_ACTION_STATE['sync'] = True
        try: pick_action.value = 'scanA'
        finally: _PICK_ACTION_STATE['sync'] = False
        if render:
            _render_scan_panel(); render_viewer(); refresh()
    def _clear_pair(_):
        if not _view_is_editable(): return
        _reset_pair()
    clear_pair = W.Button(description='Clear',
                          tooltip='Clear the atom pair being prepared.',
                          layout=W.Layout(width='64px', flex='0 0 64px'))
    clear_pair.on_click(_clear_pair)
    pair_controls = W.HBox(
        [pick_a, pick_b, clear_pair],
        layout=W.Layout(width='100%', flex_flow='row nowrap', align_items='stretch'))
    pair_controls.add_class('rxscan-pair-controls')
    pair_html = W.HTML(
        '<div class="rxscan-pair-readout">'
        '<span><b>A</b>: <code>%s</code></span>'
        '<span><b>B</b>: <code>%s</code></span></div>'
        '<div class="rxscan-pair-state">%s</div>' %
        (html.escape(display_a), html.escape(display_b), pair_state))
    pair_html.add_class('rxscan-pair-html')
    sub_now = _wv('dd_subcmd', S['subcmd'])
    if sub_now in ('scan2d', 'scan3d'):
        need = 2 if sub_now == 'scan2d' else 3
        low = W.BoundedFloatText(value=1.2, min=0.3, max=6.0, step=0.10,
                                 description='③ Set low Å', layout=W.Layout(width='178px'))
        high = W.BoundedFloatText(value=3.0, min=0.3, max=8.0, step=0.10,
                                  description='High Å', layout=W.Layout(width='150px'))
        axis_note = W.HTML()
        def _add_axis(_):
            if not _view_is_editable(): return
            if not pair_ready:
                axis_note.value = '<small role="alert" style="color:#991b1b">Pick two different atoms first.</small>'; return
            if low.value >= high.value:
                axis_note.value = '<small role="alert" style="color:#991b1b">Low must be smaller than high.</small>'; return
            if len(S['scan_axes']) >= need:
                axis_note.value = '<small role="alert" style="color:#991b1b">All axes are already defined.</small>'; return
            key = tuple(sorted((a.get('index'), b.get('index'))))
            existing = {tuple(sorted((axis['a'].get('index'), axis['b'].get('index'))))
                        for axis in S['scan_axes']}
            if key in existing:
                axis_note.value = '<small role="alert" style="color:#991b1b">That atom pair is already an axis.</small>'; return
            S['scan_axes'].append({'a': a, 'b': b, 'lo': low.value, 'hi': high.value})
            _reset_pair(False); _render_scan_panel(); render_viewer(); refresh()
        add_axis = W.Button(description='4  Add axis', button_style='info',
                            layout=W.Layout(width='120px'),
                            disabled=(not pair_ready or len(S['scan_axes']) >= need))
        add_axis.on_click(_add_axis)
        axis_cards = []
        for axis_index, axis in enumerate(S['scan_axes']):
            lo_edit = W.BoundedFloatText(
                value=float(axis['lo']), min=0.3,
                max=max(0.3, float(axis['hi']) - 0.10), step=0.10,
                description='low Å', layout=W.Layout(width='145px'))
            hi_edit = W.BoundedFloatText(
                value=float(axis['hi']),
                min=min(8.0, float(axis['lo']) + 0.10), max=8.0, step=0.10,
                description='high Å', layout=W.Layout(width='150px'))
            up = W.Button(description='↑', tooltip='Move axis up',
                          disabled=axis_index == 0, layout=W.Layout(width='44px'))
            down = W.Button(description='↓', tooltip='Move axis down',
                            disabled=axis_index == len(S['scan_axes']) - 1,
                            layout=W.Layout(width='44px'))
            remove = W.Button(description='×', tooltip='Remove axis',
                              layout=W.Layout(width='44px'))
            replace = W.Button(description='Use picked pair',
                               disabled=not pair_ready, layout=W.Layout(width='125px'))
            def _edit_axis(_change=None, i=axis_index, lo=lo_edit, hi=hi_edit):
                if lo.value >= hi.value: return
                S['scan_axes'][i]['lo'] = float(lo.value)
                S['scan_axes'][i]['hi'] = float(hi.value)
                lo.max = max(lo.min, float(hi.value) - 0.10)
                hi.min = min(hi.max, float(lo.value) + 0.10)
                refresh()
            lo_edit.observe(_edit_axis, names='value'); hi_edit.observe(_edit_axis, names='value')
            def _move_axis(_button, i=axis_index, delta=-1):
                j = i + delta
                S['scan_axes'][i], S['scan_axes'][j] = S['scan_axes'][j], S['scan_axes'][i]
                _render_scan_panel(); refresh()
            up.on_click(lambda button, i=axis_index: _move_axis(button, i, -1))
            down.on_click(lambda button, i=axis_index: _move_axis(button, i, 1))
            def _remove_axis(_button, i=axis_index):
                S['scan_axes'].pop(i); _render_scan_panel(); render_viewer(); refresh()
            remove.on_click(_remove_axis)
            def _replace_axis(_button, i=axis_index):
                if not pair_ready: return
                S['scan_axes'][i]['a'], S['scan_axes'][i]['b'] = a, b
                _reset_pair(False); _render_scan_panel(); render_viewer(); refresh()
            replace.on_click(_replace_axis)
            card = W.VBox([
                W.HTML('<b>Axis %d</b> · <code>%s</code> ↔ <code>%s</code>' %
                       (axis_index + 1, html.escape(str(_aspec(axis['a']))),
                        html.escape(str(_aspec(axis['b']))))),
                W.HBox([lo_edit, hi_edit, replace, up, down, remove],
                       layout=W.Layout(flex_flow='row wrap'))])
            card.add_class('rxscan-stage'); axis_cards.append(card)
        axes_status = W.HTML(
            '<small style="color:%s"><b>Axes %d/%d</b></small>' %
            ('#166534' if len(S['scan_axes']) == need else '#92400e',
             len(S['scan_axes']), need))
        scan_panel.children = [
            W.HTML('<b>%s coordinate grid</b> <small>· define %d atom-pair axes</small>' %
                   (sub_now, need)),
            pair_controls, pair_html, W.HBox([low, high, add_axis]), axis_note,
            axes_status, W.VBox(axis_cards), b_clear_scan]
        return
    target = W.BoundedFloatText(value=float(S['scan_target']), min=0.3, max=6.0, step=0.5,
                                description='③ Set target Å', style={'description_width': '125px'},
                                layout=W.Layout(width='100%'))
    target.add_class('rxhalf-step')
    def _set_target(_):
        if not _view_is_editable(): return
        S['scan_target'] = target.value; refresh()
    target.observe(_set_target, names='value')
    def _add_stage(_):
        if not _view_is_editable() or not pair_ready: return
        S['scan_stages'].append([{'a': a, 'b': b, 't': float(target.value)}])
        _reset_pair(False); _render_scan_panel(); render_viewer(); refresh()
    def _add_concerted(_):
        if not _view_is_editable() or not pair_ready: return
        if not S['scan_stages']: S['scan_stages'].append([])
        stage = S['scan_stages'][-1]
        pair_key = frozenset((_pick_key(a), _pick_key(b)))
        if any(frozenset((_pick_key(item['a']), _pick_key(item['b']))) == pair_key
               for item in stage):
            S.update(_last_pick_message='That coordinate is already in the current stage.',
                     _last_pick_tone='warn')
            _render_last_pick_status()
            return
        stage.append({'a': a, 'b': b, 't': float(target.value)})
        _reset_pair(False); _render_scan_panel(); render_viewer(); refresh()
    add_stage = W.Button(description='Add sequential stage', button_style='info',
                         tooltip='Start a new stage after the current one.',
                         layout=W.Layout(width='auto', flex='1 1 0', min_width='0'),
                         disabled=not pair_ready)
    add_concerted = W.Button(description='Add to stage', button_style='info',
                             tooltip='Add this coordinate to the current stage.',
                             layout=W.Layout(width='auto', flex='1 1 0', min_width='0'),
                             disabled=not pair_ready)
    add_stage.on_click(_add_stage); add_concerted.on_click(_add_concerted)
    stage_cards = []
    for stage_index, stage in enumerate(S['scan_stages']):
        stage_up = W.Button(description='↑', tooltip='Move stage up',
                            disabled=stage_index == 0, layout=W.Layout(width='40px'))
        stage_down = W.Button(description='↓', tooltip='Move stage down',
                              disabled=stage_index == len(S['scan_stages']) - 1,
                              layout=W.Layout(width='40px'))
        stage_remove = W.Button(description='×', tooltip='Remove stage',
                                layout=W.Layout(width='40px'))
        def _move_stage(_button, i=stage_index, delta=-1):
            j = i + delta
            S['scan_stages'][i], S['scan_stages'][j] = S['scan_stages'][j], S['scan_stages'][i]
            _render_scan_panel(); refresh()
        stage_up.on_click(lambda button, i=stage_index: _move_stage(button, i, -1))
        stage_down.on_click(lambda button, i=stage_index: _move_stage(button, i, 1))
        def _remove_stage(_button, i=stage_index):
            S['scan_stages'].pop(i); _render_scan_panel(); render_viewer(); refresh()
        stage_remove.on_click(_remove_stage)
        bond_rows = []
        for bond_index, bond in enumerate(stage):
            target_edit = W.BoundedFloatText(
                value=float(bond['t']), min=0.3, max=6.0, step=0.5,
                description='target Å', layout=W.Layout(width='150px'))
            target_edit.add_class('rxhalf-step')
            bond_remove = W.Button(description='×', tooltip='Remove coordinate',
                                   layout=W.Layout(width='36px', flex='0 0 36px'))
            def _edit_target(change, si=stage_index, bi=bond_index):
                S['scan_stages'][si][bi]['t'] = float(change['new']); refresh()
            target_edit.observe(_edit_target, names='value')
            def _remove_bond(_button, si=stage_index, bi=bond_index):
                S['scan_stages'][si].pop(bi)
                if not S['scan_stages'][si]: S['scan_stages'].pop(si)
                _render_scan_panel(); render_viewer(); refresh()
            bond_remove.on_click(_remove_bond)
            label = W.HTML(
                '<code>%s</code> ↔ <code>%s</code>' %
                (html.escape(str(_aspec(bond['a']))), html.escape(str(_aspec(bond['b'])))))
            label.layout = W.Layout(width='auto', flex='1 1 90px', min_width='70px')
            coordinate_children = [label, target_edit]
            if len(stage) > 1:
                bond_remove.add_class('rxscan-coordinate-remove')
                coordinate_children.append(bond_remove)
            coordinate_row = W.HBox(
                coordinate_children,
                layout=W.Layout(width='100%', flex_flow='row nowrap', align_items='center'))
            coordinate_row.add_class('rxscan-coordinate-row')
            row = W.VBox([coordinate_row], layout=W.Layout(width='100%'))
            row.add_class('rxscan-bond'); bond_rows.append(row)
        stage_title = W.HTML(
            '<b>Stage %d</b> <small>· coordinates in this stage run together</small>' %
            (stage_index + 1))
        stage_title.layout = W.Layout(flex='1 1 auto', min_width='0')
        stage_title.add_class('rxscan-stage-title')
        stage_actions = W.HBox(
            [stage_up, stage_down, stage_remove],
            layout=W.Layout(width='auto', flex_flow='row nowrap', align_items='center'))
        stage_actions.add_class('rxscan-stage-actions')
        stage_head = W.HBox(
            [stage_title, stage_actions],
            layout=W.Layout(width='100%', flex_flow='row nowrap',
                            align_items='center', justify_content='space-between'))
        stage_head.add_class('rxscan-stage-head')
        card = W.VBox([stage_head, *bond_rows])
        card.add_class('rxscan-stage'); stage_cards.append(card)
    note = W.HTML('' if len(S.get('inputs', [])) == 1 else
                  '<small style="color:#92400e">A scan coordinate is used only by a single-structure scan.</small>')
    target_row = W.HBox(
        [target, note],
        layout=W.Layout(width='100%', flex_flow='row wrap', align_items='center'))
    target_row.add_class('rxscan-target-row')
    add_actions = W.HBox(
        [add_concerted, add_stage],
        layout=W.Layout(width='100%', flex_flow='row nowrap', align_items='stretch'))
    add_actions.add_class('rxscan-add-actions')
    stages_box = W.VBox(stage_cards, layout=W.Layout(width='100%'))
    stages_box.add_class('rxscan-stages')
    b_clear_scan.add_class('rxscan-clear-all')
    scan_help = (
        'Choose atom A and atom B, set the target distance, then add the coordinate. '
        '<b>Sequential stages</b> run from top to bottom; coordinates in the same stage run together.')
    scan_panel.children = [
        _hdr('<b>Scan coordinate</b> <small>· Pick atom pairs A–B, and set target distances</small>',
             scan_help),
        pair_controls, pair_html, target_row, add_actions,
        stages_box, b_clear_scan]

def _render_freeze_panel():
    fa, fb = S['freeze_buf']
    def fmt(x): return str(_aspec(x)) if x else '— pick —'
    def _addpair(_):
        if fa and fb and not _same_atom(fa, fb):
            S['freeze_pairs'].append({'a': fa, 'b': fb, 't': None}); S['freeze_buf'] = [None, None]
            _render_freeze_panel(); refresh()
    def _clrpairs(_): S['freeze_pairs'] = []; _render_freeze_panel(); refresh()
    def _clratoms(_): S['freeze_atoms'] = []; _render_freeze_panel(); render_viewer(); refresh()
    def _toggle_frozen_atoms(_):
        active = bool(_PICK_ACTION_STATE['freeze_active'])
        if not active and not _view_is_editable(): return
        _PICK_ACTION_STATE['freeze_active'] = not active
        _PICK_ACTION_STATE['sync'] = True
        try:
            if active and 'center' in {value for _label, value in pick_action.options}:
                pick_action.value = 'center'
            elif not active:
                pick_action.value = 'freezeatom'
        finally: _PICK_ACTION_STATE['sync'] = False
        _render_freeze_panel()
        _render_pick_hint()
    b_ap = W.Button(description='add freeze pair', icon='plus', layout=W.Layout(width='160px'))
    b_cp = W.Button(description='clear pairs', layout=W.Layout(width='110px'))
    freeze_pick_active = bool(_PICK_ACTION_STATE['freeze_active'])
    b_pick_atoms = W.Button(
        description='Done picking' if freeze_pick_active else 'Pick frozen atoms',
        icon='check' if freeze_pick_active else 'mouse-pointer',
        layout=W.Layout(width='auto', max_width='100%', flex='0 1 250px'),
        tooltip=('Finish frozen-atom picking and return clicks to extraction-center selection.'
                 if freeze_pick_active else
                 'Choose atoms to keep fixed. Click a selected atom again to remove it.'))
    b_pick_atoms.button_style = 'primary' if freeze_pick_active else ''
    b_pick_atoms.disabled = not _view_is_editable()
    b_ca = W.Button(description='Clear', layout=W.Layout(width='90px'))
    b_ap.on_click(_addpair); b_cp.on_click(_clrpairs)
    b_pick_atoms.on_click(_toggle_frozen_atoms); b_ca.on_click(_clratoms)
    pairs = '; '.join('%s↔%s' % (_aspec(p['a']), _aspec(p['b'])) for p in S['freeze_pairs']) or '(none)'
    sub = _wv('dd_subcmd', S.get('subcmd', 'all'))
    children = []
    if sub == 'opt':
        children.extend([
            W.HTML('<b>Distance restraints</b> <small>(<code>--dist-freeze</code>) — set pick mode to '
                   '“Freeze pair · A/B”, click two atoms, then add</small>'),
            W.HTML('A: <code>%s</code> &nbsp; B: <code>%s</code>' % (fmt(fa), fmt(fb))),
            W.HBox([b_ap, b_cp]),
            W.HTML('<small>pairs: <code>%s</code></small>' % pairs)])
    atom_heading = ('<hr style="margin:5px 0"><b>Freeze atoms</b> <small>· optional</small>' if children
                    else '<b>Freeze atoms</b> <small>· optional</small>')
    atom_info = _info_control(
        'Click atoms to keep them fixed. Click a selected atom again, or its × chip, to remove it. '
        'Clear removes all fixed atoms.')
    children.extend([
        W.HBox([W.HTML(atom_heading + ' <small>(<code>--freeze-atoms</code>)</small>'), atom_info],
               layout=W.Layout(width='100%', align_items='center', justify_content='space-between')),
        W.HBox([b_pick_atoms, b_ca],
               layout=W.Layout(width='100%', flex_flow='row wrap', align_items='center'))])
    freeze_panel.children = children

def _view_role(index, total):
    reaction_order = (_wv('dd_subcmd', S.get('subcmd', 'all')) in ('path-opt', 'path-search') or
                      (_wv('dd_subcmd', S.get('subcmd', 'all')) == 'all' and
                       _wv('all_mode', 'mep') == 'mep'))
    if total <= 1 or not reaction_order: return 'Input %d' % (index + 1)
    if index == 0: return 'Reactant'
    if index == total - 1: return 'Product'
    return 'Intermediate %d' % index

def _sync_view_input_widget():
    paths = list(S.get('inputs', []))
    index = max(0, min(int(S.get('_view_input_index', 0)), max(0, len(paths) - 1)))
    S['_view_input_index'] = index
    opts = [('%s · %s' % (_view_role(i, len(paths)), os.path.basename(path)), i)
            for i, path in enumerate(paths)]
    _view_input_guard['active'] = True
    try:
        view_input.options = opts
        if opts: view_input.value = index
    finally:
        _view_input_guard['active'] = False

def _atom_signatures(metadata):
    return [(str(m.get('chain') or ''), str(m.get('resname') or '').upper(),
             str(m.get('resseq')), str(m.get('icode') or ''),
             str(m.get('name') or '').upper(), str(m.get('serial') or ''),
             str(m.get('element') or '').upper()) for m in metadata]

def _resolve_atom_query(query):
    query = str(query or '').strip()
    if not query: raise ValueError('Enter a 1-based atom index or CHAIN:RESNAME:RESSEQ:ATOM.')
    metadata = S.get('_atom_meta', [])
    if query.isdigit():
        index = int(query) - 1
        if 0 <= index < len(metadata): return index
        raise ValueError('Atom index %s is outside 1–%d.' % (query, len(metadata)))
    parts = [part.strip() for part in query.split(':')]
    if len(parts) == 4: chain, resn, resi, atom = parts
    elif len(parts) == 3: chain, (resn, resi, atom) = '', parts
    else: raise ValueError('Use CHAIN:RESNAME:RESSEQ[ICODE]:ATOM (or omit CHAIN).')
    found = []
    for meta in metadata:
        meta_resi = str(meta.get('resseq')) + str(meta.get('icode') or '')
        if (str(meta.get('chain') or '') == chain and
                str(meta.get('resname') or '').upper() == resn.upper() and
                meta_resi == resi and str(meta.get('name') or '').upper() == atom.upper()):
            found.append(int(meta['index']))
    if len(found) == 1: return found[0]
    if len(found) > 1: raise ValueError('That atom is ambiguous; include the chain ID.')
    raise ValueError('No retained atom matches %s.' % query)

def _apply_exact_atom(_=None):
    try:
        index = _resolve_atom_query(exact_atom.value)
    except ValueError as exc:
        exact_atom_msg.value = '<small role="alert" style="color:#991b1b">%s</small>' % html.escape(str(exc))
        return
    exact_atom_msg.value = ''
    on_click(str(index))
exact_atom_btn.on_click(_apply_exact_atom)

def _view_is_editable():
    return int(S.get('_view_input_index', 0)) == 0 or bool(S.get('_view_mapping_ok'))

def _set_primary_editor_enabled(enabled):
    disabled = not bool(enabled)
    if center_widget is not None: center_widget.disabled = disabled
    for name in ('prep_radius', 'selected_resn', 'b_pick_selected_resn', 'b_done_selected_resn'):
        widget = globals().get(name)
        if widget is not None: widget.disabled = disabled
    if charge_rows is not None:
        for row in charge_rows.values():
            row['use'].disabled = disabled or row.get('auto', False)
            row['val'].disabled = disabled or row.get('auto', False)

def _set_widget_tree_disabled(widget, disabled):
    for child in getattr(widget, 'children', ()):
        if hasattr(child, 'disabled'): child.disabled = bool(disabled)
        _set_widget_tree_disabled(child, disabled)

def _set_selection_editor_enabled(enabled):
    _set_primary_editor_enabled(enabled)
    for panel in (scan_panel, freeze_panel): _set_widget_tree_disabled(panel, not enabled)
    for button in getattr(chips_box, 'children', ()):
        if hasattr(button, 'disabled'): button.disabled = not enabled
    for name in ('b_extract', 'b_clear_center', 'b_clear_scan'):
        button = globals().get(name)
        if button is not None: button.disabled = not enabled

def build_selection():
    """Load and commit one view atomically; primary editors remain primary-owned."""
    global center_widget, charge_rows
    if S['mode'] not in ('pdb', 'mmcif', 'small') or not S['inputs']:
        render_viewer()
        return False
    _sync_view_input_widget()
    view_index = int(S.get('_view_input_index', 0))
    path = S['inputs'][view_index]
    is_small = S.get('mode') == 'small'
    try:
        if is_small:
            text, metadata, viewer_path = _load_small_view_structure(path)
            allr, het = [], []
            atoms = {(str(meta['index']),): meta['xyz'] for meta in metadata}
        else:
            text, metadata, viewer_path = _load_view_structure(path)
            allr, het = parse_residues(text, metadata)
            atoms = parse_atoms(text, metadata)
        signatures = _atom_signatures(metadata)
    except Exception as exc:
        viewer_out.value = ('<div role="alert" style="padding:12px;color:#991b1b">'
                            'Could not prepare structure for the viewer: <code>%s</code></div>' %
                            html.escape(str(exc)))
        viewer_signal_out.value = ''
        input_msg.value = '❌ structure load failed: <code>%s</code>' % html.escape(str(exc))
        return False
    suffix = Path(path).suffix.lower()
    source_format = ('xyz' if is_small else
                     ('mmcif' if suffix in ('.cif', '.mmcif') else 'pdb'))
    S.update(_pdb_path=viewer_path, _pdb_text=text,
             _view_format='xyz' if is_small else 'pdb',
             _view_source_format=source_format, _atom_meta=metadata,
             _hetero=het, _atoms=atoms)
    if view_index == 0:
        S['_primary_atom_signatures'] = signatures
        S['_primary_atom_meta'] = [dict(meta) for meta in metadata]
        S['_view_mapping_ok'] = True
        view_input_note.value = ''
    else:
        S['_view_mapping_ok'] = bool(S.get('_primary_atom_signatures')) and signatures == S['_primary_atom_signatures']
        view_input_note.value = ('' if S['_view_mapping_ok'] else
                                 ('<small role="alert" style="color:#92400e">View-only: atom identifiers/order '
                                  'differ from input 1. Clicks still highlight atoms and residues, but cannot '
                                  'change the workflow; return to R/input 1 to edit selections.</small>'))
    if view_index == 0 and not is_small:
        initial_center = tuple(x for x in S.get('center', []) if x in het)
        residue_copies = {}
        for meta in metadata:
            rn = str(meta.get('resname') or '').upper()
            if rn in het:
                residue_copies.setdefault(rn, set()).add(
                    (str(meta.get('chain') or ''), str(meta.get('resseq')),
                     str(meta.get('icode') or '')))
        _center_opts = [('%s · all %d copies' % (r, len(residue_copies[r])), r)
                        if len(residue_copies.get(r, ())) > 1 else (r, r) for r in het]
        center_widget = W.SelectMultiple(options=_center_opts, value=initial_center,
                                         rows=min(5, max(2, len(het))),
                                         layout=W.Layout(width='100%', max_width='100%'))
        charge_rows = {}; rows = []
        for rn in het:
            auto = rn in _ION_CHARGES
            pre = _ION_CHARGES[rn] if auto else S.get('lcharge', {}).get(rn, 0)
            use = W.Checkbox(value=(auto or rn in S.get('lcharge', {})),
                             description=('%s charge (auto)' if auto else 'set %s charge') % rn,
                             disabled=auto, indent=False, layout=W.Layout(width='150px'))
            val = W.BoundedFloatText(value=float(pre), min=-9, max=9, step=1,
                                     description='', style={'description_width': '0px'},
                                     disabled=auto, layout=W.Layout(width='105px'))
            charge_rows[rn] = {'use': use, 'val': val, 'auto': auto}; rows.append(W.HBox([use, val]))
        def _sync_center(change=None):
            if not _view_is_editable(): return
            _invalidate_charge_confirmation()
            S['center'] = list(center_widget.value)
            _render_center_ids()
            if not _INCREMENTAL_PICK['active']: render_viewer()
            refresh()
        def _sync_charge_rows(_=None):
            if not _view_is_editable(): return
            updated = {r: x['val'].value for r, x in charge_rows.items()
                       if x['use'].value and not x.get('auto')}
            if updated != S.get('lcharge', {}):
                S['lcharge'] = updated
                _invalidate_charge_confirmation()
            sync = globals().get('_sync_charge_controls')
            if sync is not None: sync()
            refresh()
        center_widget.observe(_sync_center, names='value')
        for x in charge_rows.values():
            x['use'].observe(_sync_charge_rows, names='value'); x['val'].observe(_sync_charge_rows, names='value')
        center_picker = (center_widget if het else
                         W.HTML('<small>No ligand names found — click an exact residue in 3D.</small>'))
        selected_resn_row = _flag_row(
            selected_resn,
            'Force-include residues in addition to <code>-c</code>. Type comma- or space-separated IDs, '
            'or use <b>Pick force-included residues</b> and click the sequence or 3D viewer; those clicks update this field.',
            rich=True)
        selected_resn_row.add_class('rxselected-resn-row')
        center_actions = W.HBox([b_pick_selected_resn, b_clear_center],
                                   layout=W.Layout(width='100%', flex_flow='row wrap', align_items='center',
                                                   column_gap='6px', row_gap='6px'))
        center_actions.add_class('rxcenter-actions')
        center_panel.children = ([
            _hdr('<b>Active-site cluster center <code>-c</code></b>',
                 'Choose the residue(s) at the center of the active-site cluster. Selecting a residue name uses every matching copy; use <i>Set extraction center (-c)</i> to pick one exact copy in the 3D viewer.')]
            + ([center_usage_guide] if 'center_usage_guide' in globals() else [])
            + [center_picker, center_ids_html,
               _flag_row(prep_radius_control,
                         'Set the extraction radius in Å around <code>-c</code>. At <code>-r 0</code>, '
                         'the model is built from residues selected by <code>-c</code> and <code>--selected-resn</code>, without radius-based expansion.',
                         rich=True),
               selected_resn_row, center_actions])
        charge_list = (W.VBox(rows, layout=W.Layout(width='100%'))
                       if rows else W.HTML('<i>no hetero ligands</i>'))
        charge_panel.children = [
            _hdr('<b>ligand charges <code>-l</code></b>',
                 'Known ions use built-in charges. Set charges only for other ligands. The system charge is calculated automatically unless you override it below.'),
            charge_list]
        _sync_charge_rows()
    _render_center_ids(); _render_scan_panel(); _render_freeze_panel()
    _set_selection_editor_enabled(_view_is_editable())
    if view_index == 0 or S.get('_view_mapping_ok'):
        _remap_stored_atom_coordinates(metadata)
    render_viewer()
    return True

def _on_view_input(change):
    if _view_input_guard['active'] or change.get('new') is None: return
    previous = {key: S.get(key) for key in ('_view_input_index', '_last_pick', '_pick_history',
                                            '_last_pick_message', '_last_pick_tone')}
    previous_note = view_input_note.value
    S.update(_view_input_index=int(change['new']), _last_pick=None, _pick_history=[],
             _last_pick_message='', _last_pick_tone='ok')
    if not build_selection():
        S.update(previous); view_input_note.value = previous_note
        _sync_view_input_widget(); render_viewer()
    _render_last_pick_status()
    contract_renderer = globals().get('_render_workflow_contract')
    if contract_renderer is not None: contract_renderer()
    refresh()
view_input.observe(_on_view_input, names='value')

# Keep one explicit representation choice beside Mol*; Mol* still owns colour,
# camera, measurement, focus, components, and screenshot controls.
dd_rep = W.Dropdown(options=[('Cartoon', 'cartoon'), ('Stick', 'stick')],
                    value='cartoon', description='Representation',
                    style={'description_width': 'initial'},
                    layout=W.Layout(width='230px'))
def _on_representation(change):
    if _REP_SYNC['active']:
        return
    S['rep'] = _normalize_representation(change.get('new'))
    _rep_user_set['value'] = True
    if not _SESSION_APPLY['active']:
        render_viewer()
dd_rep.observe(_on_representation, names='value')
dd_col = W.Dropdown(options=['element', 'chain', 'spectrum'], value='element')
cb_water = W.Checkbox(value=True, description='water', indent=False,
                      layout=W.Layout(width='82px'))
def _on_water(change):
    S['show_water'] = bool(change['new'])
    if not _SESSION_APPLY['active']: render_viewer()
cb_water.observe(_on_water, names='value')
cb_surf = W.Checkbox(value=False, description='surface')
cb_spin = W.Checkbox(value=False, description='spin')
dd_size = W.Dropdown(options=[640, 720, 800], value=720)
for _legacy_view_widget in (dd_col, cb_surf, cb_spin, dd_size):
    _legacy_view_widget.layout.display = 'none'
view_controls = W.HBox([
    dd_rep, cb_water,
    _info_control('Proteins use Cartoon automatically; small or fragmented structures use Stick. '
                  'Mol* starts with water hidden.')
], layout=W.Layout(flex_flow='row nowrap', align_items='center'))
view_controls.add_class('rxview-controls')

_sel_help = ''
summary_html = W.HTML()
def _render_summary():
    cw = center_widget
    cen = ','.join(list(cw.value) if cw is not None else S.get('center', []))
    ids = ','.join(S.get('center_ids', []))
    lc = ','.join('%s:%g' % (k, v) for k, v in S['lcharge'].items())
    auto_lc = ','.join('%s:%+g' % (name, row['val'].value) for name, row in (charge_rows or {}).items()
                       if row.get('auto'))
    sc = ' ; '.join(scan_literals())
    fp = '; '.join('%s↔%s' % (_aspec(p['a']), _aspec(p['b'])) for p in S['freeze_pairs'])
    fa = ','.join(str(i) for i in S['freeze_atoms'])
    selected = ', '.join(_selected_resn_tokens())
    rows = []
    sub = _wv('dd_subcmd', S.get('subcmd', 'all'))
    if cen or ids:
        center_label = 'extraction center -c'
        center_usage = ''
        if sub == 'extract':
            center_usage = ' · <code>extract</code> Run writes the cluster around this center'
        elif sub != 'all':
            center_usage = ' · prepare the cluster input below before Run'
        center_values = [value for value in (cen, ids) if value]
        rows.append('%s: <code>%s</code>%s' %
                    (center_label, html.escape(','.join(center_values)), center_usage))
    if selected:
        rows.append('force-included --selected-resn: <code>%s</code>' % html.escape(selected))
    if lc: rows.append('charges -l: <code>%s</code>' % html.escape(lc))
    if auto_lc: rows.append('built-in ion charge: <code>%s</code>' % html.escape(auto_lc))
    if sc and 'scan' in SPEC.get(sub, {}).get('panels', ()):
        rows.append('scan -s: <code>%s</code>' % html.escape(sc))
    freeze_parts = []
    if sub == 'opt' and fp: freeze_parts.append(html.escape(fp))
    if 'freeze' in SPEC.get(sub, {}).get('panels', ()) and fa:
        freeze_parts.append('atoms ' + html.escape(fa))
    if freeze_parts:
        rows.append('freeze: <code>%s</code>' % ' · '.join(freeze_parts))
    summary_html.value = (
        '<div style="border:1px solid #cdd;border-radius:8px;padding:5px 7px;background:#f7fbff;">'
        '<b>Selection summary</b><br><small>%s</small></div>' %
        ('<br>'.join(rows) if rows else 'No selections yet.'))

# Colab may otherwise collapse this row to zero height when Setup is rebuilt.
chips_box = W.HBox(layout=W.Layout(width='100%', min_height='38px',
                                      overflow='visible', flex='0 0 auto',
                                      flex_flow='row wrap', align_items='center'))
chips_box.add_class('rxchip')
def _render_chips():
    def mk(kind, key):
        def _rm(_):
            if kind == 'center' and center_widget is not None:
                center_widget.value = tuple(x for x in center_widget.value if x != key)
            elif kind == 'id':
                S['center_ids'] = [x for x in S['center_ids'] if x != key]
                _invalidate_charge_confirmation()
                _render_center_ids()
            elif kind == 'selected':
                selected_resn.value = ', '.join(
                    token for token in _selected_resn_tokens() if token != key)
            elif kind == 'atom':
                S['freeze_atoms'] = [x for x in S['freeze_atoms'] if x != key]
                _render_freeze_panel()
            _render_chips(); render_viewer(); refresh()
        return _rm
    btns = []
    cur = list(center_widget.value) if center_widget is not None else S.get('center', [])
    editable = _view_is_editable()
    for rn in cur:
        b = W.Button(description='%s ✕' % rn, button_style='info', disabled=not editable,
                     layout=W.Layout(width='auto'))
        b.on_click(mk('center', rn)); btns.append(b)
    for rid in S.get('center_ids', []):
        b = W.Button(description='%s ✕' % rid, button_style='info', disabled=not editable,
                     layout=W.Layout(width='auto'))
        b.on_click(mk('id', rid)); btns.append(b)
    for token in _selected_resn_tokens():
        b = W.Button(description='🔒 %s ✕' % token,
                     disabled=not editable, layout=W.Layout(width='auto'))
        b.on_click(mk('selected', token)); btns.append(b)
    for fa in S.get('freeze_atoms', []):
        b = W.Button(description='⚓ %d ✕' % fa, disabled=not editable, layout=W.Layout(width='auto'))
        b.on_click(mk('atom', fa)); btns.append(b)
    chips_box.children = btns
    chips_box.layout.display = 'flex' if btns else 'none'

# Keep optional atom freezing visible in the right-side Setup flow.  The
# compatibility alias is retained because capability routing uses freeze_acc.
freeze_acc = freeze_panel
viewer_out.layout = W.Layout(width='100%')
last_pick_html = W.HTML()
last_pick_info = _info_control('')
last_pick_html.layout = W.Layout(flex='1 1 auto', min_width='0')
last_pick_row = W.HBox([last_pick_html],
                       layout=W.Layout(width='100%', flex_flow='row wrap', align_items='center'))
last_pick_row.add_class('rxpick-footer')
_PICK_HINT = {
    'center': 'Click any atom in a residue to add that whole residue as an exact extraction center (-c).',
    'ligand': 'Click a ligand/cofactor to inspect or enable its charge.',
    'selectedresn': 'Click an additional residue in the sequence or 3D viewer to add or remove its residue number from --selected-resn.',
    'scanA': 'Click the first atom of the scan coordinate.', 'scanB': 'Click the second atom of the scan coordinate.',
    'freezeA': 'Click endpoint A of an opt distance restraint.', 'freezeB': 'Click endpoint B of an opt distance restraint.',
    'freezeatom': 'Click atoms to freeze or unfreeze them by 1-based Cartesian index.',
}
def _active_pick_hint():
    return _PICK_HINT.get(pick_action.value, 'Choose a click action.')
def _pick_info_text():
    return (_active_pick_hint() +
            ' Mol* keeps its standard focus, selection, and empty-canvas behavior.')
def _render_pick_hint(_=None):
    _set_info_text(last_pick_info, _pick_info_text())
    status_renderer = globals().get('_render_last_pick_status')
    if status_renderer is not None: status_renderer()
pick_action.observe(_render_pick_hint, names='value'); _render_pick_hint()
def _render_last_pick_status():
    pick = S.get('_last_pick')
    _set_info_text(last_pick_info, _pick_info_text())
    if not pick:
        last_pick_html.value = (
            '<div role="status" aria-live="polite" style="border-left:4px solid #E0DDD4;'
            'background:#F3F2EE;padding:6px 9px;color:#332B1F"><small>'
            '<b style="color:#B66D1D">Click sets:</b> %s</small></div>' %
            html.escape(_active_pick_hint()))
        return
    message = html.escape(str(S.get('_last_pick_message') or 'selected'))
    last_pick_html.value = (
        '<div role="status" aria-live="polite" style="border-left:4px solid #E0DDD4;'
        'background:#F3F2EE;padding:6px 9px;color:#332B1F"><small>'
        '<b style="color:#B66D1D">Last click:</b> '
        '<code style="color:#332B1F">%s</code> — '
        '<span style="color:#B66D1D">%s</span></small></div>' %
        (_pick_text(), message))
_render_last_pick_status()
# Keep the exact-atom fallback callable for tests and accessibility recovery,
# but do not expose a rarely-used disclosure in the normal Setup toolbar.
viewer_more = W.VBox([
    W.HBox([exact_atom, exact_atom_btn], layout=W.Layout(flex_flow='row wrap')),
    exact_atom_msg], layout=W.Layout(display='none'))
viewer_more.layout.display = 'none'
viewer_toolbar = W.HBox(
    [view_input, pick_action, last_pick_info, view_controls],
    layout=W.Layout(width='100%', flex_flow='row nowrap', align_items='flex-start'))
viewer_toolbar.add_class('rxviewer-toolbar')
viewer_col = W.VBox([viewer_status, viewer_out, viewer_signal_out, last_pick_row])
viewer_col.add_class('rxviewer')
# Compute commands operate on the current model. Prepare a cluster here when a
# full protein was loaded; `all` can instead perform the same extraction itself.
extract_msg = W.HTML()
b_extract = W.Button(description='Extract cluster & use it', icon='scissors',
                     button_style='info', layout=W.Layout(width='230px'),
                     tooltip='Build and use an active-site model from the current structure.')
def _do_extract_worker():
    try:
        if not S['inputs']: raise ValueError('Load a structure in the Input tab first.')
        cen = _center_cli_selectors()
        if not cen: raise ValueError('Set at least one extraction center (-c): click a residue in 3D or choose a residue name in the list.')
        r = float(prep_radius.value or 0.0)
        model_dir = _runtime_path('prepared_models')
        os.makedirs(model_dir, exist_ok=True)
        outs = [_unique_path(os.path.join(model_dir, '%02d_%s_cluster.pdb' % (i + 1, Path(path).stem)))
                for i, path in enumerate(S['inputs'])]
        cmd = [CLI, 'extract', '-i', *S['inputs'], '-o', *outs, '-c', ','.join(cen)]
        cmd += ['-r', str(r)]
        selected = str(selected_resn.value or '').strip()
        if selected: cmd += ['--selected-resn', selected]
        lc = _ligand_charge_cli(require_confirmation=True)
        if lc: cmd += ['-l', lc]
        extract_msg.value = '<small>running: <code>%s</code></small>' % ' '.join(cmd)
        p = subprocess.run(cmd, capture_output=True, text=True)
        if p.returncode != 0 or not all(os.path.exists(path) for path in outs):
            raise RuntimeError((p.stdout + p.stderr)[-400:] or 'extract failed')
        previous = {'inputs': list(S['inputs']), 'mode': S['mode'], 'parm': S.get('parm')}
        load_pdb(outs, None, keep_subcmd=True)
        # load_pdb deliberately invalidates an older preparation transaction;
        # install this new transaction only after the new inputs finish loading.
        S['_pre_extract'] = previous
        b_revert.layout.display = ''
        input_msg.value = '✅ <b>%s</b> (extracted cluster model%s)' % (
            ', '.join(outs), 's' if len(outs) != 1 else '')
        extract_msg.value = ('<small style="color:#181">✅ %d model(s) prepared outside the run output; '
                             'they are now the ordered inputs.</small>' % len(outs))
    except Exception as e:
        extract_msg.value = '<small style="color:#a00">extract failed: %s</small>' % e
_EXTRACT_TASK = {'thread': None}
def _do_extract(_):
    current = _EXTRACT_TASK.get('thread')
    if current is not None and current.is_alive():
        return
    b_extract.disabled = True
    def work():
        try:
            _do_extract_worker()
        finally:
            _dispatch_ui(lambda: setattr(b_extract, 'disabled', False))
    current = threading.Thread(target=work, name='pdb2reaction-cluster-extract', daemon=True)
    _EXTRACT_TASK['thread'] = current
    current.start()
b_extract.on_click(_do_extract)
b_revert = W.Button(description='Revert', icon='undo', layout=W.Layout(width='110px'),
                    tooltip='Restore the structure used before the last preparation.')
b_revert.layout.display = 'none'
def _do_revert(_):
    prev = S.get('_pre_extract')
    if not prev: return
    load_pdb(list(prev['inputs']), prev.get('parm'), mode=prev['mode'], keep_subcmd=True)
    input_msg.value = '↩ reverted to <b>%s</b>' % ', '.join(prev['inputs'])
    extract_msg.value = '<small>reverted to the pre-extraction structure</small>'
b_revert.on_click(_do_revert)
_CENTER_USAGE_GUIDE = (
    'Choose the ligand/cofactor residue(s) that define the center of the active-site cluster.')
_CENTER_USAGE_DETAILS = (
    'Select a residue name for every matching copy, or pick one copy in the Structure Viewer. '
    'The radius sets the cluster size; force-included residues can add residues outside it.')
center_usage_guide_text = W.HTML(_CENTER_USAGE_GUIDE)
center_usage_info = _info_control(_CENTER_USAGE_DETAILS, rich=True)
center_usage_guide = W.HBox(
    [center_usage_guide_text, center_usage_info],
    layout=W.Layout(width='100%', flex_flow='row wrap', align_items='center',
                    justify_content='flex-start'))
center_usage_guide.add_class('rxcenter-guide')
extract_panel = W.VBox([
    _hdr('<b>Prepare input cluster</b> <small>— standalone compute only</small>',
         'Use this for standalone <code>opt</code>, <code>tsopt</code>, <code>freq</code>, <code>irc</code>, <code>scan</code>, or <code>path-opt</code> runs. <code>all</code> and <code>extract</code> prepare the cluster automatically.'),
    W.HTML('<small>Choose <code>-c</code> above, set radius <code>-r</code>, then prepare. '
           'The generated cluster replaces the current input; Revert restores the full protein.</small>'),
    W.HBox([b_extract, b_revert], layout=W.Layout(flex_flow='row wrap')),
    extract_msg])
extract_panel.add_class('rxcard')
selection_inspector = W.VBox([
    summary_html, chips_box,
    center_panel, charge_panel, system_charge_panel, scan_panel, freeze_panel,
    extract_panel])
selection_inspector.add_class('rxinspector')
def _sync_active_selection_card():
    active = pick_action.value
    targets = ((center_panel, {'center', 'selectedresn'}), (charge_panel, {'ligand'}),
               (scan_panel, {'scanA', 'scanB'}),
               (freeze_panel, {'freezeA', 'freezeB', 'freezeatom'}))
    for panel, modes in targets:
        panel.remove_class('rxactive-card')
        if active in modes and not (panel is freeze_panel and S.get('mode') == 'small'):
            panel.add_class('rxactive-card')
    system_charge_panel.remove_class('rxactive-card')
    if S.get('mode') == 'small':
        system_charge_panel.add_class('rxactive-card')
_sync_active_selection_card()
workspace = W.HBox([viewer_col, selection_inspector]); workspace.add_class('rxworkspace')
selection_help = W.HTML(_sel_help)
selection_help.layout.display = 'none'
selection_route = W.HTML()
selection_route.layout.display = 'none'
def _sync_select_availability():
    has_files = bool(S.get('inputs'))
    is_utility = S.get('mode') == 'utility'
    viewable = has_files and not is_utility
    workspace.layout.display = '' if viewable else 'none'
    selection_inspector.layout.display = '' if viewable else 'none'
    selection_route.layout.display = 'none' if viewable else ''
    selection_route.value = (
        '<div class="rxcard"><b>Utility input</b><br><small>No 3D selection is needed.</small></div>'
        if is_utility else
        '<div class="rxcard"><b>No structure loaded</b></div>')
selection_box = W.VBox([selection_route, workspace])

# ============================================================== OPTIONS
dd_subcmd = W.Dropdown(options=_sub_options(SUBS), value='all', description='workflow',
                       style={'description_width': 'initial'},
                       layout=W.Layout(width='390px', max_width='100%', flex='0 1 390px'))
subreq = W.HTML(); subcmd_note = W.HTML()
def _workflow_contract(sub):
    """Resolve the visible input/output contract for the active mode and stages."""
    spec = SPEC.get(sub, {})
    req, outputs = spec.get('req', '(complete the command line)'), list(spec.get('out', ()))
    if sub == 'all':
        mode = _wv('all_mode', 'mep')
        if mode == 'mep':
            req = '2 or more structures in reaction order'
            outputs = ['summary.log', 'mep.pdb', 'energy_diagram_MEP.png', 'segments/seg_NN/']
        elif mode == 'scan':
            selected = _effective_run_inputs('all')
            req = 'current viewer: %s + a scan bond picked in Setup' % (os.path.basename(selected[0]) if selected else 'no structure')
            outputs = ['summary.log', 'scan/', 'segments/seg_NN/']
        else:
            selected = _effective_run_inputs('all')
            req = 'current viewer: %s as the TS candidate' % (os.path.basename(selected[0]) if selected else 'no structure')
            outputs = ['summary.log', 'segments/seg_01/{reactant,ts,product}', 'ts/', 'irc/']
        if _wv('w_ts', False) and mode != 'tsonly': outputs += ['ts/', 'irc/']
        if _wv('w_th', False): outputs += ['thermoanalysis.yaml', 'energy_diagram_Gibbs.png']
        if _wv('adv_dft', False): outputs += ['dft/', 'energy_diagram_DFT.png']
    return req, tuple(dict.fromkeys(outputs))

def _render_workflow_contract():
    sub = _wv('dd_subcmd', S.get('subcmd', 'all')) or 'all'
    req, outputs = _workflow_contract(sub)
    subreq.value = '<small><b>needs:</b> %s</small>' % html.escape(req)
    target = globals().get('outputs_html')
    if target is not None:
        target.value = ('<small><b>produces:</b> %s</small>' %
                        ' · '.join('<code>%s</code>' % html.escape(o) for o in outputs))

def _on_sub(_):
    sub = dd_subcmd.value
    previous_sub = S.get('subcmd')
    previous_scope = (S.get('_charge_scope')
                      if S.get('charge_explicit') else None)
    S['subcmd'] = sub
    if previous_sub != sub:
        _apply_subcommand_output_default(sub)
        # Click feedback belongs to the workflow that produced it.
        S.update(_last_pick=None, _pick_history=[], _last_pick_message='',
                 _last_pick_tone='ok')
        _rlp = globals().get('_render_last_pick_status')
        if _rlp is not None: _rlp()
    if (previous_scope is not None and
            previous_scope != _charge_scope_fingerprint(sub)):
        _invalidate_charge_confirmation()
    _render_workflow_contract()
    subcmd_note.value = ('' if (sub in COMPUTE or sub == 'extract') else
                         '<small style="color:#166534">utility template filled from the current input; review it below</small>'
                         if sub in AUTOFILL_UTILS else
                         '<small style="color:#7c5c00">Set values and output in Advanced flags.</small>'
                         if sub == 'energy-diagram' else
                         '<small style="color:#7c5c00">utility subcommand — finish it in the command line below</small>')
    _rsp = globals().get('_render_scan_panel')
    if _rsp is not None: _rsp()                          # scan vs scan2d/3d builder
    _scc = globals().get('_sync_capability_controls')
    if _scc is not None: _scc()
    refresh()
dd_subcmd.observe(_on_sub, names='value')
_on_sub(None)   # initialise the requirement hint for the default subcommand
all_mode = W.ToggleButtons(
    options=[('MEP', 'mep'), ('Scan', 'scan'), ('TS-only', 'tsonly')],
    value='mep', style={'button_width': '74px', 'description_width': '0px'},
    layout=W.Layout(width='240px'))
_ALL_MODE_STATE = {'last': 'mep', 'tsopt_before_tsonly': True, 'user': False, 'sync': False}
_SESSION_APPLY = {'active': False}
def _on_mode(change):
    if _SESSION_APPLY['active']:
        _ALL_MODE_STATE['last'] = all_mode.value
        return
    old = change.get('old', _ALL_MODE_STATE['last']) if isinstance(change, dict) else _ALL_MODE_STATE['last']
    new = all_mode.value
    if not _ALL_MODE_STATE.get('sync') and old != new: _ALL_MODE_STATE['user'] = True
    # MEP/scan/TS-only changes the calculation path, not the confirmed
    # chemical system or region charge.
    if new == 'tsonly' and old != 'tsonly':
        _ALL_MODE_STATE['tsopt_before_tsonly'] = bool(w_ts.value)
        w_ts.value = True
    elif old == 'tsonly' and new != 'tsonly':
        w_ts.value = _ALL_MODE_STATE['tsopt_before_tsonly']
    _ALL_MODE_STATE['last'] = new
    view_sync = globals().get('_sync_view_input_widget')
    if view_sync is not None: view_sync()
    contract_renderer = globals().get('_render_workflow_contract')
    if contract_renderer is not None: contract_renderer()
    sync = globals().get('_sync_capability_controls')
    if sync is not None: sync()
    refresh()
all_mode.observe(_on_mode, names='value')

_bk0 = BACKEND if BACKEND in MODELS else 'mace'
dd_backend = W.Dropdown(options=[('%s · installed' % _bk0, _bk0)], value=_bk0,
                        description='backend', disabled=True,
                        style={'description_width': 'initial'},
                        layout=W.Layout(width='190px'))
dd_model = W.Dropdown(options=MODELS[_bk0], value=S.get('model', DEFAULT_MODEL[_bk0]),
                      description='model', style={'description_width': 'initial'},
                      layout=W.Layout(width='230px'))
def _bk(_):
    changed = S.get('backend') != dd_backend.value
    S['backend'] = dd_backend.value
    dd_model.options = MODELS[dd_backend.value]
    if _SESSION_APPLY['active']: return
    dd_model.value = DEFAULT_MODEL[dd_backend.value]
    S['model'] = dd_model.value
    if changed: _invalidate_last_run('Backend changed; validate and run again.')
    refresh()
def _mdl(_):
    changed = S.get('model') != dd_model.value
    S['model'] = dd_model.value
    if _SESSION_APPLY['active']: return
    if changed: _invalidate_last_run('Backend model changed; validate and run again.')
    refresh()
dd_backend.observe(_bk, names='value'); dd_model.observe(_mdl, names='value')
w_ts = W.Checkbox(value=True, description='--tsopt (TS + IRC)', indent=False)
w_th = W.Checkbox(value=True, description='--thermo (vibrations + ΔG)', indent=False)
w_out = W.Text(value='./result_all/', description='out dir', layout=W.Layout(width='260px'))
output_note = W.HTML()
w_q = W.IntText(value=0, description='system charge (-q)', style={'description_width': 'initial'},
                layout=W.Layout(width='220px'))
w_charge_ok = W.Checkbox(value=False, description='overwrite system charge', indent=False)
charge_mode_note = W.HTML()
def _next_numbered_output_dir(path):
    """Return result(1), result(2), ... without touching prior output."""
    requested = os.path.normpath(str(path or _SUBCOMMAND_OUT_DEFAULTS.get(S.get('subcmd'), './result_all/')))
    directory, name = os.path.split(requested)
    base_name, number = name, 1
    if name.endswith(')') and '(' in name:
        prefix, suffix = name.rsplit('(', 1)
        digits = suffix[:-1]
        if prefix and digits.isdigit():
            base_name, number = prefix, int(digits) + 1
    while True:
        candidate = os.path.join(directory, '%s(%d)' % (base_name, number))
        if not os.path.lexists(candidate):
            return candidate
        if os.path.isdir(candidate) and not os.listdir(candidate):
            return candidate
        number += 1

def _render_output_note():
    out = (_effective_out_dir() if '_effective_out_dir' in globals() else
           (w_out.value or _SUBCOMMAND_OUT_DEFAULTS.get(S.get('subcmd'), './result_all/')))
    occupied = os.path.isdir(out) and bool(os.listdir(out))
    if occupied:
        fallback = _next_numbered_output_dir(out)
        output_note.value = (
            '<small style="color:#8a5a00">Existing <code>%s</code> will be preserved; '
            'Run will use <code>%s</code>.</small>' %
            (html.escape(str(out)), html.escape(str(fallback))))
    else:
        output_note.value = ''
_STAGE_SYNC = {'active': False}
def _sync_stage_dependency(change=None):
    if _STAGE_SYNC['active']: return
    _STAGE_SYNC['active'] = True
    try:
        owner = change.get('owner') if isinstance(change, dict) else None
        switched_on = bool(change.get('new')) if isinstance(change, dict) else False
        dft_widget = globals().get('adv_dft')
        # The parent stage remains user-editable. Turning TSopt off clears its
        # dependent stages; turning either dependent stage on enables TSopt.
        if owner is w_ts and not switched_on:
            if w_th.value: w_th.value = False
            if dft_widget is not None and dft_widget.value: dft_widget.value = False
        elif switched_on and (owner is w_th or owner is dft_widget):
            if not w_ts.value: w_ts.value = True
        elif (w_th.value or bool(getattr(dft_widget, 'value', False))) and not w_ts.value:
            w_ts.value = True
        w_ts.disabled = False
        S['tsopt'] = bool(w_ts.value); S['thermo'] = bool(w_th.value)
    finally:
        _STAGE_SYNC['active'] = False
def _sync_run(_=None):
    if _STAGE_SYNC['active']: return
    # During a session transaction, accept both stored stage values before
    # enforcing thermo -> tsopt. Otherwise the old thermo=True default can
    # overwrite a stored tsopt=False while values are applied in sequence.
    if _SESSION_APPLY['active']:
        S['tsopt'] = w_ts.value; S['thermo'] = w_th.value
        S['out_dir'] = w_out.value or _SUBCOMMAND_OUT_DEFAULTS.get(S.get('subcmd'), './result_all/'); S['charge'] = w_q.value
        return
    _sync_stage_dependency(_)
    S['tsopt'] = w_ts.value; S['thermo'] = w_th.value
    S['out_dir'] = w_out.value or _SUBCOMMAND_OUT_DEFAULTS.get(S.get('subcmd'), './result_all/'); S['charge'] = w_q.value
    sync = globals().get('_sync_capability_controls')
    if sync is not None: sync()
    refresh()
def _sync_charge_controls():
    charge_values, _unconfirmed = _current_ligand_charge_state()
    has_l = bool(charge_values)
    if not has_l and w_charge_ok.value and not _CHARGE_VERIFY_GUARD['active']:
        _CHARGE_VERIFY_GUARD['active'] = True
        try: w_charge_ok.value = False
        finally: _CHARGE_VERIFY_GUARD['active'] = False
        S['charge_explicit'] = False; S['_charge_scope'] = None
    w_charge_ok.disabled = not has_l
    w_q.disabled = has_l and not w_charge_ok.value
    charge_mode_note.value = (
        '<small><code>-l</code> is active: the full system charge is calculated automatically. '
        'Tick <b>overwrite system charge</b> only to emit an explicit <code>-q</code>.</small>'
        if has_l else
        '<small>No <code>-l</code> charge source is active, so <code>-q</code> is used directly.</small>')
def _sync_charge(_=None):
    S['charge'] = w_q.value
    if not _SESSION_APPLY['active']: refresh()
def _sync_charge_ok(change):
    if _CHARGE_VERIFY_GUARD['active']: return
    override = bool(change['new'])
    S['charge_explicit'] = override
    S['_charge_scope'] = _charge_scope_fingerprint() if override else None
    _sync_charge_controls()
    if not _SESSION_APPLY['active']: refresh()
for w in (w_ts, w_th, w_out): w.observe(_sync_run, names='value')
w_q.observe(_sync_charge, names='value')
w_charge_ok.observe(_sync_charge_ok, names='value')

# Advanced flags are derived from this repository's live Click command.
adv_mult = W.IntText(value=1, description='-m mult', style={'description_width': 'initial'}, layout=W.Layout(width='160px'))
adv_prec = W.Dropdown(options=[('default: per backend', 'auto'), ('fp32', 'fp32'), ('fp64', 'fp64')], value='auto', description='--precision', style={'description_width': 'initial'}, layout=W.Layout(width='420px', max_width='92%', min_width='0'))
adv_det = W.Checkbox(value=False, description='--deterministic', indent=False)
adv_mep = W.Dropdown(options=[('default: gsm', '(default)'), ('gsm', 'gsm')] + ([('dmf', 'dmf')] if DMF_READY else []), value='(default)', description='--mep-mode', style={'description_width': 'initial'}, layout=W.Layout(width='420px', max_width='92%', min_width='0'))
adv_dmf = W.Dropdown(options=[('default: gpu', '(default)'), ('gpu', 'gpu'), ('cpu', 'cpu')], value='(default)', description='--dmf-backend', style={'description_width': 'initial'}, layout=W.Layout(width='420px', max_width='92%', min_width='0'))
_THRESH_OPTIONS = [('default: gau', '(default)'), ('gau_loose', 'gau_loose'), ('gau', 'gau'), ('gau_tight', 'gau_tight'), ('gau_vtight', 'gau_vtight'), ('baker', 'baker'), ('never', 'never')]
adv_thresh = W.Dropdown(options=_THRESH_OPTIONS, value='(default)', description='--thresh', style={'description_width': 'initial'}, layout=W.Layout(width='420px', max_width='92%', min_width='0'))
adv_thresh_post = W.Dropdown(options=[('default: baker', '(default)')] + _THRESH_OPTIONS[1:], value='(default)', description='--thresh-post', style={'description_width': 'initial'}, layout=W.Layout(width='420px', max_width='92%', min_width='0'))
adv_radius = W.BoundedFloatText(value=2.6, min=0.0, max=1000000.0, step=0.5, description='-r radius Å', style={'description_width': 'initial'}, layout=W.Layout(width='360px', max_width='92%', min_width='0'))
adv_radius_control = _make_radius_stepper(adv_radius)
def _sync_radius_widgets(change):
    target = adv_radius if change['owner'] is prep_radius else prep_radius
    if target.value != change['new']: target.value = change['new']
    if not _SESSION_APPLY['active']:
        _invalidate_charge_confirmation()
prep_radius.observe(_sync_radius_widgets, names='value')
adv_radius.observe(_sync_radius_widgets, names='value')
adv_dft = W.Checkbox(value=False, description='--dft (DFT single-point energies)', indent=False)
adv_dftfb = W.Text(value='', description='--dft-func-basis', placeholder='wb97m-v/def2-tzvpd', style={'description_width': 'initial'}, layout=W.Layout(width='420px', max_width='92%', min_width='0'))
adv_flatten = W.Checkbox(value=False, description='--flatten', indent=False)
adv_refine = W.Checkbox(value=False, description='--refine-path (MEP refinement & multi-step reaction detection)', indent=False,
                        layout=W.Layout(width='auto', max_width='100%'))

adv_search = W.Text(value='', placeholder='filter flags…', description='search',
                    style={'description_width': 'initial'}, layout=W.Layout(width='360px', max_width='100%'))
adv_count = W.HTML()
b_reset_flags = W.Button(description='Reset flags', icon='undo',
                               tooltip='Restore the default values for all optional settings.',
                               layout=W.Layout(width='132px'))
adv_rows_box = W.VBox(layout=W.Layout(width='100%',
                                      border='1px solid #e2e8f0', padding='4px'))

_FRIENDLY_OPTION_HELP = {
    'tsopt': 'Optimize each transition-state candidate, run IRC in both directions, and refine the connected reactant and product structures.',
    'thermo': 'Run vibrational and QRRHO thermochemistry analyses, verify the imaginary mode, and build the ΔG diagram.',
    'dft': 'Calculate DFT single-point energies for each optimized reactant, transition state, and product. With Thermo, also build a ΔG diagram.',
    'refine_path': 'Search for additional MEP segments to identify multistep reactions. This requires more MEP calculations than the default.',
}

def _option_help(name, sub='all', fallback='See the command help for this option.'):
    if name in _FRIENDLY_OPTION_HELP: return _FRIENDLY_OPTION_HELP[name]
    for param in _advanced_options(sub):
        if getattr(param, 'name', None) == name and getattr(param, 'help', None): return param.help
    return fallback

def _widget_values(widget):
    # Dropdown options are (label, value) pairs so each control can name the
    # default it falls back to; validation compares the values, not the labels.
    return tuple(option[1] if isinstance(option, tuple) else option for option in widget.options)

def _cli_default_label(param):
    # show_default carries the effective default for options declared None so
    # that an explicit value stays distinguishable from an omission.
    shown = getattr(param, 'show_default', None)
    default = getattr(param, 'default', None)
    if isinstance(shown, str) and shown: value = shown
    elif default is None: value = 'None'
    elif isinstance(default, bool): value = 'on' if default else 'off'
    elif isinstance(default, (tuple, list)): value = ', '.join(map(str, default)) or 'empty'
    else: value = str(default)
    return 'default: %s' % value

_THRESH_HELP = (
    '<b>Optimization convergence</b><br>'
    '<code>--thresh</code> controls the main and pre-optimization convergence '
    'preset (CLI default: <code>gau</code>). When it is overridden, '
    '<code>--thresh-post</code> appears separately for post-IRC endpoint '
    'reoptimization (CLI default: <code>baker</code>).'
    '<div class="rxhelp-table-scroll"><table><thead><tr><th>preset</th><th>max force</th><th>RMS force</th>'
    '<th>max step</th><th>RMS step</th></tr></thead><tbody>'
    '<tr><td><code>gau_loose</code></td><td>2.5e-3</td><td>1.7e-3</td><td>1.0e-2</td><td>6.7e-3</td></tr>'
    '<tr><td><code>gau</code></td><td>4.5e-4</td><td>3.0e-4</td><td>1.8e-3</td><td>1.2e-3</td></tr>'
    '<tr><td><code>gau_tight</code></td><td>1.5e-5</td><td>1.0e-5</td><td>6.0e-5</td><td>4.0e-5</td></tr>'
    '<tr><td><code>gau_vtight</code></td><td>2.0e-6</td><td>1.0e-6</td><td>6.0e-6</td><td>4.0e-6</td></tr>'
    '<tr><td><code>baker</code></td><td>3.0e-4</td><td>2.0e-4</td><td>3.0e-4</td><td>2.0e-4</td></tr>'
    '<tr><td><code>never</code></td><td colspan="4">disable convergence stopping</td></tr>'
    '</tbody></table></div>')

def _set_advanced_override(sub, name, value):
    by_sub = S.setdefault('advanced_overrides', {}).setdefault(sub, {})
    if value in (None, ''): by_sub.pop(name, None)
    else: by_sub[name] = value
    refresh()

_ADVANCED_DEFAULT = '(default)'

def _advanced_widget(sub, param):
    overrides = S.setdefault('advanced_overrides', {}).setdefault(sub, {})
    _numeric_widget = False
    flag = _advanced_flag(param); saved = overrides.get(param.name)
    is_bool = param.is_bool_flag or isinstance(param.type, click.types.BoolParamType)
    # A non-None sentinel keeps the default selected across ipywidgets versions.
    selected = _ADVANCED_DEFAULT if saved is None else saved
    if param.name == 'verbose':
        if (saved is not None and
                (isinstance(saved, bool) or not isinstance(saved, int) or not 0 <= saved <= 3)):
            # Pre-v0.4.12/v0.3.3 sessions could store this generic text field
            # as an arbitrary string. Fall back to the CLI default instead of
            # making the whole Options pane fail to render.
            overrides.pop(param.name, None)
            saved = None
            selected = _ADVANCED_DEFAULT
        widget = W.Dropdown(
            options=[('default: 2', _ADVANCED_DEFAULT), ('silent · 0', 0), ('errors · 1', 1),
                     ('normal · 2', 2), ('detailed · 3', 3)],
            description=flag, style={'description_width': 'initial'},
            value=selected,
            layout=W.Layout(width='360px', max_width='92%'))
    elif is_bool:
        widget = W.Dropdown(options=[(_cli_default_label(param), _ADVANCED_DEFAULT), ('on', True), ('off', False)],
                            description=flag, style={'description_width': 'initial'},
                            value=selected,
                            layout=W.Layout(width='360px', max_width='92%'))
    elif isinstance(param.type, click.Choice):
        choices = list(param.type.choices)
        widget = W.Dropdown(options=[(_cli_default_label(param), _ADVANCED_DEFAULT)] + [(str(choice), choice) for choice in choices],
                            description=flag, style={'description_width': 'initial'},
                            value=selected,
                            layout=W.Layout(width='420px', max_width='92%'))
    elif (not param.multiple and int(getattr(param, 'nargs', 1)) == 1 and
          isinstance(param.type, (click.types.IntParamType, click.types.FloatParamType))):
        _numeric_widget = True
        _minimum = getattr(param.type, 'min', None)
        _maximum = getattr(param.type, 'max', None)
        _raw = saved if saved is not None else param.default
        if _raw is None:
            _raw = _minimum if _minimum is not None else 0
        if isinstance(param.type, click.types.IntParamType):
            _value = int(_raw)
            if _minimum is not None or _maximum is not None:
                _lo = int(_minimum) if _minimum is not None else -2147483648
                _hi = int(_maximum) if _maximum is not None else 2147483647
                _value = min(_hi, max(_lo, _value))
                widget = W.BoundedIntText(value=_value, min=_lo, max=_hi, step=1,
                                          description=flag, style={'description_width': 'initial'},
                                          layout=W.Layout(width='360px', max_width='92%'))
            else:
                widget = W.IntText(value=_value, step=1, description=flag,
                                   style={'description_width': 'initial'},
                                   layout=W.Layout(width='360px', max_width='92%'))
        else:
            _value = float(_raw)
            if _minimum is not None or _maximum is not None:
                _lo = float(_minimum) if _minimum is not None else -1.0e12
                _hi = float(_maximum) if _maximum is not None else 1.0e12
                _value = min(_hi, max(_lo, _value))
                widget = W.BoundedFloatText(value=_value, min=_lo, max=_hi,
                                            description=flag, style={'description_width': 'initial'},
                                            layout=W.Layout(width='360px', max_width='92%'))
            else:
                widget = W.FloatText(value=_value, description=flag,
                                     style={'description_width': 'initial'},
                                     layout=W.Layout(width='360px', max_width='92%'))
    else:
        default = param.default
        placeholder = _cli_default_label(param)
        nargs = max(1, int(getattr(param, 'nargs', 1)))
        if param.multiple and nargs > 1:
            placeholder += ' · repeated groups of %d space-separated values' % nargs
        elif param.multiple:
            placeholder += ' · quote each repeated value'
        elif nargs > 1:
            placeholder += ' · %d space-separated values' % nargs
        widget = W.Text(value='' if saved is None else str(saved), description=flag, placeholder=placeholder,
                        style={'description_width': 'initial'}, layout=W.Layout(width='520px', max_width='92%'))
    if _numeric_widget:
        widget.observe(lambda change, s=sub, n=param.name, d=param.default: _set_advanced_override(
            s, n, None if d is not None and change['new'] == d else change['new']), names='value')
    else:
        widget.observe(lambda change, s=sub, n=param.name: _set_advanced_override(
            s, n, None if change['new'] == _ADVANCED_DEFAULT else change['new']), names='value')
    row = _flag_row(widget, param.help or 'No additional help is supplied by this command.')
    row._rx_tier = 'advanced' if bool(getattr(param, 'hidden', False)) else 'key'
    row._rx_search = ('%s %s %s %s' %
                      (row._rx_tier, flag, param.name, param.help or '')).lower()
    return row

def _render_advanced_rows(_=None):
    _close_info()
    sub = _wv('dd_subcmd', S.get('subcmd', 'all')) or 'all'
    query = adv_search.value.strip().lower()
    # Every user-editable Click option for the selected subcommand remains
    # visible. Stage toggles affect execution, not option discoverability.
    rows = [_advanced_widget(sub, param) for param in _advanced_options(sub)
            if _advanced_status(sub, param) == 'rendered']
    shown = [row for row in rows if not query or query in row._rx_search]
    key_rows = [row for row in shown if row._rx_tier == 'key']
    advanced_rows = [row for row in shown if row._rx_tier == 'advanced']
    children = []
    if key_rows:
        children += [W.HTML('<div class="rxflag-tier"><b>Standard flags</b>'
                            '<small>shown in the standard CLI help</small></div>'), *key_rows]
    if advanced_rows:
        children += [W.HTML('<div class="rxflag-tier"><b>Advanced flags</b>'
                            '<small>shown by <code>--help-advanced</code></small></div>'), *advanced_rows]
    adv_rows_box.children = children or [W.HTML('<small>No matching flags.</small>')]
    adv_count.value = ('<small><b>%d flag%s</b> · %d standard · %d advanced%s</small>' %
                       (len(shown), '' if len(shown) == 1 else 's',
                        len(key_rows), len(advanced_rows),
                        (' · %d total' % len(rows)) if query else ''))

def _reset_advanced_flags(_=None):
    sub = _wv('dd_subcmd', S.get('subcmd', 'all')) or 'all'
    S.setdefault('advanced_overrides', {}).pop(sub, None)
    _invalidate_last_run('CLI flags reset; validate and run again.')
    _render_advanced_rows()
    refresh()

b_reset_flags.on_click(_reset_advanced_flags)
adv_search.observe(_render_advanced_rows, names='value')

def _sync_capability_controls(_=None):
    sub = dd_subcmd.value
    # The threshold preset differs by command (for example, opt uses gau
    # while tsopt/scan2d/scan3d use baker).  Read the label from the live
    # Click option instead of leaving a misleading global default.
    _thresh_param = next((param for param in _advanced_options(sub)
                          if param.name == 'thresh'), None)
    _thresh_label = (_cli_default_label(_thresh_param)
                     if _thresh_param is not None else 'default: None')
    _thresh_options = list(adv_thresh.options)
    if (_thresh_options and _thresh_options[0][0] != _thresh_label):
        _thresh_value = adv_thresh.value
        adv_thresh.options = [(_thresh_label, '(default)')] + _THRESH_OPTIONS[1:]
        if _thresh_value in {value for _label, value in adv_thresh.options}:
            adv_thresh.value = _thresh_value
    adv_mep.disabled = sub not in TOOL_CAPABILITIES['mep_mode']
    adv_thresh.disabled = sub not in TOOL_CAPABILITIES['threshold']
    show_thresh_post = sub == 'all' and adv_thresh.value != '(default)'
    adv_thresh_post.disabled = not show_thresh_post
    _set_flag_visible(adv_thresh_post, show_thresh_post)
    _set_flag_visible(adv_dft, sub == 'all' and DFT_READY)
    radius_applies = (sub == 'extract' or
                      (sub == 'all' and S.get('mode') in ('pdb', 'mmcif')))
    adv_radius.disabled = not radius_applies
    _set_flag_visible(adv_radius, radius_applies)
    # Every surfaced flag is shown only where the CLI accepts it (FLAG_SUBS).
    for _wn, _subs in FLAG_SUBS.items():
        _w = globals().get(_wn)
        if _w is not None and _wn not in ('adv_dft', 'adv_radius'):
            _set_flag_visible(_w, sub in _subs)
    _set_flag_visible(adv_dmf, sub in FLAG_SUBS['adv_dmf'] and adv_mep.value == 'dmf')
    mode = _wv('all_mode', 'mep') if sub == 'all' else None
    _sync_stage_dependency()
    if sub == 'all':
        path_active = mode != 'tsonly'
        for _w in (adv_mep, adv_thresh, adv_refine):
            _set_flag_visible(_w, path_active)
        _set_flag_visible(adv_dmf, path_active and adv_mep.value == 'dmf')
        _set_flag_visible(adv_flatten, mode == 'tsonly' or w_ts.value)
    # "all workflow" mode and the depth switches only exist on `all`; hide them for
    # every other subcommand instead of showing controls the command cannot use.
    for _name in ('all_mode_box', 'depth_box'):
        _b = globals().get(_name)
        if _b is not None: _b.layout.display = '' if sub == 'all' else 'none'
    # Select-tab panels follow the same table.
    _panels = SPEC.get(sub, {}).get('panels', ())
    if sub == 'all' and _wv('all_mode', 'mep') != 'scan':
        _panels = tuple(panel for panel in _panels if panel != 'scan')
    _atom_selectable = bool(S.get('inputs')) and S.get('mode') in ('pdb', 'mmcif', 'small')
    _residue_selectable = bool(S.get('inputs')) and S.get('mode') in ('pdb', 'mmcif')
    _center_active = sub in _PREP_SUBS and _residue_selectable
    # all extracts internally; extract itself is already the preparation command.
    # The extra preparation button is only for standalone compute workflows.
    _prep_active = _center_active and sub not in ('all', 'extract')
    _select_panels = set(_panels) if _atom_selectable else set()
    if not _residue_selectable: _select_panels.discard('center')
    if _center_active: _select_panels.add('center')
    for _name, _key in (('scan_panel', 'scan'), ('freeze_acc', 'freeze'),
                        ('center_panel', 'center'), ('charge_panel', 'center')):
        _b = globals().get(_name)
        if _b is not None: _b.layout.display = '' if _key in _select_panels else 'none'
    _extract_panel = globals().get('extract_panel')
    if _extract_panel is not None: _extract_panel.layout.display = '' if _prep_active else 'none'
    _guide = globals().get('center_usage_guide')
    _guide_text = globals().get('center_usage_guide_text')
    if _guide_text is not None: _guide_text.value = _CENTER_USAGE_GUIDE
    if _guide is not None:
        _guide.layout.display = '' if _center_active else 'none'
    _electronic = globals().get('system_charge_panel')
    if _electronic is not None:
        _electronic.layout.display = '' if sub in COMPUTE else 'none'
    _charge_sync = globals().get('_sync_charge_controls')
    if _charge_sync is not None: _charge_sync()
    _sync_select_availability()
    _freeze_renderer = globals().get('_render_freeze_panel')
    if _freeze_renderer is not None: _freeze_renderer()
    _current_pick = pick_action.value
    _pick_options = [
        (label, value) for label, value, panel in _PICK_ACTIONS
        if (panel is None or panel in _select_panels)
        and (value not in ('freezeA', 'freezeB') or sub == 'opt')]
    _valid_picks = {value for _label, value in _pick_options}
    _PICK_ACTION_STATE['sync'] = True
    try:
        pick_action.options = _pick_options
        if _PICK_ACTION_STATE['user'] and _current_pick in _valid_picks:
            pick_action.value = _current_pick
        elif 'scan' in _select_panels and 'scanA' in _valid_picks:
            pick_action.value = 'scanA'
        elif 'center' in _select_panels and 'center' in _valid_picks:
            pick_action.value = 'center'
        elif _pick_options:
            pick_action.value = _pick_options[0][1]
    finally:
        _PICK_ACTION_STATE['sync'] = False
    _sync_active_selection_card()
    _has_pick_actions = bool(_pick_options)
    for _control in (pick_action, last_pick_info):
        _control.layout.display = '' if _has_pick_actions else 'none'
    last_pick_row.layout.display = '' if _has_pick_actions else 'none'
    _kb = globals().get('key_opts_box')
    _bb = globals().get('backend_box')
    if _bb is not None: _bb.layout.display = '' if sub in MLIP_COMPUTE else 'none'
    _output_row = globals().get('output_run_row')
    _file_output = sub in COMPUTE or sub == 'extract' or sub in AUTOFILL_UTILS - {'bond-summary'}
    if _output_row is not None: _output_row.layout.display = '' if _file_output else 'none'
    w_out.layout.display = '' if sub in COMPUTE or sub == 'extract' else 'none'
    _hint = globals().get('utility_output_hint')
    if _hint is not None:
        _hint.layout.display = '' if _file_output and sub not in COMPUTE and sub != 'extract' else 'none'
    _run_box = globals().get('run_settings_box')
    if _run_box is not None: _run_box.layout.display = '' if (sub in COMPUTE or _file_output) else 'none'
    adv_dftfb.description = '--func-basis' if sub == 'dft' else '--dft-func-basis'
    _dftfb_applicable = DFT_READY and (sub == 'dft' or (sub == 'all' and adv_dft.value))
    adv_dftfb.disabled = not _dftfb_applicable
    _set_flag_visible(adv_dftfb, _dftfb_applicable)
    if _kb is not None:
        _key_widgets = (adv_mep, adv_dmf, adv_thresh, adv_thresh_post, adv_prec, adv_det,
                        adv_flatten, adv_refine, adv_radius, adv_dftfb)
        # On a same-kernel rerun, key_opts_box can still refer to the previous
        # GUI while the newly-created widgets do not have their rows yet.
        _key_rows = [getattr(w, '_rx_flag_row', None) for w in _key_widgets]
        _key_rows = [row for row in _key_rows if row is not None]
        if _key_rows:
            _kb.layout.display = '' if any(row.layout.display != 'none'
                                           for row in _key_rows) else 'none'
    _render_workflow_contract()
    queue_renderer = globals().get('_render_input_queue')
    if queue_renderer is not None: queue_renderer()
    label = globals().get('depth_label')
    if label is not None:
        label.value = ('<b>Stages</b> · TS optimization required'
                       if sub == 'all' and mode == 'tsonly' else
                       '<b>Optional stages</b>')
    render_advanced = globals().get('_render_advanced_rows')
    if render_advanced is not None: render_advanced()
    _set_selection_editor_enabled(_view_is_editable())

for w in (adv_mult, adv_prec, adv_det, adv_mep, adv_dmf, adv_thresh, adv_thresh_post, adv_radius, adv_dftfb,
          adv_flatten, adv_refine):
    if w is adv_radius:
        w.observe(lambda change: refresh(change, surface_only=True), names='value')
    else:
        w.observe(refresh, names='value')
adv_dft.observe(_sync_run, names='value')
adv_mep.observe(_sync_capability_controls, names='value')
adv_thresh.observe(_sync_capability_controls, names='value')
_sync_capability_controls()
adv_box = W.VBox([W.HBox([adv_search, adv_count, b_reset_flags],
                              layout=W.Layout(align_items='center', flex_flow='row wrap')),
                     adv_rows_box])
adv_acc = _collapsible('All flags', adv_box)
all_mode_box = all_mode
depth_label = W.HTML('<b>Optional stages</b>')
optional_stage_note = W.HTML(
    '<small style="color:#92400e">DFT is hidden (rerun Installation with install_dft enabled)</small>'
    if not DFT_READY else '')
depth_box = W.VBox([depth_label, W.HBox([
    _flag_row(w_ts, _option_help('tsopt', fallback='Optimize each TS candidate, trace the EulerPC IRC in both directions, and refine the connected R/P endpoints for every reactive segment.')),
    _flag_row(w_th, _option_help('thermo', fallback='After --tsopt, run vibrational analysis and QRRHO thermochemistry for each segment\'s R/TS/P stationary points; validate the TS imaginary mode and build the MLIP Gibbs free-energy (ΔG) diagram.')),
    _flag_row(adv_dft, _option_help('dft', fallback='After --tsopt, run energy-only DFT single-points on each segment\'s R/TS/P stationary-point structures and build the DFT//MLIP energy (ΔE) diagram. With --thermo, combine the MLIP thermal corrections to build the DFT//MLIP Gibbs free-energy (ΔG) diagram.')),
    _flag_row(adv_refine, _option_help('refine_path', fallback='Run recursive path-search over the ordered series instead of one path-opt GSM per adjacent pair. Each additional MEP segment it finds is a separate elementary step, so this is how a multi-step reaction is detected. It costs more MEP work than the single-pass default.'))]), optional_stage_note])
depth_box.add_class('rxcard')
depth_box.add_class('rxoptional-card')
outputs_html = W.HTML()
subreq.layout = W.Layout(flex='1 1 220px', min_width='0')
outputs_html.layout = W.Layout(flex='3 1 560px', min_width='0')
_adv_radius_row = _flag_row(adv_radius_control, _option_help('radius'))
adv_radius._rx_flag_row = _adv_radius_row
key_opts_content = W.VBox([
    _flag_row(adv_mep, _option_help('mep_mode')),
    _flag_row(adv_dmf, _option_help('dmf_backend')),
    _flag_row(adv_thresh, _THRESH_HELP, rich=True),
    _flag_row(adv_thresh_post, _THRESH_HELP, rich=True),
    _flag_row(adv_prec, _option_help('precision')),
    _flag_row(adv_det, _option_help('deterministic')),
    _flag_row(adv_flatten, _option_help('flatten')),
    _adv_radius_row,
    _flag_row(adv_dftfb, _option_help('dft_func_basis', fallback='Override the functional/basis used by the --dft stage, for example wb97m-v/def2-tzvpd.'))])
key_opts_box = _collapsible('Key options', key_opts_content)
_missing_features = []
if not DMF_READY: _missing_features.append('DMF is hidden (pydmf + cyipopt are not installed)')
dependency_note = W.HTML('<small style="color:#92400e">%s</small>' % ' · '.join(_missing_features)
                         if _missing_features else '')
backend_box = W.VBox([
    _hdr('<b>MLIP backend &amp; model</b>',
         'The backend is selected during Installation. Restart the runtime and rerun Installation to switch it.'),
    W.HBox([dd_backend, dd_model]), dependency_note])
backend_box.add_class('rxcard')
workflow_controls = W.HBox([dd_subcmd, all_mode_box],
                            layout=W.Layout(flex_flow='row wrap', align_items='center'))
workflow_contract_row = W.HBox([subreq, outputs_html],
                                layout=W.Layout(flex_flow='row wrap'))
workflow_controls.add_class('rxworkflow-controls')
workflow_contract_row.add_class('rxworkflow-contract')
workflow_box = W.VBox([
    _hdr(
        '<b>Workflow</b>',
        '<code>all</code> runs the full workflow; the other entries run one step. '
        '<br><br><b>Whole-PDB MEP:</b> choose '
        '<code>--mep-mode dmf</code> and turn off <code>--tsopt</code>, '
        '<code>--thermo</code>, and DFT. The full PDB can be used directly without '
        'cluster extraction. It may take several hours. Recommended <code>--max-nodes</code>: '
        'DMF <b>50–100</b>; GSM <b>20</b>.'),
    workflow_controls, subcmd_note])
workflow_box.add_class('rxcard')
workflow_box.add_class('rxworkflow-card')
viewer_box = W.VBox([workflow_box, viewer_toolbar, view_input_note, selection_box])
charge_info = _info_control(
    'Ligand charges calculate the system charge automatically. Select <b>overwrite system charge</b> '
    'only to enter <code>-q</code> yourself. <code>-m</code> sets spin multiplicity.',
    rich=True)
viewer_charge_row = W.HBox([w_q, w_charge_ok, adv_mult, charge_info],
                           layout=W.Layout(flex_flow='row wrap', align_items='center'))
system_charge_panel.children = [
    _hdr('<b>System charge &amp; multiplicity</b>',
         'Set the charge and spin multiplicity used for the calculation.'),
    viewer_charge_row, charge_mode_note]
_sync_charge_controls()
utility_output_hint = W.HTML('<small>The utility output path is shown in the generated command.</small>')
output_run_row = W.HBox([w_out, utility_output_hint], layout=W.Layout(flex_flow='row wrap', align_items='center'))
run_settings_box = W.VBox([
    _hdr('<b>Run &amp; output</b>',
         'Existing results are never overwritten; occupied directories automatically use result(1), result(2), and so on.'),
    output_run_row, output_note])
run_settings_box.add_class('rxcard')
options_box = W.VBox([backend_box, depth_box, key_opts_box, run_settings_box, adv_acc])
_sync_capability_controls()   # apply all-only visibility now that the boxes exist

# ============================================================== RESULTS tab
res_out = W.HTML(layout={'width': '100%', 'min_width': '0'})
# Colab does not reliably retain Output-captured HTML when Results are built
# after a background run. HTML widgets keep iframe markup in synced state.
traj_out = W.HTML(layout={'width': '100%', 'min_width': '0'})
traj_signal_out = W.HTML(layout={'height': '0px', 'min_height': '0px',
                                 'overflow': 'hidden'})
plot_out = W.HTML(layout={'width': '100%', 'min_width': '0'})
# Warm the pinned Plotly asset when the GUI launches. Profile iframes use the
# same immutable URLs, so MEP/IRC switches are normally served from the browser
# cache instead of starting a new network transfer. The visible loader below
# also fails over promptly if a CDN request is ever left pending by Colab.
_PLOTLY_SCRIPT_URLS = (
    'https://cdn.jsdelivr.net/npm/plotly.js-dist-min@2.35.2/plotly.min.js',
    'https://unpkg.com/plotly.js-dist-min@2.35.2/plotly.min.js',
)
_plotly_preload_document = r"""<!doctype html><html><head><meta charset="utf-8"></head><body>
<script>
(function(){
 const urls=__URLS__; let index=0;
 function next(){
   if(index>=urls.length)return;
   const script=document.createElement('script'), timer=setTimeout(()=>{
     script.remove(); index+=1; next();
   },4500);
   script.src=urls[index]; script.async=true;
   script.onload=()=>clearTimeout(timer);
   script.onerror=()=>{clearTimeout(timer);script.remove();index+=1;next();};
   document.head.appendChild(script);
 }
 next();
})();
</script></body></html>""".replace(
    '__URLS__', json.dumps(list(_PLOTLY_SCRIPT_URLS), separators=(',', ':')))
plotly_preload_out = W.HTML(
    value=_document_iframe(
        _plotly_preload_document,
        'title="Energy plot asset preload" aria-hidden="true"',
        'position:absolute;width:1px;height:1px;opacity:0;pointer-events:none;border:0;'),
    layout={'height': '0px', 'min_height': '0px', 'overflow': 'hidden'})
result_context, traj_label, frame_state, trajectory_intro, primary_result = W.HTML(), W.HTML(), W.HTML(), W.HTML(), W.HTML()
def _stable_selection_index(widget, proposal, empty):
    """Ignore stale frontend indices instead of selecting another result."""
    value = proposal['value']
    if value is None: return empty
    size = len(getattr(widget, '_options_labels', ()) or ())
    requested = value if isinstance(value, int) and not isinstance(value, bool) else -1
    if 0 <= requested < size: return requested
    current = getattr(widget, 'index', empty)
    try: current = int(current) if current is not None else None
    except (TypeError, ValueError): current = None
    if current is not None and 0 <= current < size: return current
    return empty if size == 0 else 0

def _safe_frontend_index_state(widget, sync_data, empty):
    """Normalize only integer-valued frontend noise before trait type checks."""
    if not isinstance(sync_data, dict) or 'index' not in sync_data: return sync_data
    raw = sync_data.get('index')
    if raw is None and empty is None: return sync_data
    normalized = raw if isinstance(raw, int) and not isinstance(raw, bool) else None
    if normalized is None and isinstance(raw, float):
        try: nearest = int(round(raw))
        except (OverflowError, ValueError): nearest = None
        if nearest is not None and abs(raw - nearest) <= 1e-9: normalized = nearest
    size = len(getattr(widget, '_options_labels', ()) or ())
    if normalized is None or not (0 <= normalized < size):
        normalized = getattr(widget, 'index', empty)
    if normalized is None and empty is not None: normalized = empty
    state = dict(sync_data); state['index'] = normalized
    return state

class _SafeResultDropdown(W.Dropdown):
    def set_state(self, sync_data):
        return super().set_state(_safe_frontend_index_state(self, sync_data, None))
    @_trait_validate('index')
    def _validate_index(self, proposal):
        return _stable_selection_index(self, proposal, None)

class _SafeResultSelectionSlider(W.SelectionSlider):
    def set_state(self, sync_data):
        return super().set_state(_safe_frontend_index_state(self, sync_data, 0))
    @_trait_validate('index')
    def _validate_index(self, proposal):
        return _stable_selection_index(self, proposal, 0)
energy_panel_title = W.HTML('<div class="rxpath-panel-title">Energy profile</div>')
energy_choice = _SafeResultDropdown(
    options=[], description='View', disabled=True,
    style={'description_width': 'initial'},
    layout=W.Layout(width='100%', max_width='520px'))
energy_choice.add_class('rxenergy-choice')
_ENERGY = {'mep_trajectory': None, 'aggregate_irc': None, 'views': {}}
_RESULT_SET_GENERATION = {'value': 0}
_ENERGY_LIVE_MEP = '__live_mep__'
_ENERGY_LABELS = {
    'dft_g': 'DFT//MLIP ΔG',
    'mlip_g': 'MLIP ΔG',
    'dft_e': 'DFT//MLIP ΔE',
    'mlip_e': 'MLIP ΔE',
    'mep': 'MEP',
    'irc': 'IRC',
}
artifact_out = W.HTML(layout={'width': '100%', 'min_width': '0'})
artifact_choice = _SafeResultDropdown(options=[], description='File',
                             style={'description_width': 'initial'},
                             layout=W.Layout(width='620px', max_width='100%'))
traj_choice = _SafeResultDropdown(options=[], description='Result',
                         style={'description_width': 'initial'},
                         layout=W.Layout(width='100%', max_width='720px', min_width='0'))
traj_choice.add_class('rxresult-choice')
result_selector_meta = W.HTML()
result_selector_meta.add_class('rxresult-count')
result_selector_row = W.HBox(
    [traj_choice, result_selector_meta],
    layout=W.Layout(width='100%', align_items='center'))
result_selector_row.add_class('rxresult-selector')
result_selector_row.layout.display = 'none'
result_loading = W.HTML()
result_loading.add_class('rxresult-loading')
result_loading.layout.display = 'none'
def _set_result_loading(label=None, active=False):
    if active:
        safe = html.escape(str(label or 'result view'))
        result_loading.value = ('<div class="rxresult-loading-banner" role="status" '
                                'aria-live="polite" aria-atomic="true">'
                                '<span class="rxtab-spinner" aria-hidden="true"></span>'
                                'Loading <b>%s</b>…</div>' % safe)
        result_loading.layout.display = ''
    else:
        result_loading.value = ''
        result_loading.layout.display = 'none'
    try:
        result_loading.send_state(); result_loading.layout.send_state()
    except Exception:
        pass
_result_pick_guard = {'active': False}
frame_prev = W.Button(description='‹', disabled=True,
                      tooltip='Show the preceding trajectory frame.',
                      layout=W.Layout(width='44px'))
frame_next = W.Button(description='›', disabled=True,
                      tooltip='Show the next trajectory frame.',
                      layout=W.Layout(width='44px'))
frame_slider = W.IntSlider(min=0, max=1, value=0, description='',
                           continuous_update=True, readout=False, disabled=True,
                           style={'description_width': 'initial'},
                           layout=W.Layout(width='100%', min_width='0'))
_SCAN_AXIS_GUARD = {'active': False}
scan_axis_sliders, scan_axis_rows = [], []
for _axis in range(3):
    _axis_label = W.HTML('<b>Coordinate %d</b> (Å)' % (_axis + 1))
    _axis_slider = _SafeResultSelectionSlider(
        # Colab's noUiSlider backend rejects a SelectionSlider whose option
        # range collapses to one index, even when the widget is disabled.
        options=[('—', 0.0), ('', None)], value=0.0, continuous_update=True, disabled=True, readout=True,
        layout=W.Layout(width='100%', min_width='0'))
    _axis_row = W.HBox([_axis_label, _axis_slider], layout=W.Layout(width='100%', align_items='center'))
    _axis_row.add_class('rxscan-axis-row')
    scan_axis_sliders.append(_axis_slider); scan_axis_rows.append(_axis_row)
scan_axis_controls = W.VBox(scan_axis_rows, layout=W.Layout(width='100%'))
scan_axis_controls.add_class('rxscan-axis-controls'); scan_axis_controls.layout.display = 'none'
_PLAYBACK_DEFAULT_INTERVAL_MS = 850
_PROFILE_PLAYBACK_INTERVAL_MS = 50
_VIBRATION_PLAYBACK_INTERVAL_MS = _PROFILE_PLAYBACK_INTERVAL_MS
playback_interval_signal = W.HTML(
    '<span data-rx-playback-interval="%d" hidden></span>' % _PLAYBACK_DEFAULT_INTERVAL_MS)
playback_interval_signal.add_class('rxplayback-interval')

def _playback_interval_ms(path=None, token=None):
    kind = str(token or '').strip().lower()
    name = os.path.basename(str(path or '')).lower()
    if (kind in ('mep', 'irc', 'vibration') or 'mep' in name or 'irc' in name or
            'mode_' in name or 'imag_' in name):
        return _PROFILE_PLAYBACK_INTERVAL_MS
    return _PLAYBACK_DEFAULT_INTERVAL_MS

def _sync_playback_interval(path=None, token=None):
    interval = _playback_interval_ms(path=path, token=token)
    frame_play.interval = interval
    playback_interval_signal.value = (
        '<span data-rx-playback-interval="%d" hidden></span>' % interval)

frame_play = W.Play(value=0, min=0, max=1, step=1, interval=_PLAYBACK_DEFAULT_INTERVAL_MS,
                    description='', disabled=True, repeat=True, show_repeat=True,
                    layout=W.Layout(width='116px', min_width='116px'))
# Playback has one browser-side writer. Combining W.link with W.jslink lets a
# delayed kernel echo overwrite a newer 50 ms MEP/IRC frame.
_frame_play_jslink = W.jslink((frame_play, 'value'), (frame_slider, 'value'))
frame_prev.add_class('rxframe-prev'); frame_next.add_class('rxframe-next')
frame_play.add_class('rxframe-play')
_TRAJ_VIEWER_GENERATION = {'value': 0}
_TRAJ_VIEWER_MOUNTED = {'value': False}
_TRAJ = {'frames': [], 'energies': [], 'energy_provenance': [],
         'energy_unit': 'hartree', 'quantity_label': 'ΔE', 'path': None, 'semantics': {},
         'state_labels': [], 'structure_paths': [], 'mode': 'trajectory',
         'generation': 0, 'frame_message': {}, 'structure_message': {}}

def _step_frame(delta):
    if frame_slider.disabled: return
    frame_slider.value = max(frame_slider.min, min(frame_slider.max, frame_slider.value + delta))

frame_prev.on_click(lambda _: _step_frame(-1))
frame_next.on_click(lambda _: _step_frame(1))

def _parse_trj(path):
    from pdb2reaction.io.xyz_trajectory import read_xyz_trajectory
    return read_xyz_trajectory(path, require_energies=False)

def _rel_kcal():
    es = _TRAJ['energies']
    if _TRAJ.get('energy_unit') == 'kcal/mol':
        return [float(value) if value is not None else None for value in es]
    # A relative profile is defined against frame 1 only. A later surviving
    # value must never silently become a new zero after a damaged first frame.
    if not es or es[0] is None: return None
    base = es[0]
    return [((e - base) * 627.5094740631) if e is not None else None for e in es]

def _irc_trajectory_role(path):
    name = os.path.basename(str(path or '')).lower()
    for role in ('finished', 'forward', 'backward'):
        suffix = role + '_irc_trj.xyz'
        if name == suffix or name.endswith('_' + suffix): return role
    return None

def _trajectory_result_metadata(path, n_frames=None):
    if not path or _irc_trajectory_role(path) != 'finished':
        return {}
    parent = Path(path).parent
    for candidate in (parent / 'result.json', parent.parent / 'result.json'):
        if not candidate.is_file(): continue
        try:
            with open(candidate, encoding='utf-8') as fh: data = json.load(fh)
            if not isinstance(data, dict): continue
            n_forward = int(data.get('n_frames_forward'))
            n_backward = int(data.get('n_frames_backward'))
            n_total = int(data.get('n_frames_total'))
        except (OSError, ValueError, TypeError, KeyError):
            continue
        result = {
            'forward_converged': data.get('forward_converged'),
            'backward_converged': data.get('backward_converged'),
            'scientific_status': data.get('scientific_status'),
            'scientific_status_reasons': data.get('scientific_status_reasons') or [],
            'n_forward': n_forward, 'n_backward': n_backward,
        }
        counts_valid = (
            n_forward >= 0 and n_backward >= 0 and
            n_total == n_forward + 1 + n_backward and
            (n_frames is None or n_total == int(n_frames)))
        if counts_valid:
            result.update(ts_index=n_forward, ts_label='input TS')
        else:
            result['metadata_warning'] = 'IRC frame metadata mismatch'
        branch = []
        for label, count, value in (
                ('forward', n_forward, result['forward_converged']),
                ('backward', n_backward, result['backward_converged'])):
            requested = data.get(label + '_requested')
            if requested is False or (requested is None and count == 0): continue
            branch.append(('%s ✓' if value is True else
                           '%s ✗' if value is False else '%s ?') % label)
        status = str(result.get('scientific_status') or 'status unavailable')
        result['trajectory_status'] = status + ((' · ' + ' · '.join(branch)) if branch else '')
        return result
    return {}

def _trajectory_segment_ranges(path, n_frames=None):
    name = os.path.basename(str(path or '')).lower()
    if 'mep_trj' not in name: return {}
    parent = Path(path).parent
    for candidate in (parent / 'summary.json', parent.parent / 'summary.json'):
        if not candidate.is_file(): continue
        try:
            with open(candidate, encoding='utf-8') as fh: data = json.load(fh)
            segments = data.get('segments') if isinstance(data, dict) else None
            if not isinstance(segments, list): continue
            bridge_ranges, segment_ranges = [], []
            bridge_seen = False
            for segment in segments:
                if not isinstance(segment, dict): continue
                kind = str(segment.get('kind') or 'seg').lower()
                if kind == 'bridge': bridge_seen = True
                ranges = segment.get('frame_ranges')
                if ranges is None and segment.get('frame_start') is not None:
                    ranges = [[segment.get('frame_start'), segment.get('frame_stop')]]
                if ranges is None:
                    if kind == 'bridge':
                        return {'extrema': False,
                                'metadata_warning': 'bridge frame metadata unavailable'}
                    continue
                if not isinstance(ranges, list): return {'extrema': False}
                for pair in ranges:
                    if not isinstance(pair, (list, tuple)) or len(pair) != 2:
                        return {'extrema': False}
                    start, stop = int(pair[0]), int(pair[1])
                    if start < 0 or stop <= start or (
                            n_frames is not None and stop > int(n_frames)):
                        return {'extrema': False,
                                'metadata_warning': 'segment frame metadata mismatch'}
                    record = (start, stop, kind, segment.get('index'))
                    segment_ranges.append(record)
                    if kind == 'bridge': bridge_ranges.append((start, stop))
            if bridge_seen and not bridge_ranges:
                return {'extrema': False,
                        'metadata_warning': 'bridge frame metadata unavailable'}
            ordered = sorted(segment_ranges, key=lambda item: (item[0], item[1]))
            discontinuous = any(ordered[index][0] != ordered[index - 1][1]
                                for index in range(1, len(ordered)))
            if ordered and n_frames is not None:
                discontinuous = (discontinuous or ordered[0][0] != 0 or
                                 ordered[-1][1] != int(n_frames))
            result = {'segment_ranges': segment_ranges, 'bridge_ranges': bridge_ranges}
            if discontinuous:
                result.update(extrema=False,
                              metadata_warning='segment frame ranges contain a gap or overlap')
            return result
        except (OSError, ValueError, TypeError):
            continue
    return {}

def _trajectory_semantics(sub=None, path='', n_frames=None):
    """Describe a trajectory without assigning unsupported reaction identities."""
    sub = str(sub or S.get('_last_subcmd') or S.get('subcmd') or '').lower()
    raw = str(path or '').replace('\\', '/').lower()
    name = os.path.basename(raw)
    if 'imag_' in name or 'mode_' in name or '/vib/' in raw:
        return {'title': 'Vibrational-mode trajectory', 'start': 'negative phase', 'end': 'positive phase',
                'x': 'phase frame', 'extrema': False}
    if 'scan' in name or sub in ('scan', 'scan2d', 'scan3d'):
        return {'title': 'Scan trajectory', 'start': 'scan start', 'end': 'scan end',
                'x': 'scan frame', 'extrema': False}
    if 'irc' in name or sub == 'irc':
        role = _irc_trajectory_role(path)
        if role == 'forward':
            return {'title': 'Forward IRC branch', 'start': 'near TS',
                    'end': 'forward endpoint', 'x': 'Forward IRC step', 'extrema': False}
        if role == 'backward':
            return {'title': 'Backward IRC branch', 'start': 'near TS',
                    'end': 'backward endpoint', 'x': 'Backward IRC step', 'extrema': False}
        semantics = {'title': 'IRC trajectory', 'start': 'IRC start',
                     'end': 'IRC end', 'x': 'IRC point', 'extrema': False}
        if role == 'finished':
            semantics.update(title='Combined IRC trajectory', start='forward endpoint',
                             end='backward endpoint')
            semantics.update(_trajectory_result_metadata(path, n_frames=n_frames))
            if semantics.get('n_forward') == 0 and semantics.get('n_backward', 0) > 0:
                semantics.update(title='Backward IRC trajectory', start='input TS')
            elif semantics.get('n_backward') == 0 and semantics.get('n_forward', 0) > 0:
                semantics.update(title='Forward IRC trajectory', end='input TS')
        return semantics
    if 'tsopt' in name or sub == 'tsopt':
        return {'title': 'TS-refinement trajectory', 'start': 'initial candidate', 'end': 'refined candidate',
                'x': 'optimization step', 'extrema': False}
    if ('mep' in name or 'path' in name or 'segment' in name or
            sub in ('all', 'path-opt', 'path-search')):
        semantics = {'title': 'Reaction-path trajectory', 'start': 'R', 'end': 'P',
                     'x': 'Path image', 'extrema': False}
        semantics.update(_trajectory_segment_ranges(path, n_frames=n_frames))
        return semantics
    if 'opt' in name or sub == 'opt':
        return {'title': 'Optimization trajectory', 'start': 'initial', 'end': 'optimized',
                'x': 'optimization step', 'extrema': False}
    return {'title': 'Trajectory', 'start': 'initial', 'end': 'final',
            'x': 'frame', 'extrema': False}

def _trajectory_frame_context(index, semantics=None):
    """Return compact current-range text plus scientific details for an info control."""
    semantics = semantics or _TRAJ.get('semantics') or {}
    label = ''
    for start, stop, kind, segment_index in (semantics.get('segment_ranges') or []):
        if int(start) <= int(index) < int(stop):
            kind = str(kind or '').lower()
            if kind == 'bridge':
                label = 'bridge'
            elif segment_index is not None:
                try: label = 'seg %02d' % int(segment_index)
                except (TypeError, ValueError): label = 'segment'
            else:
                label = kind or 'segment'
            break
    details = []
    status = semantics.get('trajectory_status')
    if status: details.append('Trajectory status: %s.' % status)
    reasons = semantics.get('scientific_status_reasons') or []
    if not isinstance(reasons, (list, tuple)): reasons = [reasons]
    if reasons:
        details.append('Details: %s.' % '; '.join(str(reason) for reason in reasons))
    warning = semantics.get('metadata_warning')
    if warning: details.append('Metadata: %s.' % warning)
    if label: details.insert(0, 'Current range: %s.' % label)
    return {'label': label, 'details': details}

def _stationary(ys, semantics=None):
    """Energy-only profile candidates; no frequency or IRC certification is implied."""
    semantics = semantics or _TRAJ.get('semantics') or _trajectory_semantics()
    n = len(ys)
    if not n: return []
    if n == 1: return [(0, 'only frame')]
    points = {0: semantics['start'], n - 1: semantics['end']}
    bridge_ranges = list(semantics.get('bridge_ranges') or [])
    def in_bridge(index):
        return any(int(start) <= index < int(stop)
                   for start, stop in bridge_ranges)
    # A validated TS supplied by result metadata is labelled TS. Otherwise an
    # interior MEP maximum is explicitly only a TS candidate; it must not be
    # presented as a certified transition state before TS optimization/IRC.
    try: ts_index = int(semantics.get('ts_index'))
    except (TypeError, ValueError): ts_index = -1
    if 0 <= ts_index < n:
        points[ts_index] = semantics.get('ts_label') or 'TS'
    elif str(semantics.get('title') or '').startswith('Reaction-path') and n > 2:
        ranges = [(int(start), int(stop)) for start, stop, kind, _index
                  in (semantics.get('segment_ranges') or []) if kind != 'bridge']
        if not ranges: ranges = [(0, n)]
        candidates = []
        for start, stop in ranges:
            interior = [index for index in range(max(1, start + 1), min(n - 1, stop - 1))
                        if not in_bridge(index) and ys[index] == ys[index]]
            if interior:
                candidate = max(interior, key=lambda index: ys[index])
                boundaries = [index for index in (max(0, start), min(n - 1, stop - 1))
                              if ys[index] == ys[index]]
                if boundaries and ys[candidate] <= max(ys[index] for index in boundaries):
                    continue
                if candidate not in candidates: candidates.append(candidate)
        for order, candidate in enumerate(candidates, 1):
            points[candidate] = ('TS candidate' if len(candidates) == 1
                                 else 'TS candidate %d' % order)
    return sorted(points.items())

def _energy_plot_document(energies, semantics, generation, linked=True):
    values = [None if value is None or not math.isfinite(float(value)) else float(value)
              for value in energies]
    finite_values = [value if value is not None else float('nan') for value in values]
    observed = [value for value in values if value is not None]
    if observed:
        y_min, y_max = min(observed), max(observed)
        y_span = y_max - y_min
        y_pad = max(1.0, y_span * 0.12, abs(y_max) * 0.02, abs(y_min) * 0.02)
        y_range = [y_min - y_pad, y_max + y_pad]
    else:
        y_range = [-1.0, 1.0]
    x_range = [0.5, max(1.5, len(values) + 0.5)]
    x_tick_step = 1 if len(values) <= 12 else (5 if len(values) <= 40 else 10)
    stationary = [{'index': int(index), 'label': str(label)}
                  for index, label in _stationary(finite_values, semantics)
                  if 0 <= int(index) < len(values) and values[int(index)] is not None]
    config = {
        'energies': values,
        'xTitle': str(semantics.get('x') or 'frame'),
        'xRange': x_range, 'yRange': y_range, 'xTickStep': x_tick_step,
        'bridges': [[int(start), int(stop)]
                    for start, stop in (semantics.get('bridge_ranges') or [])],
        'stationary': stationary,
        'generation': int(generation),
        'callback': 'pdb2reaction_gui.set_frame',
        'linked': bool(linked),
        'plotlySources': list(_PLOTLY_SCRIPT_URLS),
    }
    template = r"""<!doctype html>
<html><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<style>
html,body,#energy-plot{width:100%;height:100%;margin:0;overflow:hidden;background:#fff}
body{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Helvetica,Arial,sans-serif}
#plot-state{position:absolute;inset:0;display:flex;align-items:center;justify-content:center;
 color:#475569;background:#f8fafc;font-size:13px}
#plot-state.error{color:#991b1b;padding:16px;text-align:center;box-sizing:border-box}
</style></head><body>
<div id="energy-plot"></div><div id="plot-state" role="status">Loading energy profile…</div>
<script>
(async function(){
 'use strict';
 const cfg=__CONFIG__, host=document.getElementById('energy-plot');
 const state=document.getElementById('plot-state');
 function loadScript(url,timeoutMs){
   return new Promise((resolve,reject)=>{
     const script=document.createElement('script'); let settled=false;
     const finish=(error)=>{
       if(settled)return; settled=true; clearTimeout(timer);
       if(error){script.remove();reject(error);}else resolve();
     };
     const timer=setTimeout(()=>finish(new Error('timed out')),timeoutMs);
     script.src=url; script.async=true;
     script.onload=()=>finish(); script.onerror=()=>finish(new Error('failed'));
     document.head.appendChild(script);
   });
 }
 async function ensurePlotly(){
   if(window.Plotly)return;
   const errors=[];
   for(const url of cfg.plotlySources||[]){
     state.textContent='Preparing energy profile…';
     try{
       await loadScript(url,4500);
       if(window.Plotly)return;
       throw new Error('library unavailable');
     }catch(error){errors.push(url+': '+String(error&&error.message||error));}
   }
   throw new Error('Energy profile could not be loaded. Re-select MEP or IRC to retry.');
 }
 const xs=cfg.energies.map((_value,index)=>index+1);
 let current=0, pending=0, scheduled=false;
 function kernel(){
   const candidates=[window,window.parent,window.top];
   for(const candidate of candidates){
     try{
       if(candidate&&candidate.google&&candidate.google.colab&&candidate.google.colab.kernel)
         return {api:candidate.google.colab.kernel,scope:candidate};
     }catch(_error){}
   }
   return null;
 }
 function invoke(index){
   if(!cfg.linked)return Promise.resolve(null);
   const bridge=kernel();
   if(!bridge)return Promise.resolve(null);
   const kwargs=bridge.scope.JSON.parse('{}');
   return bridge.api.invokeFunction(cfg.callback,[cfg.generation,index],kwargs);
 }
 function broadcast(index){
   if(!cfg.linked)return;
   try{
     const hostDocument=window.parent&&window.parent.document;if(!hostDocument)return;
     const selector='iframe[data-rx-channel="trajectory"][data-rx-generation="'+String(cfg.generation)+'"]';
     const message={type:'rx-set-frame',generation:Number(cfg.generation),index:Number(index)};
     for(const target of hostDocument.querySelectorAll(selector)){
       if(target&&target.contentWindow&&target.contentWindow!==window)target.contentWindow.postMessage(message,'*');
     }
   }catch(_error){}
 }
 function cursorTrace(index){
   return {x:[xs[index]],y:[cfg.energies[index]],mode:'markers',hoverinfo:'skip',
     marker:{size:13,color:'#be185d',line:{width:2,color:'#fff'}},
     showlegend:false,name:'current'};
 }
 const shapes=cfg.bridges.map(pair=>({
   type:'rect',xref:'x',yref:'paper',x0:pair[0]+0.5,x1:pair[1]+0.5,y0:0,y1:1,
   fillcolor:'rgba(148,163,184,.18)',line:{width:0},layer:'below'
 }));
 const annotations=cfg.stationary.map(item=>({
   x:item.index+1,y:cfg.energies[item.index],text:'<b>'+item.label+'</b>',showarrow:false,
   xanchor:item.index===0?'left':(item.index===cfg.energies.length-1?'right':'center'),
   xshift:item.index===0?4:(item.index===cfg.energies.length-1?-4:0),
   yshift:16,font:{size:14,color:'#26364a'},bgcolor:'rgba(255,255,255,.82)',
    borderpad:2

 }));
 const traces=[
   {x:xs,y:cfg.energies,mode:'lines+markers',connectgaps:false,
     line:{color:'#0f766e',width:3,shape:'linear'},marker:{size:6,color:'#0f766e',
     line:{width:1,color:'#fff'}},name:'ΔE',
     hovertemplate:cfg.xTitle+' %{x}<br>ΔE %{y:.2f} kcal/mol<extra></extra>'},
   cursorTrace(0)
 ];
 const layout={margin:{l:92,r:34,t:40,b:64},paper_bgcolor:'#fff',plot_bgcolor:'#fbfdfd',
   font:{family:'-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,sans-serif',color:'#334155'},
   showlegend:false,hovermode:'closest',xaxis:{title:{text:cfg.xTitle,font:{size:18,color:'#253047'},standoff:12},tick0:0,dtick:cfg.xTickStep,
     range:cfg.xRange,autorange:false,
     showticklabels:true,ticks:'',ticklen:0,tickfont:{size:14},zeroline:false,gridcolor:'#eef2f7',linecolor:'#cbd5e1',automargin:true},
   yaxis:{title:{text:'Relative energy, ΔE (kcal mol⁻¹)',font:{size:18,color:'#253047'},standoff:14},range:cfg.yRange,autorange:false,gridcolor:'#e5eaf0',zerolinecolor:'#cbd5e1',linecolor:'#cbd5e1',tickfont:{size:14},automargin:true},
   shapes:shapes,annotations:annotations,uirevision:'rx-energy-'+cfg.generation};
 function applyFrame(index){
   current=Math.max(0,Math.min(cfg.energies.length-1,Number(index)||0));
   Plotly.restyle(host,{x:[[xs[current]]],y:[[cfg.energies[current]]]},[1]);
 }
 function queueFrame(index){
   pending=index;
   if(scheduled)return;
   scheduled=true;
   requestAnimationFrame(()=>{scheduled=false;applyFrame(pending);});
 }
 let draggingCursor=false, lastDragIndex=-1;
 function pointerIndex(event){
   const axis=host._fullLayout&&host._fullLayout.xaxis;
   if(!axis)return current;
   const rect=host.getBoundingClientRect();
   const pixel=event.clientX-rect.left-Number(axis._offset||0);
   const value=typeof axis.p2d==='function'?axis.p2d(pixel):xs[current];
   return Math.max(0,Math.min(cfg.energies.length-1,Math.round(Number(value))-1));
 }
 function pointerNearCursor(event){
   const layout=host._fullLayout, rect=host.getBoundingClientRect();
   const xa=layout&&layout.xaxis, ya=layout&&layout.yaxis;
   if(!xa||!ya||typeof xa.d2p!=='function'||typeof ya.d2p!=='function')return false;
   const cx=Number(xa._offset||0)+xa.d2p(xs[current]);
   const cy=Number(ya._offset||0)+ya.d2p(cfg.energies[current]);
   return Math.hypot(event.clientX-rect.left-cx,event.clientY-rect.top-cy)<=20;
 }
 function publishDraggedFrame(index){
   queueFrame(index); broadcast(index);
   if(index===lastDragIndex)return;
   lastDragIndex=index;
   invoke(index);
 }
 window.addEventListener('message',event=>{
   if(!cfg.linked)return;
   const message=event&&event.data;
   if(!message||message.type!=='rx-set-frame')return;
   if(Number(message.generation)!==Number(cfg.generation))return;
   queueFrame(message.index);
 });
 try{
   await ensurePlotly();
   await Plotly.newPlot(host,traces,layout,{responsive:true,displayModeBar:false});
   state.style.display='none';
   host.style.touchAction='none';
   host.addEventListener('pointerdown',event=>{
     if(!pointerNearCursor(event))return;
     draggingCursor=true;lastDragIndex=-1;host.style.cursor='grabbing';
     try{host.setPointerCapture(event.pointerId);}catch(_error){}
     event.preventDefault();event.stopPropagation();
   },true);
   host.addEventListener('pointermove',event=>{
     if(draggingCursor){
       publishDraggedFrame(pointerIndex(event));
       event.preventDefault();event.stopPropagation();
     }else{
       host.style.cursor=pointerNearCursor(event)?'grab':'';
     }
   },true);
   const finishDrag=event=>{
     if(!draggingCursor)return;
     publishDraggedFrame(pointerIndex(event));
     draggingCursor=false;host.style.cursor='';
     try{host.releasePointerCapture(event.pointerId);}catch(_error){}
     event.preventDefault();event.stopPropagation();
   };
   host.addEventListener('pointerup',finishDrag,true);
   host.addEventListener('pointercancel',finishDrag,true);
   host.on('plotly_click',event=>{
     const point=event&&event.points&&event.points[0];
     if(!point)return;
     const index=Math.max(0,Math.min(cfg.energies.length-1,Math.round(Number(point.x))-1));
     queueFrame(index); broadcast(index); invoke(index);
   });
 }catch(error){
   state.className='error';state.textContent=String(error&&error.message||error);
 }
})();
</script></body></html>"""
    return template.replace(
        '__CONFIG__', json.dumps(config, separators=(',', ':')).replace('<', '\\u003c'))

def _energy_plot_iframe(energies, semantics, generation, linked=True):
    document = _energy_plot_document(energies, semantics, generation, linked=linked)
    attrs = 'class="rxenergy-frame" title="Interactive energy profile"'
    if linked:
        attrs += ' data-rx-channel="trajectory" data-rx-generation="%d"' % int(generation)
    return _document_iframe(
        document, attrs,
        'display:block;width:100%;aspect-ratio:4/3;border:0;background:#fff;')

def _send_trajectory_frame(i):
    generation = int(_TRAJ.get('generation') or 0)
    message = {'type': 'rx-set-frame', 'generation': generation, 'index': int(i)}
    _TRAJ['frame_message'] = dict(message)
    encoded = json.dumps(message, separators=(',', ':')).replace('<', '\\u003c')
    structure_message = _TRAJ.get('structure_message') or None
    structure_encoded = (json.dumps(structure_message, separators=(',', ':')).replace('<', '\\u003c')
                         if structure_message else 'null')
    _TRAJ['structure_message'] = {}
    document = ("<!doctype html><html><body><script>(function(){"
                "const selector='iframe[data-rx-channel=\"trajectory\"]';"
                "const host=window.parent&&window.parent.document;"
                "if(!host)return;"
                "for(const target of host.querySelectorAll(selector)){"
                "if(!target||!target.contentWindow)continue;"
                "const load=%s;if(load){target.dataset.rxGeneration=String(load.generation);"
                "target.contentWindow.postMessage(load,'*');}"
                "target.contentWindow.postMessage(%s,'*');"
                "}"
                "})();</script></body></html>" % (structure_encoded, encoded))
    # HTML widget values inserted with innerHTML do not execute bare scripts.
    # A zero-size srcdoc iframe executes once and posts into both linked panes.
    traj_signal_out.value = _document_iframe(
        document, 'class="rxtrajectory-signal" aria-hidden="true"',
        'position:absolute;width:1px;height:1px;opacity:0;pointer-events:none;border:0;')
    try: traj_signal_out.send_state()
    except Exception: pass

def _set_frame_from_browser(generation, index):
    try:
        generation = int(generation); index = int(index)
    except (TypeError, ValueError):
        return
    if generation != int(_TRAJ.get('generation') or 0) or frame_slider.disabled:
        return
    if getattr(frame_play, '_playing', False):
        return
    target = max(frame_slider.min, min(frame_slider.max, index))
    if frame_slider.value != target:
        frame_slider.value = target
    elif _TRAJ.get('mode') == 'grid':
        _sync_axes = globals().get('_sync_scan_axis_controls')
        if callable(_sync_axes): _sync_axes(target)

try:
    from google.colab import output as _results_colab_output
    _results_colab_output.register_callback(
        'pdb2reaction_gui.set_frame', _set_frame_from_browser)
except Exception:
    _results_colab_output = None

def _show_frame(i):
    fr = _TRAJ['frames']
    if not fr: return
    i = max(0, min(i, len(fr) - 1))
    path_name = os.path.basename(str(_TRAJ.get('path') or '')).lower()
    if _TRAJ.get('mode') == 'levels': position_name = 'State'
    elif _TRAJ.get('mode') == 'grid': position_name = 'Grid point'
    elif 'irc' in path_name: position_name = 'IRC point'
    elif 'mep' in path_name: position_name = 'Image'
    else: position_name = 'Frame'
    quantity_label = str(_TRAJ.get('quantity_label') or 'ΔE')
    if _TRAJ.get('mode') == 'grid':
        _sync_axes = globals().get('_sync_scan_axis_controls')
        if callable(_sync_axes): _sync_axes(i)
    if _UPLOAD_MODE != 'colab':
        _send_trajectory_frame(i)
    rk = _rel_kcal()
    semantics = _TRAJ.get('semantics') or _trajectory_semantics(
        path=_TRAJ.get('path'), n_frames=len(fr))
    bridge_ranges = list(semantics.get('bridge_ranges') or [])
    bridge_frame = any(int(start) <= i < int(stop)
                       for start, stop in bridge_ranges)
    suffix = (' · bridge (stitching)' if bridge_frame else '')
    context = _trajectory_frame_context(i, semantics)
    range_label = context['label']
    details = context['details']
    trajectory_info.layout.display = '' if details else 'none'
    if details: _set_info_text(trajectory_info, ' '.join(details))
    if rk is None:
        later = any(value is not None for value in _TRAJ.get('energies', [])[1:])
        reason = ('frame 1 energy unavailable; profile not re-referenced'
                  if later else 'per-frame energies unavailable')
        label_text = (' · ' + range_label) if range_label else ''
        frame_state.value = ('<div role="status" aria-live="polite" aria-atomic="true">'
                             '<b>%s %d of %d</b>%s · %s%s</div>' %
                             (position_name, i + 1, len(fr), html.escape(label_text),
                              reason, html.escape(suffix)))
        return
    ys = [value if value is not None else float('nan') for value in rk]
    stationary = ('bridge · non-reactive stitching' if bridge_frame
                  else dict(_stationary(ys, semantics)).get(i, ''))
    state_labels = list(_TRAJ.get('state_labels') or [])
    if i < len(state_labels): stationary = state_labels[i]
    labels = []
    for value in (range_label, stationary):
        if value and value not in labels: labels.append(value)
    current_label = ' · '.join(labels)
    raw_energies = list(_TRAJ.get('energies') or [])
    if len(fr) == 1 and i < len(raw_energies) and raw_energies[i] is not None:
        energy_text = 'Energy = %.8f Ha' % float(raw_energies[i])
    else:
        energy_text = (('%s = %.1f kcal/mol' % (quantity_label, ys[i]))
                       if ys[i] == ys[i] else '%s unavailable' % quantity_label)
    label_text = (' · ' + current_label) if current_label else ''
    frame_state.value = ('<div role="status" aria-live="polite" aria-atomic="true">'
                         '<b>%s %d of %d</b>%s · %s%s</div>' %
                         (position_name, i + 1, len(rk), html.escape(label_text),
                          energy_text, html.escape(suffix)))

def _on_frame_change(change):
    frame_prev.disabled = frame_slider.disabled or frame_slider.value <= frame_slider.min
    frame_next.disabled = frame_slider.disabled or frame_slider.value >= frame_slider.max
    _show_frame(int(frame_slider.value))

frame_slider.observe(_on_frame_change, names='value')

def _summary_html(summary_path=None):
    if not summary_path or not os.path.exists(summary_path): return ''
    try:
        with open(summary_path, encoding='utf-8') as fh: data = json.load(fh)
    except (OSError, ValueError) as exc:
        return '<div role="alert" style="color:#991b1b">Could not parse <code>%s</code>: %s</div>' % (html.escape(os.path.basename(summary_path)), html.escape(str(exc)))
    if not isinstance(data, dict):
        return '<div role="alert" style="color:#991b1b">Summary root must be a JSON object.</div>'
    def val(value):
        try:
            number = float(value)
            return '%.1f' % number if math.isfinite(number) else '—'
        except (TypeError, ValueError): return '—'
    scientific = str(data.get('scientific_status') or '').lower()
    complete = scientific in ('success', 'complete', 'pass', 'passed')
    failed = scientific in ('failed', 'error')
    partial = bool(scientific) and not complete and not failed
    # The Results content already conveys completion. Keep only an actual
    # failure badge; partial scientific details remain available on demand.
    badge_text = scientific if failed else ''
    status_html = (('<span style="display:inline-block;background:#991b1b;color:white;border-radius:999px;'
                    'padding:2px 9px;margin-bottom:6px">%s</span>' %
                    html.escape(badge_text)) if badge_text else '')
    reasons = data.get('scientific_status_reasons') or []
    if not isinstance(reasons, (list, tuple)): reasons = [reasons]
    if reasons:
        status_html += (' <details style="display:inline-block"><summary>status details</summary><ul>%s</ul></details>' %
                        ''.join('<li>%s</li>' % html.escape(str(reason)) for reason in reasons))
    ts_only = str(data.get('pipeline_mode') or '').lower() == 'tsopt-only'
    segments = data.get('segments')
    if not isinstance(segments, list):
        metric_labels = (
            ('n_cycles', 'cycles'), ('cycles', 'cycles'),
            ('n_modes', 'modes'), ('n_imaginary', 'imaginary modes'),
            ('n_frames_total', 'frames'), ('n_frames', 'frames'),
            ('n_points', 'points'), ('energy_ha', 'energy (Ha)'),
            ('energy_au', 'energy (Ha)'), ('energy_hartree', 'energy (Ha)'),
            ('final_energy_hartree', 'energy (Ha)'),
        )
        metrics = []
        seen_labels = set()
        for key, label in metric_labels:
            if key in data and data.get(key) is not None and label not in seen_labels:
                metrics.append('%s: <b>%s</b>' %
                               (html.escape(label), html.escape(str(data[key]))))
                seen_labels.add(label)
        return status_html + (
            '<br><small>%s</small>' % ' · '.join(metrics) if metrics else '')
    reactive = [segment for segment in segments
                if isinstance(segment, dict) and str(segment.get('kind') or '').lower() != 'bridge']
    post = {row.get('index'): row for row in (data.get('post_segments') or [])
            if isinstance(row, dict)}
    highest = data.get('rate_limiting_step') or {}
    if not isinstance(highest, dict): highest = {}
    raw_method = str(highest.get('method') or ('MLIP' if ts_only else 'MEP'))
    method = {
        'mep': 'MEP', 'mlip': 'MLIP', 'mlip_gibbs': 'MLIP_Gibbs',
        'dft': 'DFT',
        'dft//mlip_gibbs': 'DFT//MLIP_Gibbs',
    }.get(raw_method.lower(), raw_method)
    method_key = {
        'DFT//MLIP_Gibbs': 'gibbs_dft_mlip',
        'DFT//MLIP/MM_Gibbs': 'gibbs_dft_mlip', 'DFT': 'dft',
        'MLIP_Gibbs': 'gibbs_mlip', 'MLIP': 'mlip',
    }.get(method)
    display_method = method
    gibbs_method = bool(method_key and 'gibbs' in method_key)
    barrier_head = 'ΔG‡' if gibbs_method else 'ΔE‡'
    delta_head = 'ΔG' if gibbs_method else 'ΔE'
    rows = ''
    for position, segment in enumerate(reactive):
        index = segment.get('index', position + 1)
        selected = (post.get(index) or {}).get(method_key) if method_key else segment
        if (not isinstance(selected, dict) or not selected) and ts_only and method == 'MLIP':
            selected = segment
        selected = selected if isinstance(selected, dict) else {}
        imag = ((post.get(index) or {}).get('ts_imag') or {}).get('n_imag')
        if ts_only:
            rows += ('<tr><td>seg %02d</td><td>%s</td><td align=right>%s</td>'
                     '<td align=right>%s</td><td align=right>%s</td></tr>') % (
                         index, html.escape(display_method), val(selected.get('barrier_kcal')),
                         val(selected.get('delta_kcal')),
                         '—' if imag is None else html.escape(str(imag)))
        else:
            rows += ('<tr><td>seg %02d</td><td>%s</td><td align=right>%s</td>'
                     '<td align=right>%s</td><td align=right>%s</td><td align=right>%s</td></tr>') % (
                         index, html.escape(display_method), val(selected.get('barrier_kcal')),
                         val(selected.get('delta_kcal')), val(segment.get('barrier_kcal')),
                         '—' if imag is None else html.escape(str(imag)))
    raw_head = '' if ts_only else '<th scope="col">raw MEP ΔE‡</th>'
    table = ('<table style="border-collapse:collapse;" border=1 cellpadding=5>'
             '<caption style="text-align:left;font-weight:600">Segments · one common method (kcal/mol)</caption>'
             '<tr><th scope="col">segment</th><th scope="col">method</th>'
             '<th scope="col">%s</th><th scope="col">%s</th>%s'
             '<th scope="col">n<sub>imag</sub></th></tr>%s</table>' %
             (barrier_head, delta_head, raw_head, rows))
    result = status_html
    if rows:
        result += '<div style="max-width:100%%;overflow-x:auto">%s</div>' % table
    highest_value = highest.get('barrier_kcal')
    if highest_value is not None and not failed:
        result += ('<br><b style="font-size:15px;">Highest local barrier: %s kcal/mol</b> '
                   '<small>(%s%s)</small>' %
                   (val(highest_value), html.escape(display_method), ''))
    if ts_only:
        assignment = data.get('endpoint_assignment') or {}
        if isinstance(assignment, dict) and assignment.get('chemical_direction_known') is False:
            result += ('<br><small>R/P labels use endpoint energy order; '
                       'chemical direction is unassigned.</small>')
    footer = []
    if data.get('status') is not None:
        footer.append('status: %s' % html.escape(str(data['status'])))
    count_label = 'IRC frames' if ts_only else 'images'
    if data.get('n_images') is not None:
        footer.append('%s: %s' %
                      (count_label, html.escape(str(data['n_images']))))
    backend = data.get('mlip_backend')
    model = data.get('mlip_model')
    if backend is not None or model is not None:
        footer.append('backend/model: %s / %s' %
                      (html.escape(str(backend or '—')),
                       html.escape(str(model or '—'))))
    if footer:
        result += '<br><small>%s</small>' % ' · '.join(footer)
    return result

def _current_run_files(out):
    """Return only files changed by the last GUI-launched run for this root."""
    if not S.get('_last_out_dir') or os.path.abspath(out) != os.path.abspath(S['_last_out_dir']):
        return []
    return sorted(path for path in S.get('_last_files', []) if os.path.isfile(path))

def _select_status_json(current, sub):
    """Prefer the aggregate summary for composite workflows, otherwise a leaf result."""
    order = ('summary.json', 'result.json') if sub in ('all', 'path-search') else ('result.json', 'summary.json')
    for name in order:
        candidates = sorted((p for p in current if os.path.basename(p) == name),
                            key=lambda p: (p.count(os.sep), p))
        if candidates: return candidates[0]
    return None

def _result_context_html(out):
    sub = S.get('_last_subcmd') or S.get('subcmd') or 'all'
    current = _current_run_files(out)
    status_words, detail_bits = [], []
    def add_status(value):
        token = str(value or '').strip().lower()
        if not token: return
        label = {'success': 'Completed', 'complete': 'Completed', 'completed': 'Completed',
                 'pass': 'Completed', 'passed': 'Completed', 'converged': 'Converged',
                 'failed': 'Failed', 'failure': 'Failed', 'error': 'Failed',
                 'cancelled': 'Cancelled', 'canceled': 'Cancelled',
                 'partial': 'Partial'}.get(token, token.replace('_', ' ').capitalize())
        if label not in status_words: status_words.append(label)
    status_json = _select_status_json(current, sub)
    if status_json:
        try:
            with open(status_json, encoding='utf-8') as fh: data = json.load(fh)
            if not isinstance(data, dict): raise ValueError('status JSON root is not an object')
            aggregate = os.path.basename(status_json) == 'summary.json'
            keys = (('execution_status', 'status') if aggregate else
                    ('execution_status', 'scientific_status', 'status', 'converged'))
            for key in keys:
                if key in data:
                    value = data[key]
                    if key == 'converged':
                        converged = value is True or str(value).strip().lower() in ('true', 'yes', '1', 'converged')
                        add_status('converged' if converged else 'not converged')
                    else: add_status(value)
                    detail_bits.append('%s: <b>%s</b>' % (key, html.escape(str(value))))
            if not aggregate:
                n_imag = next((data.get(key) for key in
                               ('n_imaginary', 'n_imaginary_modes', 'n_imag')
                               if data.get(key) is not None), None)
                if n_imag is not None:
                    detail_bits.append('imaginary modes: <b>%s</b>' % html.escape(str(n_imag)))
                frequencies = data.get('imaginary_frequencies_cm')
                if isinstance(frequencies, (list, tuple)) and frequencies:
                    shown_freqs = ', '.join('%.1f' % float(value) for value in frequencies[:3])
                    if len(frequencies) > 3: shown_freqs += ', …'
                    detail_bits.append('imaginary ν: <b>%s cm⁻¹</b>' % html.escape(shown_freqs))
                reasons = data.get('scientific_status_reasons') or []
                if reasons:
                    if not isinstance(reasons, (list, tuple)): reasons = [reasons]
                    detail_bits.extend(html.escape(str(reason)) for reason in reasons)
        except Exception:
            detail_bits.append('%s could not be parsed' % os.path.basename(status_json))
    manifest = S.get('_last_manifest') or {}
    if manifest:
        add_status(manifest.get('status') or 'recorded')
        if manifest.get('exit_code') is not None:
            detail_bits.append('exit code: <b>%s</b>' % html.escape(str(manifest['exit_code'])))
    rels = [os.path.relpath(path, out) for path in current]
    stdout_only = bool((S.get('_last_manifest') or {}).get('stdout_only'))
    if stdout_only:
        artifacts = 'standard output'
    elif rels:
        artifacts = '<b>%d generated file%s</b>' % (len(rels), '' if len(rels) == 1 else 's')
        detail_bits.extend('<code>%s</code>' % html.escape(item) for item in rels[:5])
        if len(rels) > 5: detail_bits.append('and %d more files' % (len(rels) - 5))
    else:
        artifacts = 'no files recorded'
    details = ('<details style="margin-top:5px"><summary>Run details</summary><ul>%s</ul></details>' %
               ''.join('<li>%s</li>' % item for item in detail_bits)) if detail_bits else ''
    return ('<div style="border:1px solid #dbeafe;border-radius:11px;padding:7px 9px;background:#f8fbff;">'
            '<b>%s</b> · <span style="font-weight:700;color:#334155">%s</span> · '
            '<small>%s</small>%s</div>' %
            (html.escape(sub), ' · '.join(status_words) or 'Status unavailable', artifacts, details))

_ARTIFACT_KINDS = {'.png': 'image', '.jpg': 'image', '.jpeg': 'image',
                   '.svg': 'SVG', '.html': 'interactive HTML',
                   '.csv': 'CSV table', '.pdf': 'PDF', '.json': 'JSON',
                   '.yaml': 'YAML', '.yml': 'YAML', '.txt': 'text',
                   '.log': 'text', '.out': 'text', '.md': 'text',
                   '.gjf': 'text', '.com': 'text', '.inp': 'text', '.prms': 'text',
                   '.pdb': 'structure', '.ent': 'structure', '.cif': 'structure',
                   '.mmcif': 'structure'}
_TEXT_PREVIEW_LIMIT = 512 * 1024
_STRUCTURE_PREVIEW_LIMIT = 5 * 1024 * 1024
_HTML_PREVIEW_LIMIT = 5 * 1024 * 1024
_IMAGE_PREVIEW_LIMIT = 12 * 1024 * 1024
_TRAJECTORY_PREVIEW_LIMIT = 24 * 1024 * 1024
_TRAJECTORY_FRAME_LIMIT = 10000
_TRAJECTORY_ATOM_FRAME_LIMIT = 2000000

def _artifact_kind(path):
    path = Path(path); name = path.name.lower(); suffix = path.suffix.lower()
    if suffix == '.xyz':
        if 'trj' not in name and 'trajectory' not in name: return 'structure'
        if 'mep' in name or 'final_geometries' in name: return 'MEP profile'
        if 'irc' in name: return 'IRC trajectory'
        if 'mode_' in name or 'imag_' in name or 'imaginary_' in name: return 'vibrational mode'
        return None
    if suffix in ('.pdb', '.ent', '.cif', '.mmcif') and (
            'mode_' in name or 'imag_' in name or 'imaginary_' in name):
        return 'frequency structure'
    return _ARTIFACT_KINDS.get(suffix)

def _csv_preview_html(path, max_rows=50, max_cols=20):
    rows = []
    with open(path, newline='', encoding='utf-8', errors='replace') as fh:
        reader = csv.reader(fh)
        for index, row in enumerate(reader):
            if index > max_rows: break
            rows.append(row[:max_cols])
    if not rows: return '<i>empty CSV</i>'
    head, body = rows[0], rows[1:max_rows + 1]
    table = '<table border="1" cellpadding="4" style="border-collapse:collapse;max-width:100%;">'
    table += '<thead><tr>%s</tr></thead>' % ''.join('<th>%s</th>' % html.escape(cell) for cell in head)
    table += '<tbody>%s</tbody></table>' % ''.join(
        '<tr>%s</tr>' % ''.join('<td>%s</td>' % html.escape(cell) for cell in row) for row in body)
    if len(rows) > max_rows: table += '<small>Preview truncated after %d rows.</small>' % max_rows
    return '<div style="max-width:100%%;overflow-x:auto">%s</div>' % table

def _text_preview_html(path, kind):
    size = os.path.getsize(path)
    with open(path, encoding='utf-8', errors='replace') as fh:
        source = fh.read(_TEXT_PREVIEW_LIMIT + 1)
    truncated = len(source) > _TEXT_PREVIEW_LIMIT or size > _TEXT_PREVIEW_LIMIT
    source = source[:_TEXT_PREVIEW_LIMIT]
    if kind == 'JSON':
        try: source = json.dumps(json.loads(source), indent=2, ensure_ascii=False)
        except (ValueError, TypeError): pass
    note = ('<small>Preview truncated at 512 KiB; download the result for the complete file.</small>'
            if truncated else '')
    return ('<pre style="max-height:520px;max-width:100%%;overflow:auto;white-space:pre-wrap;'
            'word-break:break-word;background:#0f172a;color:#e2e8f0;padding:10px;border-radius:8px;">%s</pre>%s'
            % (html.escape(source), note))

def _structure_preview(path):
    """Show one bounded molecular structure without treating trajectories as files."""
    size = os.path.getsize(path)
    if size > _STRUCTURE_PREVIEW_LIMIT:
        print('Structure preview skipped (%.1f MiB); download the result for the complete file.' %
              (size / 1048576.0))
        return
    suffix = Path(path).suffix.lower()
    fmt = ('pdb' if suffix in ('.pdb', '.ent') else
           ('mmcif' if suffix in ('.cif', '.mmcif') else 'xyz'))
    with open(path, encoding='utf-8', errors='replace') as fh: source = fh.read()
    display(HTML(_molstar_iframe(source, fmt, show_sequence=(fmt != 'xyz'))))

def _responsive_artifact_document(source):
    """Fit standalone Plotly HTML into its result card without clipping."""
    bridge = r'''<style id="rx-artifact-fit">
html,body{width:100%!important;max-width:100%!important;margin:0!important;box-sizing:border-box!important;}
body{overflow-x:hidden!important;overflow-y:hidden!important;}
body>div,.plotly-graph-div,.js-plotly-plot,.plot-container,.svg-container{width:100%!important;max-width:100%!important;min-width:0!important;box-sizing:border-box!important;overflow:hidden!important;}
</style><script>(function(){
function fit(){try{var height=Math.max(320,Math.min(500,window.innerHeight-16));
document.querySelectorAll('.js-plotly-plot').forEach(function(plot){
plot.style.width='100%';plot.style.maxWidth='100%';plot.style.minWidth='0';plot.style.height=height+'px';
if(window.Plotly){try{Plotly.relayout(plot,{autosize:true,height:height});}catch(_error){}if(Plotly.Plots)Plotly.Plots.resize(plot);}});}catch(_error){}}
addEventListener('load',function(){fit();setTimeout(fit,80);});
addEventListener('resize',fit);
if(window.ResizeObserver){new ResizeObserver(function(){requestAnimationFrame(fit);}).observe(document.documentElement);}
setTimeout(fit,240);
})();</script>'''
    lower = source.lower(); marker = lower.rfind('</head>')
    return (source[:marker] + bridge + source[marker:]) if marker >= 0 else (bridge + source)

def _artifact_preview_html(path, out):
    if not path or not os.path.isfile(path): return ''
    rel = os.path.relpath(path, out)
    header = '<div style="margin:0 0 8px"><b>Displayed artifact:</b> <code>%s</code></div>' % html.escape(rel)
    kind = _artifact_kind(path)
    try:
        size = os.path.getsize(path)
        if kind in ('MEP profile', 'IRC trajectory', 'vibrational mode'):
            if size > _TRAJECTORY_PREVIEW_LIMIT:
                return header + '<div role="status">Trajectory preview skipped (%.1f MiB); download the result for the complete file.</div>' % (size / 1048576.0)
            parsed = _parse_trj(path); frames = parsed.get('frames') or []
            if not frames:
                return header + '<div role="status">No coordinate frame was found in this trajectory.</div>'
            if len(frames) > _TRAJECTORY_FRAME_LIMIT:
                return header + '<div role="status">Trajectory preview skipped (%d frames); download the result for the complete file.</div>' % len(frames)
            energies = list(parsed.get('energies_ha') or [])
            note = ('<div style="margin:0 0 8px;color:#475569"><b>%d frames</b> · '
                    '%s. Use the Results selector for playback.</div>' %
                    (len(frames), 'vibrational mode' if kind == 'vibrational mode' else 'reaction path'))
            structure_preview = note + _molstar_iframe(
                ''.join(frames), 'xyz', show_sequence=False, channel='artifact',
                generation=_TRAJ_VIEWER_GENERATION['value'])
            if kind == 'MEP profile' and energies and energies[0] is not None:
                base = float(energies[0])
                relative = [((float(value) - base) * 627.5094740631)
                            if value is not None else None for value in energies]
                semantics = _trajectory_semantics(
                    S.get('_last_subcmd'), path, n_frames=len(frames))
                plot_preview = ('<div style="margin:16px 0 8px;color:#334155">'
                                '<b>Energy profile</b></div>' +
                                _energy_plot_iframe(
                                    relative, semantics,
                                    _TRAJ_VIEWER_GENERATION['value'], linked=False))
                return header + structure_preview + plot_preview
            return header + structure_preview
        if kind == 'image':
            if size > _IMAGE_PREVIEW_LIMIT:
                return header + '<div role="status">Image preview skipped (%.1f MiB); download the result for the complete file.</div>' % (size / 1048576.0)
            suffix = Path(path).suffix.lower()
            mime = {'.png': 'image/png', '.jpg': 'image/jpeg', '.jpeg': 'image/jpeg',
                    '.gif': 'image/gif', '.webp': 'image/webp'}.get(suffix, 'application/octet-stream')
            with open(path, 'rb') as fh: data = base64.b64encode(fh.read()).decode('ascii')
            return (header + '<div class="rxartifact-image-wrap">'
                    '<img class="rxartifact-image" alt="%s" src="data:%s;base64,%s"></div>' %
                    (html.escape(os.path.basename(path), quote=True), mime, data))
        if kind in ('structure', 'frequency structure'):
            if size > _STRUCTURE_PREVIEW_LIMIT:
                return header + '<div role="status">Structure preview skipped (%.1f MiB); download the result for the complete file.</div>' % (size / 1048576.0)
            suffix = Path(path).suffix.lower()
            fmt = ('pdb' if suffix in ('.pdb', '.ent') else
                   ('mmcif' if suffix in ('.cif', '.mmcif') else 'xyz'))
            with open(path, encoding='utf-8', errors='replace') as fh: source = fh.read()
            return header + _molstar_iframe(source, fmt, show_sequence=(fmt != 'xyz'))
        if kind in ('SVG', 'interactive HTML'):
            if size > _HTML_PREVIEW_LIMIT:
                return header + '<div role="status">%s preview skipped (%.1f MiB); download the result for the complete file.</div>' % (kind, size / 1048576.0)
            with open(path, encoding='utf-8', errors='replace') as fh: source = fh.read(_HTML_PREVIEW_LIMIT + 1)
            if kind == 'interactive HTML': source = _responsive_artifact_document(source)
            title = '%s: %s' % (kind, os.path.basename(path))
            frame = _plotly_document_iframe if kind == 'interactive HTML' else _document_iframe
            return header + frame(
                source, 'sandbox="allow-scripts allow-same-origin" title="%s"' % html.escape(title, quote=True),
                'display:block;width:100%;max-width:100%;min-width:0;height:auto;min-height:320px;max-height:520px;aspect-ratio:16/10;overflow:hidden;border:1px solid #e6eaf0;border-radius:10px;box-sizing:border-box;')
        if kind == 'CSV table': return header + _csv_preview_html(path)
        if kind in ('JSON', 'YAML', 'text'): return header + _text_preview_html(path, kind)
        if kind == 'PDF':
            if size > 5 * 1024 * 1024:
                return header + '<div role="status">PDF preview skipped (%.1f MiB); use the results download.</div>' % (size / 1048576.0)
            with open(path, 'rb') as fh: data = base64.b64encode(fh.read()).decode('ascii')
            return header + _binary_iframe(
                'application/pdf', data, 'title="PDF result"',
                'width:100%;height:560px;border:1px solid #e6eaf0;border-radius:10px;')
        return header + '<div role="status">No inline preview is available for this file type.</div>'
    except (OSError, ValueError, RuntimeError) as exc:
        return header + '<div role="alert" style="color:#991b1b">Could not embed artifact: %s</div>' % html.escape(str(exc))

def _render_artifact(_=None):
    if _result_pick_guard['active']: return
    if ('artifact_fold' in globals() and
            artifact_fold._rx_body.layout.display == 'none'):
        return
    out = S.get('_last_out_dir') or _effective_result_root()
    artifact_out.value = _artifact_preview_html(artifact_choice.value, out)

def _energy_diagram_kind(path):
    """Return the final-result energy family represented by an output image."""
    name = os.path.basename(str(path)).lower()
    if 'irc_plot' in name:
        return 'irc'
    if 'energy_diagram_' not in name:
        return None
    if 'g_dft_plus_mlip' in name or 'gibbs_dft' in name:
        return 'dft_g'
    if 'g_mlip' in name or 'gibbs_mlip' in name:
        return 'mlip_g'
    if 'dft' in name:
        return 'dft_e'
    if 'mlip' in name or 'uma' in name:
        return 'mlip_e'
    if 'mep' in name:
        return 'mep'
    return None

def _energy_payload_kind(name):
    token = str(name or '').lower().replace('-', '_')
    if 'g_dft_plus_mlip' in token or 'gibbs_dft' in token or 'dft//mlip' in token and 'gibbs' in token:
        return 'dft_g'
    if 'g_mlip' in token or 'gibbs_mlip' in token or ('mlip' in token and 'gibbs' in token):
        return 'mlip_g'
    if 'dft' in token:
        return 'dft_e'
    if 'mlip' in token or 'uma' in token:
        return 'mlip_e'
    if 'mep' in token:
        return 'mep'
    return None

def _normalise_energy_payload(payload, current=(), out=None):
    if not isinstance(payload, dict):
        return None
    labels = payload.get('labels')
    energies = payload.get('energies_kcal')
    if not isinstance(labels, (list, tuple)) or not isinstance(energies, (list, tuple)):
        return None
    if len(labels) != len(energies) or not labels:
        return None
    try:
        values = [float(value) for value in energies]
    except (TypeError, ValueError):
        return None
    if not all(math.isfinite(value) for value in values):
        return None
    view = {'mode': 'levels', 'labels': [str(label) for label in labels],
            'energies_kcal': values, 'ylabel': str(payload.get('ylabel') or 'Relative energy (kcal/mol)')}
    raw_structures = payload.get('structures')
    if isinstance(raw_structures, dict):
        def declared(label):
            if label in raw_structures: return raw_structures[label]
            upper = str(label).upper()
            return raw_structures.get('TS') if upper.startswith('TS') else raw_structures.get(upper)
        raw_structures = [declared(label) for label in labels]
    if isinstance(raw_structures, (list, tuple)) and len(raw_structures) == len(labels):
        resolved = []
        for raw in raw_structures:
            raw = str(raw or '')
            candidates = [raw] if os.path.isabs(raw) else [os.path.join(str(out or ''), raw)]
            normal = raw.replace('\\', '/')
            if '/segments/' in normal and out:
                candidates.append(os.path.join(str(out), 'segments', normal.split('/segments/', 1)[1]))
            path = next((os.path.abspath(candidate) for candidate in candidates if os.path.isfile(candidate)), None)
            if path is None:
                suffix = normal.rsplit('/segments/', 1)[-1] if '/segments/' in normal else normal
                matches = [candidate for candidate in current
                           if str(candidate).replace('\\', '/').endswith(suffix)]
                path = matches[0] if len(matches) == 1 else None
            resolved.append(path)
        if all(resolved): view['structures'] = resolved
    return view

def _energy_summary_views(current, out):
    """Read final energy values from the canonical summary, even without PNG export."""
    ranked = {}
    summaries = sorted((path for path in current if os.path.basename(path) in ('summary.json', 'result.json')),
                       key=lambda path: (0 if os.path.basename(path) == 'summary.json' else 1,
                                         len(Path(os.path.relpath(path, out)).parts), path))
    def offer(kind, payload, rank):
        view = _normalise_energy_payload(payload, current=current, out=out)
        if kind and view and (kind not in ranked or rank < ranked[kind][0]):
            ranked[kind] = (rank, view)
    for source_rank, summary_path in enumerate(summaries):
        try:
            with open(summary_path, encoding='utf-8') as fh: data = json.load(fh)
        except (OSError, ValueError, TypeError):
            continue
        if not isinstance(data, dict):
            continue
        post_segments = [item for item in (data.get('post_segments') or [])
                         if isinstance(item, dict)]
        is_root = len(Path(os.path.relpath(summary_path, out)).parts) == 1
        for item_rank, payload in enumerate(data.get('energy_diagrams') or []):
            if not isinstance(payload, dict): continue
            name = payload.get('name') or payload.get('title')
            kind = _energy_payload_kind(name)
            aggregate = '_all' in str(name or '').lower()
            if aggregate:
                offer(kind, payload, (source_rank, 0, item_rank))
            elif is_root and len(post_segments) <= 1:
                offer(kind, payload, (source_rank, 1, item_rank))
        if is_root and len(post_segments) == 1:
            segment = post_segments[0]
            for field_rank, (field, kind) in enumerate((
                    ('gibbs_dft_mlip', 'dft_g'), ('gibbs_mlip', 'mlip_g'),
                    ('dft', 'dft_e'), ('mlip', 'mlip_e'))):
                offer(kind, segment.get(field), (source_rank, 2, field_rank))
    return {kind: view for kind, (_, view) in ranked.items()}

def _stationary_structure_series(current, out):
    """Return canonical stationary-point structures in diagram order."""
    groups = {}
    roles = {'reactant': 'R', 'ts': 'TS', 'product': 'P'}
    for path in current:
        suffix = Path(path).suffix.lower()
        if suffix not in ('.xyz', '.pdb', '.cif', '.mmcif', '.gjf', '.com'):
            continue
        role = roles.get(Path(path).stem.lower())
        if role is None:
            continue
        try:
            parts = Path(os.path.relpath(path, out)).parts
        except ValueError:
            continue
        if len(parts) < 3 or parts[0] != 'segments' or not parts[1].startswith('seg_'):
            continue
        try: segment = int(parts[1][4:])
        except ValueError: continue
        # Prefer the canonical public XYZ, then the lossless working XYZ,
        # before coordinate-only public structure formats.
        rank = (0 if len(parts) == 3 and suffix == '.xyz' else
                1 if suffix == '.xyz' else
                2 if len(parts) == 3 else 3)
        previous = groups.setdefault(segment, {}).get(role)
        if previous is None or rank < previous[0]:
            groups[segment][role] = (rank, path)
    complete = [(segment, groups[segment]) for segment in sorted(groups)
                if all(role in groups[segment] for role in ('R', 'TS', 'P'))]
    paths = []
    for _, group in complete:
        paths.extend(group[role][1] for role in ('R', 'TS', 'P'))
    return paths

def _result_segment_number(path, out):
    try: parts = Path(os.path.relpath(path, out)).parts
    except ValueError: parts = ()
    for part in parts:
        match = re.match(r'^seg(?:ment)?[_-]?(\d+)', part, re.IGNORECASE)
        if match: return int(match.group(1))
    match = re.search(r'(?:mep_)?seg[_-]?(\d+)', os.path.basename(str(path)), re.IGNORECASE)
    return int(match.group(1)) if match else None

def _aggregate_irc_trajectory(current, out):
    """Return one whole-reaction IRC trajectory for the linked Results view."""
    def depth(path):
        try: return len(Path(os.path.relpath(path, out)).parts)
        except ValueError: return 999
    root = [path for path in current
            if depth(path) == 1 and _irc_trajectory_role(path) == 'finished']
    if root: return min(root)
    chosen = {}
    for path in current:
        segment = _result_segment_number(path, out)
        if segment is None or _irc_trajectory_role(path) != 'finished': continue
        candidate = (depth(path), path)
        if segment not in chosen or candidate < chosen[segment]: chosen[segment] = candidate
    if not chosen: return None
    reverse = {}
    summaries = [path for path in current
                 if depth(path) == 1 and os.path.basename(path) == 'summary.json']
    if summaries:
        try:
            with open(min(summaries), encoding='utf-8') as fh: payload = json.load(fh)
            for item in payload.get('post_segments') or []:
                if not isinstance(item, dict): continue
                assignment = item.get('endpoint_assignment') or {}
                reverse[int(item.get('index'))] = bool(assignment.get('reversed'))
        except (OSError, ValueError, TypeError, KeyError):
            reverse = {}
    frames = []
    for segment in sorted(chosen):
        try: part = list(_parse_trj(chosen[segment][1])['frames'])
        except (OSError, ValueError, RuntimeError, KeyError): continue
        if reverse.get(segment): part.reverse()
        frames.extend(part)
    if not frames: return None
    cache = _RUNTIME_DIR / 'results'
    cache.mkdir(parents=True, exist_ok=True)
    key = hashlib.sha256(os.path.abspath(out).encode()).hexdigest()[:12]
    target = cache / ('aggregate_irc_%s_finished_irc_trj.xyz' % key)
    target.write_text('\n'.join(frame.rstrip('\n') for frame in frames) + '\n', encoding='utf-8')
    return str(target)

def _energy_diagram_options(current, trajectories, out, sub=None):
    """Expose one aggregate, structure-linked Results view per method."""
    priority = ('dft_g', 'mlip_g', 'dft_e', 'mlip_e', 'mep', 'irc')
    views = _energy_summary_views(current, out)
    def depth(path):
        try: return len(Path(os.path.relpath(path, out)).parts)
        except ValueError: return 999
    def artifact_rank(path):
        name = os.path.basename(path).lower()
        return (0 if '_all.' in name else 1, depth(path), name, path)
    for path in current:
        kind = _energy_diagram_kind(path)
        if (not kind or kind in ('mep', 'irc') or depth(path) != 1 or
                _artifact_kind(path) != 'image'):
            continue
        if kind not in views: views[kind] = {'mode': 'image', 'path': path}
    sub = str(sub or S.get('_last_subcmd') or '').lower()
    mep_paths = [path for path in trajectories if 'mep_trj' in os.path.basename(path).lower()]
    if not mep_paths and sub in ('all', 'path-opt', 'path-search'):
        mep_paths = [path for path in trajectories
                     if 'irc' not in os.path.basename(path).lower() and
                     any(token in os.path.basename(path).lower()
                         for token in ('final_geometries_trj', 'path', 'segment'))]
    mep_root = [path for path in mep_paths if depth(path) == 1]
    mep_path = min(mep_root, key=artifact_rank) if mep_root else None
    if sub == 'irc':
        standalone = [path for path in trajectories
                      if depth(path) == 1 and _irc_trajectory_role(path) == 'finished']
        irc_path = min(standalone, key=artifact_rank) if standalone else None
    else:
        irc_path = _ENERGY.get('aggregate_irc')
        if irc_path not in trajectories: irc_path = None
    if mep_path: views['mep'] = {'mode': 'profile', 'path': mep_path}
    if irc_path: views['irc'] = {'mode': 'profile', 'path': irc_path}
    stationary_paths = _stationary_structure_series(current, out)
    for kind, view in views.items():
        labels = list((view or {}).get('labels') or [])
        if (view.get('mode') == 'levels' and stationary_paths and
                len(labels) == len(stationary_paths) and not view.get('structures')):
            view['structures'] = list(stationary_paths)
    ordered = [kind for kind in priority if kind in views]
    _ENERGY['views'] = {'energy:' + kind: views[kind] for kind in ordered}
    options = [(_ENERGY_LABELS[kind], 'energy:' + kind) for kind in ordered]
    return options, mep_path

def _energy_levels_document(payload, generation, current_index=0):
    """Render an interactive chemical diagram linked to the stationary structures."""
    labels = list(payload.get('labels') or [])
    values = list(payload.get('energies_kcal') or [])
    if not labels or len(labels) != len(values):
        return '<!doctype html><html><body>Energy diagram data are incomplete.</body></html>'
    width, height = 760, 520
    left, right, top, bottom = 96, 28, 40, 72
    vmin, vmax = min(values), max(values)
    pad = max(2.0, (vmax - vmin) * 0.12)
    vmin -= pad; vmax += pad
    usable_w, usable_h = width - left - right, height - top - bottom
    xs = [left + usable_w * (index + 0.5) / len(values) for index in range(len(values))]
    ys = [top + (vmax - value) / max(vmax - vmin, 1e-9) * usable_h for value in values]
    palette = ('#0f766e', '#be185d', '#7c3aed', '#2563eb', '#d97706', '#0891b2')
    parts = ['<svg viewBox="0 0 %d %d" role="img" aria-label="Chemical energy diagram">' % (width, height)]
    zero_y = top + (vmax - 0.0) / max(vmax - vmin, 1e-9) * usable_h
    if top <= zero_y <= height - bottom:
        parts.append('<line x1="%d" x2="%d" y1="%.1f" y2="%.1f" stroke="#e2e8f0" stroke-width="1"/>' % (left, width-right, zero_y, zero_y))
    for index in range(len(values) - 1):
        parts.append('<line x1="%.1f" y1="%.1f" x2="%.1f" y2="%.1f" stroke="#94a3b8" stroke-width="2" stroke-dasharray="5 7"/>' %
                     (xs[index] + 68, ys[index], xs[index + 1] - 68, ys[index + 1]))
    for index, (label, value, x, y) in enumerate(zip(labels, values, xs, ys)):
        color = palette[index % len(palette)]
        parts.append('<g class="rxlevel" data-index="%d" role="button" tabindex="0" aria-label="Show %s structure">' %
                     (index, html.escape(str(label), quote=True)))
        parts.append('<line class="rxlevel-hit" x1="%.1f" x2="%.1f" y1="%.1f" y2="%.1f"/>' %
                     (x - 78, x + 78, y, y))
        parts.append('<line class="rxlevel-halo" x1="%.1f" x2="%.1f" y1="%.1f" y2="%.1f" stroke="%s"/>' %
                     (x - 74, x + 74, y, y, color))
        parts.append('<line class="rxlevel-bar" x1="%.1f" x2="%.1f" y1="%.1f" y2="%.1f" stroke="%s"/>' %
                     (x - 68, x + 68, y, y, color))
        label_y = min(height - 42, y + 36)
        value_y = max(28, y - 24)
        parts.append('<text class="rxlevel-label" x="%.1f" y="%.1f" text-anchor="middle" font-family="system-ui,sans-serif" font-size="18" font-weight="750" fill="#172033">%s</text>' %
                     (x, label_y, html.escape(str(label))))
        parts.append('<text class="rxlevel-value" x="%.1f" y="%.1f" text-anchor="middle" font-family="system-ui,sans-serif" font-size="24" font-weight="780" fill="%s">%+.1f</text>' %
                     (x, value_y, color, value))
        parts.append('</g>')
    ylabel = html.escape(str(payload.get('ylabel') or 'Relative energy (kcal/mol)'))
    parts.append('<text x="25" y="%d" transform="rotate(-90 25 %d)" text-anchor="middle" font-family="system-ui,sans-serif" font-size="21" font-weight="750" fill="#253047">%s</text>' %
                 (height // 2, height // 2, ylabel))
    parts.append('</svg>')
    config = {
        'generation': int(generation),
        'current': max(0, min(len(values) - 1, int(current_index or 0))),
        'callback': ('pdb2reaction_gui.set_frame' if IS_CLUSTER else 'mlmm_gui.set_frame'),
    }
    template = r"""<!doctype html><html><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<style>
html,body{width:100%;height:100%;margin:0;overflow:hidden;background:#fff}
body{display:flex;align-items:center;justify-content:center;font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif}
svg{display:block;width:100%;height:100%}
.rxlevel{cursor:pointer;outline:none}
.rxlevel-hit{stroke:transparent;stroke-width:44;pointer-events:stroke}
.rxlevel-bar{stroke-width:9;stroke-linecap:round;transition:stroke-width .14s ease,filter .14s ease}
.rxlevel-halo{stroke-width:18;stroke-linecap:round;opacity:0;transition:opacity .14s ease}
.rxlevel-value,.rxlevel-label{pointer-events:none;transition:font-size .14s ease,font-weight .14s ease}
.rxlevel:hover .rxlevel-halo{opacity:.14}
.rxlevel:focus-visible .rxlevel-halo{opacity:.38}
.rxlevel.active .rxlevel-halo{opacity:.20}
.rxlevel.active .rxlevel-bar{stroke-width:11;filter:drop-shadow(0 2px 2px rgba(15,23,42,.16))}
.rxlevel.active .rxlevel-value{font-size:26px;font-weight:850}
.rxlevel.active .rxlevel-label{font-weight:850}
</style></head><body>__SVG__
<script>(function(){'use strict';const cfg=__CONFIG__;
const levels=Array.from(document.querySelectorAll('.rxlevel'));let current=-1;
function kernel(){for(const candidate of [window,window.parent,window.top]){try{if(candidate&&candidate.google&&candidate.google.colab&&candidate.google.colab.kernel)return {api:candidate.google.colab.kernel,scope:candidate};}catch(_error){}}return null;}
function invoke(index){const bridge=kernel();if(!bridge)return Promise.resolve(null);const kwargs=bridge.scope.JSON.parse('{}');return bridge.api.invokeFunction(cfg.callback,[cfg.generation,index],kwargs);}
function broadcast(index){try{const host=window.parent&&window.parent.document;if(!host)return;const selector='iframe[data-rx-channel="trajectory"][data-rx-generation="'+String(cfg.generation)+'"]';const message={type:'rx-set-frame',generation:Number(cfg.generation),index:Number(index)};for(const target of host.querySelectorAll(selector)){if(target&&target.contentWindow&&target.contentWindow!==window)target.contentWindow.postMessage(message,'*');}}catch(_error){}}
function apply(index){current=Math.max(0,Math.min(levels.length-1,Number(index)||0));levels.forEach((node,i)=>{const active=i===current;node.classList.toggle('active',active);node.setAttribute('aria-pressed',active?'true':'false');});}
function choose(index){apply(index);broadcast(index);invoke(index).catch(error=>console.error(error));}
levels.forEach((node,index)=>{node.addEventListener('click',()=>choose(index));node.addEventListener('keydown',event=>{if(event.key==='Enter'||event.key===' '){event.preventDefault();choose(index);}});});
window.addEventListener('message',event=>{const message=event&&event.data;if(!message||message.type!=='rx-set-frame'||Number(message.generation)!==Number(cfg.generation))return;apply(message.index);});
apply(cfg.current);})();</script></body></html>"""
    return template.replace('__SVG__', ''.join(parts)).replace(
        '__CONFIG__', json.dumps(config, separators=(',', ':')).replace('<', '\\u003c'))

def _energy_levels_iframe(payload, generation, current_index=0):
    document = _energy_levels_document(payload, generation, current_index)
    return _document_iframe(
        document,
        'class="rxenergy-frame" title="Interactive energy diagram" '
        'data-rx-channel="trajectory" data-rx-generation="%d"' % int(generation),
        'display:block;width:100%;aspect-ratio:4/3;border:0;background:#fff;')

def _energy_image_html(path):
    try:
        size = os.path.getsize(path)
        if size > _IMAGE_PREVIEW_LIMIT:
            return ('<div role="status" style="padding:18px;color:#64748b">'
                    'Energy diagram preview skipped (%.1f MiB). Download the result to view it.</div>'
                    % (size / 1048576.0))
        with open(path, 'rb') as fh:
            data = base64.b64encode(fh.read()).decode('ascii')
        ext = Path(path).suffix.lower()
        mime = {'jpg': 'jpeg', 'svg': 'svg+xml'}.get(ext.lstrip('.'), ext.lstrip('.') or 'png')
        return ('<div class="rxartifact-image-wrap" style="padding:6px">'
                '<img class="rxartifact-image" alt="Selected energy diagram" src="data:image/%s;base64,%s"></div>'
                % (mime, data))
    except OSError as exc:
        return ('<div role="alert" style="padding:18px;color:#991b1b">'
                'Could not read energy diagram: %s</div>' % html.escape(str(exc)))

def _result_structure_frame(path, label):
    """Read one canonical stationary structure as an XYZ model."""
    suffix = Path(path).suffix.lower()
    if suffix in ('.gjf', '.com'):
        frame, _, _ = _load_small_view_structure(path)
        return frame if frame.endswith('\n') else frame + '\n'
    if suffix == '.xyz':
        parsed = _parse_trj(path)
        frames = list(parsed.get('frames') or [])
        if not frames: raise ValueError('empty XYZ structure: %s' % path)
        frame = frames[0]
        return frame if frame.endswith('\n') else frame + '\n'
    from ase.io import read as ase_read
    atoms = ase_read(str(path), index=0)
    rows = [str(len(atoms)), str(label)]
    rows.extend('%s %.12g %.12g %.12g' %
                (atom.symbol, atom.position[0], atom.position[1], atom.position[2])
                for atom in atoms)
    return '\n'.join(rows) + '\n'

def _mount_result_models(frames, generation):
    """Mount one fresh Mol* instance for exactly this model collection."""
    source = ''.join(frames)
    traj_signal_out.value = ''
    traj_out.value = _molstar_iframe(
        source, 'xyz', show_sequence=False, channel='trajectory',
        generation=generation, frame_count=len(frames))
    _TRAJ_VIEWER_MOUNTED['value'] = True
    _TRAJ['structure_message'] = {}

def _load_energy_structures(view, token, label):
    """Install R/TS/P models as the active linked Results structure."""
    _sync_playback_interval(token=token)
    paths = [str(path) for path in (view.get('structures') or [])]
    labels = [str(value) for value in (view.get('labels') or [])]
    values = [float(value) for value in (view.get('energies_kcal') or [])]
    if not paths or len(paths) != len(labels) or len(paths) != len(values):
        return False
    try:
        frames = [_result_structure_frame(path, state)
                  for path, state in zip(paths, labels)]
        atom_counts = [int(frame.splitlines()[0]) for frame in frames]
        if len(set(atom_counts)) != 1:
            raise ValueError('stationary structures have different atom counts')
    except (OSError, ValueError, RuntimeError, IndexError) as exc:
        view['structure_error'] = str(exc)
        return False
    same_models = (_TRAJ.get('mode') == 'levels' and
                   list(_TRAJ.get('structure_paths') or []) == paths and
                   len(_TRAJ.get('frames') or []) == len(frames))
    if not same_models:
        _TRAJ_VIEWER_GENERATION['value'] += 1
        generation = _TRAJ_VIEWER_GENERATION['value']
    else:
        generation = int(_TRAJ.get('generation') or 0)
    semantics = {'title': '%s stationary points' % label,
                 'start': labels[0], 'end': labels[-1],
                 'x': 'stationary point', 'extrema': False}
    quantity_label = 'ΔG' if 'ΔG' in str(label) else 'ΔE'
    _TRAJ.update(frames=frames, energies=[value / 627.5094740631 for value in values],
                 energy_provenance=[label] * len(frames), energy_unit='hartree',
                 quantity_label=quantity_label,
                 path='energy-states:' + str(token), semantics=semantics,
                 state_labels=labels, structure_paths=paths, mode='levels',
                 generation=generation, frame_message={}, structure_message={})
    if not same_models:
        # Replacing the iframe is intentional: Colab may deliver sibling-frame
        # postMessages out of order, leaving the previous 22/51-model trajectory
        # in Mol*. A fresh instance makes R/TS/P count deterministic.
        _mount_result_models(frames, generation)
    last = len(frames) - 1
    current = min(max(0, int(frame_slider.value)), last) if same_models else 0
    safe_last = max(1, last)
    frame_slider.max = safe_last; frame_play.max = safe_last
    frame_slider.disabled = last == 0; frame_play.disabled = last == 0
    frame_slider.value = current; frame_play.value = current
    frame_prev.disabled = current <= 0; frame_next.disabled = current >= last
    results_empty.layout.display = 'none'; trajectory_box.layout.display = ''
    trajectory_content.layout.display = ''; frame_controls.layout.display = ''
    structure_panel.layout.display = ''
    traj_choice.layout.display = '' if len(traj_choice.options) > 1 else 'none'
    traj_label.layout.display = 'none'; trajectory_info.layout.display = 'none'
    trajectory_intro.value = ('<div><span class="rxpath-title">Energy diagram &amp; stationary structures</span>'
                              '<span class="rxpath-subtitle">Select a state to inspect its structure and relative energy.</span></div>')
    _show_frame(current)
    return True

def _matching_result_path(token, view=None):
    view = view or {}
    if view.get('mode') == 'profile' and view.get('path'):
        return view.get('path')
    return None

def _sync_result_selector(path):
    values = [value for _label, value in traj_choice.options]
    if path not in values: return
    _result_pick_guard['active'] = True
    try: traj_choice.value = path
    finally: _result_pick_guard['active'] = False

def _render_energy_choice_now(_=None):
    if _result_pick_guard['active']:
        return
    energy_panel.layout.display = ''
    scan_axis_controls.layout.display = 'none'
    path_grid.remove_class('rxstructure-only')
    token = energy_choice.value
    view = (_ENERGY.get('views') or {}).get(token)
    if not view:
        _sync_playback_interval()
        energy_panel_title.value = '<div class="rxpath-panel-title">Energy result</div>'
        plot_out.value = ('<div class="rxlevel-diagram" role="status" style="color:#64748b">'
                          'No energy result was produced by this run.</div>')
        return
    _sync_playback_interval(path=view.get('path'), token=token)
    label = next((text for text, value in energy_choice.options if value == token), 'Energy result')
    mode = view.get('mode')
    matching_path = _matching_result_path(token, view)
    if matching_path:
        _sync_result_selector(matching_path)
    result_selector_row.layout.display = ('' if matching_path and len(traj_choice.options) > 1
                                          else 'none')
    if mode == 'profile':
        energy_panel_title.value = ('<div class="rxpath-panel-title">%s energy profile</div>' % html.escape(label))
        profile_path = view.get('path')
        if profile_path and _TRAJ.get('path') != profile_path:
            _load_trajectory(profile_path)
        relative_energies = _rel_kcal()
        if relative_energies is None:
            plot_out.value = ('<div class="rxlevel-diagram" style="color:#64748b">%s energy profile unavailable.</div>' % html.escape(label))
        else:
            semantics = _TRAJ.get('semantics') or _trajectory_semantics(
                path=_TRAJ.get('path'), n_frames=len(_TRAJ.get('frames') or []))
            plot_out.value = _energy_plot_iframe(
                relative_energies, semantics, _TRAJ.get('generation', 0))
        return
    energy_panel_title.value = ('<div class="rxpath-panel-title">%s energy diagram</div>' %
                                html.escape(label))
    if mode == 'levels':
        linked = _load_energy_structures(view, token, label)
        if linked:
            plot_out.value = _energy_levels_iframe(
                view, _TRAJ.get('generation', 0), int(frame_slider.value))
            _show_frame(int(frame_slider.value))
        else:
            structure_panel.layout.display = 'none'; frame_controls.layout.display = 'none'
            plot_out.value = _energy_levels_iframe(view, 0, 0)
    elif mode == 'image':
        if matching_path and _TRAJ.get('path') != matching_path:
            _load_trajectory(matching_path)
        plot_out.value = _energy_image_html(view.get('path'))
    else:
        plot_out.value = '<div class="rxlevel-diagram" style="color:#64748b">Energy diagram unavailable.</div>'

def _render_energy_choice(_=None):
    if _result_pick_guard['active']:
        return
    label = next((text for text, value in energy_choice.options
                  if value == energy_choice.value), 'result view')
    _set_result_loading(label, True)
    try:
        return _render_energy_choice_now(_)
    finally:
        _set_result_loading(label, False)
energy_choice.observe(_render_energy_choice, names='value')

def _single_point_energy_ha(out=None):
    """Read the scalar energy associated with an sp/dft structure preview."""
    if str(S.get('_last_subcmd') or '').lower() not in ('sp', 'dft'):
        return None
    out = out or S.get('_last_out_dir') or _effective_result_root()
    current = _current_run_files(out)
    ordered = sorted((path for path in current if os.path.basename(path) in ('result.json', 'summary.json')),
                     key=lambda path: (0 if os.path.basename(path) == 'result.json' else 1, path))
    for path in ordered:
        try:
            with open(path, encoding='utf-8') as fh: payload = json.load(fh)
        except (OSError, ValueError, TypeError):
            continue
        if not isinstance(payload, dict):
            continue
        for key in ('energy_hartree', 'energy_ha', 'energy_au', 'energy'):
            try: value = float(payload.get(key))
            except (TypeError, ValueError): continue
            if math.isfinite(value): return value
    return None

def _load_trajectory(path=None, out=None):
    path = path or traj_choice.value
    out = out or S.get('_last_out_dir') or _effective_result_root()
    _sync_playback_interval(path=path)
    path_grid.remove_class('rxplot-only'); path_grid.remove_class('rxstructure-only')
    scan_axis_controls.layout.display = 'none'
    energy_panel.layout.display = ''
    _TRAJ_VIEWER_GENERATION['value'] += 1
    generation = _TRAJ_VIEWER_GENERATION['value']
    _TRAJ.update(frames=[], energies=[], energy_provenance=[], energy_unit='hartree', quantity_label='\u0394E',
                 path=path, semantics={}, state_labels=[], structure_paths=[],
                 mode='trajectory', generation=generation,
                 frame_message={}, structure_message={})
    # A failed preview must not strand the user on an unchangeable candidate.
    traj_choice.disabled = not bool(traj_choice.options)
    parse_error = None
    if path and os.path.isfile(path):
        try:
            size = os.path.getsize(path)
            if size > _TRAJECTORY_PREVIEW_LIMIT:
                raise ValueError(
                    'file is %.1f MiB; inline trajectory preview is limited to %.0f MiB' %
                    (size / 1048576.0, _TRAJECTORY_PREVIEW_LIMIT / 1048576.0))
            suffix = Path(path).suffix.lower()
            if suffix == '.xyz':
                parsed = _parse_trj(path)
                frames = parsed['frames']
            else:
                frames = [_result_structure_frame(path, Path(path).stem)]
                parsed = {'frames': frames, 'energies_ha': [None],
                          'energy_provenance': [], 'energy_unit': 'hartree'}
            if len(frames) > _TRAJECTORY_FRAME_LIMIT:
                raise ValueError('trajectory has %d frames; inline preview is limited to %d' %
                                 (len(frames), _TRAJECTORY_FRAME_LIMIT))
            try: atom_count = int(frames[0].splitlines()[0]) if frames else 0
            except (IndexError, ValueError): atom_count = 0
            if atom_count * len(frames) > _TRAJECTORY_ATOM_FRAME_LIMIT:
                raise ValueError(
                    'trajectory has %d atom-frames; inline preview is limited to %d' %
                    (atom_count * len(frames), _TRAJECTORY_ATOM_FRAME_LIMIT))
            if len(frames) == 1 and not any(value is not None for value in parsed['energies_ha']):
                scalar_energy = _single_point_energy_ha(out)
                if scalar_energy is not None:
                    parsed = dict(parsed)
                    parsed['energies_ha'] = [scalar_energy]
                    parsed['energy_provenance'] = ['result.json']
                    parsed['energy_unit'] = 'hartree'
            _TRAJ.update(frames=frames, energies=parsed['energies_ha'],
                         energy_provenance=parsed['energy_provenance'],
                         energy_unit=parsed['energy_unit'])
        except (OSError, ValueError, RuntimeError) as exc:
            parse_error = str(exc)
            traj_label.value = '<small role="alert" style="color:#991b1b">Could not read trajectory: %s</small>' % html.escape(parse_error)
    if _TRAJ['frames']:
        semantics = _trajectory_semantics(
            S.get('_last_subcmd'), path, n_frames=len(_TRAJ['frames']))
        _TRAJ['semantics'] = semantics
        semantics_title = str(semantics.get('title') or '')
        if 'IRC' in semantics_title:
            _sync_playback_interval(token='irc')
        elif semantics_title.startswith('Reaction-path'):
            _sync_playback_interval(token='mep')
        sub = str(S.get('_last_subcmd') or '').lower()
        is_vibrational_view = (sub == 'freq' or
                               'mode_' in os.path.basename(str(path)).lower() or
                               'imag_' in os.path.basename(str(path)).lower())
        if is_vibrational_view:
            _sync_playback_interval(token='vibration')
            intro_title, intro_text = ('Vibrational mode &amp; structure',
                                       'Use the playback controls or slider to inspect the vibration.')
        elif sub in ('opt', 'tsopt') and len(_TRAJ['frames']) == 1:
            intro_title, intro_text = (('Final optimized structure' if sub == 'opt' else 'Refined transition-state structure'),
                                       'The final structure produced by this run is shown below.')
        elif sub in ('opt', 'tsopt'):
            intro_title, intro_text = (('Optimization progress &amp; structures' if sub == 'opt' else 'TS refinement progress &amp; structures'),
                                       'Select an optimization step to inspect its structure and relative energy.')
        elif sub in ('sp', 'dft') and len(_TRAJ['frames']) == 1:
            intro_title, intro_text = ('Evaluated input structure',
                                       'The single-point result corresponds to this input structure.')
        elif sub == 'scan':
            intro_title, intro_text = ('Scan profile &amp; structures',
                                       'Select a scan point to inspect its structure and relative energy.')
        elif 'irc' in os.path.basename(str(path)).lower():
            intro_title, intro_text = ('IRC profile &amp; structures',
                                       'Select an IRC point to inspect its structure and relative energy.')
        elif 'mep' in os.path.basename(str(path)).lower():
            intro_title, intro_text = ('MEP profile &amp; structures',
                                       'Select a path image to inspect its structure and relative energy.')
        else:
            intro_title, intro_text = ('Trajectory &amp; structures',
                                       'Select a frame to inspect its structure and relative energy.')
        trajectory_intro.value = ('<div><span class="rxpath-title">%s</span>'
                                  '<span class="rxpath-subtitle">%s</span></div>' %
                                  (intro_title, intro_text))
        provenance = list(dict.fromkeys(_TRAJ.get('energy_provenance') or []))
        provenance_text = ', '.join(provenance) if provenance else 'unavailable'
        truth = semantics.get('trajectory_status')
        truth_text = (' · %s' % html.escape(str(truth))) if truth else ''
        traj_label.value = ('<small><code>%s</code> · energy: %s%s</small>' %
                            (html.escape(os.path.relpath(path, out)),
                             html.escape(provenance_text), truth_text))
        # The selected view and file are already shown beside the plot. Avoid
        # a second 'Primary trajectory' card that incorrectly competes with
        # higher-level energy diagrams for visual priority.
        primary_result.value = ''
        last = max(0, len(_TRAJ['frames']) - 1)
        _mount_result_models(_TRAJ['frames'], generation)
        safe_last = max(1, last)
        frame_slider.max = safe_last; frame_play.max = safe_last
        frame_slider.value = 0; frame_play.value = 0
        frame_slider.disabled = last == 0; frame_play.disabled = last == 0
        frame_prev.disabled = True; frame_next.disabled = last == 0
        results_empty.layout.display = 'none'; trajectory_box.layout.display = ''
        trajectory_content.layout.display = ''
        single_static_result = last == 0
        frame_controls.layout.display = 'none' if single_static_result else ''
        structure_panel.layout.display = ''
        traj_choice.layout.display = ''; traj_label.layout.display = ''
        if single_static_result or is_vibrational_view:
            energy_panel.layout.display = 'none'
            path_grid.add_class('rxstructure-only')
            plot_out.value = ''
        else:
            profile_name = ('IRC' if 'irc' in os.path.basename(str(path)).lower() else
                            ('MEP' if 'mep' in os.path.basename(str(path)).lower() else 'Trajectory'))
            energy_panel_title.value = ('<div class="rxpath-panel-title">%s energy profile</div>' % profile_name)
            relative_energies = _rel_kcal()
            if relative_energies is None:
                plot_out.value = ''
                energy_panel.layout.display = 'none'
                path_grid.add_class('rxstructure-only')
            else:
                plot_out.value = _energy_plot_iframe(relative_energies, semantics, generation)
        _show_frame(0)
    else:
        frame_slider.max = 1; frame_play.max = 1
        frame_slider.value = 0; frame_play.value = 0
        frame_slider.disabled = True; frame_play.disabled = True
        frame_prev.disabled = True; frame_next.disabled = True
        frame_state.value = ''
        trajectory_info.layout.display = 'none'
        has_energy = bool(energy_choice.options)
        trajectory_content.layout.display = '' if has_energy else 'none'
        trajectory_box.layout.display = '' if (traj_choice.options or has_energy) else 'none'
        frame_controls.layout.display = 'none'; structure_panel.layout.display = 'none'
        traj_choice.layout.display = '' if traj_choice.options else 'none'
        traj_label.layout.display = '' if traj_choice.options else 'none'
        if has_energy:
            trajectory_intro.value = ('<div><span class="rxpath-title">Energy results</span>'
                                      '<span class="rxpath-subtitle">Select an available energy view.</span></div>')
            plot_out.value = ''
        else:
            plot_out.value = ''
        traj_out.value = ''
        _TRAJ_VIEWER_MOUNTED['value'] = False
        traj_signal_out.value = ''
        primary_result.value = ''
        if parse_error:
            results_empty.layout.display = ''
            results_empty.value = ('<div role="alert" style="padding:10px;border:1px solid #fecaca;'
                                   'border-radius:10px;color:#991b1b">Trajectory unavailable: %s</div>' %
                                   html.escape(parse_error))

_SCAN_GRID = {'token': None, 'view': None}

def _last_input_paths():
    """Recover existing structure inputs from the command that produced this run."""
    raw = []
    argv = list(S.get('_last_argv') or [])
    for index, token in enumerate(argv[:-1]):
        if token in ('-i', '--input', '--structure', '--structures'):
            raw.append(argv[index + 1])
    raw.extend(S.get('inputs') or [])
    paths = []
    for value in raw:
        if isinstance(value, dict): value = value.get('path') or value.get('name')
        if not value: continue
        candidate = os.path.abspath(os.path.expanduser(str(value)))
        if os.path.isfile(candidate) and candidate not in paths: paths.append(candidate)
    return paths

def _frequency_mode_record(path):
    """Parse the export order and signed frequency from a freq trajectory."""
    name = os.path.basename(str(path))
    order_match = re.search(r'mode_(\d+)_', name, flags=re.IGNORECASE)
    frequency_match = re.search(r'_([+-]?\d+(?:\.\d+)?)cm-1_trj\.xyz$', name, flags=re.IGNORECASE)
    order = int(order_match.group(1)) if order_match else None
    try: frequency = float(frequency_match.group(1)) if frequency_match else None
    except (TypeError, ValueError): frequency = None
    return {'path': path, 'order': order, 'frequency': frequency}

def _frequency_mode_views(paths, depth):
    records = [_frequency_mode_record(path) for path in paths]
    records.sort(key=lambda item: (
        item['order'] is None, item['order'] if item['order'] is not None else 10**9,
        depth(item['path']), os.path.basename(item['path']).lower()))
    total = len(records); views = []
    for position, item in enumerate(records, 1):
        frequency = item['frequency']
        if frequency is None:
            frequency_text, kind = 'frequency unavailable', ''
        else:
            frequency_text = ('−%.2f' % abs(frequency) if frequency < 0 else '%.2f' % frequency) + ' cm⁻¹'
            kind = ' · imaginary' if frequency < 0 else ''
        views.append(('Mode %d of %d · %s%s' %
                      (position, total, frequency_text, kind), item['path']))
    return views

def _imaginary_mode_views(trajectories, out, depth):
    """Return every distinct imaginary-mode trajectory for each TS."""
    dedicated = [path for path in trajectories
                 if os.path.basename(path).lower().startswith('imag_') and
                 os.path.basename(path).lower().endswith('_trj.xyz')]
    fallback = [path for path in trajectories
                if re.search(r'mode_\d+_-', os.path.basename(path), re.IGNORECASE) and
                any(part.upper() == 'TS' for part in Path(path).parts)]
    # Prefer a dedicated imag_* trajectory per TS, but fall back per TS rather
    # than globally. Mixed runs can contain a new dedicated file for one TS
    # and only a negative-frequency mode_* file for another.
    grouped = {}
    for kind, candidates in (('dedicated', dedicated), ('fallback', fallback)):
        for path in candidates:
            parts = Path(os.path.relpath(path, out)).parts
            numbered = next((part for part in parts if
                               re.fullmatch(r'(?:seg|ts)[_-]?\d+', part, re.IGNORECASE)), '')
            number = int(re.search(r'\d+', numbered).group()) if numbered else None
            if numbered:
                key = numbered.lower()
            else:
                ts_index = next((index for index, part in enumerate(parts)
                                 if part.lower() == 'ts'), None)
                key = ('/'.join(parts[:ts_index + 1]).lower() if ts_index is not None
                       else '__standalone_ts__')
            group = grouped.setdefault(key, {'number': number, 'dedicated': [], 'fallback': []})
            group[kind].append(path)
    records = []
    for group in grouped.values():
        candidates = group['dedicated'] or group['fallback']
        def _imaginary_rank(path):
            match = re.search(r'_([+-]?\d+(?:\.\d+)?)cm-1_trj\.xyz$',
                              os.path.basename(path), re.IGNORECASE)
            frequency = float(match.group(1)) if match else None
            return (frequency is None, frequency if frequency is not None else 0.0,
                    depth(path), os.path.basename(path).lower())
        for path in sorted(candidates, key=_imaginary_rank):
            frequency_match = re.search(r'_([+-]?\d+(?:\.\d+)?)cm-1_trj\.xyz$',
                                        os.path.basename(path), re.IGNORECASE)
            frequency = float(frequency_match.group(1)) if frequency_match else None
            records.append((group['number'], frequency, depth(path), path))
    records.sort(key=lambda item: (item[0] is None, item[0] or 0, item[1] is None, item[1] or 0.0, item[2], item[3]))
    views = []
    for position, (segment_number, frequency, _path_depth, path) in enumerate(records, 1):
        ts_label = ('TS%d' % segment_number if segment_number is not None else
                    'TS%d' % position if len(records) > 1 else '')
        bits = ['Imaginary mode']
        if ts_label: bits.append(ts_label)
        if frequency is not None: bits.append('−%.2f cm⁻¹' % abs(frequency))
        views.append((' · '.join(bits), path))
    return views

def _result_view_candidates(current, out, sub):
    """Choose the scientifically useful structural view for each subcommand."""
    sub = str(sub or '').lower()
    structures = [path for path in current if Path(path).suffix.lower() in
                  ('.xyz', '.pdb', '.cif', '.mmcif', '.gjf', '.com')]
    trajectories = [path for path in structures
                    if Path(path).suffix.lower() == '.xyz' and
                    ('trj' in os.path.basename(path).lower() or
                     'trajectory' in os.path.basename(path).lower())]
    def depth(path):
        try: return len(Path(os.path.relpath(path, out)).parts)
        except ValueError: return 999
    def best_named(names, pool=structures):
        rank = {str(name).lower(): index for index, name in enumerate(names)}
        found = [path for path in pool if os.path.basename(path).lower() in rank]
        return min(found, key=lambda path: (rank[os.path.basename(path).lower()], depth(path), path)) if found else None
    if sub in ('scan2d', 'scan3d') and _SCAN_GRID.get('view'):
        view = _SCAN_GRID['view']
        return [('%s scan grid · %d structures' %
                 ('2D' if sub == 'scan2d' else '3D', len(view.get('paths') or [])),
                 _SCAN_GRID['token'])]
    if sub in ('opt', 'tsopt'):
        dump_path = best_named(('optimization_all_trj.xyz', 'optimization_trj.xyz'), trajectories)
        final_path = best_named(('final_geometry.xyz', 'final_geometry.pdb', 'final_geometry.cif', 'final_geometry.gjf'))
        views = []
        if dump_path:
            views.append((('Optimization trajectory' if sub == 'opt' else 'TS-refinement trajectory'), dump_path))
        if final_path:
            views.append((('Final optimized structure' if sub == 'opt' else 'Refined transition-state structure'), final_path))
        if sub == 'tsopt': views += _imaginary_mode_views(trajectories, out, depth)
        return views
    if sub == 'dft':
        path = best_named(('input_geometry.xyz', 'input_geometry.pdb', 'input_geometry.cif'))
        if path is None:
            path = next(iter(_last_input_paths()), None)
        return [('Evaluated input structure', path)] if path else []
    if sub == 'sp':
        path = next(iter(_last_input_paths()), None)
        return [('Evaluated input structure', path)] if path else []
    if sub == 'freq':
        modes = [path for path in trajectories if
                 'mode_' in os.path.basename(path).lower() or
                 'imag_' in os.path.basename(path).lower()]
        reference = best_named(('final_geometry.xyz', 'final_geometry.pdb',
                                'final_geometry.cif', 'input_geometry.xyz'))
        views = ([('Frequency reference structure', reference)] if reference else [])
        return views + _frequency_mode_views(modes, depth)
    if sub == 'irc':
        labels = {'finished': 'Combined IRC trajectory', 'forward': 'Forward IRC branch',
                  'backward': 'Backward IRC branch'}
        order = {'finished': 0, 'forward': 1, 'backward': 2}
        irc_paths = [path for path in trajectories if _irc_trajectory_role(path)]
        if irc_paths:
            return [(labels[_irc_trajectory_role(path)], path) for path in
                    sorted(irc_paths, key=lambda path: (order[_irc_trajectory_role(path)], depth(path), path))]
    preferred = None
    if sub == 'scan':
        preferred = best_named(('scan_trj.xyz',), trajectories)
    elif sub == 'path-opt':
        preferred = best_named(('final_geometries_trj.xyz', 'mep_trj.xyz'), trajectories)
    elif sub in ('all', 'path-search'):
        candidate = best_named(('mep_trj.xyz', 'final_geometries_trj.xyz'), trajectories)
        preferred = candidate if candidate is not None and depth(candidate) == 1 else None
    imaginary_views = (_imaginary_mode_views(trajectories, out, depth)
                       if sub == 'all' else [])
    aggregate_irc = _ENERGY.get('aggregate_irc') if sub == 'all' else None
    views = []
    if preferred:
        label = ('IRC trajectory' if sub == 'irc' else
                 'Scan trajectory' if sub == 'scan' else 'Reaction-path trajectory')
        views.append((label, preferred))
    if aggregate_irc: views.append(('IRC trajectory', aggregate_irc))
    if views or imaginary_views: return [*views, *imaginary_views]
    if trajectories:
        path = min(trajectories, key=lambda value: (depth(value), value))
        return [('Trajectory · ' + os.path.basename(path), path)]
    usable = [path for path in structures if '%sgrid%s' % (os.sep, os.sep) not in path]
    if usable:
        path = min(usable, key=lambda value: (depth(value),
                                               0 if Path(value).suffix.lower() == '.xyz' else 1, value))
        return [('Result structure · ' + os.path.basename(path), path)]
    return []

def _scan_grid_view(current, out, sub):
    """Build an owned scan-grid view; HTML remains useful without structures."""
    dims = 2 if sub == 'scan2d' else 3
    html_name = 'scan2d_landscape.html' if dims == 2 else 'scan3d_density.html'
    root = os.path.abspath(out)
    def owned(name):
        found = next((path for path in current if os.path.basename(path) == name), None)
        if found: return found
        candidate = os.path.abspath(os.path.join(root, name))
        same_run = (S.get('_last_out_dir') and
                    os.path.abspath(S['_last_out_dir']) == root)
        return candidate if same_run and os.path.isfile(candidate) else None
    html_path = owned(html_name)
    if not html_path: return None
    csv_path = owned('surface.csv')
    result_path = next((path for path in current
                        if os.path.basename(path) == 'result.json'), None)
    result_payload = {}
    if result_path:
        try:
            with open(result_path, encoding='utf-8') as fh: result_payload = json.load(fh)
            if not isinstance(result_payload, dict): result_payload = {}
        except (OSError, ValueError):
            result_payload = {}

    def number(value):
        try: return float(value)
        except (TypeError, ValueError): return None
    def vector(value, length):
        values = list(value or []) if isinstance(value, (list, tuple)) else []
        parsed = [number(item) for item in values[:length]]
        return parsed if len(parsed) == length and all(
            item is not None and math.isfinite(item) for item in parsed) else None
    csv_rows, csv_by_index = [], {}
    if csv_path:
        try:
            with open(csv_path, newline='', encoding='utf-8', errors='replace') as fh:
                for row in csv.DictReader(fh):
                    if str(row.get('is_preopt') or '').strip().lower() in ('true', '1', 'yes'): continue
                    indices = []
                    for key in ('i', 'j', 'k')[:dims]:
                        try: indices.append(int(float(row.get(key))))
                        except (TypeError, ValueError): indices = []; break
                    if len(indices) != dims or any(index < 0 for index in indices): continue
                    coords = [number(row.get('d%d_A' % axis) or row.get('target_d%d_A' % axis))
                              for axis in range(1, dims + 1)]
                    targets = [number(row.get('target_d%d_A' % axis) or row.get('d%d_A' % axis))
                               for axis in range(1, dims + 1)]
                    if any(value is None or not math.isfinite(value) for value in coords + targets): continue
                    record = {'index': tuple(indices), 'coords': coords, 'targets': targets,
                              'energy': number(row.get('energy_kcal'))}
                    csv_rows.append(record); csv_by_index[tuple(indices)] = record
        except OSError:
            csv_rows = []; csv_by_index = {}

    grid_paths = {}
    for path in current:
        name = os.path.basename(path).lower()
        if name.startswith('point_') and Path(path).suffix.lower() == '.xyz':
            grid_paths[name] = path
    grid_dir = Path(root) / 'grid'
    if grid_dir.is_dir():
        for path in grid_dir.glob('point_*.xyz'):
            grid_paths.setdefault(path.name.lower(), str(path.resolve()))

    def geometry_path(raw):
        if not raw: return None
        raw = str(raw)
        candidates = ([raw] if os.path.isabs(raw) else
                      [os.path.join(os.path.dirname(result_path), raw) if result_path else '',
                       os.path.join(root, raw)])
        for candidate in candidates:
            if candidate and os.path.isfile(candidate): return os.path.abspath(candidate)
        return None
    def fallback_path(record):
        def tag(value): return '%03d' % int(round(float(value) * 100.0))
        stem = 'point_' + '_'.join('%s%s' % (axis, tag(value))
                                   for axis, value in zip(('i', 'j', 'k')[:dims], record['targets']))
        names = [stem + '.xyz', stem + '_grid_' +
                 '_'.join('%03d' % index for index in record['index']) + '.xyz']
        path = next((grid_paths.get(name.lower()) for name in names if grid_paths.get(name.lower())), None)
        if path: return path
        return next((value for name, value in grid_paths.items()
                     if name.startswith(stem.lower())), None)

    linked = []
    manifest_points = result_payload.get('grid_points') or []
    if isinstance(manifest_points, list) and manifest_points:
        for item in manifest_points:
            if not isinstance(item, dict): continue
            try: indices = tuple(int(value) for value in item.get('index', []))
            except (TypeError, ValueError): continue
            if len(indices) != dims: continue
            row = csv_by_index.get(indices)
            coords = row['coords'] if row else vector(item.get('distances_angstrom'), dims)
            targets = row['targets'] if row else vector(item.get('targets_angstrom'), dims)
            if coords is None: coords = targets
            if targets is None: targets = coords
            if coords is None or targets is None: continue
            path = geometry_path(item.get('geometry_file'))
            if path is None: path = fallback_path({'index': indices, 'targets': targets})
            if path is None: continue
            linked.append({'coords': coords, 'targets': targets, 'energy': row['energy'] if row else None,
                           'path': path, 'index': indices})
    else:
        for row in csv_rows:
            path = fallback_path(row)
            if path:
                linked.append({**row, 'path': path})

    points, paths, labels, energies = [], [], [], []
    for item in linked:
        energy = item.get('energy')
        # Controls represent the requested Cartesian grid, while the white
        # plot markers retain the measured optimized distances.
        control_coords = list(item['targets'])
        plot_coords = list(item['coords'])
        label = ' \u2194 '.join('d%d %.2f \u00c5' % (axis, value)
                           for axis, value in enumerate(control_coords, 1))
        points.append({'coords': control_coords, 'plot_coords': plot_coords,
                       'energy': energy, 'label': label})
        paths.append(item['path']); labels.append(label); energies.append(energy)
    return {'sub': sub, 'dims': dims, 'html': html_path, 'csv': csv_path,
            'points': points, 'paths': paths, 'labels': labels, 'energies': energies,
            'linked': bool(paths)}

def _scan_axis_values(slider):
    values = []
    for option in tuple(slider.options or ()): 
        value = option[1] if isinstance(option, (tuple, list)) and len(option) == 2 else option
        if value is not None: values.append(value)
    return values

def _sync_scan_axis_controls(index):
    view = _SCAN_GRID.get('view') or {}
    points = view.get('points') or []
    if not points: return
    index = max(0, min(int(index), len(points) - 1))
    coords = list(points[index].get('coords') or [])
    _SCAN_AXIS_GUARD['active'] = True
    try:
        for axis, slider in enumerate(scan_axis_sliders):
            if axis < len(coords) and slider.options:
                values = _scan_axis_values(slider)
                slider.value = min(values, key=lambda value: abs(float(value) - float(coords[axis])))
    finally:
        _SCAN_AXIS_GUARD['active'] = False

def _on_scan_axis_change(_=None):
    if _SCAN_AXIS_GUARD['active'] or _TRAJ.get('mode') != 'grid': return
    view = _SCAN_GRID.get('view') or {}
    dims = int(view.get('dims') or 0); points = view.get('points') or []
    raw = [scan_axis_sliders[axis].value for axis in range(dims)]
    if not points or any(value is None for value in raw): return
    # A scan grid is a Cartesian product.  Resolve the complete coordinate
    # tuple instead of choosing a nearest point, which could silently move
    # axes that the user did not touch.  Rounding removes harmless CSV
    # representations such as 1.5999999999999999 without changing the grid.
    key = tuple(round(float(value), 10) for value in raw)
    lookup = {tuple(round(float(value), 10) for value in point['coords'][:dims]): index
              for index, point in enumerate(points)}
    best = lookup.get(key)
    if best is None: return
    if frame_slider.value != best: frame_slider.value = best
    else: _sync_scan_axis_controls(best)

def _configure_scan_axis_controls(view):
    dims = max(0, min(3, int((view or {}).get('dims') or 0)))
    points = (view or {}).get('points') or []
    _SCAN_AXIS_GUARD['active'] = True
    try:
        for axis, (row, slider) in enumerate(zip(scan_axis_rows, scan_axis_sliders)):
            visible = axis < dims and bool(points)
            row.layout.display = '' if visible else 'none'
            values = sorted({float(point['coords'][axis]) for point in points}) if visible else []
            if len(values) >= 2:
                slider.options = [('%.2f \u00c5' % value, value) for value in values]
            elif values:
                value = values[0]
                # Keep the real coordinate selected and pad only the inert
                # frontend index range; this control remains disabled.
                slider.options = [('%.2f \u00c5' % value, value), ('', None)]
            else:
                slider.options = [('—', 0.0), ('', None)]
            slider.disabled = len(values) <= 1
            if values: slider.value = values[0]
    finally:
        _SCAN_AXIS_GUARD['active'] = False
    scan_axis_controls.layout.display = '' if dims and points else 'none'
    if points: _sync_scan_axis_controls(0)

for _scan_axis_slider in scan_axis_sliders:
    _scan_axis_slider.observe(_on_scan_axis_change, names='value')

def _scan_grid_document(view, generation):
    # Keep the CLI-authored RBF surface/isosurface as the single source of
    # truth.  This linked copy gets a small clickable grid overlay; the
    # generated-file preview remains the original CLI-authored document.
    with open(view['html'], encoding='utf-8', errors='replace') as fh:
        source = _responsive_artifact_document(fh.read())
    config = {'generation': int(generation), 'dims': int(view['dims']),
              'points': view['points'],
              'callback': ('pdb2reaction_gui.set_frame' if IS_CLUSTER else 'mlmm_gui.set_frame')}
    bridge = r"""<script>(function(){'use strict';const cfg=__CONFIG__;let graph=null,tries=0;
function kernel(){for(const candidate of [window,window.parent,window.top]){try{if(candidate&&candidate.google&&candidate.google.colab&&candidate.google.colab.kernel)return {api:candidate.google.colab.kernel,scope:candidate};}catch(_error){}}return null;}
function invoke(index){const bridge=kernel();if(!bridge)return Promise.resolve(null);const kwargs=bridge.scope.JSON.parse('{}');return bridge.api.invokeFunction(cfg.callback,[cfg.generation,index],kwargs);}
function choose(index){if(!cfg.points.length)return;index=Math.max(0,Math.min(cfg.points.length-1,Number(index)||0));invoke(index).catch(error=>console.error(error));}
function pointTrace(){const coords=axis=>cfg.points.map(item=>Number((item.plot_coords||item.coords)[axis]));const trace={type:'scatter3d',mode:'markers',x:coords(0),y:coords(1),customdata:cfg.points.map((_item,index)=>index),meta:'rx-grid-points',name:'Computed grid points',showlegend:false,marker:{size:1.8,color:'#ffffff',line:{color:'#64748b',width:0.45}},hovertemplate:'d1 %{x:.3f} Å<br>d2 %{y:.3f} Å'+(cfg.dims===2?'<br>ΔE %{z:.2f} kcal/mol':'<br>d3 %{z:.3f} Å')+'<extra></extra>'};trace.z=cfg.dims===2?cfg.points.map(item=>Number(item.energy)):coords(2);return trace;}
function attach(){graph=document.querySelector('.plotly-graph-div');if(!graph||!window.Plotly||typeof graph.on!=='function'){if(++tries<125)setTimeout(attach,80);return;}const surfaces=[];(graph.data||[]).forEach((trace,index)=>{if(['surface','mesh3d','isosurface','volume'].includes(String(trace.type||'').toLowerCase()))surfaces.push(index);});window.Plotly.relayout(graph,{'title.text':'','margin.t':20});if(surfaces.length)window.Plotly.restyle(graph,{hoverinfo:'skip'},surfaces);if(!cfg.points.length)return;window.Plotly.addTraces(graph,[pointTrace()]);graph.on('plotly_click',event=>{const point=event&&event.points&&event.points[0];if(!point||!point.data||point.data.meta!=='rx-grid-points')return;const direct=Number(point.customdata);if(Number.isInteger(direct)&&direct>=0&&direct<cfg.points.length)choose(direct);});}
attach();})();</script>""".replace('__CONFIG__', json.dumps(config, separators=(',', ':')).replace('<', '\u003c'))
    marker = source.lower().rfind('</body>')
    return source[:marker] + bridge + source[marker:] if marker >= 0 else source + bridge

def _load_scan_grid():
    _sync_playback_interval()
    view = _SCAN_GRID.get('view')
    if not view: return
    _TRAJ_VIEWER_GENERATION['value'] += 1
    generation = _TRAJ_VIEWER_GENERATION['value']
    link_error = ''
    try:
        frames = [_result_structure_frame(path, label)
                  for path, label in zip(view['paths'], view['labels'])]
        atom_counts = [int(frame.splitlines()[0]) for frame in frames]
        if frames and len(set(atom_counts)) != 1:
            raise ValueError('scan structures have different atom counts')
    except (OSError, ValueError, RuntimeError, IndexError) as exc:
        frames = []; link_error = str(exc)
    document = _scan_grid_document(view, generation)
    plot_out.value = _plotly_document_iframe(
        document,
        'class="rxenergy-frame rxscan-pes-frame" title="Interactive scan landscape"',
        'display:block;width:100%;max-width:100%;min-width:0;height:auto;min-height:320px;max-height:520px;aspect-ratio:16/10;box-sizing:border-box;overflow:hidden;border:0;background:#fff;')
    results_empty.layout.display = 'none'; trajectory_box.layout.display = ''
    trajectory_content.layout.display = ''; energy_panel.layout.display = ''
    energy_choice.layout.display = 'none'
    traj_choice.layout.display = ''; traj_label.layout.display = ''
    energy_panel_title.value = '<div class="rxpath-panel-title">Interactive %dD PES</div>' % view['dims']
    path_grid.remove_class('rxplot-only'); path_grid.remove_class('rxstructure-only')

    if not frames:
        _TRAJ.update(frames=[], energies=[], energy_provenance=[], energy_unit='kcal/mol',
                     quantity_label='\u0394E', path=_SCAN_GRID['token'], semantics={},
                     state_labels=[], structure_paths=[], mode='grid', generation=generation,
                     frame_message={}, structure_message={})
        frame_slider.max = 1; frame_play.max = 1
        frame_slider.value = 0; frame_play.value = 0
        frame_slider.disabled = True; frame_play.disabled = True
        frame_prev.disabled = True; frame_next.disabled = True
        frame_controls.layout.display = 'none'; scan_axis_controls.layout.display = 'none'; structure_panel.layout.display = 'none'
        path_grid.add_class('rxplot-only')
        traj_out.value = ''; traj_signal_out.value = ''
        _TRAJ_VIEWER_MOUNTED['value'] = False
        trajectory_info.layout.display = 'none'
        trajectory_intro.value = ('<div><span class="rxpath-title">Interactive %dD potential-energy surface</span>'
                                  '<span class="rxpath-subtitle">Rotate, zoom, and inspect the Plotly result directly.</span></div>' % view['dims'])
        detail = (' · %s' % html.escape(link_error)) if link_error else ''
        frame_state.value = ('<div role="status"><b>Interactive Plotly surface</b> · '
                             'Structure linking is unavailable for this output%s.</div>' % detail)
        traj_label.value = ('<small><code>%s</code> · interactive plot</small>' %
                            html.escape(os.path.basename(view['html'])))
        return

    values = [float(value) if value is not None and math.isfinite(float(value)) else None
              for value in view['energies']]
    semantics = {'title': '%dD scan grid' % view['dims'], 'start': 'grid start',
                 'end': 'grid end', 'x': 'grid point', 'extrema': False}
    _TRAJ.update(frames=frames, energies=values, energy_provenance=['surface.csv'] * len(frames),
                 energy_unit='kcal/mol', quantity_label='\u0394E', path=_SCAN_GRID['token'],
                 semantics=semantics, state_labels=view['labels'],
                 structure_paths=view['paths'], mode='grid', generation=generation,
                 frame_message={}, structure_message={})
    _mount_result_models(frames, generation)
    last = len(frames) - 1; safe_last = max(1, last)
    frame_slider.max = safe_last; frame_play.max = safe_last
    frame_slider.value = 0; frame_play.value = 0
    frame_slider.disabled = last == 0; frame_play.disabled = True
    frame_prev.disabled = True; frame_next.disabled = last == 0
    frame_controls.layout.display = 'none'; structure_panel.layout.display = ''
    _configure_scan_axis_controls(view)
    trajectory_intro.value = ('<div><span class="rxpath-title">%dD PES &amp; optimized grid structures</span>'
                              '<span class="rxpath-subtitle">Select a computed point in the plot or use the linked controls.</span></div>' % view['dims'])
    traj_label.value = ('<small><code>%s</code> · %d linked structures</small>' %
                        (html.escape(os.path.basename(view['html'])), len(frames)))
    _show_frame(0)

def _on_traj_choice(_=None):
    if _result_pick_guard['active']: return
    label = next((text for text, value in traj_choice.options
                  if value == traj_choice.value), 'result view')
    _set_result_loading(label, True)
    try:
        matching_energy = next((token for token, view in (_ENERGY.get('views') or {}).items()
                                if view.get('mode') == 'profile' and
                                view.get('path') == traj_choice.value), None)
        _result_pick_guard['active'] = True
        try:
            if matching_energy:
                energy_choice.value = matching_energy
                energy_choice.layout.display = ''
            else:
                energy_choice.layout.display = 'none'
        finally:
            _result_pick_guard['active'] = False
        if traj_choice.value and traj_choice.value == _SCAN_GRID.get('token'):
            _load_scan_grid()
        else:
            _load_trajectory()
    finally:
        _set_result_loading(label, False)
artifact_choice.observe(_render_artifact, names='value')
traj_choice.observe(_on_traj_choice, names='value')

def _reset_result_presentation():
    """Begin a result-set transaction with no state from the previous run."""
    _TRAJ_VIEWER_GENERATION['value'] += 1
    _RESULT_SET_GENERATION['value'] += 1
    _TRAJ.update(frames=[], energies=[], energy_provenance=[], energy_unit='hartree',
                 quantity_label='ΔE', path=None, semantics={}, state_labels=[],
                 structure_paths=[], mode='trajectory',
                 generation=_TRAJ_VIEWER_GENERATION['value'], frame_message={},
                 structure_message={})
    _ENERGY.update(mep_trajectory=None, aggregate_irc=None, views={})
    _SCAN_GRID.update(token=None, view=None)
    _result_pick_guard['active'] = True
    try:
        energy_choice.options = []; energy_choice.disabled = True
        artifact_choice.options = []; artifact_choice.disabled = True
        traj_choice.options = []; traj_choice.disabled = True
    finally:
        _result_pick_guard['active'] = False
    energy_choice.layout.display = 'none'; result_selector_row.layout.display = 'none'
    result_selector_meta.value = ''; artifact_out.value = ''
    traj_out.value = ''; plot_out.value = ''; traj_signal_out.value = ''
    _TRAJ_VIEWER_MOUNTED['value'] = False
    trajectory_box.layout.display = 'none'; scan_axis_controls.layout.display = 'none'

def _results(out=None):
    out = out or S.get('_last_out_dir') or _effective_result_root()
    _reset_result_presentation()
    path_grid.remove_class('rxplot-only'); path_grid.remove_class('rxstructure-only')
    scan_axis_controls.layout.display = 'none'
    current = _current_run_files(out)
    manifest = S.get('_last_manifest') or {}
    result_context.value = _result_context_html(out)
    if manifest.get('restored'):
        result_context.value += ('<div style="margin-top:6px;color:#475569;font-size:12px">'
                                 'Recovered existing result · read-only preview</div>')
    result_messages = []
    rejected_claims = list(manifest.get('rejected_claims') or [])
    if rejected_claims:
        shown = ', '.join(os.path.basename(str(path)) for path in rejected_claims[:3])
        more = ' (+%d more)' % (len(rejected_claims) - 3) if len(rejected_claims) > 3 else ''
        result_messages.append('Declared output unavailable: %s%s.' % (shown, more))
    if True:  # keep status and summary construction local; publish once below
        if not current and manifest.get('stdout_only'):
            result_messages.append('This utility writes to standard output; inspect the run log below.')
        elif not current and manifest.get('status') == 'failed':
            result_messages.append('Command failed (exit %s). Inspect the run log; no current-run artifact was produced.' % manifest.get('exit_code', '?'))
        elif not current and manifest.get('status') == 'cancelled':
            result_messages.append('Command was cancelled (exit 130). Inspect the run log; no current-run artifact was produced.')
        elif not current and manifest:
            result_messages.append('The current run finished without a file artifact. Inspect the run log and command status.')
        elif not current:
            result_messages.append('No current GUI run is recorded for %s. Run a workflow first; files from previous runs are intentionally not shown.' % out)
        sub = S.get('_last_subcmd') or S.get('subcmd') or 'all'
        def _preview_priority(path):
            name = os.path.basename(path).lower(); kind = _artifact_kind(path)
            if sub in ('all', 'path-opt', 'path-search', 'irc', 'scan', 'scan2d', 'scan3d') and kind in ('image', 'interactive HTML', 'MEP profile', 'IRC trajectory'): return 0
            if sub in ('opt', 'tsopt', 'extract', 'fix-altloc', 'add-elem-info') and kind == 'structure': return 0
            if sub == 'freq' and ('frequenc' in name or 'thermo' in name): return 0
            return {'result.json': 2, 'result.yaml': 3, 'result.yml': 3,
                    'thermoanalysis.yaml': 4, 'frequencies_cm-1.txt': 5,
                    'summary.json': 20}.get(name, 10)
        visuals = sorted((p for p in current if _artifact_kind(p)),
                         key=lambda p: (_preview_priority(p), p))
        summaries = sorted((p for p in current if os.path.basename(p) == 'summary.json'),
                           key=lambda p: p.count(os.sep))
        sh = _summary_html(summaries[0] if summaries else None)
        if sh: result_messages.append(sh)
    res_out.value = ''.join(
        item if str(item).lstrip().startswith('<') else
        '<div role="status" style="margin:4px 0">%s</div>' % html.escape(str(item))
        for item in result_messages)
    last_sub = S.get('_last_subcmd') or ''
    prefer_irc = last_sub in ('tsopt', 'irc')
    composite = last_sub in ('all', 'path-opt', 'path-search')
    def _trajectory_rank(path):
        name = os.path.basename(path).lower()
        try: depth = len(Path(os.path.relpath(path, out)).parts)
        except ValueError: depth = 999
        is_mep = 'mep_trj' in name
        is_irc = 'finished_irc_trj' in name
        if composite:
            # The root mechanism path is the default overview. Segment IRCs
            # remain selectable without replacing that overview.
            group = 0 if is_mep and depth == 1 else (1 if is_mep else (2 if is_irc else 3))
        else:
            group = 0 if (prefer_irc and is_irc) else (1 if is_mep else (2 if is_irc else 3))
        return (group, depth, path)
    cand = sorted((p for p in current if Path(p).suffix.lower() == '.xyz' and
                   ('trj' in os.path.basename(p).lower() or 'trajectory' in os.path.basename(p).lower())),
                  key=_trajectory_rank)
    aggregate_irc = _aggregate_irc_trajectory(current, out) if composite else None
    _ENERGY['aggregate_irc'] = aggregate_irc
    if aggregate_irc and aggregate_irc not in cand: cand.append(aggregate_irc)
    _SCAN_GRID['view'] = _scan_grid_view(current, out, last_sub) if last_sub in ('scan2d', 'scan3d') else None
    _SCAN_GRID['token'] = ('scan-grid:%s:%s' % (last_sub, os.path.abspath(out))
                           if _SCAN_GRID['view'] else None)
    result_views = _result_view_candidates(current, out, last_sub)
    energy_options, mep_trajectory = _energy_diagram_options(current, cand, out, last_sub)
    is_frequency_result = last_sub == 'freq'
    traj_choice.description = 'Mode' if is_frequency_result else 'Result'
    show_result_selector = bool(result_views) and (is_frequency_result or len(result_views) > 1)
    result_selector_row.layout.display = '' if show_result_selector else 'none'
    result_selector_meta.value = (
        '<span data-rx-result-generation="%d" hidden></span>' %
        _RESULT_SET_GENERATION['value'])
    _result_pick_guard['active'] = True
    try:
        _ENERGY['mep_trajectory'] = mep_trajectory
        energy_choice.options = energy_options
        energy_choice.disabled = not energy_options
        energy_choice.layout.display = '' if energy_options else 'none'
        if energy_options: energy_choice.value = energy_options[0][1]
        artifact_choice.options = [('%s · %s' % (_artifact_kind(p), os.path.relpath(p, out)), p)
                                   for p in visuals]
        artifact_choice.disabled = not visuals
        if visuals: artifact_choice.value = visuals[0]
        primary_path = result_views[0][1] if result_views else None
        traj_choice.options = result_views
        traj_choice.disabled = not bool(result_views)
        if primary_path: traj_choice.value = primary_path
    finally:
        _result_pick_guard['active'] = False
    artifact_fold.layout.display = '' if visuals else 'none'
    # A linked structure/energy result is the primary presentation. Generic
    # images remain available on demand without being emitted beside the GUI.
    artifact_fold._rx_set_open(False)
    artifact_fold._rx_set_open(bool(visuals and not result_views and not energy_options))
    # Render the selected result first. Loading a primary MEP/IRC trajectory
    # before a stationary-state diagram briefly mounted 22/51 models and made
    # the following asynchronous R/TS/P replacement order-dependent.
    if energy_options:
        selected_energy_view = (_ENERGY.get('views') or {}).get(energy_choice.value) or {}
        if primary_path and selected_energy_view.get('mode') == 'image':
            # A static energy image does not replace the linked structure.
            # Reload the selected primary result so loading cannot leave a
            # previous trajectory (for example an imaginary mode) mounted.
            _load_trajectory(primary_path, out)
            _render_energy_choice()
        else:
            _render_energy_choice()
    else:
        if primary_path == _SCAN_GRID.get('token'):
            _load_scan_grid()
        else:
            _load_trajectory(primary_path, out)
    if not result_views and not energy_options:
        results_empty.layout.display = ''
        state = manifest.get('status')
        results_empty.value = ('<div role="status" style="padding:10px;border:1px dashed #64748b;border-radius:10px;">%s</div>' %
                               ('No trajectory was produced; the generated file preview is below.' if current and visuals else
                                'This result has no XYZ trajectory. Inspect its status and files above.' if current else
                                'The command failed before producing a trajectory; inspect the run log.' if state == 'failed' else
                                'The command was cancelled before producing a trajectory.' if state == 'cancelled' else
                                'This run produced no XYZ trajectory.' if manifest else
                                'Run a workflow to inspect its current trajectory here.'))
        traj_label.value = '<small>no XYZ trajectory found for this result</small>'
    res_btn.disabled = False
    dl_btn.disabled = not bool(current or manifest or S.get('_last_log'))
    S['_results_presented_dir'] = os.path.abspath(out)
def _begin_results_attempt(out, sub):
    """Detach the prior run presentation before a new real attempt starts."""
    S['_results_presented_dir'] = None
    _RESULT_SET_GENERATION['value'] += 1
    _TRAJ_VIEWER_GENERATION['value'] += 1
    _TRAJ.update(frames=[], energies=[], energy_provenance=[],
                 energy_unit='hartree', path=None, semantics={},
                 state_labels=[], structure_paths=[], mode='trajectory',
                 generation=_TRAJ_VIEWER_GENERATION['value'],
                 frame_message={}, structure_message={})
    _result_pick_guard['active'] = True
    try:
        energy_choice.options = []; energy_choice.disabled = True
        energy_choice.layout.display = 'none'; _ENERGY.update(mep_trajectory=None, aggregate_irc=None, views={})
        artifact_choice.options = []; artifact_choice.disabled = True
        traj_choice.options = []; traj_choice.disabled = True
        frame_slider.max = 1; frame_play.max = 1
        frame_slider.value = 0; frame_play.value = 0
        frame_slider.disabled = True; frame_play.disabled = True
        frame_prev.disabled = True; frame_next.disabled = True
    finally:
        _result_pick_guard['active'] = False
    res_out.value = ''; artifact_out.value = ''; result_selector_meta.value = ''
    result_selector_row.layout.display = 'none'
    for panel in (traj_out, plot_out, traj_signal_out):
        panel.value = ''
    _TRAJ_VIEWER_MOUNTED['value'] = False
    artifact_fold.layout.display = 'none'; trajectory_box.layout.display = 'none'
    scan_axis_controls.layout.display = 'none'
    result_context.value = ('<div style="border:1px solid #fde68a;border-radius:11px;'
                            'padding:7px 9px;background:#fffbeb"><b>%s</b> · '
                            '<small>run in progress · <code>%s</code></small></div>' %
                            (html.escape(str(sub)), html.escape(str(out))))
    results_empty.layout.display = ''
    results_empty.value = ('<div role="status" style="padding:10px;border:1px dashed #d97706;'
                           'border-radius:10px;color:#92400e">Run in progress…</div>')
    traj_label.value = ''; frame_state.value = ''; trajectory_intro.value = ''
    res_btn.disabled = True; dl_btn.disabled = True

results_dir = W.Text(value='', placeholder='/content/result_all',
                     description='Results directory',
                     style={'description_width': 'initial'},
                     layout=W.Layout(width='420px', max_width='100%', min_width='0'))
res_btn = W.Button(description='Load results', icon='folder-open', disabled=False,
                   tooltip='Load an existing results directory.',
                   layout=W.Layout(width='180px'))
def _load_results(_):
    raw = str(results_dir.value or '').strip()
    out = os.path.abspath(os.path.expanduser(raw or _effective_result_root()))
    if not os.path.isdir(out):
        res_out.value = ('<div role="alert" style="color:#991b1b">Results directory not found: '
                         '<code>%s</code></div>' % html.escape(out))
        return
    if not _recover_existing_results(out, replace=True):
        res_out.value = ('<div role="alert" style="color:#991b1b">No completed '
                         'pdb2reaction result manifest was found in <code>%s</code>.</div>' %
                         html.escape(out))
        return
    results_dir.value = out
    _results(out)
res_btn.on_click(_load_results)
dl_btn = W.Button(description='Download current run (.zip)', icon='download', disabled=True,
                  tooltip='Download the files and log from the current run.',
                  layout=W.Layout(width='300px', max_width='100%'))
def _download(_):
    out = S.get('_last_out_dir') or _effective_result_root()
    current = _current_run_files(out)
    manifest = S.get('_last_manifest') or {}
    transcript = S.get('_last_log') or ''
    if not (current or manifest or transcript):
        res_out.value = '<div role="alert">No current run is available to bundle.</div>'
        return
    stem = Path(os.path.abspath(out)).name or 'results'
    z = _unique_path(_runtime_path('downloads', stem + '_current_run.zip'))
    with zipfile.ZipFile(z, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        archived = set()
        external_index = 0
        for path in current:
            rel = os.path.relpath(path, out)
            if rel.startswith('..'):
                external_index += 1
                arcname = 'other_outputs/%02d_%s' % (external_index, os.path.basename(path))
            else:
                arcname = rel
            archive.write(path, arcname)
            archived.add(arcname)
        archive.writestr('colab_run.json', json.dumps(manifest, indent=2))
        if transcript and 'run.log' not in archived: archive.writestr('run.log', transcript)
    try:
        from google.colab import files as _f; _f.download(z)
    except Exception as e:
        res_out.value = ('<div role="status">Download needs Colab; zip saved at <code>%s</code> (%s)</div>' %
                         (html.escape(str(z)), html.escape(str(e))))
dl_btn.on_click(_download)
results_empty = W.HTML(
    '<div role="status" style="padding:10px;border:1px dashed #64748b;border-radius:10px;">'
    'No results yet.</div>')
artifact_box = W.VBox([artifact_choice, artifact_out])
artifact_fold = _collapsible('Generated file preview', artifact_box, on_open=_render_artifact)
artifact_fold.layout.display = 'none'
artifact_fold.add_class('rxartifact')
trajectory_info = _info_control('Trajectory context.')
trajectory_info.layout.display = 'none'
results_energy_info = _info_control(
    '<code>ΔE</code> is relative electronic energy; <code>ΔG</code> includes thermochemistry. '
    'MEP and IRC show path energies. Treat an MEP maximum as a transition state only after '
    'TS optimization finds one imaginary mode and IRC confirms both endpoints. Values are '
    'relative to the first point shown.', rich=True)
trajectory_state = W.HBox(
    [frame_state, trajectory_info],
    layout=W.Layout(align_items='center', flex_flow='row wrap',
                    max_width='100%', min_width='0'))
trajectory_state.add_class('rxpath-state')
trajectory_intro_group = W.HBox([trajectory_intro, results_energy_info],
                                  layout=W.Layout(align_items='center'))
trajectory_head = W.VBox([trajectory_intro_group, result_loading, result_selector_row, trajectory_state],
                          layout=W.Layout(width='100%', align_items='stretch'))
trajectory_head.add_class('rxpath-head')
frame_controls = W.HBox([frame_prev, frame_next, frame_play, frame_slider, playback_interval_signal],
                         layout=W.Layout(width='100%', align_items='center'))
frame_controls.add_class('rxpath-controls')
structure_panel_title = W.HTML('<div class="rxpath-panel-title">Structure Viewer</div>')
structure_panel_info = _info_control(
    'The displayed structure follows the selected result or grid point. '
    'Distance and angle measurements update when the selected point changes.')
structure_panel_head = W.HBox(
    [structure_panel_title, structure_panel_info],
    layout=W.Layout(width='100%', align_items='center', flex_flow='row wrap'))
structure_panel_head.add_class('rxstructure-panel-head')
structure_panel = W.VBox([structure_panel_head, traj_out, traj_signal_out])
energy_panel_head = W.HBox([energy_panel_title, energy_choice],
                          layout=W.Layout(width='100%', align_items='center'))
energy_panel_head.add_class('rxpath-panel-head')
energy_panel = W.VBox([energy_panel_head, plot_out])
structure_panel.add_class('rxpath-panel'); energy_panel.add_class('rxpath-panel')
path_grid = W.HBox([structure_panel, energy_panel], layout=W.Layout(
    width='100%', max_width='100%', min_width='0', overflow='hidden'))
path_grid.add_class('rxpath-grid')
trajectory_content = W.VBox([frame_controls, scan_axis_controls, path_grid])
trajectory_box = W.VBox([trajectory_head, trajectory_content])
trajectory_box.layout.display = 'none'
trajectory_box.add_class('rxresults'); trajectory_box.add_class('rxpath')
results_box = W.VBox([
    W.HBox([results_dir, res_btn, dl_btn], layout=W.Layout(
        width='100%', flex_flow='row wrap', align_items='center',
        justify_content='flex-start')),
    trajectory_box, res_out, results_empty, result_context, artifact_fold])
results_box.add_class('rxresults')

# ============================================================== command line + run (bottom, PyMOL-style)
_RUN_LOG_BUFFER = {'text': ''}
_RUN_LOG_PENDING = {'scheduled': False}
_RUN_LOG_LAST_VIEW = {'text': None}
_RUN_LOG_PATCH_SEQ = {'value': 0}
_RUN_LOG_RECOVERY = {'last_snapshot': 0.0}
_RUN_LOG_LOCK = threading.Lock()
_RUN_LOG_INTERVAL = 0.20

def _run_log_markup(text):
    body = html.escape(text or "Ready to run. Click 'Validate' or 'Run' to begin.")
    return '<pre role="log" aria-live="polite">%s</pre>' % body

if _HAS_DROP_WIDGET and (IN_COLAB or not IS_COLAB_FRONTEND):
    class _RunLogWidget(anywidget.AnyWidget):
        _esm = r"""
        export default { render({ model, el }) {
          const shell = document.createElement("div");
          shell.className = "rxlog-shell";
          const host = document.createElement("div");
          host.className = "rxlog-console";
          host.dataset.rxFollowLog = "smart";
          const rail = document.createElement("div");
          rail.className = "rxlog-scrollbar";
          rail.setAttribute("role", "scrollbar");
          rail.setAttribute("aria-orientation", "vertical");
          const thumb = document.createElement("div");
          thumb.className = "rxlog-scroll-thumb";
          rail.appendChild(thumb);
          const style = document.createElement("style");
          style.textContent = `
            .rxlog-shell{position:relative;box-sizing:border-box;width:100%;}
            .rxlog-console{box-sizing:border-box;width:100%;height:clamp(280px,52vh,420px);min-height:180px;overflow-y:scroll;overflow-x:hidden;scrollbar-width:none;overscroll-behavior:contain;border:1px solid #263449;border-radius:10px;background:#0f172a;}
            .rxlog-console pre{box-sizing:border-box;width:100%;min-height:100%;margin:0;padding:10px 28px 10px 12px;white-space:pre-wrap;overflow-wrap:anywhere;border:0;background:transparent;color:#d1fae5;font:12px/1.5 "DejaVu Sans Mono","Liberation Mono",Consolas,"Courier New",monospace;}
            .rxlog-console::-webkit-scrollbar{width:0;height:0;}
            .rxlog-scrollbar{position:absolute;z-index:3;top:7px;right:5px;bottom:7px;width:11px;box-sizing:border-box;border:1px solid #475569;border-radius:999px;background:#1e293b;cursor:pointer;}
            .rxlog-scrollbar.rxinactive{opacity:.45;}
            .rxlog-scroll-thumb{position:absolute;left:1px;right:1px;top:0;min-height:32px;border-radius:999px;background:#94a3b8;box-shadow:0 0 0 1px rgba(15,23,42,.5);cursor:grab;touch-action:none;}
            .rxlog-scroll-thumb:active{cursor:grabbing;background:#cbd5e1;}
          `;
          const pre = document.createElement("pre");
          pre.setAttribute("role", "log");
          pre.setAttribute("aria-live", "polite");
          host.appendChild(pre);
          shell.append(host, rail);
          el.replaceChildren(style, shell);
          const isAtBottom = () => Math.max(0, host.scrollHeight - host.scrollTop - host.clientHeight) <= 24;
          const updateThumb = () => {
            const track = Math.max(1, rail.clientHeight);
            const maximum = Math.max(0, host.scrollHeight - host.clientHeight);
            const thumbHeight = maximum ? Math.max(32, Math.round(track * host.clientHeight / host.scrollHeight)) : track;
            const travel = Math.max(0, track - thumbHeight);
            const top = maximum ? Math.round(travel * host.scrollTop / maximum) : 0;
            thumb.style.height = `${thumbHeight}px`;
            thumb.style.transform = `translateY(${top}px)`;
            rail.classList.toggle("rxinactive", maximum === 0);
            rail.setAttribute("aria-valuemin", "0");
            rail.setAttribute("aria-valuemax", String(Math.round(maximum)));
            rail.setAttribute("aria-valuenow", String(Math.round(host.scrollTop)));
          };
          host.addEventListener("scroll", updateThumb, { passive: true });
          rail.addEventListener("pointerdown", event => {
            if (event.target === thumb) return;
            const rect = rail.getBoundingClientRect();
            const ratio = Math.max(0, Math.min(1, (event.clientY - rect.top) / Math.max(1, rect.height)));
            host.scrollTop = ratio * Math.max(0, host.scrollHeight - host.clientHeight);
          });
          let drag = null;
          thumb.addEventListener("pointerdown", event => {
            drag = { y: event.clientY, scrollTop: host.scrollTop };
            thumb.setPointerCapture(event.pointerId);
            event.preventDefault();
          });
          thumb.addEventListener("pointermove", event => {
            if (!drag) return;
            const maximum = Math.max(0, host.scrollHeight - host.clientHeight);
            const travel = Math.max(1, rail.clientHeight - thumb.offsetHeight);
            host.scrollTop = drag.scrollTop + (event.clientY - drag.y) * maximum / travel;
            event.preventDefault();
          });
          const stopDrag = () => { drag = null; };
          thumb.addEventListener("pointerup", stopDrag);
          thumb.addEventListener("pointercancel", stopDrag);
          const resizeObserver = new ResizeObserver(updateThumb);
          resizeObserver.observe(host);
          resizeObserver.observe(pre);
          const settleScroll = (shouldFollow) => {
            if (!shouldFollow) return;
            const follow = () => { host.scrollTop = host.scrollHeight; updateThumb(); };
            requestAnimationFrame(() => {
              follow();
              setTimeout(follow, 0);
              setTimeout(follow, 60);
            });
          };
          const apply = () => {
            const shouldFollow = !pre.textContent || isAtBottom();
            pre.textContent = String(model.get("snapshot") || "");
            settleScroll(shouldFollow);
            requestAnimationFrame(updateThumb);
          };
          const applyPatch = () => {
            let payload;
            try { payload = JSON.parse(String(model.get("patch") || "")); }
            catch (_error) { return; }
            const shouldFollow = isAtBottom();
            if (payload.reset) pre.textContent = String(payload.text || "");
            else if (payload.text) pre.append(document.createTextNode(String(payload.text)));
            settleScroll(shouldFollow);
            requestAnimationFrame(updateThumb);
          };
          model.on("change:snapshot", apply);
          model.on("change:patch", applyPatch);
          apply();
          return () => {
            resizeObserver.disconnect();
            model.off("change:snapshot", apply);
            model.off("change:patch", applyPatch);
          };
        } };
        """
        snapshot = traitlets.Unicode(
            "Ready to run. Click 'Validate' or 'Run' to begin.").tag(sync=True)
        patch = traitlets.Unicode('').tag(sync=True)
        active = traitlets.Bool(False).tag(sync=True)
    try:
        logbox = _RunLogWidget(layout=W.Layout(width='100%'))
        _RUN_LOG_INCREMENTAL = True
    except Exception:
        # A partially mocked or unavailable custom-widget manager must not
        # prevent the GUI from launching; the browser bridge follows this log.
        logbox = W.HTML(value=_run_log_markup(''),
                        layout=W.Layout(width='100%', overflow='visible'))
        _RUN_LOG_INCREMENTAL = False
else:
    logbox = W.HTML(
        value=_run_log_markup(''),
        layout=W.Layout(width='100%', overflow='visible'))
    _RUN_LOG_INCREMENTAL = False
def _stop_child(process):
    if process.poll() is not None: return
    try:
        if os.name == 'posix': os.killpg(os.getpgid(process.pid), signal.SIGTERM)
        else: process.terminate()
    except ProcessLookupError:
        pass
    try: process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        try:
            if os.name == 'posix': os.killpg(os.getpgid(process.pid), signal.SIGKILL)
            else: process.kill()
        except ProcessLookupError:
            pass
        process.wait(timeout=5)

def _dispatch_ui(callback):
    """Run immediately on the live GUI loop; marshal only across loops."""
    if not IN_COLAB or threading.current_thread() is threading.main_thread():
        return callback()
    worker = _RUN_EXECUTION.get('thread') if '_RUN_EXECUTION' in globals() else None
    if worker is not None and threading.current_thread() is worker:
        # Standard ipywidgets must publish from Colab's live UI loop.  The
        # custom run-log widget can stream from this worker, but direct
        # Button/HTML/Layout trait writes can be silently dropped.
        try:
            if _UI_IOLOOP is not None:
                _UI_IOLOOP.add_callback(callback)
                return None
        except Exception:
            pass
        try:
            if _UI_ASYNC_LOOP is not None and _UI_ASYNC_LOOP.is_running():
                _UI_ASYNC_LOOP.call_soon_threadsafe(callback)
                return None
        except Exception:
            pass
        return callback()
    try:
        if (_UI_ASYNC_LOOP is not None and _UI_ASYNC_LOOP.is_running() and
                asyncio.get_running_loop() is _UI_ASYNC_LOOP):
            # Colab button callbacks may own the GUI loop from a non-main
            # kernel thread. Re-queueing onto that same loop can strand the
            # completion callback after a long child process exits.
            return callback()
    except RuntimeError:
        pass
    try:
        if _UI_ASYNC_LOOP is not None and _UI_ASYNC_LOOP.is_running():
            _UI_ASYNC_LOOP.call_soon_threadsafe(callback)
            return None
    except Exception:
        pass
    try:
        if _UI_IOLOOP is not None:
            _UI_IOLOOP.add_callback(callback)
            return None
    except Exception:
        pass
    return callback()

def _visible_run_log_text():
    return _RUN_LOG_BUFFER['text']

def _flush_run_log(force=False):
    """Publish deltas quickly and occasionally resend a full recovery snapshot."""
    with _RUN_LOG_LOCK:
        _RUN_LOG_PENDING['scheduled'] = False
        view = _visible_run_log_text()
    now = time.monotonic()
    recovery_reset = bool(force and now - _RUN_LOG_RECOVERY['last_snapshot'] >= 15.0)
    if view == _RUN_LOG_LAST_VIEW['text'] and not recovery_reset:
        return
    try:
        if _RUN_LOG_INCREMENTAL:
            previous = _RUN_LOG_LAST_VIEW['text']
            reset = recovery_reset or previous is None or not view.startswith(previous)
            delta = view if reset else view[len(previous):]
            _RUN_LOG_PATCH_SEQ['value'] += 1
            logbox.patch = json.dumps({'seq': _RUN_LOG_PATCH_SEQ['value'],
                                       'reset': reset, 'text': delta}, separators=(',', ':'))
            logbox.send_state('patch')
        else:
            logbox.value = _run_log_markup(view)
            logbox.send_state()
    except Exception:
        # Do not mark the snapshot as delivered. A low-rate recovery poll will
        # retry it instead of leaving the visible log permanently stale.
        return
    _RUN_LOG_LAST_VIEW['text'] = view
    if recovery_reset:
        _RUN_LOG_RECOVERY['last_snapshot'] = now

def _arm_run_log_flush():
    try:
        if _UI_ASYNC_LOOP is not None and _UI_ASYNC_LOOP.is_running():
            _UI_ASYNC_LOOP.call_later(_RUN_LOG_INTERVAL, _flush_run_log)
            return
    except Exception:
        pass
    try:
        if _UI_IOLOOP is not None:
            _UI_IOLOOP.call_later(_RUN_LOG_INTERVAL, _flush_run_log)
            return
    except Exception:
        pass
    _flush_run_log()

def _run_log_emit(text, clear=False, flush=False):
    """Coalesce output into responsive snapshots while retaining every line."""
    chunk = str(text)
    with _RUN_LOG_LOCK:
        if clear: _RUN_LOG_BUFFER['text'] = chunk
        else: _RUN_LOG_BUFFER['text'] += chunk
        schedule = not _RUN_LOG_PENDING['scheduled']
        if schedule: _RUN_LOG_PENDING['scheduled'] = True
    if flush:
        return _dispatch_ui(lambda: _flush_run_log(force=True))
    if schedule:
        return _dispatch_ui(_arm_run_log_flush)

def _command_option(argv, *flags):
    for index, token in enumerate(argv[:-1]):
        if token in flags: return argv[index + 1]
    return None

def _stream(cmd):
    _open_run_log()
    command_line = '$ ' + ' '.join(shlex.quote(c) for c in cmd) + '\n\n'
    transcript = [command_line]
    _run_log_emit(command_line, clear=True, flush=True)
    # GPU4PySCF reports its normal CuPy contraction fallback in every child
    # process. Keep that one informational warning out of the user run log.
    _tensor_warning_filter = ('ignore:using cupy as the tensor contraction '
                              'engine.:UserWarning')
    _warning_filters = [os.environ.get('PYTHONWARNINGS', '').strip(),
                        _tensor_warning_filter]
    _child_env = {**os.environ, 'PYTHONUNBUFFERED': '1',
                  'PYTHONIOENCODING': 'utf-8',
                  'PYTHONWARNINGS': ','.join(value for value in _warning_filters if value)}
    try:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             encoding='utf-8', errors='replace', bufsize=1,
                             env=_child_env,
                             start_new_session=(os.name == 'posix'))
    except OSError as exc:
        line = '[launch failed] %s\n[exit 127]\n' % exc
        transcript.append(line); _run_log_emit(line)
        return 127, ''.join(transcript)
    _RUN_EXECUTION['process'] = p
    try:
        if _RUN_EXECUTION['cancel'].is_set():
            _stop_child(p)
        # Read stdout on a daemon thread so an inherited pipe held by a
        # backend helper cannot keep the GUI waiting after the CLI exits.
        lines = queue.Queue()
        reader_done = threading.Event()
        reader_error = []
        def _read_stdout():
            try:
                for item in p.stdout:
                    lines.put(item)
            except Exception as exc:
                reader_error.append(exc)
            finally:
                reader_done.set()
        threading.Thread(target=_read_stdout, name='%s-log-reader' % TOOL,
                         daemon=True).start()
        exit_seen = None
        while True:
            try:
                line = lines.get(timeout=0.10)
                transcript.append(line); _run_log_emit(line)
            except queue.Empty:
                pass
            while True:
                try: line = lines.get_nowait()
                except queue.Empty: break
                transcript.append(line); _run_log_emit(line)
            if reader_done.is_set() and reader_error:
                raise reader_error[0]
            if _RUN_EXECUTION['cancel'].is_set():
                _stop_child(p)
                break
            if p.poll() is not None:
                if exit_seen is None: exit_seen = time.monotonic()
                if reader_done.is_set() or time.monotonic() - exit_seen >= 0.50:
                    break
        if p.poll() is None: p.wait()
        while True:
            try: line = lines.get_nowait()
            except queue.Empty: break
            transcript.append(line); _run_log_emit(line)
        if _RUN_EXECUTION['cancel'].is_set():
            line = '\n[cancel] stopping the current task …\n'
            transcript.append(line); _run_log_emit(line)
            _stop_child(p)
            line = '[exit 130 · cancelled]\n'
            transcript.append(line); _run_log_emit(line)
            return 130, ''.join(transcript)
    except KeyboardInterrupt:
        line = '\n[interrupt] stopping the current task …\n'
        transcript.append(line); _run_log_emit(line)
        _stop_child(p)
        line = '[exit 130 · cancelled]\n'
        transcript.append(line); _run_log_emit(line)
        return 130, ''.join(transcript)
    except Exception as exc:
        _stop_child(p)
        line = '\n[stream failed] %s\n[exit 125]\n' % exc
        transcript.append(line); _run_log_emit(line)
        return 125, ''.join(transcript)
    finally:
        if p.stdout is not None: p.stdout.close()
        if _RUN_EXECUTION.get('process') is p:
            _RUN_EXECUTION['process'] = None
    # A successful run is announced by the status chip and the Results tab;
    # only a non-zero code needs the reader's attention here.
    if p.returncode:
        trailer = '\n[exit %d]\n' % p.returncode
        transcript.append(trailer); _run_log_emit(trailer, flush=True)
    else:
        _run_log_emit('', flush=True)
    return p.returncode, ''.join(transcript)

def _argv():
    line = cmd_box.value.strip()
    if not line or line.startswith('#'):
        _run_log_emit('No command yet — load an input (Input tab).\n', clear=True)
        _open_run_log()
        return None
    try:
        return shlex.split(line)   # execute exactly what the editable command line shows
    except ValueError as exc:
        _run_log_emit('Invalid command line: %s\n' % exc, clear=True)
        _set_run_status('✗ invalid command', 'error', 'invalid')
        _open_run_log()
        return None

def _normalized_scope_argv(argv):
    """Normalize argv exactly as the root CLI before classifying outputs."""
    args = list(argv or [])
    if len(args) < 2: return args
    first = args[1]
    if first in ('-h', '--help', '--version'): return args
    try:
        from pdb2reaction.cli.bool_compat import normalize_argv_option_names
        args = [args[0], *normalize_argv_option_names(args[1:])]
        first = args[1]
    except Exception:
        pass
    try:
        import click
        from pdb2reaction.cli import cli as root_cli
        root_ctx = click.Context(root_cli, info_name=CLI, resilient_parsing=True)
        if root_cli.get_command(root_ctx, first) is None:
            top_level = set()
            for param in root_cli.params:
                top_level.update(getattr(param, 'opts', ()) or ())
                top_level.update(getattr(param, 'secondary_opts', ()) or ())
            if first.startswith('--'):
                is_top_level = first.split('=', 1)[0] in top_level
            elif first.startswith('-') and len(first) >= 2:
                is_top_level = first[:2] in top_level
            else:
                is_top_level = False
            if first.startswith('-') and not is_top_level:
                args.insert(1, 'all')
        if len(args) > 1 and not args[1].startswith('-'):
            command_name = args[1]
            value_opts, toggle_opts, negative_aliases, single_flags = (
                root_cli._resolve_bool_options(root_ctx, command_name))
            normalized, _legacy = root_cli._normalize_bool_argv(
                args[1:], {command_name: value_opts},
                {command_name: toggle_opts}, {command_name: negative_aliases},
                {command_name: single_flags})
            args = [args[0], *normalized]
    except Exception:
        pass
    return args

def _effective_out_dir(argv=None):
    """Return the last output path in the editable command, with GUI fallback."""
    if argv is None:
        line = cmd_box.value.strip()
        try: argv = shlex.split(line) if line and not line.startswith('#') else []
        except ValueError: argv = []
    argv = _normalized_scope_argv(argv)
    out = None
    flags = ('-o', '--out', '--out-dir', '--output')
    for i, token in enumerate(argv):
        if token in flags and i + 1 < len(argv):
            out = argv[i + 1]
        elif token.startswith('-o') and not token.startswith('--') and token != '-o':
            out = token[2:]
        else:
            for flag_name in flags:
                if token.startswith(flag_name + '='):
                    out = token.split('=', 1)[1]
    return (out or _click_output_default(argv) or S.get('out_dir') or
            _SUBCOMMAND_OUT_DEFAULTS.get(S.get('subcmd'), './result_all/'))

def _persist_run_log(out, transcript):
    """Atomically keep the complete GUI transcript beside compute outputs."""
    root = os.path.abspath(os.path.expanduser(str(out)))
    path = os.path.join(root, 'run.log')
    temporary = path + '.tmp'
    try:
        os.makedirs(root, exist_ok=True)
        with open(temporary, 'w', encoding='utf-8', newline='') as handle:
            handle.write(transcript)
        os.replace(temporary, path)
        return os.path.abspath(path)
    except Exception as exc:
        try:
            if os.path.isfile(temporary): os.remove(temporary)
        except OSError:
            pass
        _run_log_emit('\n[run.log] could not be saved: %s\n' % exc, flush=True)
        return None

def _click_subcommand_params(argv):
    """Parse the pinned command without invoking callbacks or validating files."""
    try:
        argv = _normalized_scope_argv(argv)
        import click
        from pdb2reaction.cli import cli as root_cli
        root_ctx = click.Context(root_cli, info_name=CLI, resilient_parsing=True)
        command = root_cli.get_command(root_ctx, argv[1])
        if command is None: return {}
        sub_ctx = click.Context(command, info_name=argv[1], parent=root_ctx,
                                resilient_parsing=True)
        parsed, _remaining, _order = command.make_parser(sub_ctx).parse_args(
            args=list(argv[2:]))
        return parsed
    except Exception:
        return {}

def _click_output_default(argv):
    """Read the selected command's own output-option default from Click."""
    try:
        import click
        from pdb2reaction.cli import cli as root_cli
        argv = _normalized_scope_argv(argv)
        root_ctx = click.Context(root_cli, info_name=CLI, resilient_parsing=True)
        command = root_cli.get_command(root_ctx, argv[1])
        output_flags = {'-o', '--out', '--out-dir', '--output'}
        for param in command.params if command is not None else ():
            if output_flags.intersection(getattr(param, 'opts', ()) or ()):
                value = getattr(param, 'default', None)
                if isinstance(value, (str, os.PathLike)) and str(value):
                    return str(value)
    except Exception:
        pass
    return None

def _trj2fig_output_targets(argv):
    """Parse every trj2fig output accepted by this pinned CLI release."""
    parsed = _click_subcommand_params(argv)
    outs = parsed.get('outs')
    extra_outs = parsed.get('extra_outs')
    parsed_outputs = (list(outs) if isinstance(outs, (list, tuple)) else [])
    parsed_outputs += (list(extra_outs) if isinstance(extra_outs, (list, tuple)) else [])
    if parsed_outputs:
        return list(dict.fromkeys(str(path) for path in parsed_outputs))
    value_flags = {'-v', '--verbose', '-i', '--input', '--unit', '-r', '--reference',
                   '-q', '--charge', '-m', '--multiplicity', '-b', '--backend',
                   '--solvent', '--solvent-model', '--backend-model', '--precision'}
    output_exts = {'.png', '.jpg', '.jpeg', '.html', '.svg', '.pdf', '.csv'}
    flagged, positional = [], []
    args = list(argv[2:])
    positional_only = False
    i = 0
    while i < len(args):
        token = args[i]
        if token == '--':
            positional_only = True; i += 1; continue
        if not positional_only and token in ('-o', '--out'):
            if i + 1 < len(args): flagged.append(args[i + 1])
            i += 2; continue
        if not positional_only and token.startswith('--out='):
            flagged.append(token.split('=', 1)[1]); i += 1; continue
        if not positional_only and token.startswith('-o') and token != '-o':
            flagged.append(token[2:]); i += 1; continue
        if not positional_only and token in value_flags:
            i += 2; continue
        if not positional_only and any(token.startswith(flag + '=') for flag in value_flags if flag.startswith('--')):
            i += 1; continue
        if positional_only or (not token.startswith('-') and Path(token).suffix.lower() in output_exts):
            positional.append(token)
        i += 1
    found = flagged + positional
    if not found: found = ['energy.png']
    return list(dict.fromkeys(found))

def _flag_enabled(argv, positive, negative):
    enabled = False
    for token in argv:
        if token == positive: enabled = True
        elif token == negative: enabled = False
    return enabled

def _force_dry_run(argv):
    """Append the validating flag once, before a positional-only ``--`` marker."""
    args = list(argv)
    if _flag_enabled(args, '--dry-run', '--no-dry-run'):
        return args
    index = args.index('--') if '--' in args else len(args)
    args.insert(index, '--dry-run')
    return args

def _grouped_option_values(argv, flags, unique=True):
    """Collect repeated or space-grouped values for legacy variadic options."""
    found = []
    i = 2
    while i < len(argv):
        token = argv[i]
        if token in flags:
            i += 1
            while i < len(argv) and not argv[i].startswith('-'):
                found.append(argv[i]); i += 1
            continue
        matched = False
        for flag in flags:
            if flag.startswith('--') and token.startswith(flag + '='):
                found.append(token.split('=', 1)[1]); matched = True; break
        if not matched and '-o' in flags and token.startswith('-o') and token != '-o':
            found.append(token[2:]); matched = True
        i += 1
    return list(dict.fromkeys(found)) if unique else found

def _parsed_path(parsed, key):
    """Return one explicitly parsed CLI path, excluding Click sentinels."""
    value = parsed.get(key)
    return str(value) if isinstance(value, (str, os.PathLike)) and str(value) else None

def _input_needs_cif_companion(inputs):
    """Match the product's mmCIF/PDB-overflow normalization boundary."""
    for value in inputs:
        path = Path(value)
        if path.suffix.lower() in ('.cif', '.mmcif'): return True
        if path.suffix.lower() == '.pdb' and path.is_file():
            try:
                from pdb2reaction.io.structure_formats import pdb_requires_normalization
                if pdb_requires_normalization(path): return True
            except Exception:
                pass
    return False

def _exact_output_scope(outputs, include_json=False, companions=(), expand_user=False):
    """Build a current-run scope from the exact files a command may own."""
    def _absolute(path):
        value = os.path.expanduser(str(path)) if expand_user else str(path)
        return os.path.abspath(value)
    paths = [_absolute(path) for path in outputs]
    paths = list(dict.fromkeys(paths))
    root = os.path.dirname(paths[0]) or '.'
    tracked = list(paths)
    tracked.extend(_absolute(path) for path in companions)
    if include_json:
        tracked.extend((os.path.join(root, 'result.json'),
                        os.path.join(root, 'summary.json')))
    return {'target': paths[0], 'targets': paths, 'root': root,
            'shallow': True, 'prefix': None,
            'exact_targets': list(dict.fromkeys(tracked)),
            'direct_current': True}

def _output_scope(argv=None):
    """Resolve exact file targets or the directory owned by this argv."""
    if argv is None:
        try: argv = shlex.split(cmd_box.value.strip())
        except ValueError: argv = []
    argv = _normalized_scope_argv(argv)
    # Click's eager information flags exit after printing and never own output
    # files. Stop scanning at ``--`` so a positional token is not misread.
    visible = argv[1:argv.index('--')] if '--' in argv else argv[1:]
    if any(token in ('-h', '--help', '--help-advanced', '--version')
           for token in visible):
        return {'target': 'standard output', 'targets': [], 'root': os.getcwd(),
                'shallow': True, 'prefix': None, 'exact_targets': [],
                'direct_current': True, 'stdout_only': True}
    target = _effective_out_dir(argv)
    sub = argv[1] if len(argv) > 1 else S.get('subcmd')
    if sub == 'bond-summary':
        return {'target': 'standard output', 'targets': [], 'root': os.getcwd(),
                'shallow': True, 'prefix': None, 'exact_targets': [],
                'direct_current': True, 'stdout_only': True}
    if sub == 'trj2fig':
        outputs = [os.path.abspath(os.path.expanduser(path))
                   for path in _trj2fig_output_targets(argv)]
        return _exact_output_scope(
            outputs, _flag_enabled(argv, '--out-json', '--no-out-json'),
            expand_user=True)
    if sub == 'extract':
        raw_outputs = _grouped_option_values(argv, ('-o', '--output'), unique=False)
        inputs = _grouped_option_values(argv, ('-i', '--input'), unique=False)
        if len(inputs) == 1 and raw_outputs:
            raw_outputs = raw_outputs[:1]
        if not raw_outputs and inputs:
            raw_outputs = (['model.pdb'] if len(inputs) == 1 else
                           ['model_%s.pdb' % Path(path).stem for path in inputs])
        if raw_outputs:
            outputs = list(raw_outputs)
            companions = []
            if len(outputs) == len(inputs) and len(outputs) > 1:
                companions = [Path(output).with_suffix('.cif')
                              for output, input_path in zip(outputs, inputs)
                              if _input_needs_cif_companion([input_path])]
            elif outputs and inputs and _input_needs_cif_companion([inputs[0]]):
                companions = [Path(outputs[0]).with_suffix('.cif')]
            return _exact_output_scope(
                outputs, _flag_enabled(argv, '--out-json', '--no-out-json'),
                companions)
    if sub == 'energy-diagram':
        parsed = _click_subcommand_params(argv)
        output = _parsed_path(parsed, 'output_path') or 'energy_diagram.png'
        if not Path(output).suffix: output += '.png'
        return _exact_output_scope(
            [output], _flag_enabled(argv, '--out-json', '--no-out-json'))
    if sub == 'add-elem-info':
        parsed = _click_subcommand_params(argv)
        input_path = _parsed_path(parsed, 'in_pdb')
        output = _parsed_path(parsed, 'out_pdb')
        if input_path and not output:
            source = Path(input_path)
            if _flag_enabled(argv, '--overwrite', '--no-overwrite'):
                output = str(source)
            elif source.name.lower().endswith('.pdb'):
                output = str(source.with_name(source.name[:-4] + '_add_elem.pdb'))
            else:
                output = str(source.with_name(source.name + '_add_elem.pdb'))
        if output: return _exact_output_scope([output])
    if sub == 'fix-altloc':
        parsed = _click_subcommand_params(argv)
        input_path = _parsed_path(parsed, 'input_path')
        output = _parsed_path(parsed, 'out')
        inplace = _flag_enabled(argv, '--inplace', '--no-inplace')
        if input_path and os.path.isdir(input_path):
            source = os.path.abspath(input_path)
            owned = source if inplace else (
                os.path.abspath(output) if output else
                str(Path(source).with_name(Path(source).name + '_clean')))
            return {'target': owned, 'targets': [], 'root': owned,
                    'shallow': False, 'prefix': None, 'exact_targets': [],
                    'direct_current': False}
        if input_path:
            source = Path(input_path)
            if inplace:
                return _exact_output_scope(
                    [source, source.with_suffix(source.suffix + '.bak')])
            if output:
                candidate = Path(output)
                output = candidate if candidate.suffix.lower() == '.pdb' else candidate / source.name
            else:
                output = source.with_name(source.stem + '_clean.pdb')
            return _exact_output_scope([output])
    file_output_subs = {'extract', 'fix-altloc', 'add-elem-info', 'energy-diagram',
                        'trj2fig', 'oniom-export', 'oniom-import'}
    if sub in file_output_subs and Path(target).suffix:
        absolute = os.path.abspath(target)
        tracked = [absolute]
        if _flag_enabled(argv, '--out-json', '--no-out-json'):
            parent = os.path.dirname(absolute) or '.'
            tracked.extend((os.path.join(parent, 'result.json'),
                            os.path.join(parent, 'summary.json')))
        return {'target': target, 'targets': [absolute],
                'root': os.path.dirname(absolute) or '.', 'shallow': True,
                'prefix': None, 'exact_targets': tracked, 'direct_current': True}
    return {'target': target, 'targets': [], 'root': target, 'shallow': False,
            'prefix': None, 'exact_targets': [], 'direct_current': False}

def _effective_result_root(argv=None):
    """Resolve the primary directory Results should inspect for this argv."""
    return _output_scope(argv)['root']

def _matches_output_scope(path, scope):
    exact = scope.get('exact_targets') or []
    return not exact or os.path.abspath(path) in set(exact)

def _snapshot_files(root, shallow=False):
    root = os.path.abspath(root)
    if not os.path.isdir(root): return {}
    paths = glob.glob(os.path.join(root, '*')) if shallow else glob.glob(os.path.join(root, '**', '*'), recursive=True)
    snap = {}
    for path in paths:
        if not os.path.isfile(path): continue
        st = os.stat(path); snap[os.path.abspath(path)] = (st.st_size, st.st_mtime_ns)
    return snap

def _snapshot_output_scope(scope):
    if scope.get('stdout_only'): return {}
    exact = scope.get('exact_targets') or []
    if not exact: return _snapshot_files(scope['root'], shallow=scope['shallow'])
    snap = {}
    for path in exact:
        if not os.path.isfile(path): continue
        st = os.stat(path); snap[os.path.abspath(path)] = (st.st_size, st.st_mtime_ns)
    return snap

def _output_scope_collision(scope):
    if scope.get('stdout_only'): return False
    exact = scope.get('exact_targets') or []
    if exact: return any(os.path.lexists(path) for path in exact)
    root = scope['root']
    return os.path.isdir(root) and bool(os.listdir(root))

def _replace_output_dir_arg(argv, output_dir):
    """Mutate an argv list so command, validation, and execution agree."""
    flags = ('-o', '--out', '--out-dir', '--output', '--out-prefix')
    stop = argv.index('--') if '--' in argv else len(argv)
    for index, token in enumerate(argv[:stop]):
        if token in flags and index + 1 < len(argv):
            argv[index + 1] = output_dir
            return
        for flag in flags:
            if token.startswith(flag + '='):
                argv[index] = flag + '=' + output_dir
                return
    argv[stop:stop] = ['-o', output_dir]

def _can_number_output_scope(argv, scope):
    if scope.get('stdout_only') or scope.get('shallow') or scope.get('prefix'):
        return False
    if scope.get('exact_targets'):
        return False
    return os.path.abspath(scope['root']) == os.path.abspath(_effective_out_dir(argv))

def _publish_output_fallback(argv, previous, fallback):
    S['out_dir'] = fallback
    old_stage_guard = _STAGE_SYNC['active']
    old_command_guard = _auto['guard']
    _STAGE_SYNC['active'] = True
    _auto['guard'] = True
    try:
        if w_out.value != fallback:
            w_out.value = fallback
        cmd_box.value = shlex.join(argv)
    finally:
        _STAGE_SYNC['active'] = old_stage_guard
        _auto['guard'] = old_command_guard
    _sync_action_enabled()
    output_note.value = (
        '<small style="color:#166534">Existing <code>%s</code> was preserved; '
        'this run uses <code>%s</code>.</small>' %
        (html.escape(str(previous)), html.escape(str(fallback))))

def _preflight_output_scope(argv):
    """Resolve occupied compute outputs before validation or execution."""
    scope = _output_scope(argv)
    if not _output_scope_collision(scope):
        return scope
    if _can_number_output_scope(argv, scope):
        previous = scope['root']
        fallback = _next_numbered_output_dir(previous)
        _replace_output_dir_arg(argv, fallback)
        scope = _output_scope(argv)
        _publish_output_fallback(argv, previous, fallback)
        return scope
    target = scope['target']; out = scope['root']
    _set_run_status('✗ output exists', 'error', 'collision')
    _run_log_emit(
        'Run not started: refusing to overwrite the existing output: %s\n'
        'Choose a new output path for this utility command.\n' %
        (target if scope['shallow'] else out), clear=True)
    _open_run_log()
    _tab_go(2)
    _set_running(False)
    return None

def _important_result_companion(path, root):
    """Keep compact scientific companions that aggregate summaries omit."""
    path = os.path.abspath(path); root = os.path.abspath(root)
    try: parts = tuple(part.lower() for part in Path(path).relative_to(root).parts)
    except ValueError: return False
    if not os.path.isfile(path): return False
    name = parts[-1] if parts else ''
    if 'freq' in parts and name in ('frequencies_cm-1.txt', 'thermoanalysis.yaml'):
        return True
    if 'vib' in parts and name.startswith('imag_') and name.endswith(('.pdb', '_trj.xyz')):
        return True
    if re.fullmatch(r'mep_seg_\d+_trj\.xyz', name, re.IGNORECASE):
        return True
    if name in ('mep_trj.xyz', 'finished_irc_trj.xyz', 'irc_trj.xyz'):
        return True
    if ((name.startswith('energy_diagram_') or name.startswith('irc_plot')) and
            Path(name).suffix.lower() in ('.png', '.jpg', '.jpeg', '.svg')):
        return True
    segment_path = any(re.match(r'^seg(?:ment)?[_-]?\d+', part, re.IGNORECASE)
                       for part in parts[:-1])
    if segment_path and name in ('mep_trj.xyz', 'final_geometries_trj.xyz'):
        return True
    return bool('freq' in parts and 'ts' in parts and
                re.match(r'mode_\d+_-.*(?:\.pdb|_trj\.xyz)$', name, re.IGNORECASE))

def _structured_current_paths(root, changed):
    """Use machine-readable claims when present; otherwise use the file delta."""
    root = os.path.abspath(root); claimed = set(); rejected = set(); status_json = set(); has_claims = False
    S['_pending_rejected_claims'] = []
    changed_abs = {os.path.abspath(path) for path in changed}
    def _add(value, base=None):
        if isinstance(value, dict):
            for nested in value.values(): _add(nested, base)
            return
        if isinstance(value, (list, tuple, set)):
            for nested in value: _add(nested, base)
            return
        if not isinstance(value, str) or not value: return
        if os.path.isabs(value): candidates = [value]
        else:
            candidates = []
            if base is not None: candidates.append(os.path.join(base, value))
            candidates.extend((os.path.join(root, value), os.path.abspath(value)))
        candidates = list(dict.fromkeys(os.path.abspath(path) for path in candidates))
        path = next((path for path in candidates
                     if path in changed_abs and os.path.isfile(path)), None)
        if path is not None: claimed.add(path)
        else: rejected.add(value)
    for path in list(changed):
        if os.path.basename(path) not in ('result.json', 'summary.json'): continue
        status_json.add(os.path.abspath(path))
        try:
            with open(path) as fh: payload = json.load(fh)
        except Exception:
            continue
        metadata_base = os.path.dirname(os.path.abspath(path))
        if 'current_output_paths' in payload:
            has_claims = True
            for value in payload.get('current_output_paths') or []: _add(value, metadata_base)
        if 'output_files' in payload:
            has_claims = True
            for value in payload.get('output_files') or []: _add(value, metadata_base)
        if 'files' in payload:
            has_claims = True
            values = payload.get('files') or {}
            values = values.values() if isinstance(values, dict) else values
            for value in values: _add(value, metadata_base)
        if 'key_output_files' in payload:
            has_claims = True
            for key, value in (payload.get('key_output_files') or {}).items():
                if isinstance(value, dict):
                    base = os.path.join(root, 'segments', key) if str(key).startswith('seg_') else root
                    for rel in value.get('files') or []: _add(rel, base)
                else:
                    _add(key)
    companions = {path for path in changed_abs if _important_result_companion(path, root)}
    S['_pending_rejected_claims'] = sorted(rejected)
    return sorted((claimed | status_json | companions) if has_claims else set(changed))

def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for block in iter(lambda: fh.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()

def _command_input_option_flags(argv):
    """Derive existing-file option arity from pdb2reaction's selected command."""
    fallback_multi = {
        '-i', '--input', '--ref-pdb', '--ref-full-pdb', '-s', '--scan-lists',
    }
    fallback_single = {'--config', '--calc-file', '--ref-mode', '--csv'}
    args = _normalized_scope_argv(argv)
    try:
        import click
        from pdb2reaction.cli import cli as root_cli
        if len(args) < 2 or args[1].startswith('-'):
            return fallback_multi, fallback_single
        root_ctx = click.Context(root_cli, info_name=CLI, resilient_parsing=True)
        command = root_cli.get_command(root_ctx, args[1])
        if command is None:
            return fallback_multi, fallback_single
        multi, single = set(), set()
        for param in command.params:
            value_type = getattr(param, 'type', None)
            flags = set(getattr(param, 'opts', ()) or ())
            flags.update(getattr(param, 'secondary_opts', ()) or ())
            # scan-lists and center deliberately accept either inline selectors or
            # files while Click exposes them as STRING. Hash values only when
            # they resolve to regular files.
            if flags.intersection({'-s', '--scan-lists'}):
                multi.update(flags)
                continue
            if flags.intersection({'-c', '--center'}):
                single.update(flags)
                continue
            if not (isinstance(value_type, click.Path) and value_type.exists and value_type.file_okay):
                continue
            if getattr(param, 'multiple', False) or getattr(param, 'nargs', 1) != 1:
                multi.update(flags)
            else:
                single.update(flags)
        return (multi, single) if (multi or single) else (fallback_multi, fallback_single)
    except Exception:
        return fallback_multi, fallback_single

def _click_parsed_input_files(argv):
    """Read option and positional input paths through Click's parser, without invoking."""
    args = _normalized_scope_argv(argv)
    try:
        import click
        from pdb2reaction.cli import cli as root_cli
        if len(args) < 2 or args[1].startswith('-'): return []
        root_ctx = click.Context(root_cli, info_name=CLI, resilient_parsing=True)
        command = root_cli.get_command(root_ctx, args[1])
        if command is None: return []
        command_ctx = click.Context(command, info_name=args[1], parent=root_ctx,
                                    resilient_parsing=True)
        parsed, _remaining, _order = command.make_parser(command_ctx).parse_args(args[2:])
        found = []
        for param in command.params:
            value_type = getattr(param, 'type', None)
            if not (isinstance(value_type, click.Path) and value_type.exists and value_type.file_okay):
                continue
            value = parsed.get(param.name)
            values = value if isinstance(value, (tuple, list)) else (value,)
            for path in values:
                if path is not None and os.path.isfile(os.fspath(path)):
                    found.append(os.path.abspath(os.fspath(path)))
        return found
    except Exception:
        return []

def _command_input_files(argv):
    """Collect existing command inputs without importing the Click command tree.

    The GUI command already contains concrete paths.  Scanning those paths is
    complete for validation invalidation and avoids a multi-second CLI import
    on every Validate/Run click in Colab.
    """
    args = _normalized_scope_argv(argv)
    output_flags = {'-o', '--out', '--out-dir', '--output', '--out-prefix'}
    found = []
    positional_only = False
    skip_next = False
    for index, token in enumerate(args[2:], start=2):
        if skip_next:
            skip_next = False
            continue
        if token == '--':
            positional_only = True
            continue
        if not positional_only and token in output_flags:
            skip_next = True
            continue
        if not positional_only and any(
                token.startswith(flag + '=') for flag in output_flags):
            continue
        candidates = []
        if not token.startswith('-') or positional_only:
            candidates.append(token)
        elif '=' in token:
            candidates.append(token.split('=', 1)[1])
        for candidate in candidates:
            if os.path.isfile(candidate):
                found.append(os.path.abspath(candidate))
    return sorted(set(found))

def _validation_fingerprint(argv):
    files = []
    for path in _command_input_files(argv):
        try: digest = _sha256(path)
        except OSError: digest = '<missing>'
        files.append((os.path.abspath(path), digest))
    return (_command_fingerprint(cmd_box.value), tuple(files))
b_rebuild = W.Button(description='Rebuild', icon='magic', layout=W.Layout(width='120px'),
                     tooltip='Rebuild the command from the current settings.')
b_rebuild.on_click(lambda _: (_auto.__setitem__('on', True), refresh()))
b_copy = W.Button(description='Copy', icon='copy', layout=W.Layout(width='100px'), tooltip='Copy the command line.')
def _copy(_):
    try:
        from google.colab import output as _o
        _o.eval_js('navigator.clipboard.writeText(%s)' % json.dumps(cmd_box.value))
        toast.value = '<small style="color:#1f7a3d">copied ✓</small>'
    except Exception as e:
        toast.value = '<small>copy needs Colab (%s)</small>' % e
b_copy.on_click(_copy)
b_clear = b_clear_center  # compatibility alias for older saved/test code
def _clear_center_sel(_):
    if not _view_is_editable(): return
    S['center'] = []; S['lcharge'] = {}; selected_resn.value = ''
    if center_widget is not None: center_widget.value = ()
    if charge_rows is not None:
        for x in charge_rows.values(): x['use'].value = bool(x.get('auto'))
    _invalidate_charge_confirmation()
    S.update(center_ids=[], _last_pick=None, _pick_history=[],
             _last_pick_message='', _last_pick_tone='ok')
    _render_center_ids(); _render_last_pick_status()
    render_viewer(); refresh()
def _clear_scan_sel(_):
    if not _view_is_editable(): return
    S.update(scan_atoms=[None, None], scan_preset='', scan_stages=[], scan_axes=[],
             _last_pick=None, _pick_history=[],
             _last_pick_message='', _last_pick_tone='ok')
    _render_scan_panel(); _render_last_pick_status()
    render_viewer(); refresh()
def _clear_sel(_):
    _clear_center_sel(None); _clear_scan_sel(None)
b_clear_center.on_click(_clear_center_sel)
b_clear_scan.on_click(_clear_scan_sel)
b_validate = W.Button(description='Validate', button_style='info', icon='check', layout=W.Layout(width='130px'),
                      tooltip='Validate the command shown above without starting a calculation.')
b_run = W.Button(description='Run', button_style='danger', icon='play', layout=W.Layout(width='130px'),
                 tooltip='Execute the command shown above.')
b_cancel = W.Button(description='Cancel', button_style='warning', icon='stop',
                    disabled=True, layout=W.Layout(width='110px'),
                    tooltip='Stop the current validation or run.')
b_cancel.layout.display = 'none'
b_cancel.add_class('rxcancel-run')
_RUN_WIDGET_DISABLED = []
def _run_control_widgets():
    """Return only controls that can mutate the command or launch another job."""
    seen = set()
    for name in ('cmd_box', 'b_rebuild', 'b_validate', 'b_run',
                 'b_save', 'up_sess'):
        widget = globals().get(name)
        if widget is not None and id(widget) not in seen and hasattr(widget, 'disabled'):
            seen.add(id(widget)); yield widget

def _publish_run_widget_state():
    """Publish the small run-control surface instead of repainting the whole GUI."""
    widgets = list(_run_control_widgets())
    widgets.extend(widget for widget in (
        globals().get('b_cancel'), globals().get('ready_chip'))
        if widget is not None)
    for widget in widgets:
        try: widget.send_state()
        except Exception: pass
        try: widget.layout.send_state()
        except Exception: pass

def _publish_result_widget_state():
    """Rebind Results layouts after long Colab workers can stale LayoutModels."""
    results_empty.layout = W.Layout(display=('none' if results_empty.layout.display == 'none' else ''))
    trajectory_box.layout = W.Layout(display=('none' if trajectory_box.layout.display == 'none' else 'flex'), width='100%')
    trajectory_content.layout = W.Layout(display=('none' if trajectory_content.layout.display == 'none' else 'flex'), width='100%')
    frame_controls.layout = W.Layout(display=('none' if frame_controls.layout.display == 'none' else 'flex'), width='100%', align_items='center')
    structure_panel.layout = W.Layout(display=('none' if structure_panel.layout.display == 'none' else 'flex'))
    artifact_fold.layout = W.Layout(display=('none' if artifact_fold.layout.display == 'none' else ''))
    stack = [results_box]; seen = set()
    while stack:
        widget = stack.pop()
        if id(widget) in seen: continue
        seen.add(id(widget))
        try: widget.send_state()
        except Exception: pass
        try: widget.layout.send_state()
        except Exception: pass
        stack.extend(getattr(widget, 'children', ()))

def _handle_run_log_poll(widget, content, buffers):
    if content.get('kind') != 'poll':
        return
    # This is a low-rate recovery path. Normal output is pushed by Python;
    # repainting all controls on every log poll can saturate Colab comms.
    _flush_run_log(force=True)

if _RUN_LOG_INCREMENTAL:
    logbox.on_msg(_handle_run_log_poll)

def _set_running(value, _on_ioloop=False):
    value = bool(value)
    worker = _RUN_EXECUTION.get('thread') if '_RUN_EXECUTION' in globals() else None
    if (IN_COLAB and not _on_ioloop and worker is not None and
            threading.current_thread() is worker):
        return _dispatch_ui(lambda value=value: _set_running(value, True))
    if (IN_COLAB and not _on_ioloop and
            threading.current_thread() is not threading.main_thread()):
        return _dispatch_ui(lambda value=value: _set_running(value, True))
    changed = value != _ACTION_STATE['running']
    _ACTION_STATE['running'] = value
    chip = globals().get('ready_chip')
    if value:
        _RUN_STATE['kind'] = 'running'
        if changed:
            _ACTION_STATE['ready_before_run'] = chip.value if chip is not None else ''
            _RUN_WIDGET_DISABLED.clear()
            for widget in _run_control_widgets():
                _RUN_WIDGET_DISABLED.append((widget, bool(widget.disabled)))
                widget.disabled = True
        if chip is not None: chip.value = _RUNNING
    else:
        for widget, disabled in _RUN_WIDGET_DISABLED:
            widget.disabled = disabled
        _RUN_WIDGET_DISABLED.clear()
        _RUN_EXECUTION['argv'] = None
        _flush_deferred_upload_deletes()
        if chip is not None and _RUN_STATE.get('kind') in ('', 'running', 'validating', 'cancelling'):
            previous = _ACTION_STATE.pop('ready_before_run', '')
            if previous: chip.value = previous
        else:
            _ACTION_STATE.pop('ready_before_run', None)
    cancel = globals().get('b_cancel')
    if cancel is not None:
        cancel.disabled = not value
        cancel.layout.display = '' if value else 'none'
    _sync_action_enabled()
    live_log = globals().get('logbox')
    if _RUN_LOG_INCREMENTAL and live_log is not None:
        if not value:
            _flush_run_log(force=True)
            try: live_log.send_state('snapshot')
            except Exception: pass
        live_log.active = value
        try: live_log.send_state('active')
        except Exception: pass
    _publish_run_widget_state()
def _open_run_log():
    toggle = globals().get('w_show_run_log')
    if toggle is not None:
        toggle.value = True
        logbox.layout.display = ''

def _validate_command(a):
    try:
        _verify_bound_session_identities()
    except ValueError as exc:
        _run_log_emit('Session file check failed: %s\n' % exc, clear=True)
        _set_run_status('✗ session file changed', 'error', 'invalid')
        _open_run_log(); return False
    if _preflight_output_scope(a) is None: return False
    fingerprint = _validation_fingerprint(a)
    effective = _normalized_scope_argv(a)
    sub = effective[1] if len(effective) > 1 else ''
    if sub not in COMPUTE:
        _run_log_emit(
            'Validate is available for compute subcommands; use the command help for this utility.\n',
            clear=True)
        _set_run_status('validation is for compute workflows', 'warn', 'utility')
        _open_run_log()
        return False
    dry_argv = _force_dry_run(list(a))
    _RUN_EXECUTION['argv'] = list(dry_argv)
    owns_running = not _ACTION_STATE['running']
    if owns_running: _set_running(True)
    try:
        _set_run_status('🔎 validating…', 'info', 'validating')
        rc, validation_log = _stream(dry_argv)
        _RUN_STATE['validation_log'] = validation_log
        if rc == 0:
            _RUN_STATE['validated_fingerprint'] = fingerprint
            _set_run_status('✓ valid', 'ok', 'valid')
            return True
        _RUN_STATE['validated_fingerprint'] = None
        if _RUN_EXECUTION['cancel'].is_set():
            _set_run_status('■ cancelled', 'warn', 'cancelled')
        else:
            _set_run_status('✗ validation failed', 'error', 'invalid')
        _open_run_log()
        return False
    finally:
        if owns_running: _set_running(False)

def _do_validate_sync(_):
    a = _argv()
    if a: _validate_command(a)

def _do_validate(_):
    if not IN_COLAB:
        return _do_validate_sync(_)
    active = _RUN_EXECUTION.get('thread')
    if active is not None and active.is_alive(): return
    a = _argv()
    if a is None: return
    _RUN_EXECUTION['argv'] = list(a)
    _RUN_EXECUTION['cancel'].clear()
    _set_running(True)
    _set_run_status('🔎 validating…', 'info', 'validating')
    _run_log_emit('[validate] preparing command and inputs …\n',
                  clear=True, flush=True)
    def _worker():
        try:
            _validate_command(list(a))
        except Exception as exc:
            _run_log_emit('\n[GUI validation worker failed] %s\n' % exc, flush=True)
            _set_run_status('✕ validation worker failed', 'error', 'failed')
            _open_run_log()
        finally:
            _RUN_EXECUTION['process'] = None
            _RUN_EXECUTION['task'] = None
            _RUN_EXECUTION['cancel'].clear()
            _set_running(False, True)
            if _RUN_EXECUTION.get('thread') is threading.current_thread():
                _RUN_EXECUTION['thread'] = None
    worker = threading.Thread(target=_worker, name='%s-colab-validate' % TOOL, daemon=True)
    _RUN_EXECUTION['task'] = None
    _RUN_EXECUTION['thread'] = worker
    worker.start()
b_validate.on_click(_do_validate)
def _do_run_sync(_, a=None):
    a = list(a) if a is not None else _argv()
    if not a: return
    try:
        _verify_bound_session_identities()
    except ValueError as exc:
        _run_log_emit('Session file check failed: %s\n' % exc, clear=True)
        _set_run_status('✗ session file changed', 'error', 'invalid')
        _open_run_log(); return
    scope = _preflight_output_scope(a)
    if scope is None: return
    effective = _normalized_scope_argv(a)
    sub = effective[1] if len(effective) > 1 else ''
    validated = _RUN_STATE.get('validated_fingerprint')
    requested_dry_run = sub in COMPUTE and _flag_enabled(
        effective, '--dry-run', '--no-dry-run')
    validated_now = False
    if sub in COMPUTE and validated != _validation_fingerprint(a):
        _RUN_STATE['validated_fingerprint'] = None
        if not _validate_command(a): return
        validated_now = True
    if requested_dry_run and validated_now:
        _set_run_status('✓ dry run valid', 'ok', 'valid')
        return
    if sub not in COMPUTE and sub != 'extract' and not _utility_autofill_complete(a) and _auto['on']:
        _run_log_emit(
            'This utility needs command-specific arguments.\n'
            'Edit the command line using its --help, then run the exact edited command.\n',
            clear=True)
        _set_run_status('◆ finish utility command', 'warn', 'utility')
        _open_run_log()
        return
    out = scope['root']
    before = _snapshot_output_scope(scope)
    started = time.time()
    input_files = _command_input_files(a)
    real_run = not _flag_enabled(effective, '--dry-run', '--no-dry-run')
    # Utilities may edit an input in place. Capture provenance before execution.
    input_records = ([{'path': path, 'sha256': _sha256(path)}
                      for path in input_files] if real_run else [])
    if real_run:
        S.update(_last_out_dir=out, _last_subcmd=(sub or S.get('subcmd')),
                 _last_argv=list(a), _last_files=[], _last_log='',
                 _last_manifest={'tool': TOOL, 'subcommand': sub, 'argv': list(a),
                                 'status': 'running', 'output_root': out,
                                 'stdout_only': bool(scope.get('stdout_only'))})
        dl_btn.disabled = True
        _begin_results_attempt(out, sub)
    _RUN_EXECUTION['argv'] = list(a)
    _set_running(True)
    run_log_path = None
    try:
        _RUN_STATE['validated_fingerprint'] = None
        _set_run_status('⏳ running…', 'warn', 'running')
        rc, transcript = _stream(a)
        if real_run:
            S['_last_log'] = transcript
            if sub in COMPUTE: run_log_path = _persist_run_log(out, transcript)
        else: _RUN_STATE['validation_log'] = transcript
        if rc == 0: _set_run_status('✓ done', 'ok', 'done')
        elif rc == 130: _set_run_status('■ cancelled', 'warn', 'cancelled')
        else: _set_run_status('✗ failed (exit %d)' % rc, 'error', 'failed')
        if rc != 0: _open_run_log()
        if real_run:
            after = _snapshot_output_scope(scope)
            changed = sorted(path for path, stat in after.items()
                             if before.get(path) != stat and _matches_output_scope(path, scope))
            S['_pending_rejected_claims'] = []
            current = changed if scope.get('direct_current') else _structured_current_paths(out, changed)
            if run_log_path: current = sorted(set(current) | {run_log_path})
            try:
                from importlib.metadata import version as _version
                package = 'pdb2reaction'
                installed = _version(package)
            except Exception:
                installed = None
            S['_last_out_dir'] = out
            S['_last_subcmd'] = sub or S.get('subcmd')
            S['_last_argv'] = list(a)
            S['_last_files'] = current
            S['_last_manifest'] = {
                'tool': TOOL, 'version': installed, 'subcommand': S['_last_subcmd'],
                'argv': list(a), 'started_unix': started, 'finished_unix': time.time(),
                'output_root': out, 'exit_code': rc,
                'stdout_only': bool(scope.get('stdout_only')),
                'status': 'success' if rc == 0 else ('cancelled' if rc == 130 else 'failed'),
                'inputs': input_records,
                'rejected_claims': list(S.pop('_pending_rejected_claims', [])),
                'current_files': [os.path.relpath(path, out) for path in current],
            }
            def _render_completed_results():
                # Restore the controls before Results enables its own widgets;
                # otherwise the pre-run disabled snapshot wins after rendering.
                # This callback runs on the UI loop, where an exception is
                # swallowed by the loop: without the guard a failure here left
                # the run 'running' for ever with nothing said about why.
                _set_running(False, True)
                try:
                    _results(out); _tab_go(3); _publish_result_widget_state()
                except Exception as exc:
                    _run_log_emit('\n[results] could not be rendered: %r\n' % (exc,),
                                  flush=True)
                    _open_run_log()
                    _set_run_status('✕ results rendering failed', 'error', 'failed')
                finally:
                    _set_running(False, True)
                    _publish_run_widget_state()
            _dispatch_ui(_render_completed_results)
    finally:
        _set_running(False)
def _do_run(_):
    if not IN_COLAB:
        return _do_run_sync(_)
    active = _RUN_EXECUTION.get('thread')
    if active is not None and active.is_alive(): return
    a = _argv()
    if a is None: return
    _RUN_EXECUTION['argv'] = list(a)
    _RUN_EXECUTION['cancel'].clear()
    _set_running(True)
    _set_run_status('⏳ running…', 'warn', 'running')
    _open_run_log()
    _run_log_emit('[run] preparing command and inputs …\n',
                  clear=True, flush=True)
    def _worker():
        try:
            _do_run_sync(None, list(a))
        except Exception as exc:
            _run_log_emit('\n[GUI run worker failed] %s\n' % exc, flush=True)
            _set_run_status('✕ GUI run failed', 'error', 'failed')
            _open_run_log()
        finally:
            _RUN_EXECUTION['process'] = None
            _RUN_EXECUTION['task'] = None
            _RUN_EXECUTION['cancel'].clear()
            _set_running(False, True)
            if _RUN_EXECUTION.get('thread') is threading.current_thread():
                _RUN_EXECUTION['thread'] = None
    worker = threading.Thread(target=_worker, name='%s-colab-run' % TOOL, daemon=True)
    _RUN_EXECUTION['task'] = None
    _RUN_EXECUTION['thread'] = worker
    worker.start()

def _cancel_run(_):
    task = _RUN_EXECUTION.get('task')
    thread = _RUN_EXECUTION.get('thread')
    task_active = task is not None and not task.done()
    thread_active = thread is not None and thread.is_alive()
    if not task_active and not thread_active:
        _set_running(False, True)
        _publish_run_widget_state()
        return
    _RUN_EXECUTION['cancel'].set()
    _set_run_status('■ cancelling…', 'warn', 'cancelling')
    process = _RUN_EXECUTION.get('process')
    if task_active:
        if process is not None:
            try:
                if os.name == 'posix': os.killpg(os.getpgid(process.pid), signal.SIGTERM)
                else: process.terminate()
            except (ProcessLookupError, AttributeError):
                pass
    else:
        if process is not None: _stop_child(process)
        # Keep the frontend poll alive until the worker records and
        # publishes the terminal cancelled state. Stopping it here leaves
        # the visible chip stranded at 'cancelling…'.

b_cancel.on_click(_cancel_run)
b_run.on_click(_do_run)
def _file_identity(path):
    if not path or not os.path.isfile(path): return None
    return {'sha256': _sha256(path), 'size': int(os.path.getsize(path)),
            'name': os.path.basename(path)}

def _normalized_identity(value, label):
    if value is None: return None
    if not isinstance(value, dict): raise ValueError('%s identity must be an object.' % label)
    digest = value.get('sha256'); size = value.get('size'); name = value.get('name', '')
    if not isinstance(digest, str) or len(digest) != 64 or any(
            character not in '0123456789abcdefABCDEF' for character in digest):
        raise ValueError('%s identity has an invalid SHA-256.' % label)
    if isinstance(size, bool) or not isinstance(size, int) or size < 0:
        raise ValueError('%s identity has an invalid size.' % label)
    if not isinstance(name, str): raise ValueError('%s identity name must be text.' % label)
    return {'sha256': digest.lower(), 'size': size, 'name': name}

def _normalize_file_identities(value):
    if value in (None, {}): return {}
    if not isinstance(value, dict): raise ValueError('file_identities must be an object.')
    normalized = {}
    for plural in ('inputs', 'ref_pdbs'):
        entries = value.get(plural, [])
        if not isinstance(entries, list): raise ValueError('file_identities.%s must be a list.' % plural)
        normalized[plural] = [_normalized_identity(item, '%s[%d]' % (plural, index))
                              for index, item in enumerate(entries)]
    for singular in ('parm', 'model_pdb'):
        normalized[singular] = _normalized_identity(value.get(singular), singular)
    return normalized

def _identity_matches(path, expected):
    if expected is None or not os.path.isfile(path): return False
    if int(os.path.getsize(path)) != int(expected.get('size', -1)): return False
    return _sha256(path).lower() == str(expected.get('sha256', '')).lower()

def _session_file_identities():
    stored = S.get('_session_file_identities') or {}
    def current_or_stored(path, key, index=None):
        current = _file_identity(path)
        if current is not None: return current
        saved = stored.get(key)
        if index is not None:
            return saved[index] if isinstance(saved, list) and index < len(saved) else None
        return saved
    return {
        'inputs': [current_or_stored(path, 'inputs', index)
                   for index, path in enumerate(S.get('inputs', []))],
        'ref_pdbs': [current_or_stored(path, 'ref_pdbs', index)
                     for index, path in enumerate(S.get('ref_pdbs', []))],
        'parm': current_or_stored(S.get('parm'), 'parm'),
        'model_pdb': current_or_stored(S.get('model_pdb'), 'model_pdb'),
    }

def _expected_session_identity(kind, index=None):
    identities = S.get('_session_file_identities') or {}
    if kind in ('input', 'ref'):
        values = identities.get('inputs' if kind == 'input' else 'ref_pdbs') or []
        return values[index] if index is not None and index < len(values) else None
    return identities.get('model_pdb' if kind == 'model' else kind)

def _session_role_paths(data):
    for index, path in enumerate(data.get('inputs') or []):
        yield 'input', index, path
    for index, path in enumerate(data.get('ref_pdbs') or []):
        yield 'ref', index, path
    for kind, key in (('parm', 'parm'), ('model', 'model_pdb')):
        path = data.get(key)
        if path: yield kind, None, path

def _verify_existing_session_identities(data):
    identities = data.get('file_identities') or {}
    for kind, index, path in _session_role_paths(data):
        if not os.path.isfile(path): continue
        if kind in ('input', 'ref'):
            values = identities.get('inputs' if kind == 'input' else 'ref_pdbs') or []
            expected = values[index] if index < len(values) else None
        else:
            expected = identities.get('model_pdb' if kind == 'model' else kind)
        if expected is None or not _identity_matches(path, expected):
            raise ValueError('Saved %s file identity does not match: %s.' %
                             (kind, os.path.basename(path)))

def _verify_bound_session_identities():
    identities = S.get('_session_file_identities') or {}
    if not identities: return True
    current = {
        'inputs': list(S.get('inputs') or []),
        'ref_pdbs': list(S.get('ref_pdbs') or []),
        'parm': S.get('parm'),
        'model_pdb': S.get('model_pdb'),
    }
    for key in ('inputs', 'ref_pdbs'):
        if len(current[key]) != len(identities.get(key) or []):
            raise ValueError('The saved %s role count changed; attach the saved files again.' % key)
    for key in ('parm', 'model_pdb'):
        if bool(current.get(key)) != bool(identities.get(key)):
            raise ValueError('The saved %s role changed; attach the saved file again.' % key)
    for kind, index, path in _session_role_paths(current):
        expected = _expected_session_identity(kind, index)
        if expected is None:
            raise ValueError('The saved %s role is no longer bound; attach it again.' % kind)
        if not os.path.isfile(path):
            raise ValueError('Saved %s file is missing: %s.' %
                             (kind, os.path.basename(path)))
        if not _identity_matches(path, expected):
            raise ValueError('Saved %s file changed after session load: %s.' %
                             (kind, os.path.basename(path)))
    _verify_session_primary_signature()
    return True

def _retire_session_binding():
    S['_session_file_identities'] = {}
    S['_session_primary_atom_signatures'] = []

def _verify_session_primary_signature():
    expected = S.get('_session_primary_atom_signatures') or []
    if not expected: return True
    actual = [list(item) for item in S.get('_primary_atom_signatures', [])]
    if actual != expected:
        raise ValueError('Primary atom identity/order does not match the saved session.')
    return True

def _require_session_file_identities(data):
    identities = data.get('file_identities') or {}
    for kind, index, path in _session_role_paths(data):
        if kind in ('input', 'ref'):
            values = identities.get('inputs' if kind == 'input' else 'ref_pdbs') or []
            expected = values[index] if index < len(values) else None
        else:
            expected = identities.get('model_pdb' if kind == 'model' else kind)
        if expected is None:
            raise ValueError(
                'Settings schema 2 requires a SHA-256 identity for every referenced file '
                '(%s: %s).' % (kind, os.path.basename(path)))

def _session_primary_signature_for(data):
    if not data.get('inputs'): return []
    mode = data.get('mode')
    primary = data['inputs'][0]
    if not os.path.isfile(primary): return None
    if mode in ('pdb', 'mmcif'):
        _text, metadata, _viewer = _load_view_structure(primary)
    elif mode == 'small':
        _text, metadata, _viewer = _load_small_view_structure(primary)
    else:
        return []
    return [list(item) for item in _atom_signatures(metadata)]

def _verify_session_primary_signature_before_commit(data):
    expected = data.get('primary_atom_signatures') or []
    if not data.get('inputs') or data.get('mode') not in (
            'pdb', 'mmcif', 'small'):
        return
    if not expected:
        raise ValueError('Settings schema 2 requires the primary ordered atom signature.')
    actual = _session_primary_signature_for(data)
    if actual is not None and actual != expected:
        raise ValueError('Primary atom identity/order does not match the saved session.')

def _session_number(value, name, integer=False, minimum=None, maximum=None):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(float(value)):
        raise ValueError('%s must be a finite number.' % name)
    if integer and int(value) != value: raise ValueError('%s must be an integer.' % name)
    value = int(value) if integer else float(value)
    if minimum is not None and value < minimum: raise ValueError('%s is below %s.' % (name, minimum))
    if maximum is not None and value > maximum: raise ValueError('%s is above %s.' % (name, maximum))
    return value

def _session_strings(value, name):
    if not isinstance(value, list) or not all(isinstance(item, str) and item for item in value):
        raise ValueError('%s must be a list of non-empty strings.' % name)
    return list(value)

def _session_atom(value, name):
    if not isinstance(value, dict): raise ValueError('%s must be an atom record.' % name)
    atom = dict(value)
    for field in ('chain', 'resn', 'resi', 'atom'):
        if field in atom and not isinstance(atom[field], str):
            raise ValueError('%s.%s must be text.' % (name, field))
    if 'index' in atom and atom['index'] is not None:
        atom['index'] = _session_number(atom['index'], name + '.index', integer=True, minimum=0)
    xyz = atom.get('xyz')
    if xyz is not None:
        if not isinstance(xyz, (list, tuple)) or len(xyz) != 3:
            raise ValueError('%s.xyz must contain three coordinates.' % name)
        atom['xyz'] = tuple(_session_number(item, name + '.xyz') for item in xyz)
    return atom

def _validate_and_normalize_session(payload):
    if not isinstance(payload, dict): raise ValueError('Settings JSON must contain one object.')
    d = dict(payload)
    if d.get('tool') not in (None, TOOL):
        raise ValueError('These settings belong to %s, not %s.' % (d.get('tool'), TOOL))
    version = d.get('schema_version', 1)
    if version != 2:
        raise ValueError(
            'Unsupported settings schema version: %s. Save the session again with this notebook.' %
            version)
    d['tool'] = TOOL; d['schema_version'] = 2
    backend = d.get('backend', S.get('backend', BACKEND))
    if backend not in MODELS: raise ValueError('Unknown backend: %s.' % backend)
    if backend != BACKEND:
        raise ValueError('Saved backend %s is not installed; restart Installation with that backend.' % backend)
    model = d.get('model', DEFAULT_MODEL[backend])
    if model not in MODELS[backend]: raise ValueError('Model %s is not available for %s.' % (model, backend))
    d['backend'] = backend; d['model'] = model
    sub = d.get('subcmd', 'all')
    if sub not in SUBS: raise ValueError('Subcommand %s is unavailable in this runtime.' % sub)
    d['subcmd'] = sub
    all_kind = d.get('all_mode', 'mep')
    if all_kind not in ('mep', 'scan', 'tsonly'): raise ValueError('Unknown all mode: %s.' % all_kind)
    d['all_mode'] = all_kind
    d['inputs'] = _session_strings(d.get('inputs', []), 'inputs')
    d['file_identities'] = _normalize_file_identities(d.get('file_identities'))
    signatures = d.get('primary_atom_signatures', [])
    if not isinstance(signatures, list) or not all(
            isinstance(item, list) and len(item) == 7 and
            all(isinstance(field, str) for field in item) for item in signatures):
        raise ValueError('primary_atom_signatures must contain seven-string atom identities.')
    d['primary_atom_signatures'] = signatures
    d['ref_pdbs'] = _session_strings(d.get('ref_pdbs', []), 'ref_pdbs')
    for name in ('parm', 'model_pdb'):
        value = d.get(name)
        if value is not None and not isinstance(value, str): raise ValueError('%s must be a path or null.' % name)
        d[name] = value
    mode = d.get('mode')
    if mode not in (None, 'pdb', 'mmcif', 'small', 'utility'): raise ValueError('Unknown input mode: %s.' % mode)
    d['mode'] = mode
    _require_session_file_identities(d)
    d['center'] = _session_strings(d.get('center', []), 'center')
    d['center_ids'] = _session_strings(d.get('center_ids', []), 'center_ids')
    charges = d.get('lcharge', {})
    if not isinstance(charges, dict) or not all(isinstance(name, str) and name for name in charges):
        raise ValueError('lcharge must map residue names to charges.')
    d['lcharge'] = {name: _session_number(value, 'lcharge.' + name, minimum=-9, maximum=9)
                    for name, value in charges.items()}
    scan_atoms = d.get('scan_atoms', [None, None])
    if not isinstance(scan_atoms, list) or len(scan_atoms) != 2:
        raise ValueError('scan_atoms must contain A and B.')
    d['scan_atoms'] = [None if atom is None else _session_atom(atom, 'scan_atoms') for atom in scan_atoms]
    stages = d.get('scan_stages', [])
    if not isinstance(stages, list): raise ValueError('scan_stages must be a list.')
    normalized_stages = []
    for si, stage in enumerate(stages):
        if not isinstance(stage, list): raise ValueError('scan_stages[%d] must be a list.' % si)
        normalized_stage = []
        for bi, bond in enumerate(stage):
            if not isinstance(bond, dict): raise ValueError('scan stage bond must be an object.')
            normalized_stage.append({'a': _session_atom(bond.get('a'), 'scan stage A'),
                                     'b': _session_atom(bond.get('b'), 'scan stage B'),
                                     't': _session_number(bond.get('t'), 'scan target', minimum=0.3, maximum=8.0)})
        normalized_stages.append(normalized_stage)
    d['scan_stages'] = normalized_stages
    axes = d.get('scan_axes', [])
    if not isinstance(axes, list): raise ValueError('scan_axes must be a list.')
    d['scan_axes'] = [
        {'a': _session_atom(axis.get('a'), 'scan axis A'),
         'b': _session_atom(axis.get('b'), 'scan axis B'),
         'lo': _session_number(axis.get('lo'), 'scan low', minimum=0.3, maximum=8.0),
         'hi': _session_number(axis.get('hi'), 'scan high', minimum=0.3, maximum=8.0)}
        for axis in axes if isinstance(axis, dict)
    ]
    if len(d['scan_axes']) != len(axes): raise ValueError('Every scan axis must be an object.')
    pairs = d.get('freeze_pairs', [])
    if not isinstance(pairs, list): raise ValueError('freeze_pairs must be a list.')
    d['freeze_pairs'] = [
        {'a': _session_atom(pair.get('a'), 'restraint A'),
         'b': _session_atom(pair.get('b'), 'restraint B'),
         't': (None if pair.get('t') is None else _session_number(pair.get('t'), 'restraint target', minimum=0.0))}
        for pair in pairs if isinstance(pair, dict)
    ]
    if len(d['freeze_pairs']) != len(pairs): raise ValueError('Every freeze pair must be an object.')
    frozen = d.get('freeze_atoms', [])
    if not isinstance(frozen, list): raise ValueError('freeze_atoms must be a list.')
    d['freeze_atoms'] = [_session_number(value, 'freeze atom', integer=True, minimum=1) for value in frozen]
    measured = d.get('measure_atoms', [])
    if not isinstance(measured, list) or len(measured) > 4: raise ValueError('measure_atoms accepts up to four atoms.')
    d['measure_atoms'] = [_session_atom(atom, 'measure atom') for atom in measured]
    d['scan_target'] = _session_number(d.get('scan_target', 1.6), 'scan_target', minimum=0.3, maximum=8.0)
    preset = d.get('scan_preset', '')
    if not isinstance(preset, str): raise ValueError('scan_preset must be text.')
    d['scan_preset'] = preset
    for name in ('tsopt', 'thermo', 'charge_explicit', 'show_water', 'surface', 'spin'):
        value = d.get(name, False)
        if not isinstance(value, bool): raise ValueError('%s must be true or false.' % name)
        d[name] = value
    # Scientific charge confirmation is an interaction, never a portable setting.
    d['charge_explicit'] = False
    d['charge'] = _session_number(d.get('charge', 0), 'charge', integer=True)
    out_dir = d.get('out_dir') or _SUBCOMMAND_OUT_DEFAULTS.get(d.get('subcmd', 'all'), './result_all/')
    if not isinstance(out_dir, str) or not out_dir.strip(): raise ValueError('out_dir must be non-empty text.')
    d['out_dir'] = out_dir
    raw_rep = d.get('rep', 'cartoon')
    if raw_rep not in ('cartoon', 'stick', 'sticks', 'ball+stick', 'ball-and-stick',
                       'spheres', 'sphere', 'line', 'lines'):
        raise ValueError('Unknown representation.')
    rep_user_set = d.get('rep_user_set')
    if rep_user_set is None:
        rep_user_set = raw_rep != 'cartoon'
    if not isinstance(rep_user_set, bool):
        raise ValueError('rep_user_set must be true or false.')
    if d.get('color', 'element') not in ('element', 'chain', 'spectrum'):
        raise ValueError('Unknown colour scheme.')
    d['rep'] = _normalize_representation(raw_rep); d['rep_user_set'] = rep_user_set
    d['color'] = d.get('color', 'element')
    width = _session_number(d.get('viewer_width', 720), 'viewer_width', integer=True)
    if width not in (640, 720, 800): raise ValueError('viewer_width must be 640, 720, or 800.')
    d['viewer_width'] = width
    selected_value = d.get('selected_resn', '')
    if not isinstance(selected_value, str): raise ValueError('selected_resn must be text.')
    d['selected_resn'] = selected_value.strip()
    overrides = d.get('advanced_overrides', {})
    if not isinstance(overrides, dict): raise ValueError('advanced_overrides must be an object.')
    normalized_overrides = {}
    for command_name, values in overrides.items():
        if command_name not in SUBS or not isinstance(values, dict):
            raise ValueError('Invalid advanced overrides for %s.' % command_name)
        params = {param.name: param for param in _advanced_options(command_name)}
        normalized_values = {}
        for name, value in values.items():
            param = params.get(name)
            if param is None or _advanced_status(command_name, param) != 'rendered':
                raise ValueError('Unknown editable advanced option %s.%s.' % (command_name, name))
            is_bool = param.is_bool_flag or isinstance(param.type, click.types.BoolParamType)
            if name == 'verbose':
                if (value is not None and
                        (isinstance(value, bool) or not isinstance(value, int) or not 0 <= value <= 3)):
                    raise ValueError('%s.verbose must be an integer from 0 to 3 or default.' % command_name)
            elif is_bool and value not in (True, False, None):
                raise ValueError('%s.%s must be true, false, or default.' % (command_name, name))
            elif isinstance(param.type, click.Choice) and value is not None and value not in param.type.choices:
                raise ValueError('Invalid choice for %s.%s.' % (command_name, name))
            elif (not param.multiple and int(getattr(param, 'nargs', 1)) == 1 and
                  isinstance(param.type, (click.types.IntParamType, click.types.FloatParamType))):
                if isinstance(value, bool):
                    raise ValueError('%s.%s must be numeric.' % (command_name, name))
                try:
                    value = param.type.convert(value, param, None)
                except (click.BadParameter, TypeError, ValueError) as exc:
                    raise ValueError('Invalid numeric value for %s.%s: %s' %
                                     (command_name, name, exc)) from exc
            elif not is_bool and not isinstance(param.type, click.Choice) and not isinstance(value, str):
                raise ValueError('%s.%s must be text.' % (command_name, name))
            normalized_values[name] = value
        normalized_overrides[command_name] = normalized_values
    d['advanced_overrides'] = normalized_overrides
    advanced = d.get('advanced', {})
    if not isinstance(advanced, dict): raise ValueError('advanced must be an object.')
    d['advanced'] = {
        'mult': _session_number(advanced.get('mult', 1), 'multiplicity', integer=True, minimum=1),
        'precision': advanced.get('precision', 'auto'), 'deterministic': advanced.get('deterministic', False),
        'mep_mode': advanced.get('mep_mode', '(default)'), 'dmf_backend': advanced.get('dmf_backend', '(default)'),
        'thresh': advanced.get('thresh', '(default)'),
        'thresh_post': advanced.get('thresh_post', '(default)'),
        'radius': _session_number(advanced.get('radius', 2.6), 'radius', minimum=0.0),
        'flatten': advanced.get('flatten', False), 'refine_path': advanced.get('refine_path', False),
        'dft': advanced.get('dft', False), 'dft_func_basis': advanced.get('dft_func_basis', ''),
    }
    if d['advanced']['precision'] not in ('auto', 'fp32', 'fp64'): raise ValueError('Invalid precision.')
    if d['advanced']['mep_mode'] not in _widget_values(adv_mep): raise ValueError('MEP mode is unavailable.')
    if d['advanced']['dmf_backend'] not in _widget_values(adv_dmf): raise ValueError('DMF backend is unavailable.')
    if d['advanced']['thresh'] not in _widget_values(adv_thresh): raise ValueError('Invalid threshold.')
    if d['advanced']['thresh_post'] not in _widget_values(adv_thresh_post): raise ValueError('Invalid post-IRC threshold.')
    for name in ('deterministic', 'flatten', 'refine_path', 'dft'):
        if not isinstance(d['advanced'][name], bool): raise ValueError('%s must be true or false.' % name)
    if not isinstance(d['advanced']['dft_func_basis'], str): raise ValueError('dft_func_basis must be text.')
    if d['advanced']['dft'] and not DFT_READY: raise ValueError('DFT support is not installed in this runtime.')
    if d['subcmd'] == 'all' and (d['thermo'] or d['advanced']['dft']):
        d['tsopt'] = True
    existing_structures = [p for p in d['inputs'] if os.path.isfile(p) and Path(p).suffix.lower() in ('.pdb','.ent','.cif','.mmcif')]
    _preflight_structures(existing_structures)
    for small_path in d['inputs']:
        if os.path.isfile(small_path) and Path(small_path).suffix.lower() in ('.xyz', '.gjf'):
            _load_small_view_structure(small_path)
    _assert_distinct_session_pairs = []
    a, b = d['scan_atoms']
    if a and b: _assert_distinct_session_pairs.append((a, b))
    for stage in d['scan_stages']:
        _assert_distinct_session_pairs.extend((bond['a'], bond['b']) for bond in stage)
    _assert_distinct_session_pairs.extend((axis['a'], axis['b']) for axis in d['scan_axes'])
    _assert_distinct_session_pairs.extend((pair['a'], pair['b']) for pair in d['freeze_pairs'])
    if any(_same_atom(first, second) for first, second in _assert_distinct_session_pairs):
        raise ValueError('Saved scan/restraint pairs must use two different atoms.')
    _verify_session_primary_signature_before_commit(d)
    return d

def _session_dict():
    if center_widget is not None: S['center'] = list(center_widget.value)
    if charge_rows is not None:
        S['lcharge'] = {r: x['val'].value for r, x in charge_rows.items()
                        if x['use'].value and not x.get('auto')}
    keys = ['tool', 'backend', 'model', 'subcmd', 'inputs', 'parm', 'model_pdb', 'ref_pdbs', 'mode', 'center', 'center_ids', 'selected_resn',
            'lcharge', 'scan_atoms', 'scan_stages', 'scan_axes', 'scan_target', 'scan_preset',
            'freeze_pairs', 'freeze_atoms', 'measure_atoms', 'charge', 'charge_explicit', 'tsopt', 'thermo',
            'out_dir', 'rep', 'color', 'show_water', 'surface', 'spin', 'viewer_width', 'advanced_overrides']
    d = {k: S.get(k) for k in keys}; d['schema_version'] = 2
    d['rep_user_set'] = bool(_rep_user_set['value'])
    # A saved checkbox is not a fresh scientific verification.
    d['charge_explicit'] = False
    d['file_identities'] = _session_file_identities()
    d['primary_atom_signatures'] = [
        list(item) for item in S.get('_primary_atom_signatures', [])]
    d['all_mode'] = _wv('all_mode', 'mep')
    if d['all_mode'] == 'tsonly':
        d['tsopt'] = bool(_ALL_MODE_STATE.get('tsopt_before_tsonly', False))
    d['advanced'] = {'mult': _wv('adv_mult', 1), 'precision': _wv('adv_prec', 'auto'),
                     'deterministic': _wv('adv_det', False), 'mep_mode': _wv('adv_mep', '(default)'),
                     'dmf_backend': _wv('adv_dmf', '(default)'),
                     'thresh': _wv('adv_thresh', '(default)'), 'thresh_post': _wv('adv_thresh_post', '(default)'),
                      'radius': _wv('adv_radius', 0.0),
                     'flatten': _wv('adv_flatten', False), 'refine_path': _wv('adv_refine', False),
                     'dft': _wv('adv_dft', False), 'dft_func_basis': _wv('adv_dftfb', '')}
    if d['subcmd'] == 'all' and (d['thermo'] or d['advanced']['dft']):
        d['tsopt'] = True
    return d
def _save_session(_):
    session_path = _runtime_path('session.json')
    try:
        payload = _session_dict()
        _require_session_file_identities(payload)
        with open(session_path, 'w') as fh: json.dump(payload, fh, indent=1)
    except Exception as exc:
        toast.value = '<small role="alert" style="color:#991b1b">session not saved: %s</small>' % html.escape(str(exc))
        return
    toast.value = '<small style="color:#1f7a3d">saved session.json</small>'
    try:
        from google.colab import files as _f; _f.download(session_path)
    except Exception:
        toast.value = '<small>saved session.json (working dir)</small>'
def _apply_session(payload):
    d = _validate_and_normalize_session(payload)
    _verify_existing_session_identities(d)
    previous_referenced = (list(S.get('inputs', [])) + list(S.get('ref_pdbs', [])) +
                           [p for p in (S.get('parm'), S.get('model_pdb')) if p])
    _bump_drop_generation()
    if _UPLOAD_MODE != 'anywidget': _reset_file_upload(upl)
    _clear_structure_bound_state(preserve_model=d.get('model_pdb'))
    S.update(_last_pick=None, _pick_history=[],
             _last_pick_message='', _last_pick_tone='ok',
             _view_input_index=0, _view_mapping_ok=True,
             _session_file_identities=dict(d.get('file_identities') or {}),
             _session_primary_atom_signatures=list(d.get('primary_atom_signatures') or []))
    _SESSION_APPLY['active'] = True
    _rep_user_set['value'] = bool(d['rep_user_set'])
    try:
        for key in ('backend', 'model', 'subcmd', 'out_dir', 'tsopt', 'thermo', 'charge', 'rep', 'color',
                    'show_water', 'surface', 'spin', 'viewer_width', 'scan_target', 'scan_preset',
                    'parm', 'model_pdb', 'ref_pdbs', 'mode'):
            S[key] = d[key]
        S['inputs'] = list(d['inputs']); S['center'] = list(d['center']); S['center_ids'] = list(d['center_ids'])
        S['selected_resn'] = d['selected_resn']; selected_resn.value = d['selected_resn']
        S['lcharge'] = dict(d['lcharge']); S['scan_atoms'] = list(d['scan_atoms'])
        S['scan_stages'] = list(d['scan_stages']); S['scan_axes'] = list(d['scan_axes'])
        S['freeze_pairs'] = list(d['freeze_pairs']); S['freeze_atoms'] = list(d['freeze_atoms'])
        S['measure_atoms'] = list(d['measure_atoms']); S['advanced_overrides'] = dict(d['advanced_overrides'])
        S['charge_explicit'] = d['charge_explicit']
        dd_backend.value = d['backend']; dd_model.options = MODELS[d['backend']]; dd_model.value = d['model']
        dd_subcmd.value = d['subcmd'] if d['subcmd'] in SUBS else 'all'
        all_mode.value = d['all_mode']; _ALL_MODE_STATE['last'] = d['all_mode']; _ALL_MODE_STATE['user'] = True
        _ALL_MODE_STATE['tsopt_before_tsonly'] = bool(d['tsopt'])
        w_ts.value = True if d['all_mode'] == 'tsonly' else bool(d['tsopt'])
        w_th.value = bool(d['thermo']); w_out.value = d['out_dir']
        w_q.value = int(d['charge']); w_charge_ok.value = bool(d['charge_explicit'])
        _REP_SYNC['active'] = True
        try: dd_rep.value = d['rep']
        finally: _REP_SYNC['active'] = False
        dd_col.value = d['color']; cb_water.value = d['show_water']
        cb_surf.value = d['surface']; cb_spin.value = d['spin']; dd_size.value = d['viewer_width']
        adv = d['advanced']
        adv_mult.value = int(adv['mult']); adv_prec.value = adv['precision']; adv_det.value = adv['deterministic']
        adv_mep.value = adv['mep_mode']; adv_dmf.value = adv['dmf_backend']; adv_thresh.value = adv['thresh']
        adv_thresh_post.value = adv['thresh_post']
        adv_radius.value = float(adv['radius']); adv_flatten.value = adv['flatten']
        adv_refine.value = adv['refine_path']
        adv_dft.value = adv['dft']; adv_dftfb.value = adv['dft_func_basis']
    finally:
        _SESSION_APPLY['active'] = False
    _auto['on'] = True
    _render_advanced_rows()
    loaded_primary = False
    if S['mode'] in ('pdb', 'mmcif', 'small') and S['inputs'] and os.path.exists(S['inputs'][0]):
        loaded_primary = build_selection()
    if loaded_primary:
        _verify_session_primary_signature()
        if center_widget is not None:
            center_widget.value = tuple(v for v in _center_values() if v in set(S['center']))
        if charge_rows is not None:
            for rn, row in charge_rows.items():
                row['use'].value = bool(row.get('auto')) or rn in S['lcharge']
                if row.get('auto'): row['val'].value = float(_ION_CHARGES[rn])
                elif rn in S['lcharge']: row['val'].value = float(S['lcharge'][rn])
    elif not S.get('_pdb_text'):
        render_viewer()
    _render_input_queue(); _sync_view_input_widget(); _render_freeze_panel()
    _render_last_pick_status(); _sync_capability_controls(); refresh()
    referenced = (list(S.get('inputs', [])) + list(S.get('ref_pdbs', [])) +
                  [p for p in (S.get('parm'), S.get('model_pdb')) if p])
    retained = {_path_identity(path) for path in referenced}
    _delete_owned_uploads(
        [path for path in previous_referenced
         if _path_identity(path) not in retained])
    return [p for p in referenced if not os.path.isfile(p)]
up_sess = W.FileUpload(accept='.json', multiple=False, description='Load session', layout=W.Layout(width='130px'))
def _accept_session_pairs(pairs):
    """Apply one uploaded session file from (name, bytes) pairs."""
    if pairs:
        for name, c in pairs:
            try:
                text = c if isinstance(c, str) else bytes(c).decode('utf-8')
                missing = _apply_session(json.loads(text))
                toast.value = ('<small role="alert" style="color:#92400e">loaded session; re-upload files: %s</small>' %
                               html.escape(', '.join(os.path.basename(p) for p in missing)) if missing else
                               '<small style="color:#1f7a3d">loaded %s</small>' % html.escape(name))
            except Exception as e:
                toast.value = '<small style="color:#a00">load failed: %s</small>' % html.escape(str(e))
def _on_load_sess(change):
    items = up_sess.value
    raw = list(items.items()) if isinstance(items, dict) else [(f['name'], f) for f in items]
    _set_operation_loading('session', True)
    try:
        _accept_session_pairs(_widget_upload_pairs(raw))
    finally:
        _reset_file_upload(up_sess)
        _set_operation_loading('session', False)
up_sess.observe(_on_load_sess, names='value')
b_save = W.Button(description='Save settings', icon='save', layout=W.Layout(width='130px'),
                  tooltip='Download the current settings as session.json.'); b_save.on_click(_save_session)
session_row = W.HBox([b_save, up_sess],
                     layout=W.Layout(flex_flow='row nowrap', align_items='center'))
session_row.add_class('rxsession-upload')
if _UPLOAD_MODE == 'colab': up_sess.layout.display = 'none'
primary_bar = W.HBox([b_validate, b_run, b_cancel]); primary_bar.add_class('rxrun')
w_show_run_log = W.Checkbox(value=True, description='Show log', indent=False)
def _toggle_run_log(change):
    logbox.layout.display = '' if change['new'] else 'none'
w_show_run_log.observe(_toggle_run_log, names='value')
logbox.layout.display = ''
logbox.add_class('rxlog')
command_head = W.HBox([
    W.HTML('<b>Command line</b>'), ready_chip, toast],
    layout=W.Layout(width='100%', flex_flow='row wrap', align_items='center'))
command_head.add_class('rxcommand-head')
command_editor = W.VBox([command_head, cmd_box])
command_editor.add_class('rxcard'); command_editor.add_class('rxcommand-editor')
command_tools = W.HBox([b_rebuild, b_copy]); command_tools.add_class('rxcommand-tools')
command_actions = W.HBox(
    [command_tools, session_row, primary_bar],
    layout=W.Layout(width='100%', flex_flow='row nowrap', align_items='center'))
command_actions.add_class('rxcommand-actions')
command_footer = W.VBox([command_actions, w_show_run_log], layout=W.Layout(width='100%'))
command_footer.add_class('rxcommand-footer')
cmdline_box = W.VBox([command_editor, command_footer, logbox])
cmdline_box.add_class('rxcommand-dock')

def _recover_existing_results(directory=None, replace=False):
    """Recover only producer-declared files from an authoritative result JSON."""
    if not replace and (S.get('_last_out_dir') or S.get('_last_manifest')):
        return False
    out = os.path.abspath(os.path.expanduser(str(directory or _effective_result_root())))
    if not os.path.isdir(out):
        return False
    root_real = os.path.realpath(out)
    version_keys = (('pdb2reaction_version',) if TOOL == 'pdb2reaction' else
                    ('mlmm_toolkit_version', 'mlmm_version'))
    def _flatten_claims(value):
        if isinstance(value, str):
            return [value]
        if isinstance(value, (list, tuple)):
            return [item for child in value for item in _flatten_claims(child)]
        if isinstance(value, dict):
            return [item for child in value.values() for item in _flatten_claims(child)]
        return []
    def _declared_paths(payload):
        # Producer metadata evolved over time.  Treat its file fields as a
        # union: current_output_paths may contain the aggregate deliverables
        # while key_output_files contains per-segment MEP/IRC results.
        claimed = []
        for field in ('current_output_paths', 'output_files', 'files'):
            claimed.extend(_flatten_claims(payload.get(field)))
        key_files = payload.get('key_output_files')
        if isinstance(key_files, (list, tuple)):
            claimed.extend(_flatten_claims(key_files))
        elif isinstance(key_files, dict):
            for prefix, details in key_files.items():
                if isinstance(details, dict) and details.get('files'):
                    base = (os.path.join('segments', str(prefix))
                            if str(prefix).startswith('seg_') else str(prefix))
                    claimed.extend(os.path.join(base, str(item))
                                   for item in _flatten_claims(details['files']))
                elif isinstance(details, str):
                    # Root key_output_files commonly maps a path to a human
                    # description; the mapping key is the actual file.
                    claimed.append(str(prefix))
                else:
                    claimed.extend(_flatten_claims(details))
        return list(dict.fromkeys(str(path) for path in claimed if str(path)))
    internal = os.path.join(out, '_work', '_run_manifest.json')
    internal_run_id = None
    internal_started_ns = None
    if os.path.isfile(internal):
        try:
            with open(internal, encoding='utf-8') as fh:
                _internal_manifest = json.load(fh) or {}
                internal_run_id = _internal_manifest.get('run_id')
                internal_started_ns = _internal_manifest.get('started_ns')
        except Exception:
            return False
    for metadata_name in ('summary.json', 'result.json'):
        metadata_path = os.path.join(out, metadata_name)
        if not os.path.isfile(metadata_path):
            continue
        try:
            with open(metadata_path, encoding='utf-8') as fh:
                payload = json.load(fh)
        except Exception:
            continue
        if not isinstance(payload, dict) or not any(payload.get(key) for key in version_keys):
            continue
        states = [str(payload.get(key, '')).strip().lower()
                  for key in ('status', 'execution_status', 'scientific_status')]
        if any(state in ('running', 'in_progress', 'pending') for state in states):
            continue
        if internal_run_id and payload.get('run_id') != internal_run_id:
            continue
        recovered, rejected = [], []
        for claim in _declared_paths(payload):
            candidate = os.path.abspath(claim if os.path.isabs(str(claim)) else
                                        os.path.join(out, str(claim)))
            candidate_real = os.path.realpath(candidate)
            try:
                inside = os.path.commonpath((root_real, candidate_real)) == root_real
            except ValueError:
                inside = False
            if inside and os.path.isfile(candidate):
                recovered.append(candidate)
            else:
                rejected.append(str(claim))
        companions = []
        if internal_started_ns is not None:
            for companion in Path(out).rglob('*'):
                try: current_enough = companion.stat().st_mtime_ns >= int(internal_started_ns)
                except (OSError, TypeError, ValueError): current_enough = False
                if current_enough and _important_result_companion(str(companion), out):
                    companions.append(str(companion.resolve()))
        recovered = sorted(set(recovered + companions + [metadata_path]))
        if len(recovered) < 2:
            continue
        command = str(payload.get('command') or '').strip()
        try:
            command_tokens = shlex.split(command)
        except ValueError:
            command_tokens = []
        inferred = (command_tokens[1] if len(command_tokens) > 1 and
                    command_tokens[0] in ('pdb2reaction', 'mlmm') else
                    command_tokens[0] if command_tokens else (S.get('subcmd') or 'all'))
        S['_last_out_dir'] = out
        S['_last_subcmd'] = inferred
        try: S['_last_argv'] = shlex.split(cmd_box.value)
        except ValueError: S['_last_argv'] = []
        S['_last_files'] = recovered
        S['_last_manifest'] = {
            'tool': TOOL, 'subcommand': inferred, 'output_root': out,
            'status': 'restored', 'restored': True,
            'source_status': next((state for state in states if state), 'completed'),
            'run_id': payload.get('run_id'),
            'rejected_claims': list(dict.fromkeys(rejected)),
            'current_files': [os.path.relpath(path, out) for path in recovered],
        }
        S['_results_presented_dir'] = None
        results_dir.value = out
        res_btn.disabled = False
        dl_btn.disabled = False
        return True
    return False

# ============================================================== assemble
# Colab's widget frontend renders ipywidgets' Tab as an empty block. Four
# ordinary buttons stay responsive while _tab_go() swaps persistently mounted
# panes; Colab's native bridge owns its visible loading transition.
_TAB_PAGES = [('① Input', input_box), ('② Setup', viewer_box),
              ('③ Options', options_box), ('④ Results', results_box)]
for _label, _page in _TAB_PAGES:
    _page.add_class('rxpage')
viewer_box.add_class('rxviewer-page')
results_box.add_class('rxresults-page')
_tab_body = W.VBox([page for _label, page in _TAB_PAGES])
_tab_body.add_class('rxpages')
_tab_status = W.HTML()
_tab_status.add_class('rxsr-only')
_tab_notice = W.HTML()
_tab_notice.add_class('rxtab-notice')
_tab_loading = W.HTML()
_tab_loading.add_class('rxtab-loading')
_TAB_NAV = {'active': 0, 'syncing': False, 'request': 0}
class _TabChoiceState:
    value = 0
_tab_choice = _TabChoiceState()
_tab_buttons = []
for _index, (_label, _pane) in enumerate(_TAB_PAGES):
    _button = W.Button(description=_label, layout=W.Layout(
        flex='1 1 0', width='auto', min_width='0'))
    _button.add_class('rxtab-button')
    if not IS_COLAB_FRONTEND:
        _button.on_click(lambda _clicked, _index=_index: _tab_go(_index))
    _tab_buttons.append(_button)
_tab_strip = W.HBox(_tab_buttons, layout=W.Layout(
    width='100%', flex_flow='row nowrap', align_items='stretch'))
_tab_strip.add_class('rxtabs')
def _set_tab_choice(i):
    _tab_choice.value = i
    for _index, _button in enumerate(_tab_buttons):
        if _index == i: _button.add_class('mod-active')
        else: _button.remove_class('mod-active')
def _set_tab_loading(i, active):
    if active:
        label = _TAB_PAGES[i][0]
        _tab_loading.value = (
            '<div class="rxtab-loading-banner" role="status" aria-live="polite" aria-atomic="true">'
            '<span class="rxtab-spinner" aria-hidden="true"></span>'
            'Loading <b>%s</b>…</div>' % html.escape(label))
    else:
        _tab_loading.value = ''
    try: _tab_loading.send_state()
    except Exception: pass
def _finish_tab_go(i, request):
    if request != _TAB_NAV['request']:
        return
    try:
        if i == 3 and (S.get('_last_manifest') or S.get('_last_files')):
            _result_root = S.get('_last_out_dir') or _effective_result_root()
            if S.get('_results_presented_dir') != os.path.abspath(_result_root):
                _results(_result_root)
        _close_info()
        # Keep every pane mounted. Replacing children aborts browser-side
        # upload queues and can recreate a live WebGL output in Colab.
        for _j, (_label, _pane) in enumerate(_TAB_PAGES):
            # Setup owns the live Mol* WebGL iframe. Keep that one pane
            # mounted off-screen so tab changes never recreate its camera.
            if _j == i:
                _pane.remove_class('rxpage-prewarm')
                _pane.layout.display = ''
            elif _j == 1:
                _pane.layout.display = ''
                _pane.add_class('rxpage-prewarm')
            else:
                _pane.remove_class('rxpage-prewarm')
                _pane.layout.display = 'none'
            try: _pane.layout.send_state()
            except Exception: pass
        _tab_status.value = ('<div role="status" aria-live="polite" aria-atomic="true" '
                             'style="color:#475569;font-size:12px;margin:0 2px 5px">'
                             'Step %d of %d · <b>%s</b></div>' %
                             (i + 1, len(_TAB_PAGES), _TAB_PAGES[i][0].split(' ', 1)[1]))
        try: _tab_status.send_state()
        except Exception: pass
        _TAB_NAV['active'] = i
    finally:
        if request == _TAB_NAV['request']:
            _tab_body.remove_class('rxpages-loading')
            _set_tab_loading(i, False)
def _tab_go(i):
    i = max(0, min(len(_TAB_PAGES) - 1, int(i)))
    if i == 1 and not S.get('inputs'):
        _tab_notice.value = ('<div role="alert"><b>Setup needs a structure.</b> ' 
                             'Upload files or load an example in ① Input first.</div>')
        _set_tab_choice(_TAB_NAV['active'])
        _set_tab_loading(i, False)
        return
    _tab_notice.value = ''
    _TAB_NAV['request'] += 1
    _request = _TAB_NAV['request']
    _set_tab_choice(i)
    _set_tab_loading(i, True)
    _tab_body.add_class('rxpages-loading')
    _finish_tab_go(i, _request)
_tab_go(0)
app = W.VBox([_tab_strip, _tab_notice, _tab_loading, _tab_status, _tab_body])
app.add_class('rxapp-main')
render_viewer()
refresh()
_recover_existing_results()
_t = "pdb2reaction"
_dist_name = "pdb2reaction"
try:
    from importlib.metadata import version as _package_version
    _tool_version = _package_version(_dist_name)
except Exception:
    _tool_version = str(globals().get('installed_version') or 'dev')
_tool_version_label = (_tool_version if _tool_version == 'dev' or _tool_version.startswith('v')
                       else 'v' + _tool_version)
header = W.HTML(
    '<div class="rxheader">'
    '<span class="rxheader-brand"><span class="rxheader-product">%s</span>'
    '<span class="rxheader-version">%s</span></span>'
    '<span class="rxheader-meta"><span class="rxheader-chip">COLAB WORKSPACE</span>'
    '<span class="rxheader-backend"><span class="rxheader-key">backend</span><b>%s</b></span>'
    '</span></div>'
    % (html.escape(_t), html.escape(_tool_version_label), html.escape(str(BACKEND))))
header.add_class('rxheader-host')
rootbox = W.VBox([header, manual_mode_notice, plotly_preload_out, app, cmdline_box])
rootbox.add_class('rxapp')
# Colab syncs widget state but drops the binary buffers a FileUpload transfers, so
# in Colab every upload moves its bytes through google.colab.kernel.invokeFunction
# instead: one browser-side zone owns its own button and <input type=file>, and the
# decoded pairs enter the same ingestion helpers the widget path uses.
_COLAB_UPLOAD_LIMITS = {'files': 64, 'file_bytes': 256 * 1024 * 1024,
                        'batch_bytes': 512 * 1024 * 1024}
_COLAB_UPLOAD_JS = r"""<script>
(function(){
  var CONFIG = __CONFIG__;
  var SCRIPT_HOST = document.currentScript && document.currentScript.parentElement;

  function colabOutputApi(){
    var scope=window;
    for(var depth=0;depth<6&&scope;depth++){
      try{
        if(scope.google&&scope.google.colab&&scope.google.colab.output)
          return scope.google.colab.output;
      }catch(_error){}
      scope=(scope.parent&&scope.parent!==scope)?scope.parent:null;
    }
    return null;
  }
  var resizePending=false, resizeSettleTimer=0, lastFrameHeight=0;
  function resizeColabFrame(){
    if(resizePending)return;
    resizePending=true;
    requestAnimationFrame(function(){
      resizePending=false;
      var app=document.querySelector('.rxapp');
      if(!app)return;
      // Measure only the app's natural size. Measuring document/body after
      // setIframeHeight creates a parent/child resize feedback loop in Colab.
      var rect=app.getBoundingClientRect();
      var height=Math.ceil(Math.max(app.scrollHeight,rect.height))+16;
      // Font/WebGL settling can alternate by a few subpixels; do not publish
      // those changes back to the outer Colab output frame.
      if(lastFrameHeight&&Math.abs(height-lastFrameHeight)<8)return;
      lastFrameHeight=height;
      var api=colabOutputApi();
      try{
        if(api&&typeof api.setIframeHeight==='function')
          api.setIframeHeight(height,true,{maxHeight:30000});
      }catch(_error){}
    });
  }
  function queueColabFrameResize(){
    // Structure replacement briefly passes through a taller loading layout.
    // Wait for the app to settle so Colab receives one final height, not a
    // grow/shrink pair that looks like UI vibration.
    clearTimeout(resizeSettleTimer);
    resizeSettleTimer=setTimeout(resizeColabFrame,900);
  }
  function statusNode(zone){
    var status = zone.querySelector('.rxnative-status');
    if(!status){
      status = document.createElement('div');
      status.className = 'rxnative-status';
      status.setAttribute('role', 'status');
      status.setAttribute('aria-live', 'polite');
      zone.appendChild(status);
    }
    return status;
  }
  function kernelBridge(){
    var scope = window;
    for(var depth = 0; depth < 6 && scope; depth++){
      if(scope.google && scope.google.colab && scope.google.colab.kernel){
        return scope.google.colab.kernel;
      }
      scope = (scope.parent && scope.parent !== scope) ? scope.parent : null;
    }
    return null;
  }
  var nativeTabRequest=0;
  function nativeTabHosts(){
    return Array.prototype.slice.call(document.querySelectorAll('.rxtabs .rxtab-button'));
  }
  function setNativeTabChoice(index){
    nativeTabHosts().forEach(function(host,hostIndex){
      host.classList.toggle('mod-active',hostIndex===index);
    });
  }
  function setNativeTabPane(index){
    var body=document.querySelector('.rxpages');
    if(!body)return false;
    var panes=Array.prototype.filter.call(body.children,function(child){
      return child.classList&&child.classList.contains('rxpage');
    });
    if(panes.length!==4)return false;
    panes.forEach(function(pane,paneIndex){
      if(paneIndex===index){
        pane.classList.remove('rxpage-prewarm'); pane.style.display='flex';
      }else if(paneIndex===1){
        pane.style.display='flex'; pane.classList.add('rxpage-prewarm');
      }else{
        pane.classList.remove('rxpage-prewarm'); pane.style.display='none';
      }
    });
    return true;
  }
  function setNativeTabLoading(index,active){
    var labels=['① Input','② Setup','③ Options','④ Results'];
    var loading=document.querySelector('.rxtab-loading .widget-html-content, .rxtab-loading');
    var pages=document.querySelector('.rxpages');
    if(loading)loading.innerHTML=active?'<div class="rxtab-loading-banner" role="status" aria-live="polite" aria-atomic="true"><span class="rxtab-spinner" aria-hidden="true"></span>Loading <b>'+labels[index]+'</b>…</div>':'';
    if(pages)pages.classList.toggle('rxpages-loading',!!active);
  }
  function setNativeOperationLoading(label,active){
    var loading=document.querySelector('.rxtab-loading .widget-html-content, .rxtab-loading');
    var pages=document.querySelector('.rxpages');
    var safe=String(label||'').replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;');
    if(loading)loading.innerHTML=active?'<div class="rxtab-loading-banner" role="status" aria-live="polite" aria-atomic="true"><span class="rxtab-spinner" aria-hidden="true"></span>Loading <b>'+safe+'</b>…</div>':'';
    if(pages)pages.classList.toggle('rxpages-loading',!!active);
  }
  function wireOperationTriggers(){
    document.querySelectorAll('.rxoperation-trigger').forEach(function(host){
      if(host.dataset.rxOperationTrigger)return; host.dataset.rxOperationTrigger='true';
      host.addEventListener('click',function(event){
        var button=host.matches('button')?host:host.querySelector('button');
        var bridge=kernelBridge();
        if(!button||button.disabled||host.dataset.rxOperationBusy==='true')return;
        if(!bridge||!CONFIG.example_callback)return;
        event.preventDefault(); event.stopImmediatePropagation();
        host.dataset.rxOperationBusy='true'; button.disabled=true;
        setNativeOperationLoading('example',true);
        function settleExample(){
          delete host.dataset.rxOperationBusy; button.disabled=false;
          setNativeOperationLoading('example',false); queueColabFrameResize();
        }
        bridge.invokeFunction(CONFIG.example_callback,[],{}).then(
          settleExample,function(error){console.error(error);settleExample();});
      },true);
    });
  }
  function setNativeResultLoading(label,active){
    var host=document.querySelector('.rxresult-loading .widget-html-content, .rxresult-loading');
    if(!host)return;
    var safe=String(label||'result view').replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;');
    host.innerHTML=active?'<div class="rxresult-loading-banner" role="status" aria-live="polite" aria-atomic="true"><span class="rxtab-spinner" aria-hidden="true"></span>Loading <b>'+safe+'</b>…</div>':'';
    var widget=host.closest('.rxresult-loading'); if(widget)widget.style.display=active?'block':'none';
  }
  function wireResultChoices(){
    var bridge=kernelBridge();
    if(!bridge||!CONFIG.result_callback)return false;
    document.querySelectorAll('.rxresult-choice select, .rxenergy-choice select').forEach(function(select){
      if(select.dataset.rxNativeResult)return;
      select.dataset.rxNativeResult='true'; select.dataset.rxLastValue=select.value;
      select.addEventListener('change',function(event){
        var previous=select.dataset.rxLastValue||'', value=select.value;
        var selectedIndex=select.selectedIndex;
        var option=select.options[select.selectedIndex];
        var label=option?option.textContent:'result view';
        var kind=select.closest('.rxenergy-choice')?'energy':'trajectory';
        clearNativeFrameTimer();
        event.stopImmediatePropagation(); setNativeResultLoading(label,true);
        var meta=document.querySelector('[data-rx-result-generation]');
        var generation=Number(meta&&meta.dataset.rxResultGeneration||0);
        bridge.invokeFunction(CONFIG.result_callback,[kind,selectedIndex,generation,label],{}).then(function(result){
          if(result&&result.ok===false)select.value=previous; else select.dataset.rxLastValue=value;
          setNativeResultLoading(label,false); queueColabFrameResize();
        },function(error){
          console.error(error); select.value=previous; setNativeResultLoading(label,false);
        });
      },true);
    });
    return true;
  }
  function wireTabs(){
    var hosts=nativeTabHosts(),bridge=kernelBridge();
    if(hosts.length!==4||!bridge)return false;
    hosts.forEach(function(host,index){
      if(host.dataset.rxNativeTab)return;
      host.dataset.rxNativeTab=String(index);
      host.addEventListener('click',function(event){
        var button=host.matches('button')?host:host.querySelector('button');
        if(button&&button.disabled)return;
        clearNativeFrameTimer();
        event.preventDefault(); event.stopImmediatePropagation();
        var active=hosts.findIndex(function(item){return item.classList.contains('mod-active');});
        var previous=active<0?0:active,request=++nativeTabRequest,shownAt=performance.now();
        setNativeTabChoice(index); setNativeTabPane(index); setNativeTabLoading(index,true);
        function settle(ok){
          var wait=Math.max(0,160-(performance.now()-shownAt));
          setTimeout(function(){
            if(request!==nativeTabRequest)return;
            if(!ok||document.querySelector('.rxtab-notice [role="alert"]')){
              setNativeTabChoice(previous); setNativeTabPane(previous);
            }else{ setNativeTabPane(index); }
            setNativeTabLoading(index,false); queueColabFrameResize();
          },wait);
        }
        bridge.invokeFunction(CONFIG.tab_callback,[index],{}).then(
          function(result){settle(!result||result.active===undefined||Number(result.active)===index);},
          function(){settle(false);});
      },true);
    });
    return true;
  }
  function installIframeKernelRelay(scope){
    var relayKey='__rxIframeKernelRelay_'+String(CONFIG.viewer_callback_base||'none').replace(/[^A-Za-z0-9_]/g,'_');
    if(scope[relayKey])return;
    scope[relayKey]=true;
    scope.addEventListener('message',function(event){
      var message=event&&event.data;
      if(!message||message.type!=='rx-kernel-invoke')return;
      var trusted=false;
      scope.document.querySelectorAll('iframe.rxmolstar-frame').forEach(function(frame){
        if(frame.contentWindow===event.source)trusted=true;
      });
      if(!trusted)return;
      var suffix=String(message.suffix||'');
      if(['on_click','clear_highlights','set_frame'].indexOf(suffix)<0)return;
      var args=Array.isArray(message.args)?message.args:[];
      if(args.length>12)return;
      var bridge=kernelBridge();
      if(!bridge||!CONFIG.viewer_callback_base)return;
      bridge.invokeFunction(CONFIG.viewer_callback_base+'.'+suffix,args,{}).catch(function(error){
        console.error(error);
      });
    },false);
  }
  function wireIframeKernelRelay(){
    var scope=window;
    for(var depth=0;depth<6&&scope;depth++){
      try{
        installIframeKernelRelay(scope);
        scope=(scope.parent&&scope.parent!==scope)?scope.parent:null;
      }catch(_error){scope=null;}
    }
  }
  function wireCancel(){
    var button=document.querySelector(
      '.rxcancel-run button, button.rxcancel-run');
    if(!button)return false;
    if(button.dataset.rxNativeCancel)return true;
    button.dataset.rxNativeCancel='true';
    button.addEventListener('click',function(){
      var bridge=kernelBridge();
      if(bridge)bridge.invokeFunction(CONFIG.cancel_callback,[],{});
    },true);
    return true;
  }
  var nativeFrameTimer=0;
  function clearNativeFrameTimer(){
    if(nativeFrameTimer){clearTimeout(nativeFrameTimer);nativeFrameTimer=0;}
  }
  function wireFrameControls(){
    var bridge=kernelBridge();
    function publishFrame(generation,index){
      var message={type:'rx-set-frame',generation:generation,index:index};
      document.querySelectorAll('iframe[data-rx-channel="trajectory"]').forEach(function(frame){
        if(frame.contentWindow)frame.contentWindow.postMessage(message,'*');
      });
    }
    var selectors=[['.rxframe-prev button,button.rxframe-prev',-1],
                   ['.rxframe-next button,button.rxframe-next',1]];
    selectors.forEach(function(spec){
      var button=document.querySelector(spec[0]);
      if(!button||button.dataset.rxNativeFrame)return;
      button.dataset.rxNativeFrame='true';
      button.addEventListener('click',function(event){
        if(button.disabled)return;
        var state=document.querySelector('.rxpath-state');
        var match=state&&String(state.textContent||'').match(/(IRC point|State|Image|Frame)\s+(\d+)\s+of\s+(\d+)/i);
        var target=document.querySelector('iframe[data-rx-channel="trajectory"][data-rx-generation]');
        if(!match||!target||!target.contentWindow||!bridge)return;
        var total=Number(match[3]), index=Math.max(0,Math.min(total-1,Number(match[2])-1+spec[1]));
        var generation=Number(target.dataset.rxGeneration||0);
        event.preventDefault(); event.stopImmediatePropagation();
        var strong=state.querySelector('b');
        if(strong)strong.textContent=match[1]+' '+String(index+1)+' of '+String(total);
        publishFrame(generation,index);
        bridge.invokeFunction(CONFIG.frame_callback,[generation,index],{}).catch(function(){});
      },true);
    });
    var slider=document.querySelector('.rxpath-controls [role="slider"]');
    if(slider&&!slider.dataset.rxNativeFrame){
      slider.dataset.rxNativeFrame='true';
      var publishSlider=function(){
        var target=document.querySelector('iframe[data-rx-channel="trajectory"][data-rx-generation]');
        if(!target||!target.contentWindow)return;
        var maximum=Math.max(0,Number(slider.getAttribute('aria-valuemax'))||0);
        var index=Math.max(0,Math.min(maximum,Math.round(Number(slider.getAttribute('aria-valuenow')))||0));
        publishFrame(Number(target.dataset.rxGeneration||0),index);
      };
      var sliderObserver=new MutationObserver(publishSlider);
      sliderObserver.observe(slider,{attributes:true,attributeFilter:['aria-valuenow']});
      slider.addEventListener('keydown',function(){requestAnimationFrame(publishSlider);},true);
    }
    var playHost=document.querySelector('.rxframe-play');
    var playButtons=playHost?playHost.querySelectorAll('button'):[];
    var repeatButton=playButtons.length>=3?playButtons[playButtons.length-1]:null;
    if(repeatButton){
      repeatButton.title='Repeat';repeatButton.setAttribute('aria-label','Repeat');
      if(!repeatButton.dataset.rxRepeatInitialized){
        repeatButton.dataset.rxRepeatInitialized='true';
        repeatButton.classList.add('mod-active');
        repeatButton.setAttribute('aria-pressed','true');
      }
    }
    function stopNativePlayback(syncKernel){
      clearNativeFrameTimer();
      if(!syncKernel||!bridge)return;
      var state=document.querySelector('.rxpath-state');
      var match=state&&String(state.textContent||'').match(/(IRC point|State|Image|Frame)\s+(\d+)\s+of\s+(\d+)/i);
      var target=document.querySelector('iframe[data-rx-channel="trajectory"][data-rx-generation]');
      if(match&&target)bridge.invokeFunction(CONFIG.frame_callback,[Number(target.dataset.rxGeneration||0),Number(match[2])-1],{}).catch(function(){});
    }
    if(playButtons.length>=2&&!playButtons[0].dataset.rxNativePlay){
      playButtons[0].dataset.rxNativePlay='true';
      playButtons[0].addEventListener('click',function(event){
        var state=document.querySelector('.rxpath-state');
        var match=state&&String(state.textContent||'').match(/(IRC point|State|Image|Frame)\s+(\d+)\s+of\s+(\d+)/i);
        var target=document.querySelector('iframe[data-rx-channel="trajectory"][data-rx-generation]');
        if(!match||!target||!bridge)return;
        event.preventDefault();event.stopImmediatePropagation();stopNativePlayback(false);
        var total=Number(match[3]),index=Number(match[2])-1,generation=Number(target.dataset.rxGeneration||0);
        var intervalNode=document.querySelector('.rxplayback-interval [data-rx-playback-interval]');
        var interval=Math.max(20,Number(intervalNode&&intervalNode.dataset.rxPlaybackInterval)||850);
        function show(index){
          var strong=state.querySelector('b'); if(strong)strong.textContent=match[1]+' '+String(index+1)+' of '+String(total);
          var handle=document.querySelector('.rxpath-controls [role="slider"]');
          var sliderHost=handle&&handle.closest('.slider');
          if(sliderHost&&sliderHost.noUiSlider)sliderHost.noUiSlider.set(index);
          publishFrame(generation,index);
        }
        function tick(){
          if(!playHost.isConnected){stopNativePlayback(false);return;}
          if(index>=total-1){
            if(!repeatButton||!repeatButton.classList.contains('mod-active')){
              stopNativePlayback(true);return;
            }
            index=0;
          }else index+=1;
          show(index); nativeFrameTimer=setTimeout(tick,interval);
        }
        nativeFrameTimer=setTimeout(tick,interval);
      },true);
      playButtons[1].dataset.rxNativePlay='true';
      playButtons[1].addEventListener('click',function(event){
        event.preventDefault();event.stopImmediatePropagation();stopNativePlayback(true);
      },true);
    }
    return true;
  }
  function wire(spec){
    var zone = document.querySelector(spec.selector);
    if(!zone) return false;
    if(zone.dataset.rxNativeUpload) return true;
    zone.dataset.rxNativeUpload = spec.role;
    var status = statusNode(zone);
    var input = document.createElement('input');
    input.type = 'file';
    input.multiple = !!spec.multiple;
    input.accept = spec.accept;
    input.className = 'rxnative-input';
    input.style.display = 'none';
    input.setAttribute('aria-label', spec.label);
    var button = document.createElement('button');
    button.type = 'button';
    button.className = 'rxnative-button';
    button.textContent = spec.label;
    // Keep the button in the middle of the zone: prompt, button, then status.
    var statusHost = status;
    while(statusHost.parentElement && statusHost.parentElement !== zone){
      statusHost = statusHost.parentElement;
    }
    if(statusHost.parentElement === zone) zone.insertBefore(button, statusHost);
    else zone.appendChild(button);
    zone.appendChild(input);
    function say(text){ status.textContent = text; }
    function send(payload){
      var bridge = kernelBridge();
      if(!bridge){ setNativeOperationLoading(spec.role,false); say('Colab kernel bridge unavailable - rerun the GUI cell.'); return; }
      bridge.invokeFunction(CONFIG.callback, [spec.role, payload], {}).then(
        function(){ setNativeOperationLoading(spec.role,false); say(spec.multiple ? 'Add more files - drop or click' : 'Uploaded.'); },
        function(){ setNativeOperationLoading(spec.role,false); say('Not added - click or drop to retry'); });
    }
    function submit(list){
      var files = Array.prototype.slice.call(list || []);
      input.value = '';
      if(!files.length) return;
      if(files.length > CONFIG.max_files){
        say('Not added - more than ' + CONFIG.max_files + ' files'); return;
      }
      var total = 0;
      for(var i = 0; i < files.length; i++) total += files[i].size;
      if(total > CONFIG.max_batch_bytes || files.some(function(file){
            return file.size > CONFIG.max_file_bytes; })){
        say('Not added - upload size limit exceeded'); return;
      }
      setNativeOperationLoading(spec.role==='model'?'ML-region PDB':(spec.role==='session'?'session':'files'),true);
      say('Adding ' + files.length + (files.length === 1 ? ' file...' : ' files...'));
      var out = new Array(files.length), left = files.length;
      files.forEach(function(file, index){
        var reader = new FileReader();
        reader.onload = function(){
          out[index] = {name: file.name, b64: String(reader.result).split(',')[1] || ''};
          if(--left === 0) send(out);
        };
        reader.onerror = function(){
          out[index] = {name: file.name, error: 'read failed'};
          if(--left === 0) send(out);
        };
        reader.readAsDataURL(file);
      });
    }
    button.addEventListener('click', function(event){ event.preventDefault(); input.click(); });
    input.addEventListener('change', function(event){ submit(event.target.files); });
    var stop = function(event){ event.preventDefault(); event.stopPropagation(); };
    zone.addEventListener('dragover', function(event){ stop(event); zone.classList.add('rxdrag'); });
    zone.addEventListener('dragleave', function(event){ stop(event); zone.classList.remove('rxdrag'); });
    zone.addEventListener('drop', function(event){
      stop(event);
      zone.classList.remove('rxdrag');
      submit(event.dataTransfer && event.dataTransfer.files);
    });
    return true;
  }
  function wireHalfStepSpinners(){
    if(document.documentElement.dataset.rxHalfStepWired)return true;
    document.documentElement.dataset.rxHalfStepWired='true';
    var inputFor=function(event){
      var input=event.target&&event.target.closest?event.target.closest('.rxhalf-step input[type="number"]'):null;
      return input;
    };
    var prime=function(input){
      if(!input.dataset.rxHalfMin)input.dataset.rxHalfMin=input.getAttribute('min')||'';
      if(!input.dataset.rxHalfMax)input.dataset.rxHalfMax=input.getAttribute('max')||'';
      input.setAttribute('value',input.value);
      input.removeAttribute('min');
      input.dataset.rxHalfPrimed='true';
    };
    var settle=function(event){
      var input=inputFor(event);
      if(!input||input.dataset.rxHalfPrimed!=='true')return;
      var value=Number(input.value), minimum=Number(input.dataset.rxHalfMin), maximum=Number(input.dataset.rxHalfMax);
      if(Number.isFinite(value)){
        if(input.dataset.rxHalfMin!==''&&Number.isFinite(minimum))value=Math.max(minimum,value);
        if(input.dataset.rxHalfMax!==''&&Number.isFinite(maximum))value=Math.min(maximum,value);
        input.value=String(value);
      }
      if(input.dataset.rxHalfMin==='')input.removeAttribute('min');else input.setAttribute('min',input.dataset.rxHalfMin);
      if(input.dataset.rxHalfMax==='')input.removeAttribute('max');else input.setAttribute('max',input.dataset.rxHalfMax);
      input.setAttribute('value',input.value);
      delete input.dataset.rxHalfPrimed;
    };
    document.addEventListener('keydown',function(event){
      if(event.key!=='ArrowUp'&&event.key!=='ArrowDown')return;
      var input=inputFor(event);if(input)prime(input);
    },true);
    document.addEventListener('pointerdown',function(event){var input=inputFor(event);if(input)prime(input);},true);
    document.addEventListener('input',settle,true);
    document.addEventListener('change',settle,true);
    return true;
  }
  function wireLog(){
    var host=document.querySelector('.rxlog .rxlog-console, .rxlog .widget-html-content');
    if(!host)return false;
    if(host.dataset.rxFollowLog==='smart')return true;
    var isAtBottom=function(){return Math.max(0,host.scrollHeight-host.scrollTop-host.clientHeight)<=24;};
    var pinned=isAtBottom();
    host.addEventListener('scroll',function(){pinned=isAtBottom();},{passive:true});
    var follow=function(){if(!pinned)return;requestAnimationFrame(function(){
      var apply=function(){host.scrollTop=host.scrollHeight;};
      apply(); setTimeout(apply,0); setTimeout(apply,60);
    });};
    if(!host.dataset.rxFollowLog){
      host.dataset.rxFollowLog='true';
      new MutationObserver(follow).observe(host,{childList:true,subtree:true,characterData:true});
    }
    follow(); return true;
  }
  function wireAll(){
    var uploadPending = 0;
    wireIframeKernelRelay();
    wireHalfStepSpinners();
    wireTabs();
    wireOperationTriggers();
    wireResultChoices();
    for(var i = 0; i < CONFIG.zones.length; i++){ if(!wire(CONFIG.zones[i])) uploadPending += 1; }
    wireCancel();
    wireFrameControls();
    wireLog();
    if(uploadPending === 0){
      var warning=document.querySelector('[data-rx-upload-warning]');
      if(warning)warning.remove();
    }
    return uploadPending === 0;
  }
  // Keep one low-rate recovery heartbeat independent of Cancel visibility.
  // During a run it accelerates; while idle it stays inexpensive.
  var runPollPending=false;
  function pollRunState(){
    if(!SCRIPT_HOST || !SCRIPT_HOST.isConnected)return;
    var button=document.querySelector('.rxcancel-run button, button.rxcancel-run');
    var active=!!(button && button.offsetParent!==null);
    if(button)wireCancel();
    if(runPollPending){setTimeout(pollRunState,active?1500:5000);return;}
    var bridge=kernelBridge();
    if(!bridge){setTimeout(pollRunState,active?1500:5000);return;}
    runPollPending=true;
    bridge.invokeFunction(CONFIG.poll_callback,[active],{}).then(
      function(){runPollPending=false;setTimeout(pollRunState,active?1500:5000);},
      function(){runPollPending=false;setTimeout(pollRunState,5000);});
  }
  setTimeout(pollRunState,1200);
  if(!wireLog()){
    var logTimer=setInterval(function(){if(wireLog())clearInterval(logTimer);},300);
    setTimeout(function(){clearInterval(logTimer);},20000);
  }
  resizeColabFrame();
  if(window.ResizeObserver){
    var resizeTarget=document.querySelector('.rxapp');
    if(resizeTarget){
      var resizeObserver=new ResizeObserver(queueColabFrameResize);
      resizeObserver.observe(resizeTarget);
    }
  }
  window.addEventListener('load',resizeColabFrame,{once:true});
  setTimeout(resizeColabFrame,500);
  setTimeout(resizeColabFrame,2500);
  if(!wireAll()){
    var timer = setInterval(function(){ if(wireAll()) clearInterval(timer); }, 400);
    setTimeout(function(){
      clearInterval(timer);
      if(!wireAll() && SCRIPT_HOST && SCRIPT_HOST.isConnected &&
          !document.querySelector('[data-rx-upload-warning]')){
        var warning = document.createElement('div');
        warning.setAttribute('data-rx-upload-warning','true');
        warning.setAttribute('role', 'alert');
        warning.style.cssText = 'margin:8px 0;padding:8px 10px;border:1px solid #f59e0b;' +
          'border-radius:7px;background:#fffbeb;color:#92400e;font:13px sans-serif';
        warning.textContent = 'File upload controls did not initialize. Rerun the Launch GUI cell.';
        var warningHost = document.querySelector('.rxdrop');
        if(warningHost && warningHost.parentElement){
          warningHost.parentElement.insertBefore(warning, warningHost.nextSibling);
        }else{ SCRIPT_HOST.appendChild(warning); }
      }
    }, 20000);
  }
})();
</script>"""

def _decode_colab_batch(files):
    """Turn one browser batch of {name, b64} entries into (name, bytes) pairs."""
    pairs, failed, total = [], [], 0
    for item in files or []:
        entry = item if isinstance(item, dict) else {}
        name = os.path.basename(str(entry.get('name') or '').replace('\\', '/'))
        if not name: continue
        if entry.get('error') or not entry.get('b64'):
            failed.append(name); continue
        try: content = base64.b64decode(entry['b64'], validate=True)
        except Exception: failed.append(name); continue
        total += len(content)
        if (len(content) > _COLAB_UPLOAD_LIMITS['file_bytes'] or
                total > _COLAB_UPLOAD_LIMITS['batch_bytes']):
            failed.append(name); continue
        pairs.append((name, content))
    return pairs, failed

def _on_colab_upload(role, files):
    """Receive one browser batch for the input queue, the model PDB or a session."""
    pairs, failed = _decode_colab_batch(files)
    if len(pairs) > _COLAB_UPLOAD_LIMITS['files']:
        pairs, failed = [], [name for name, _content in pairs]
    accepted = False
    busy_label = {'model': 'ML-region PDB', 'session': 'session'}.get(role, 'files')
    _set_operation_loading(busy_label, True)
    try:
        if pairs and role == 'model':
            _accept_model_pairs(pairs); accepted = True
        elif pairs and role == 'session':
            _accept_session_pairs(pairs); accepted = True
        elif pairs:
            accepted = bool(_accept_upload_pairs(pairs, 'browser upload'))
    except Exception as exc:
        input_msg.value = ('<div role="alert" style="color:#991b1b">Upload failed: %s; '
                           'existing files were kept.</div>' % html.escape(str(exc)))
    finally:
        _set_operation_loading(busy_label, False)
    if failed:
        input_msg.value += ('<br><span style="color:#991b1b">read failed: %s</span>' %
                            html.escape(', '.join(failed)))
    return {'ok': bool(accepted and not failed), 'n': len(pairs), 'failed': failed}

if _UPLOAD_MODE == 'colab':
    # Registered and wired after the app is in the DOM, exactly like the browser
    # callbacks the 3D viewer uses.
    _cwm.register_callback('pdb2reaction_gui.upload_files', _on_colab_upload)
    _cwm.register_callback('pdb2reaction_gui.load_example', _on_colab_example)
    _cwm.register_callback('pdb2reaction_gui.cancel_run', lambda: _cancel_run(None))
    def _on_colab_tab(index):
        _tab_go(index)
        return {'active': _TAB_NAV['active']}
    _cwm.register_callback('pdb2reaction_gui.tab_go', _on_colab_tab)
    def _on_colab_result(kind, selected_index, generation, selected_label):
        widget = energy_choice if str(kind) == 'energy' else traj_choice
        options = list(widget.options)
        try: generation = int(generation)
        except (TypeError, ValueError): generation = -1
        if generation != _RESULT_SET_GENERATION['value']:
            return {'ok': False, 'reason': 'stale result set'}
        try: selected_index = int(selected_index)
        except (TypeError, ValueError): selected_index = -1
        if selected_index < 0 or selected_index >= len(options):
            return {'ok': False, 'reason': 'unknown result view'}
        if str(options[selected_index][0]) != str(selected_label):
            return {'ok': False, 'reason': 'stale result view'}
        value = options[selected_index][1]
        _result_pick_guard['active'] = True
        try:
            widget.value = value
        finally:
            _result_pick_guard['active'] = False
        if str(kind) == 'energy':
            _render_energy_choice()
        else:
            _on_traj_choice()
        return {'ok': True}
    _cwm.register_callback('pdb2reaction_gui.select_result', _on_colab_result)
    def _colab_poll_run(frontend_running=False):
        task = _RUN_EXECUTION.get('task')
        thread = _RUN_EXECUTION.get('thread')
        process = _RUN_EXECUTION.get('process')
        task_running = bool(task is not None and not task.done())
        thread_running = bool(thread is not None and thread.is_alive())
        process_running = bool(
            process is not None and
            (process.returncode is None if hasattr(process, 'returncode')
             else process.poll() is None))
        running = bool(task_running or thread_running or process_running)
        _flush_run_log(force=True)
        manifest = S.get('_last_manifest') or {}
        out = S.get('_last_out_dir')
        terminal = bool(out and manifest.get('status') in
                        ('success', 'partial', 'failed', 'cancelled'))
        results_pending = bool(terminal and
                               S.get('_results_presented_dir') != os.path.abspath(out))
        if not running and (frontend_running or _ACTION_STATE.get('running') or
                            results_pending):
            # The browser still shows an active run, so republish from this
            # live callback even if the worker already cleared kernel state.
            _set_running(False, True)
            _set_run_status(_RUN_STATE.get('text', ''),
                            _RUN_STATE.get('tone', 'muted'),
                            _RUN_STATE.get('kind', ''), True)
            if terminal:
                _results(out)
                _tab_go(3)
                _publish_result_widget_state()
            _publish_run_widget_state()
        return {'running': running}
    _cwm.register_callback('pdb2reaction_gui.poll_run', _colab_poll_run)
    _colab_zones = [dict(selector='.rxdrop', role='input', accept=_acc,
                         multiple=True, label='Upload files')]
    if not IS_CLUSTER and 'model_upl' in globals():
        _colab_zones.append(dict(selector='.rxmodel-upload', role='model',
                                 accept='.pdb,.ent', multiple=False,
                                 label='Upload ML-region PDB'))
    _colab_zones.append(dict(selector='.rxsession-upload', role='session',
                             accept='.json', multiple=False, label='Load session'))
    # Keep the inline script in the same Colab output frame as the widget DOM.
    _colab_upload_bridge = W.Output(
        layout=W.Layout(height='1px', min_height='1px', overflow='visible'))
    with _colab_upload_bridge:
        display(HTML(_COLAB_UPLOAD_JS.replace('__CONFIG__', json.dumps({
            'callback': 'pdb2reaction_gui.upload_files',
            'example_callback': 'pdb2reaction_gui.load_example',
            'cancel_callback': 'pdb2reaction_gui.cancel_run',
            'tab_callback': 'pdb2reaction_gui.tab_go',
            'result_callback': 'pdb2reaction_gui.select_result',
            'poll_callback': 'pdb2reaction_gui.poll_run',
            'frame_callback': 'pdb2reaction_gui.set_frame',
            'viewer_callback_base': 'pdb2reaction_gui',
            'max_files': _COLAB_UPLOAD_LIMITS['files'],
            'max_file_bytes': _COLAB_UPLOAD_LIMITS['file_bytes'],
            'max_batch_bytes': _COLAB_UPLOAD_LIMITS['batch_bytes'],
            'zones': _colab_zones}, separators=(',', ':')).replace('<', '\\u003c'))))
    rootbox.children = tuple(rootbox.children) + (_colab_upload_bridge,)
_gui_launch_status.value = ''
_gui_launch_status.layout.display = 'none'
display(rootbox)
